# Max and argmax reduction

Greedy decoding picks the token with the largest logit, which is a max **and** an argmax over a long vector. Write it as a two-stage Triton reduction.

Write `vector_max(x)` returning `(value, index)`.

## Inputs
- `x`: contiguous 1-D `float32` CUDA tensor with $n \ge 1$ finite elements (no NaN).

## Output
- `value`: 0-d `float32` CUDA tensor, the maximum of `x`.
- `index`: 0-d `int64` CUDA tensor, the position of the maximum. **Ties go to the smallest index.**

## Requirements
1. **Stage 1:** `triton.cdiv(n, BLOCK)` programs. Each finds the max of its block and the *global* index of that max, and writes them to `block_max[pid]` and `block_idx[pid]`.
2. **Stage 2:** a single program reduces the partials the same way.
3. Masked lanes must never win: load them as `-inf`.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def vector_max(x):
    """x: 1-D float32 CUDA tensor, n >= 1. Return (max 0-d float32, argmax 0-d int64), ties -> smallest index."""
    raise NotImplementedError


@triton.jit
def _block_argmax_kernel(x_ptr, max_ptr, idx_ptr, n, BLOCK: tl.constexpr):
    pass


@triton.jit
def _final_argmax_kernel(max_ptr, idx_ptr, out_val_ptr, out_idx_ptr, n_blocks, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [3.0, 7.0, 1.0, 7.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": 7.0, "dtype": "float32"}, {"__nd__": 1, "dtype": "int64"}]}}, {"a": [{"__nd__": [-5.0, -2.0, -9.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": -2.0, "dtype": "float32"}, {"__nd__": 1, "dtype": "int64"}]}}, {"a": [{"__nd__": [0.5], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": 0.5, "dtype": "float32"}, {"__nd__": 0, "dtype": "int64"}]}}, {"a": [{"__nd__": [-1.7382663488388062, -1.3366427421569824, -1.3611067533493042, -0.35161712765693665, -2.3125815391540527, -0.18889719247817993, -0.9572292566299438, 0.8936001658439636, 0.9568472504615784, 1.3922582864761353, 0.767470121383667, -0.053029779344797134, 0.8597939610481262, 1.5054811239242554, -0.6535945534706116, 0.6103511452674866, -0.04267382621765137, 1.440016746520996, -0.8368950486183167, -0.3015466034412384, 0.3623386025428772, 0.25811025500297546, -1.6394479274749756, 0.36015522480010986, -0.11849769949913025, -0.23974785208702087, -0.15530166029930115, 0.21897169947624207, -1.81639564037323, 1.5524665117263794, -0.8614416718482971, -2.241367816925049, -0.08197449147701263, 1.4574804306030273, -0.5186010003089905, 1.551275610923767, 1.5569419860839844, -0.86273193359375, -2.465120792388916, -1.235182762145996, 1.1874321699142456, -0.8167721629142761, -1.510677456855774, -1.3376946449279785, 0.0001801442267606035, -0.026108482852578163, 0.8720449209213257, 0.9890499711036682, -0.9322490096092224, -0.15675869584083557, -1.1339346170425415, 0.07189067453145981, -1.1506078243255615, -1.200251579284668, 2.1234219074249268, 0.03238888829946518, 0.6431786417961121, 2.5380983352661133, 0.7858388423919678, -0.11414260417222977, 0.054589953273534775, -0.7444919943809509, 1.0943384170532227, -0.3302786946296692, 1.7627454996109009, -0.344758540391922, -0.3536229431629181, -0.4193773865699768, 0.6256938576698303, 0.18693606555461884, 1.3714324235916138, -0.5533496141433716, 0.2902381420135498, 1.9379616975784302, -0.24038168787956238, 0.5326128005981445, 1.5718822479248047, 0.5534457564353943, -0.6345051527023315, -2.062350034713745, 2.3931636810302734, 0.4303142726421356, 1.2605289220809937, -0.4763135612010956, 1.818481683731079, 0.5195393562316895, 0.04979744181036949, -0.9018172025680542, 0.3681847155094147, -0.48653003573417664, -0.22276918590068817, -1.6225107908248901, -0.49055391550064087, 0.15987902879714966, -1.250806212425232, 0.07389651238918304, 0.6632938981056213, -0.30675995349884033, -0.6207841634750366, -0.6201241612434387, 0.9411423206329346, -1.534505009651184, -0.300451397895813, -0.41940510272979736, 0.14399704337120056, -0.06957008689641953, -1.1075061559677124, 2.0400893688201904, 0.17761176824569702, -1.3420382738113403, -0.8914334774017334, 0.09758487343788147, 0.6129699349403381, -1.3421550989151, 0.7113368511199951, -1.373292088508606, -0.7943888306617737, 0.8531012535095215, 1.700997233390808, -0.08470787107944489, -0.3095386028289795, 1.2678991556167603, -1.0719789266586304, 0.2779218256473541, 0.23324556648731232, 1.0093218088150024, 0.2985784411430359, -0.26639434695243835, 0.6145545840263367, -1.126564860343933, 0.8140764236450195, -0.6726765036582947, -0.910365879535675, -0.36764639616012573, 0.04600144177675247, 0.71058589220047, -0.21267066895961761, 0.7357228398323059, 0.9750824570655823, -0.10575880855321884, -0.9815471172332764, 0.9197511076927185, 0.2754482328891754, -2.1151936054229736, -0.9882886409759521, 0.14316168427467346, -1.3719055652618408, -0.6814201474189758, -1.7129220962524414, 0.4022636115550995, 0.2415178120136261, 0.17612560093402863, 1.4474741220474243, -0.35260167717933655, -0.289180189371109, 2.019923210144043, -1.3008630275726318, 0.7291629314422607, -1.3150964975357056, 0.23625579476356506, 2.1157827377319336, 0.2713560163974762, -0.7718227505683899, 2.384216070175171, -0.543821394443512, -0.20619650185108185, 0.30437207221984863, -1.3425651788711548, 1.115098237991333, 1.618825078010559, 0.7587658166885376, 0.3303610384464264, -0.14929768443107605, 0.8867355585098267, -3.150078296661377, -0.12477026134729385, 0.25612571835517883, -1.985530138015747, 0.4261673390865326, -0.12750498950481415, -0.03456597030162811, 0.10139123350381851, 0.5751370191574097, 0.4579625427722931, -1.0373519659042358, -0.11696197837591171, -0.18209832906723022, 0.038422439247369766, -0.5910912752151489, 0.969853937625885, 1.3045483827590942, 1.5216997861862183, 1.9056522846221924, 1.4932806491851807, -0.4599316418170929, -0.6143768429756165, -0.3976389467716217, 0.5584385395050049, -0.12983064353466034, 0.9910935759544373, 1.6503914594650269, -0.4302612543106079, 0.9797077178955078, -0.3929447829723358, -1.2664974927902222, -1.2688435316085815, -0.3191296458244324, -1.89910089969635, 0.4993104934692383, 1.5106810331344604, -1.1650620698928833, -0.3343140780925751, 1.5435583591461182, 0.34723198413848877, 0.1956666260957718, -0.2615339756011963, 1.0186587572097778, 0.3292423188686371, -1.0841877460479736, -2.432093381881714, -2.681692361831665, 0.6576714515686035, -0.2660844922065735, 1.4481297731399536, 0.5099945068359375, -1.2720370292663574, 0.29038894176483154, 1.191861629486084, -0.9017567038536072, -0.5087423920631409, 1.7409871816635132, -1.746392011642456, -1.0902869701385498, 1.0500398874282837, 0.19899573922157288, -1.1424908638000488, -1.4552453756332397, 0.4634547531604767, 1.2142695188522339, 0.82658451795578, 0.4111853241920471, 1.4954732656478882, 0.31114280223846436, -0.8324772715568542, 2.0318803787231445, -1.2340540885925293, -0.13453851640224457, -0.0730673149228096, 1.4311362504959106, -0.7637032270431519, -0.2767201364040375, -1.7804064750671387, 0.3375348150730133, 0.44516247510910034, 0.6506773233413696, -0.9267338514328003, 0.23636260628700256, -0.8841448426246643, -0.47911885380744934, -0.07857993990182877, 1.2926464080810547, -0.4402814209461212, 0.17397288978099823, -0.8474342226982117, -0.5732844471931458, -0.7051883339881897, -0.16304819285869598, 0.7618706226348877, -1.1113027334213257, 0.712733805179596, -0.9589356184005737, 0.3270075023174286, -0.5608709454536438, -0.312241792678833, -0.40216922760009766, 1.0870612859725952, 0.23946614563465118, 1.0559872388839722, -0.3834002912044525, -0.2582603693008423, 1.592368721961975, -1.6704614162445068, -0.9014682173728943, -0.20147962868213654, 1.7977226972579956, -0.6333995461463928, 0.7699410915374756, 1.5583218336105347, -0.26087453961372375, -0.19837509095668793, 1.6461708545684814, -0.578785240650177, -1.0786495208740234, 0.8029630184173584, -0.5418881773948669, 1.2721842527389526, 0.5135474801063538, -1.7799098491668701, 0.2533246874809265, -0.014213846996426582, -0.689329981803894, 1.0785818099975586, -1.7978252172470093, -1.280020833015442, -0.5234267115592957, 0.76621413230896, -1.5694801807403564, -0.6661272048950195, -0.7451257705688477, -1.5424983501434326, -0.4198623299598694, -1.1857109069824219, -0.48898765444755554, 0.24868829548358917, -0.9534118175506592, -0.9142826795578003, -1.0316511392593384, -0.8933054804801941, -1.6148024797439575, -1.2263998985290527, 0.07719191908836365, -0.9243544340133667, 3.2339019775390625, -0.8600130677223206, 0.4160231947898865, -1.5521503686904907, -0.31256815791130066, -0.591259777545929, -0.12163392454385757, 1.2945406436920166, -0.20827333629131317, -1.2501996755599976, 0.08347004652023315, -1.003814935684204, -0.15295571088790894, 1.573054552078247, 0.9302760362625122, 1.067178726196289, 0.5212998986244202, -0.031584322452545166, -1.5579217672348022, -0.4846561849117279, -0.5406911373138428, 0.6760981678962708, 0.9945112466812134, 0.4785252511501312, -0.7995845675468445, 1.8504661321640015, 1.4198445081710815, -0.21764756739139557, 0.19711457192897797, 0.6402798891067505, 0.10389133542776108, -0.09853558987379074, 0.49441975355148315, 0.010406093671917915, 0.20431135594844818, -1.611350178718567, -1.6075892448425293, 1.741086721420288, -1.522369384765625, 0.4872288405895233, 0.3203769624233246, 0.2225087583065033, -0.6936047077178955, 1.0157777070999146, 0.35101813077926636, -0.3421611189842224, -1.4072133302688599, -0.22970756888389587, -0.5989564657211304, -1.2583180665969849, 0.3811205327510834, 0.5828281044960022, 0.8354563117027283, 0.12421026825904846, -0.238362118601799, 1.1490908861160278, 0.783278226852417, 2.320749044418335, 0.5771483182907104, -0.9177967309951782, 0.9884632229804993, 2.3570268154144287, 0.2993776798248291, -0.15610040724277496, -0.5687140226364136, 1.9433252811431885, -0.15373294055461884, -0.21125951409339905, 0.3418104946613312, -1.3857483863830566, -0.23304623365402222, -1.2283991575241089, -0.09942019730806351, 0.5426042079925537, 0.9742494821548462, -0.3717920482158661, 0.8710718154907227, -0.34345632791519165, -1.6070256233215332, 0.6258801817893982, -0.15362507104873657, 0.16474390029907227, -2.569173812866211, 0.2735447883605957, 0.3110918402671814, 0.3114382028579712, 1.1089649200439453, -0.29332032799720764, 2.223723888397217, 0.8031034469604492, 0.6240062117576599, -2.421482801437378, -1.8931089639663696, 1.5844553709030151, 0.47644317150115967, 1.0201116800308228, -0.29070839285850525, -0.02703980542719364, 0.685147762298584, -0.6922662258148193, 1.4292187690734863, -1.0044193267822266, 0.33179447054862976, -0.15690743923187256, 0.4732441008090973, 1.37125563621521, -0.996040940284729, 0.5844177007675171, 0.254646360874176, 0.9801493883132935, 0.11495698243379593, -0.35196763277053833, 0.6444337368011475, 1.4782006740570068, 0.9628000259399414, 0.3031577169895172, 0.5738712549209595, -1.393775224685669, -0.3917389512062073, -0.10254844278097153, -0.3870595097541809, -0.5505884885787964, 0.1027279868721962, 0.551355242729187, 2.090562105178833, -0.5512598156929016, -0.7597450613975525, 1.1461411714553833, 0.30457401275634766, -0.7397732734680176, 1.680578351020813, -0.06008252128958702, -0.17854255437850952, -0.46757954359054565, -0.7756847143173218, 0.24523374438285828, -0.814614474773407, 2.596635580062866, 0.4721570611000061, -0.11234265565872192, -0.6410627961158752, 0.3851921856403351, 1.1213703155517578, 0.3636777997016907, -1.0227874517440796, -0.11412151902914047, -1.5907870531082153, 1.546349287033081, 0.7935159206390381, 0.3170539140701294, 0.4014677405357361, -0.45886802673339844, -0.03695022687315941, 1.0700042247772217, 0.06491156667470932, 0.6977799534797668, 0.32922038435935974, 0.3413335978984833, 0.42848336696624756, 1.1508064270019531, -0.18001380562782288, 0.7919830679893494, -0.6769083738327026, -0.46685364842414856, -0.6009628772735596, 0.9694458842277527, -0.18392646312713623, 1.3453326225280762, -0.899358332157135, 0.506836473941803, -1.0331485271453857, 1.245664119720459, -0.56925368309021, 0.5801404118537903, -0.5636005997657776, -0.6410977244377136, 0.36529356241226196, -1.5916415452957153, -1.0686767101287842, 1.6341973543167114, -0.6977888941764832, 0.9383077621459961, -0.7373259663581848, -1.517554521560669, -0.29113227128982544, 0.9870496988296509, 0.09527728706598282, -0.17467331886291504, 0.9747644066810608, -0.640337347984314, 0.841088056564331, 0.4294322729110718, -0.9108722805976868, 1.0386327505111694, 1.2351939678192139, -0.13237424194812775, 0.22949165105819702, -0.17156587541103363, -1.0147161483764648, 0.8962792158126831, -0.9046406745910645, 0.628000020980835, -0.43150898814201355, -0.9284555912017822, 0.7108275890350342, 0.330066055059433, 1.9410794973373413, 0.025128886103630066, 0.24491490423679352, 0.2112748920917511, -0.15376640856266022, 1.4126092195510864, 1.6005057096481323, 0.7781341075897217, -0.9088376760482788, 0.55153489112854, 0.1603454053401947, -0.34652310609817505, -1.4476244449615479, 0.987802267074585, 1.303877592086792, 0.41584914922714233, 0.5601369142532349, 1.8627432584762573, 2.0399672985076904, 0.3273084759712219, -2.747089385986328, 0.8175570368766785, 0.25750309228897095, 0.7897226214408875, -0.4121878445148468, -0.8969552516937256, 0.6126949191093445, -0.196690633893013, 0.2041257619857788, 0.5165899991989136, 0.2013186812400818, -1.3402438163757324, 0.6684731841087341, -0.3956027030944824, -0.12074411660432816, 0.5113555788993835, -1.366544246673584, -0.183507040143013, -1.746786117553711, -0.48186153173446655, 0.4096670150756836, -0.07690535485744476, -0.22228245437145233, 0.6266838908195496, -0.3570657968521118, -1.5545387268066406, 1.0958157777786255, 0.23265428841114044, 0.8052873015403748, -0.3945046067237854, -0.4247748851776123, -0.7371426820755005, -1.9241280555725098, -0.474252313375473, -0.6128571629524231, 1.560513973236084, -1.9220070838928223, 0.0030023169238120317, -0.21344541013240814, -0.11847500503063202, -0.19713065028190613, 0.16718223690986633, 0.7973518371582031, -1.7093943357467651, 0.768265426158905, -0.3487713038921356, -1.058093786239624, -1.9339700937271118, -1.9043972492218018, -0.16302333772182465, -1.2126497030258179, -1.2184237241744995, -0.5895909667015076, -0.07720280438661575, -0.5665599703788757, -1.2274507284164429, -0.7198516726493835, -0.2678945064544678, -1.1398073434829712, 0.8141037821769714, 1.1814849376678467, 1.2072911262512207, -0.40028589963912964, -0.41727718710899353, -0.9886246919631958, 1.3025071620941162, -0.5436244010925293, 2.104508876800537, -2.098515033721924, 0.16232331097126007, 0.4405595362186432, -0.05316127464175224, 0.526418924331665, 1.9365832805633545, -0.09313882887363434, 0.7715348601341248, -0.49378636479377747, -1.294114351272583, -1.0982331037521362, 0.7948825359344482, 0.2787083089351654, -1.4689363241195679, 0.2773216664791107, 0.5684255361557007, 0.4554232656955719, 1.0979690551757812, -1.4759453535079956, -0.9629876017570496, -1.0878472328186035, 0.12799318134784698, 1.2881345748901367, -0.4503118097782135, 0.3336343765258789, 0.8592336773872375, -0.4626212418079376, 0.8013818264007568, -0.11802595108747482, 1.7629021406173706, 1.3995916843414307, -0.439474493265152, -0.524234414100647, 1.851724624633789, -0.8503212928771973, 1.1733943223953247, -0.38917219638824463, -1.130934238433838, 1.7466504573822021, 0.460079550743103, -1.004965901374817, -0.2092638611793518, -0.7027046084403992, -0.9360027313232422, -0.07686091959476471, 1.118921160697937, -1.3821264505386353, 0.5226945877075195, -1.4241079092025757, -0.04648755490779877, -1.4197629690170288, -1.3156477212905884, -0.6737267374992371, 0.9969309568405151, 0.8332121968269348, -0.1400996744632721, -0.3223450779914856, 0.10968479514122009, 2.8069612979888916, 0.33993950486183167, -0.13760988414287567, -0.49008429050445557, -0.3188677430152893, 0.6106883883476257, 0.5691229104995728, -2.393630266189575, 2.089538812637329, 0.43353843688964844, -0.10376215726137161, 2.668743133544922, -1.5537388324737549, -1.258624792098999, 1.1098220348358154, -0.006835451815277338, 0.8768751621246338, 1.2191746234893799, 1.820399284362793, 0.6593455672264099, -0.6425390839576721, -1.6820188760757446, 0.1297117918729782, -1.7652734518051147, 0.05356057733297348, -0.805429995059967, 0.3753669559955597, 0.6886500716209412, -0.04873517155647278, -1.3864518404006958, 0.14003323018550873, -0.7328979969024658, 0.30258044600486755, -0.4667275547981262, 2.3280980587005615, -0.5582310557365417, -1.9399346113204956, -0.9251581430435181, 0.07279361039400101, 2.7747459411621094, 1.3956878185272217, -0.6048130393028259, -0.0038610075134783983, -1.2178730964660645, -1.477757453918457, 0.932426929473877, 0.403515100479126, -0.7681823372840881, -0.7328890562057495, 0.41636011004447937, 0.669384777545929, -0.8196051716804504, -1.7338343858718872, 0.8910172581672668, -0.75168776512146, -1.8518257141113281, -0.2035514861345291, 0.20039072632789612, 0.9549105167388916, -0.44994020462036133, 0.2671380639076233, -1.2877706289291382, 0.4566347599029541, -1.563422679901123, 0.598377525806427, 0.1581210494041443, -1.5435774326324463, -0.6452536582946777, 1.656347393989563, -1.077950358390808, -1.6710238456726074, -0.3032878041267395, 0.15415480732917786, 0.8107074499130249, -0.15956942737102509, -0.48023444414138794, -0.24728916585445404, 0.764523983001709, -0.19225817918777466, 0.548805832862854, -1.809944987297058, -0.3737794756889343, -0.10034368187189102, 1.0286647081375122, 0.45232895016670227, 0.20530320703983307, 1.0783238410949707, 0.16320696473121643, 3.1584367752075195, -0.9232821464538574, 0.11891160905361176, 1.849683403968811, 0.7840509414672852, 0.31393128633499146, 1.0169039964675903, -0.4675537347793579, 0.12108337134122849, 0.18246309459209442, 1.9606969356536865, 0.33724597096443176, 2.0994298458099365, -0.017480092123150826, 0.2773311138153076, -1.7049813270568848, 0.6417155265808105, -1.9829516410827637, 0.6865586638450623, 0.9971675872802734, 0.41438794136047363, -1.0490918159484863, -0.28038930892944336, -2.667123794555664, -0.7223902940750122, 0.23755238950252533, 0.0031034122221171856, -0.37469497323036194, 1.3118318319320679, 0.34605672955513, -0.0095148216933012, 0.18177476525306702, -1.0836495161056519, -0.19356705248355865, -1.9889007806777954, -2.7987148761749268, -0.37729063630104065, 0.9062796235084534, -1.1366767883300781, 0.36298179626464844, -1.7897584438323975, -0.05663633719086647, -0.01177810225635767, -0.036181043833494186, -1.4821664094924927, 0.5263088941574097, 0.6722403764724731, 0.5190643668174744, 2.266423225402832, -0.3777061998844147, 0.17430755496025085, 1.207990050315857, -1.36544930934906, -0.9337140917778015, 1.0937907695770264, -0.3591651916503906, -0.8170666098594666, 0.4860116243362427, -1.3740428686141968, 0.13816285133361816, -1.050660252571106, -0.6622568368911743, -2.6806116104125977, 0.20693176984786987, 0.24180398881435394, -0.1857728213071823, -0.6757552027702332, 0.22834494709968567, 1.4790589809417725, 0.8793352842330933, -0.7165902256965637, 0.24446220695972443, -0.2520046532154083, 0.5454614162445068, -0.803826093673706, 1.502301573753357, -0.7802918553352356, -0.9982320070266724, 0.3579771816730499, 1.6611402034759521, -1.6792556047439575, 0.0731918066740036, 0.6472507119178772, 2.727679491043091, -1.493826985359192, 1.1509730815887451, -0.7175939679145813, -0.313991904258728, 0.9780725836753845, 0.07314704358577728, -1.351309061050415, -0.20304499566555023, -0.37296512722969055, 0.19463419914245605, -0.9337754249572754, -0.6872219443321228, -0.3332996964454651, 0.028917351737618446, -0.1684538722038269, 0.433274507522583, 0.6432778835296631, 0.3274223208427429, -0.6657026410102844, -1.563272476196289, -0.9012159109115601, -0.6990161538124084, -0.2803460359573364, 0.8912644386291504, -0.11162547022104263, -0.9973923563957214, -1.3665040731430054, -0.17101259529590607, -0.3123100996017456, -1.127493143081665, -1.7111409902572632, -0.41319629549980164, 0.32066670060157776, 0.13894645869731903, -1.529258370399475, -0.5116206407546997, -0.2697174549102783, -1.0454437732696533, -0.16380614042282104, -0.7201964855194092, 0.8476422429084778, -0.7316958904266357, 1.5484598875045776, 0.9457424283027649, 0.32671019434928894, 0.21312929689884186, 1.3708993196487427, -0.3915010690689087, 0.39734137058258057, -0.2660280466079712, 1.521628975868225, -0.3137664794921875, -0.26461076736450195, -2.6545071601867676, 0.2328719049692154, -0.3748224377632141, 0.5526352524757385, -1.4839873313903809, -0.8876532912254333, 1.2927701473236084, -0.7849860191345215, 0.18080474436283112, 0.8994319438934326, 0.06402359902858734, 0.046491838991642, 1.1975826025009155, -0.5458376407623291, 0.563004195690155, -0.28016120195388794, -1.1387696266174316, 0.03673404082655907, 1.3233083486557007, -0.17636507749557495, -0.7029561400413513, -0.21106581389904022, -0.48907965421676636, 0.5068619847297668, -1.007771611213684, 0.8978118896484375, 0.4113196134567261, 1.2242766618728638, -0.45999157428741455, 0.9414390921592712, -0.999001681804657, 0.6036674380302429, -0.22615057229995728, -1.1208693981170654, 0.7911710739135742, 1.4563997983932495, 1.791345477104187, 0.4552600681781769, -0.23789256811141968, 1.4815205335617065, -0.9480773210525513, 0.14320899546146393, -1.4804913997650146, 0.1446722447872162, 0.25892746448516846, -0.40554308891296387, 1.0498600006103516, -0.1267859786748886, 0.47724461555480957, 0.540884792804718, -0.12205758690834045, -2.5662927627563477, -0.4149852395057678, 0.16074281930923462, -1.0207897424697876, 0.047427691519260406, -1.5008867979049683, 0.620630145072937, 0.6164167523384094, 0.397392213344574, -0.49345797300338745, 0.28224778175354004, -1.2736107110977173, -0.7967376112937927, 0.4111950695514679, -0.6933547258377075, -0.26668670773506165, -0.03917592391371727, 1.0462981462478638, -0.6963016986846924, -0.5843914747238159, -0.7268269658088684, -1.6830404996871948, 0.16197092831134796, -2.103029489517212, -0.8465458750724792, 1.2659733295440674, -0.02627784200012684, -2.3051788806915283, -1.0430693626403809, 1.6285830736160278, -0.32531705498695374, 1.8127764463424683, -0.601321816444397, -2.4915764331817627, 0.21890464425086975, -0.2836671471595764, 0.23245204985141754, -1.1036114692687988, -1.1257827281951904, -0.6452894806861877, -1.1196917295455933, 0.8037201166152954, 1.0661982297897339, -0.27029693126678467, 0.8916670083999634, -0.7507020831108093, -0.3655409514904022, -1.7271910905838013, 1.2042310237884521, -0.5892519950866699, 1.2109616994857788, 1.2067854404449463, 0.4842170178890228, 0.19888806343078613, -1.062806248664856, 0.838993489742279, 0.597255527973175, 1.0923823118209839, 0.33935847878456116, -1.2334654331207275, -1.1412882804870605, 0.9349463582038879, 1.6422641277313232, -0.25684723258018494, -0.07043473422527313, 0.32245948910713196, 0.2091006338596344, 0.298260360956192, -0.2612937092781067, 0.18789055943489075, -0.5783578157424927, -0.12147719413042068, 0.6596133708953857, 0.03824423998594284, -0.6475821733474731, 0.27309650182724, -1.124527931213379, -0.6790493130683899, -1.0731337070465088, 0.25758224725723267, 0.6163615584373474, -0.39491161704063416, -0.9228276014328003, 0.4418676197528839, 0.22130244970321655, -1.6025326251983643, 0.9381987452507019, 0.8914095759391785, -0.294089138507843, -0.10889469087123871, -0.14413361251354218, -0.5229936838150024, 0.3858964145183563, -0.7756603360176086, -0.3592498302459717, 0.08548370003700256, -1.749052882194519, -0.5796825289726257, -1.3222270011901855, -0.5378828048706055, -0.5960327982902527, -0.28943049907684326, 0.037295665591955185, -1.1334788799285889, 0.17353443801403046, -1.8358405828475952, 0.804293155670166, -0.21222816407680511, 0.6526786088943481, -2.0498552322387695, 1.2353686094284058, -0.43580102920532227, 0.1898174285888672, -0.03460575267672539, -0.2121879756450653, -0.6091708540916443, -2.14418888092041, 1.1060833930969238, 1.7320771217346191, -0.7358713746070862, 1.220913290977478, 0.8258031010627747, 0.24383249878883362, -0.23866446316242218, 0.49645984172821045, -0.7413650155067444, -0.425754576921463, -0.6233797669410706, -0.8327327966690063, -0.6360228657722473, 0.5183205008506775, 0.5902007818222046, -0.8880091309547424, -0.14987622201442719, -0.27056118845939636, -1.8161600828170776, 1.0486866235733032, -1.1685906648635864, 1.0682004690170288, -2.397200345993042, -0.842426598072052, 0.7046791315078735, 0.3832975924015045, 0.45242074131965637, -0.09305988252162933, 2.011720657348633, 0.02196699194610119, 0.2972586154937744, -0.765188992023468, -0.8360711932182312, 1.3422855138778687, 1.3497984409332275, -1.8405654430389404, -0.5823349952697754, 1.2332276105880737, 0.0700664147734642, 0.9620758891105652, -1.8252867460250854, 0.17479492723941803, 0.007433827966451645, 0.059796616435050964, -0.6577346920967102, -0.04246719926595688, 1.7822335958480835, -1.0846375226974487, -0.0694727897644043, -0.3092872202396393, -0.2691084146499634, 1.8194668292999268, 1.5958808660507202, -0.36823025345802307, 0.5385801792144775, -0.8205536007881165, 0.3292393088340759, -0.37009963393211365, 1.495968222618103, 0.46551063656806946, -1.0564483404159546, -1.804330825805664, -1.1226171255111694, 2.1403796672821045, -0.976892352104187, -1.4260982275009155, -0.6778203845024109, -0.48983901739120483, 0.6301676034927368, -0.7599680423736572, 1.6918151378631592, 0.01788458600640297, 0.31674566864967346, 1.4498584270477295, 0.19388997554779053, 0.3390420973300934, 0.6624720096588135, -0.7225229144096375, -0.5285610556602478, 0.7007918953895569, -0.7758427858352661, -0.5649134516716003, 0.9578723907470703, 0.6158062219619751, 0.3586193919181824, -1.13657546043396, 2.007783889770508, -0.1398862600326538, -0.3081517815589905, -0.1704953908920288, -0.8389602899551392, -0.26653894782066345, 2.027139186859131, -1.5161992311477661, 2.263695478439331, -0.958862841129303, -0.959499716758728, 1.3763152360916138, 1.930783748626709, -0.40711385011672974, 0.8299251794815063, -0.797158420085907, -0.3780014216899872, -0.7820996642112732, 0.10535077750682831, -0.6876091957092285, 0.09549757093191147, -0.3923560380935669, -0.049512043595314026, 0.014896092936396599, -0.7407687306404114, -0.4537556767463684, -1.2314902544021606, 0.4334135055541992, 0.6674797534942627, -0.4507458209991455, -0.37434616684913635, -0.6441704034805298, -0.5523775219917297, 0.051353249698877335, -1.7972325086593628, -1.563706874847412, -0.03474334627389908, 0.7849850654602051, -0.8056647181510925, 0.6938956379890442, 0.8414202332496643, 0.00061203190125525, 2.12799334526062, -0.22714976966381073, 0.02641822211444378, -1.247353434562683, -1.1387319564819336, -0.005207929294556379, 0.468589723110199, -0.3257032036781311, -0.17292198538780212, -0.01665063016116619, 0.6307157874107361, -1.8342331647872925, 0.658908486366272, -0.9833194613456726, 0.3472721576690674, 0.6396923065185547, -1.556449055671692, 0.9614579677581787, -0.09901413321495056, -0.7047393321990967, 1.6174067258834839, -1.4450690746307373, -1.6921610832214355, 0.007322079036384821, 0.25054529309272766, -0.40666821599006653, 0.3890153467655182, -2.5502145290374756, 0.8137433528900146, -0.00559651292860508, -1.2616443634033203, -1.377927541732788, 0.29472121596336365, 0.23191006481647491, -0.8204362392425537, -0.046837568283081055, -0.11684013903141022, 0.9336124062538147, -1.0730763673782349, 0.8804498910903931, -0.023246577009558678, -0.07134547829627991, 0.41808629035949707, -0.5960789918899536, -1.540041446685791, -1.7476743459701538, 0.9128369688987732, -1.8464511632919312, 0.9381548166275024, -0.8126603364944458, 1.0017503499984741, -1.0244101285934448, 0.2099343240261078, 0.09134814888238907, 0.690533459186554, -0.48539993166923523, 0.3644758462905884, 1.4970744848251343, -1.444985032081604, 0.9577875137329102, -1.1349315643310547, 0.9047306776046753, -0.448495477437973, 1.384230613708496, -0.3153761923313141, -0.5612239837646484, 0.9912170767784119, -0.2962316870689392, -0.25200381875038147, 0.9129879474639893, 1.3927404880523682, 0.8301562070846558, -1.9340335130691528, 0.3177182078361511, 0.27913597226142883, 0.8592969179153442, 1.0563231706619263, 1.0304932594299316, 0.5026458501815796, 2.0529823303222656, -0.11749757081270218, -0.6291709542274475, -1.2159324884414673, -0.33208051323890686, 0.8362230062484741, 0.09022649377584457, 1.47105872631073, 0.06341973692178726, -0.4633646607398987, 1.5986789464950562, -0.11946913599967957, 1.228693962097168, -2.1568257808685303, -1.221801996231079, -0.4377862513065338, -0.9530757069587708, 0.3321724236011505, 0.1649041771888733, 0.3091050684452057, -0.7998711466789246, -0.7810963988304138, -0.5725833177566528, 1.5637717247009277, -0.22060637176036835, 0.9748853445053101, -0.7683118581771851, -1.3755064010620117, 0.1616361439228058, 0.9899457097053528, -1.6493217945098877, -0.526802122592926, 0.6281986832618713, -0.2686956226825714, -0.30805617570877075, 0.7574686408042908, 0.1481786072254181, 0.1654220074415207, -0.5176767110824585, 0.2570085823535919, -0.5238391160964966, 1.3991074562072754, -3.104443073272705, 0.09458345919847488, 0.7116065621376038, -0.9667118191719055, 0.5773001313209534, -0.7307002544403076, 0.679527997970581, 1.6355239152908325, -0.2545260190963745, 2.2056217193603516, -0.042973652482032776, -2.4412009716033936, -0.4305480420589447, 0.10364095121622086, 0.19742542505264282, -0.04644368588924408, 0.35097652673721313, -0.22435550391674042, 1.297217845916748, 0.2887120544910431, -0.17465293407440186, 1.797566294670105, -1.3163118362426758, -1.9239383935928345, -0.7279305458068848, -2.5956838130950928, 0.4604748487472534, -0.34730908274650574, 2.0632684230804443, -0.9786630272865295, -1.2366728782653809, 0.6351453065872192, 0.5907925367355347, 0.6226950883865356, -0.6258736848831177, -0.6309187412261963, 0.5020201802253723, -0.2005874216556549, 0.5315168499946594, 1.0815269947052002, -0.3271239399909973, 0.5293330550193787, -1.9626060724258423, 1.555724024772644, 1.4922387599945068, 0.14742256700992584, -0.23429110646247864, 0.14019832015037537, 0.25831419229507446, -0.05360828712582588, -0.6254017949104309, 0.31552955508232117, 1.2422159910202026, -1.886293649673462, -0.47751280665397644, -0.7891485095024109, -1.1116383075714111, -0.001025487552396953, -2.178266763687134, -0.4085840880870819, 0.8566321730613708, 0.903753399848938, 0.6848056316375732, 0.3476071357727051, 0.8362714648246765, 0.17232786118984222, 0.5112305283546448, 0.7581361532211304, 1.5258891582489014, 1.0757631063461304, 0.9365333318710327, -1.4396315813064575, 0.32189321517944336, 0.21753999590873718, -0.4869099259376526, -0.986884593963623, 0.6954519748687744, 0.12179695814847946, -0.26509660482406616, -1.061543583869934, -0.7206279039382935, -0.4973069727420807, 0.3283075988292694, 1.656814455986023, 0.2117931842803955, -1.979476809501648, 0.4335184395313263, 0.46364250779151917, -1.0189481973648071, -1.3299508094787598, -0.0038242507725954056, 0.7289733290672302, 0.39037925004959106, 1.5290594100952148, -0.16642716526985168, -0.02615181915462017, -0.9110639095306396, 0.9278238415718079, 0.15830589830875397, 0.5025833249092102, -2.321216583251953, -1.841459035873413, 0.297913134098053, 0.5362416505813599, -0.49738338589668274, -0.38602229952812195, 0.641112744808197, -0.8979023694992065, -0.07149983942508698, -1.0428835153579712, -0.680851936340332, 0.7307190299034119, -1.8332597017288208, -0.8398033976554871, -2.5819177627563477, 0.22623766958713531, 0.10617378354072571, 2.789687395095825, -0.48561322689056396, 0.1400134265422821, 1.0045424699783325, 0.597095251083374, -0.3646883964538574, 1.1571643352508545, -2.8006656169891357, 0.4018033742904663, -0.8237016797065735, -0.8602696061134338, -1.1325374841690063, 0.24729779362678528, -1.1234593391418457, 0.05651009455323219, 1.6345884799957275, -0.8203730583190918, -0.0005702564958482981, -0.7058470845222473, -1.4266785383224487, 0.4707368016242981, 1.1149262189865112, -0.7815501689910889, -0.06131651997566223, -0.256969690322876, -0.8580024242401123, 0.7094393372535706, -2.1231634616851807, 1.3732467889785767, -0.6638829708099365, 1.6404231786727905, 0.3766246736049652, -0.205758199095726, -1.8112331628799438, -1.4524770975112915, 0.541599452495575, -0.8737642765045166, 0.7500434517860413, -1.4272911548614502, 0.9326648116111755, -0.16561098396778107, 0.25196152925491333, -1.0359212160110474, -0.5797218084335327, 0.35120880603790283, 0.0049696629866957664, -1.1236529350280762, 0.04656178876757622, -1.64536452293396, 1.8568570613861084, 0.40206781029701233, -0.666921079158783, -0.47116708755493164, -1.1940113306045532, -0.08387478440999985, -2.9210379123687744, -0.19710490107536316, 1.5188907384872437, -2.281765937805176, -0.00988624058663845, -0.732597827911377, -0.3493001163005829, -1.348891019821167, 0.8098892569541931, -1.4676802158355713, 0.8046097755432129, 0.909000813961029, 9.0, 0.4366960823535919, -2.1150569915771484, -0.8103950619697571, -2.651766300201416, 0.47973477840423584, 1.0098332166671753, -1.6515239477157593, 0.06291496008634567, -0.378065824508667, -2.0521302223205566, -0.4941888749599457, -0.05300423502922058, 0.8684529662132263, -0.23716527223587036, 0.531751275062561, 1.8294305801391602, 1.0448269844055176, -0.879642903804779, 0.45014163851737976, 1.6334974765777588, 2.69873309135437, 0.7433814406394958, 1.1572036743164062, 0.8913856744766235, 2.063788652420044, 0.9464554786682129, 1.097573161125183, -0.9135310053825378, -0.328651487827301, -1.5431058406829834, -0.891595184803009, 0.3194286525249481, 1.1315746307373047, 1.0837029218673706, 0.4962824881076813, 2.2053773403167725, -1.2620899677276611, -0.0014668593648821115, 1.5251878499984741, 0.41251781582832336, 0.2051449716091156, 1.0189682245254517, -0.6945900321006775, -0.6745295524597168, -0.34352847933769226, 1.9085383415222168, -1.2610087394714355, -0.5805574655532837, 2.3309009075164795, 0.43690457940101624, -0.23002944886684418, -1.8490453958511353, 0.19427140057086945, -0.0070648472756147385, 0.9929315447807312, -0.8283033967018127, -0.6644617319107056, 0.8811376094818115, -0.9570307731628418, 1.6328628063201904, -0.15655626356601715, 0.4479122459888458, -0.5733190774917603, 1.733536958694458, -0.5630571246147156, 1.3651877641677856, -0.49156537652015686, 0.43324458599090576, -0.843168318271637, 0.8134686946868896, 1.6235227584838867, -0.483330637216568, 1.8981027603149414, 0.1410674750804901, -0.704012393951416, 0.31492453813552856, -0.4090029299259186, 0.2818307876586914, -0.4595433175563812, -1.552743673324585, 1.3016635179519653, 0.2800787687301636, -0.3084815740585327, 0.6530278325080872, -1.3492072820663452, -0.20185913145542145, 0.7462832927703857, 1.4395376443862915, 0.2309342473745346, 0.9131057262420654, -1.7256338596343994, 0.022201431915163994, 0.9180032014846802, 1.3854711055755615, -1.1503503322601318, 1.649262547492981, 2.5541348457336426, 0.250798761844635, -1.685581088066101, 0.7345517873764038, 0.713752269744873, -0.399766743183136, 1.4480029344558716, 1.8618863821029663, -0.9810760617256165, -1.0297445058822632, 0.28687480092048645, 1.2372815608978271, -1.2569130659103394, -0.7972064018249512, -0.2910524010658264, -0.8525568842887878, -0.27208369970321655, 0.9759091734886169, -0.9351677894592285, 0.698798418045044, -0.034150052815675735, -0.3127303123474121, -1.452479362487793, 0.20081216096878052, 0.5087900161743164, -0.8499066829681396, 0.899718701839447, 0.9903169870376587, 0.13557744026184082, -0.9155983924865723, -0.12511704862117767, 0.940676212310791, 0.9159737229347229, -0.0947057381272316, -0.22385026514530182, 0.7589336633682251, 0.6808295845985413, 1.0427464246749878, -1.1032891273498535, -0.11214200407266617, 0.914781928062439, -0.7259528636932373, 0.14024758338928223, -0.7236114144325256, 1.0014796257019043, 1.365478515625, -0.5018090605735779, 2.4308395385742188, 0.8847157955169678, -0.26825374364852905, 0.5766239762306213, -0.41685450077056885, 1.1021496057510376, -0.1007910817861557, -0.8258553743362427, -0.20693360269069672, -1.2314563989639282, -2.2734549045562744, -1.4075008630752563, 0.5677826404571533, -0.5621827244758606, 0.39220666885375977, -0.47659674286842346, -0.057837020605802536, 0.30218496918678284, 0.8597078919410706, -0.9902725219726562, -0.8610193729400635, 0.05302515625953674, 1.598530888557434, 0.2803572416305542, -0.18498745560646057, -0.158583402633667, -0.24420714378356934, -0.6180244088172913, 0.8967393636703491, 1.9007689952850342, 0.28914448618888855, -1.4441664218902588, -1.8221380710601807, 1.9752938747406006, -1.7421108484268188, 2.3925490379333496, 0.49584850668907166, -0.11992235481739044, 0.8765802979469299, 0.04960684850811958, 1.7051575183868408, -0.47124749422073364, 0.33798524737358093, -0.518356204032898, 0.5480863451957703, 0.8434584736824036, -1.3017253875732422, -1.6280397176742554, 1.0278490781784058, 0.8092420101165771, 0.4935280680656433, -0.6976474523544312, -1.1545417308807373, -1.0096378326416016, 0.03269349783658981, -0.9767012596130371, -0.985382616519928, 0.3889610767364502, 1.7174848318099976, 1.9794524908065796, 0.2810607850551605, 1.1954530477523804, 0.5292657017707825, 0.4156470000743866, -0.04335087537765503, -0.5846619009971619, -0.4936845898628235, 0.28548502922058105, 1.5122963190078735, -1.3202111721038818, -0.5249760746955872, -1.1332356929779053, 0.7199828028678894, 2.5237770080566406, 0.5282914638519287, -1.2871896028518677, 0.5195638537406921, 2.052410840988159, -0.7688151001930237, -2.2920584678649902, -2.012869119644165, 0.4799420237541199, -1.9279592037200928, 1.5767189264297485, -2.440887689590454, -0.1687821000814438, -1.328340768814087, 0.8688510656356812, 0.3626765012741089, 0.6329400539398193, 0.7403234243392944, -0.8294228315353394, 0.09370584040880203, -0.9430159330368042, -0.3867145776748657, 0.17581897974014282, -0.43189185857772827, -1.4463568925857544, -1.3687649965286255, -1.2172956466674805, -1.1468403339385986, -1.2032277584075928, 1.3765488862991333, 0.8269774317741394, 0.2884029150009155, 1.4013619422912598, 0.5126029849052429, -0.5105431079864502, 1.161098599433899, 0.5045958757400513, -2.152647018432617, 2.290482521057129, -0.14638717472553253, -1.7953741550445557, 1.3368653059005737, 0.114399254322052, 0.1443893313407898, 1.447786569595337, 0.8777902722358704, 0.332439661026001, 0.22543002665042877, 0.09921532869338989, -0.18316856026649475, -1.4083151817321777, 1.0969440937042236, 0.0005215710261836648, -0.11673787981271744, 0.7650635838508606, -0.7070277333259583, -0.426769882440567, 0.6876895427703857, -0.23635689914226532, -1.1324067115783691, -1.0853770971298218, 2.968015193939209, -0.5759568214416504, 2.2566492557525635, -0.7582289576530457, 0.3598344922065735, -0.4831227958202362, -1.2201143503189087, -0.35468846559524536, 2.113914728164673, -0.8151161074638367, -1.0704602003097534, 0.3783693313598633, -1.0707391500473022, -0.8701009750366211, -1.5222082138061523, 0.9571831226348877, 0.6589531898498535, -2.4903903007507324, 0.6539967060089111, -0.2932114005088806, -0.13507258892059326, -0.8150467276573181, 0.17307166755199432, 0.592769980430603, -0.8825793266296387, -0.06822184473276138, -0.10712767392396927, -0.8843346834182739, -0.6163951754570007, -0.17211298644542694, 0.7948868274688721, -0.7405109405517578, -0.04376114159822464, 0.9758882522583008, 1.3615254163742065, -1.5287258625030518, 0.16882826387882233, -0.28913363814353943, -0.6035681366920471, -0.7802486419677734, 1.6774380207061768, 0.9795106053352356, -0.3082907497882843, 0.0633917897939682, -0.11984837800264359, -0.238459974527359, 0.7963541746139526, -0.5105640292167664, 0.7052000761032104, -1.4809495210647583, 0.2326347380876541, -0.14205530285835266, -0.6014551520347595, 0.40969768166542053, 0.12235400825738907, -0.3395129442214966, 1.3385026454925537, -0.13283973932266235, -0.19740243256092072, -0.5475403666496277, 1.1089884042739868, 0.9588006138801575, -2.3794469833374023, -0.92307448387146, 0.3583967089653015, 0.9635907411575317, 0.5202352404594421, -0.0924760028719902, 1.489818811416626, -0.5038323998451233, 0.12379509955644608, 0.03115248866379261, 0.155657559633255, -0.9124954342842102, -0.24074940383434296, 0.7410072088241577, 1.052878975868225, 0.8668227195739746, 0.5209587812423706, -0.22754348814487457, 0.6937247514724731, 0.00977340992540121, 0.1842057704925537, 0.49403315782546997, -0.781653642654419, 0.6141483187675476, 1.1691720485687256, -1.5757511854171753, -0.277225524187088, -0.3295084834098816, 1.6359448432922363, 0.14454519748687744, 0.6457842588424683, 0.7772934436798096, -0.7518274784088135, -1.296535611152649, 1.1572288274765015, 1.317269206047058, 0.9073278903961182, 0.8504289984703064, -0.6898838877677917, -0.7010238170623779, 0.03048011101782322, 0.6839897632598877, -0.3079220950603485, -0.28417015075683594, 0.5093578100204468, 1.6199713945388794, -1.5618393421173096, -0.20981425046920776, -1.481415867805481, -1.7997366189956665, 1.140315055847168, -1.0079346895217896, 0.3775971829891205, 1.1373172998428345, 0.709613025188446, -0.9786646962165833, 0.38811659812927246, -0.06323745101690292, -0.6997265219688416, -0.31594061851501465, -0.5269657969474792, 1.0272201299667358, -0.011361729353666306, -0.4074234366416931, 0.409265398979187, 0.40972840785980225, 0.6106203198432922, 1.006721019744873, -0.7470558881759644, -0.5424505472183228, 1.1369471549987793, -0.6856306791305542, 0.7141010165214539, -2.208653211593628, -0.019444143399596214, -0.8592761158943176, -0.2121453434228897, 0.4046761393547058, -1.9773340225219727, -1.5359680652618408, -0.038483474403619766, -1.1908235549926758, 0.21772992610931396, 0.06183825060725212, -1.0583438873291016, 0.9126214981079102, 0.7068728804588318, 0.42292460799217224, -0.5862930417060852, -0.5588416457176208, -1.430053949356079, -2.082641839981079, -0.3879413902759552, 0.705820620059967, -0.27136221528053284, 0.567409098148346, -0.5780177116394043, -0.4732751250267029, -0.4492134153842926, 1.1999906301498413, -1.165043830871582, -0.7053467631340027, 0.7476173639297485, 0.9675102829933167, 0.6058588027954102, 2.6187477111816406, -0.12835882604122162, 0.014947405084967613, -0.5729653239250183, -0.3166317343711853, 1.7676599025726318, 1.0646635293960571, -0.3286202847957611, 1.3295977115631104, -0.9217328429222107, 1.8469785451889038, -0.3688420057296753, 0.8296511769294739, -1.2980022430419922, 0.18369299173355103, -0.3980085253715515, -0.21582765877246857, -0.09366856515407562, -1.676572561264038, -1.6425786018371582, 1.395402193069458, -1.1651421785354614, -2.2295451164245605, 0.4506939649581909, 0.007727629505097866, 0.167962446808815, -0.44870778918266296, 0.6859208941459656, 0.9246201515197754, -0.21223615109920502, 0.4161590039730072, -1.9458513259887695, -0.2596261203289032, -0.9070665836334229, -1.1445883512496948, 0.9573532938957214, -2.1532914638519287, 0.24050579965114594, -1.391554594039917, 1.1039578914642334, 0.13104437291622162, 0.9027987122535706, -0.2705051004886627, -1.4559664726257324, 2.4182679653167725, -0.17469438910484314, -1.1364880800247192, 1.1051504611968994, 2.1164324283599854, 0.031054820865392685, -0.4646923243999481, 0.6805592775344849, 0.9084532260894775, 0.9904087781906128, -0.27510881423950195, 0.014519929885864258, 0.4952469766139984, 2.0430731773376465, 1.3320729732513428, -0.5404872894287109, 0.48418986797332764, 0.8996980786323547, -0.5562095046043396, -2.8984878063201904, 0.16108956933021545, -0.4965665340423584, 0.2958228886127472, 0.4287932217121124, 0.19963321089744568, 0.08019395172595978, 1.1571928262710571, -0.3860328495502472, -1.4491393566131592, -2.2359683513641357, 0.61131352186203, -1.3019005060195923, -0.09467646479606628, 0.1336335688829422, 0.8899884223937988, 0.9627301096916199, 0.14006604254245758, 0.14260262250900269, 1.7687346935272217, -2.0664243698120117, -0.13598033785820007, -0.7333488464355469, -0.06655839830636978, 1.888277530670166, -0.4149593412876129, -1.170732855796814, -1.3799962997436523, 0.05839013308286667, -0.6304231882095337, -0.9680682420730591, 1.2672431468963623, 2.2123613357543945, -0.4208727777004242, -0.8037735819816589, 0.7130459547042847, -0.2819175124168396, 0.9412232041358948, 1.2191643714904785, 0.5848544239997864, 0.25608524680137634, -0.7983735799789429, 1.1012616157531738, 1.3302221298217773, -1.2894597053527832, -0.47199544310569763, 0.647387683391571, 0.3910370469093323, 1.3267472982406616, -0.881112277507782, -0.12134905159473419, 0.0971146672964096, 0.12740401923656464, 1.2353582382202148, 0.9495869278907776, 1.617385983467102, 0.07715406268835068, -0.3970840275287628, 0.9750033020973206, 0.5541580319404602, -1.0112385749816895, -1.058463454246521, 0.33170244097709656, 1.1074731349945068, -0.888010561466217, 1.9123514890670776, 0.7093943357467651, 0.13550519943237305, 1.3155648708343506, 0.18853268027305603, -1.3514413833618164, -0.3799613118171692, 0.960037350654602, 0.9855403304100037, -0.37621602416038513, -0.07516378164291382, 0.016601448878645897, -1.8812086582183838, -0.7023212313652039, -0.6116606593132019, 0.8450053334236145, -1.2967145442962646, 0.8070108294487, -0.1856812685728073, -0.18430912494659424, -1.8888115882873535, 1.5809805393218994, 2.3182427883148193, -1.5810447931289673, -0.464213103055954, -2.313631057739258, -0.0012579445028677583, 9.0, 0.18256303668022156, -1.4768874645233154, 2.099210262298584, 0.9407469630241394, -1.7066152095794678, -0.8193117380142212, -0.3172473907470703, -0.09235432744026184, 1.7312097549438477, -1.1291545629501343, -0.5830473899841309, -0.3142505884170532, -0.29881906509399414, -0.09507328271865845, 0.3382260501384735, 1.8656445741653442, 1.4328827857971191, 0.44796159863471985, -1.4792996644973755, -0.976367712020874, -0.9710869193077087, -0.15971128642559052, -1.186012625694275, -0.657112181186676, 0.8423916101455688, 1.312968134880066, -0.18461167812347412, 1.530304193496704, -2.9010512828826904, -0.021753733977675438, -0.6070734858512878, 0.7770478129386902, -0.14225484430789948, 0.24093976616859436, 0.522329568862915, -0.18516118824481964, 0.6187609434127808, -0.46436089277267456, -1.2634600400924683, -0.6766777038574219, 0.17882975935935974, -0.5084487795829773, -0.757121741771698, -1.7683582305908203, -0.9454478621482849, 0.6308110952377319, 0.3961038887500763, 0.7816324830055237, -0.4321557879447937, -0.22694124281406403, 0.7252547144889832, -1.1391525268554688, 3.0870745182037354, -1.0829368829727173, 0.5772818326950073, 1.647424578666687, -0.7236877083778381, -2.623116970062256, -1.5812921524047852, -2.299612522125244, -2.377865791320801, 1.8862437009811401, 0.9436936378479004, 0.8441003561019897, -0.9845811724662781, 0.03550495207309723, -0.5043834447860718, -0.3052826225757599, -0.28507158160209656, 1.092082142829895, 0.32945993542671204, 0.1716434210538864, -0.2182338982820511, -1.6855888366699219, 0.45384421944618225, 0.08069862425327301, 1.8793857097625732, -1.9127063751220703, -1.037791132926941, 0.041487909853458405, 0.44434526562690735, 0.5208432674407959, -0.08868316560983658, -0.6133303046226501, 0.09675473719835281, 1.8723728656768799, 0.9869771599769592, -1.2885255813598633, -0.5362873673439026, -1.7793198823928833, -1.196950912475586, 0.9336878061294556, 1.3571441173553467, -0.6885251998901367, 0.738249659538269, -1.235763430595398, 1.0539606809616089, -0.3965480923652649, -0.17398953437805176, -0.8247631788253784, -0.5865885615348816, 1.9506826400756836, 0.44186049699783325, 0.6722797751426697, -0.3375740051269531, -0.30087921023368835, -0.3485497832298279, -0.929820716381073, 0.7360591888427734, -1.0476163625717163, 0.7053648233413696, 1.515965223312378, 0.8597327470779419, -1.0739216804504395, 1.694413423538208, -0.87534499168396, -0.2392641305923462, -0.5219976305961609, -0.9307896494865417, -1.0655573606491089, -0.4712648391723633, -0.02953215129673481, 0.16631941497325897, -1.4830936193466187, 1.5888783931732178, 0.26761648058891296, 0.14928677678108215, 0.14908745884895325, 0.8363495469093323, 0.534059464931488, 0.30112817883491516, -0.8161541223526001, 0.7314384579658508, 0.015669049695134163, 0.016043102368712425, -0.31014975905418396, 0.4225750267505646, 0.48873403668403625, 0.3711700439453125, 1.7441829442977905, 0.9197544455528259, 0.1374603509902954, 0.5293432474136353, -1.6507924795150757, 0.9252006411552429, -0.43354347348213196, -0.7521091103553772, 0.6549360156059265, -1.0341987609863281, 0.7289068698883057, -0.6507056355476379, -1.4147638082504272, -0.17829079926013947, 0.869522213935852, -0.5642476081848145, 0.669317901134491, -0.06576628237962723, 0.04677162319421768, 1.0277013778686523, 0.07109346240758896, 2.888850212097168, -0.2960462272167206, -0.8857107758522034, -2.000452995300293, -1.2493325471878052, -0.05947785824537277, -1.8388510942459106, -1.499779462814331, 0.43125003576278687, -0.30865219235420227, -0.9949896335601807, -1.79111909866333, -0.8655440211296082, -0.25103121995925903, -0.3365599513053894, -0.27234190702438354, -0.3665618598461151, -0.4782986640930176, 0.30485063791275024, 0.9299456477165222, 0.6298050284385681, -0.5890413522720337, 0.8404924869537354, 0.4879206120967865, 0.4840703010559082, 0.6562128663063049, 0.6400593519210815, -1.1298457384109497, -0.015205185860395432, -0.39769285917282104, -1.0767841339111328, 2.9915788173675537, -0.30849528312683105, -0.8086223006248474, 0.5277032852172852, -0.2624010443687439, -0.5311663150787354, 0.2546583116054535, 0.6848768591880798, 0.41843074560165405, 0.7978110909461975, -0.8436316847801208, -0.9394998550415039, 1.4268499612808228, -1.0752431154251099, 0.5658904314041138, -0.4178893566131592, -1.4038010835647583, 0.6011890769004822, 0.819078803062439, -2.3595926761627197, -0.8681647777557373, 0.08330459147691727, -2.0798604488372803, 0.4693162441253662, -2.2365033626556396, 1.4202039241790771, -1.131225824356079, 0.07819173485040665, -1.4811094999313354, 1.2840158939361572, -0.5290544033050537, -0.9834691882133484, 0.07764142006635666, -0.33477118611335754, -2.97891902923584, -0.6663161516189575, 0.1706925928592682, -0.9926756024360657, 0.14988163113594055, -0.31460633873939514, 0.7620197534561157, -0.5711478590965271, 1.50286066532135, -0.636749267578125, -0.09794414788484573, -0.4952099025249481, -1.1998069286346436, -0.15929631888866425, -0.6882041096687317, 0.8193598985671997, -0.10393214970827103, 0.9203867316246033, 0.9004856944084167, -0.43490585684776306, -0.1658354550600052, 2.7315196990966797, 1.0146331787109375, 0.3026023805141449, 1.7902120351791382, -0.6773375868797302, -1.560037612915039, 1.0437953472137451, -0.3556532561779022, -1.6232200860977173, -0.7409505844116211, 0.1082322895526886, -1.0372803211212158, 0.10534673184156418, 0.7168010473251343, 0.5381494760513306, 1.1600446701049805, -1.468209981918335, 2.3334717750549316, -2.159125328063965, -0.7609884738922119, -1.2251083850860596, 0.6233205199241638, 0.2654283940792084, 1.4059151411056519, 1.0465642213821411, 0.8449885249137878, -0.011778539977967739, 0.4455660879611969, -0.25925546884536743, 1.1587101221084595, 0.7557337880134583, 1.3780168294906616, 0.23515091836452484, -0.08807753026485443, 0.6764244437217712, -0.40202614665031433, -0.1740283966064453, -0.8502615690231323, 0.6062297224998474, 0.299466073513031, -0.6167311668395996, 2.101966381072998, 0.6255462169647217, -0.18498627841472626, -0.877041757106781, -0.8156355023384094, -0.1084405705332756, -1.6476976871490479, -0.8926734924316406, 0.23228447139263153, 0.883387565612793, -1.1814380884170532, 1.1887305974960327, 0.821418285369873, -1.0610475540161133, -1.4602062702178955, 0.5934877991676331, 0.6491745114326477, -0.2474288046360016, 0.7570732235908508, 3.195481538772583, -0.019253332167863846, 1.4418971538543701, -0.4956215023994446, -1.401802897453308, -1.0330874919891357, -1.5973868370056152, 1.0642216205596924, 0.256679505109787, 0.46230900287628174, 0.6051137447357178, -0.6918014883995056, 0.12632103264331818, -0.4666544497013092, -0.36633044481277466, -1.1002613306045532, -1.5922808647155762, -0.003384348936378956, -0.8183835744857788, -0.7281738519668579, 0.975822925567627, 0.1575126200914383, -0.02169545367360115, -0.21771007776260376, -0.09773942083120346, 1.0379631519317627, -0.3496495187282562, 1.7192444801330566, 1.4363957643508911, -1.2837026119232178, -0.347271591424942, 0.9411511421203613, -0.7752024531364441, -1.631791591644287, 0.2841413617134094, 1.4074122905731201, -1.6620376110076904, 1.0560109615325928, -1.364511489868164, -0.9015825986862183, 0.39960071444511414, -0.39318445324897766, -0.5877174735069275, -0.4376937747001648, -1.917210578918457, 1.4061933755874634, -0.5052657723426819, -1.1286966800689697, -1.2263505458831787, 0.4764244556427002, 0.22315986454486847, -1.850036382675171, -0.5762915015220642, -1.5423592329025269, 0.31471312046051025, -0.7366553544998169, 0.527445912361145, 0.5798144936561584, 1.03426992893219, -0.4517878592014313, -0.5397548675537109, 0.10320576280355453, 0.8959557414054871, -0.43554291129112244, -0.510588526725769, 0.9423148036003113, -1.8674129247665405, 1.4490145444869995, -0.15455783903598785, 0.3131088316440582, 1.0840121507644653, -0.866771399974823, 0.236168771982193, 1.1884849071502686, 0.3026007413864136, 0.764569878578186, -0.6836226582527161, 0.7476121783256531, 0.20860591530799866, -0.40435293316841125, -1.312193751335144, 0.44075807929039, -1.058973789215088, 0.3352873921394348, 1.475382924079895, -0.35392001271247864, 0.3747059106826782, -1.1055209636688232, 0.7816861867904663, -2.0903983116149902, 0.004992483649402857, -1.1829992532730103, -1.0035533905029297, -0.9920879006385803, 1.6059378385543823, -0.9273006319999695, 1.4164402484893799, -0.10677874088287354, -0.705322802066803, -0.07182639837265015, -0.18905779719352722, -0.14127428829669952, 1.0055978298187256, -1.7217676639556885, 0.27956846356391907, -0.05227159336209297, 0.3815673589706421, -0.7717455625534058, 1.7878745794296265, 0.3613732159137726, -0.947949230670929, -0.4585036039352417, -0.26871299743652344, -0.5035516619682312, 0.9308366179466248, -1.1172558069229126, 2.533057928085327, -0.475953072309494, -0.17951014637947083, -0.18232180178165436, 0.579470157623291, -2.15073299407959, -0.058289408683776855, 0.7406378388404846, -0.3764699399471283, 0.4232959449291229, 1.767654299736023, -1.0753600597381592, -0.848827600479126, -0.03460683673620224, 0.17708256840705872, -0.07773245871067047, -0.8977365493774414, -0.7306308150291443, -0.5224047899246216, 0.43858179450035095, 2.545491933822632, 0.09363966435194016, -1.5404045581817627, 1.7066740989685059, -0.4044123888015747, 0.7959111928939819, 0.4433453679084778, -0.07938652485609055, 0.1275559514760971, -2.153319835662842, 1.0452961921691895, -2.510510206222534, 1.4828654527664185, -0.1152055487036705, -1.9664379358291626, 0.3542475402355194, 1.79624605178833, 1.7418344020843506, 0.583141565322876, 0.6407600045204163, 0.32514217495918274, 0.9597204923629761, -0.17566049098968506, 0.7701539993286133, -1.1581778526306152, 0.4867096245288849, -1.2869442701339722, -1.0996558666229248, 1.468840479850769, 0.30762553215026855, 2.066540002822876, -0.9703555107116699, -1.839718222618103, -1.018275260925293, 0.9763931035995483, 0.7727570533752441, -1.4131920337677002, -0.5381938219070435, 0.15360762178897858, -0.44252267479896545, -2.166649103164673, 2.682948589324951, 2.1731553077697754, 0.8660534024238586, -1.0353213548660278, -1.7887098789215088, 0.814258873462677, 0.6027601361274719, 1.1758415699005127, 1.344348430633545, -0.2577818036079407, -0.3018834590911865, 0.17563699185848236, -0.2630361020565033, -1.1950265169143677, 0.009526940062642097, 1.3342523574829102, 0.08321971446275711, 0.35584038496017456, 1.9895011186599731, -0.5926929116249084, 0.024405185133218765, -0.22576485574245453, -0.0181905347853899, 0.39961740374565125, -1.028568148612976, -0.12571263313293457, 2.34206223487854, -1.070081353187561, 0.3284893333911896, 1.483924388885498, 2.1679608821868896, 0.8894821405410767, -0.9892499446868896, -0.26398733258247375, 1.3874306678771973, 1.3553979396820068, -0.5408464074134827, -0.018739748746156693, 1.4831403493881226, -1.0279409885406494, 1.589324712753296, -1.8927315473556519, -1.1003401279449463, 0.6173388957977295, 0.5248202085494995, -1.0761479139328003, 1.9094786643981934, -0.8052487373352051, 0.4204876720905304, 1.9465214014053345, 0.5798414349555969, 1.2653166055679321, -1.1618152856826782, -0.5297948718070984, -1.2971699237823486, 0.27339980006217957, -0.02254510298371315, -0.7195107340812683, 0.6869572401046753, -0.4559738039970398, 1.3013619184494019, 0.5170600414276123, -0.5134586095809937, 0.804904043674469, 0.2428451031446457, 1.6464624404907227, -1.0156230926513672, 1.1249414682388306, -0.10989733785390854, -0.8234274983406067, 1.032983660697937, -1.0173662900924683, -0.2350786030292511, 0.22183066606521606, 0.8242237567901611, 0.21135929226875305, -0.8610659837722778, 0.4671412408351898, -1.5990924835205078, -1.659379482269287, -1.093247652053833, 0.6313910484313965, -0.8517835140228271, 0.310835063457489, -0.17765814065933228, -0.5683889985084534, -0.1522510051727295, 1.8533772230148315, 1.0443766117095947, 0.007292772643268108, -0.15055611729621887, -2.1148765087127686, 1.300148844718933, 0.21583877503871918, 0.7824920415878296, 1.4901349544525146, -0.41409432888031006, 1.823754072189331, -1.6731821298599243, -0.47185981273651123, -0.8678258061408997, -0.7608484625816345, 1.3775943517684937, 0.7309754490852356, 0.8548100590705872, -0.6487948298454285, 0.40046975016593933, -1.6767277717590332, 0.6918109059333801, -1.5246285200119019, -0.66365647315979, -1.1534360647201538, 0.42928245663642883, -1.154533863067627, 0.48049396276474, 0.5316933393478394, -1.5692598819732666, 1.3884589672088623, 0.6303561925888062, -0.6380527019500732, -0.5605087280273438, -0.17205435037612915, 0.2765529453754425, 0.17892253398895264, -0.798583447933197, 0.7442017197608948, -0.48277583718299866, -0.25020813941955566, -0.7661880254745483, 0.8411669135093689, 0.8119179606437683, 1.218264102935791, -0.7290377616882324, -0.003267272375524044, 0.5913137793540955, -1.0541728734970093, -1.220927119255066, 0.4856346547603607, -0.9688652753829956, -0.27514153718948364, -0.9241459965705872, 1.3227356672286987, 0.7045183181762695, 0.29376956820487976, 1.0115467309951782, 0.13438330590724945, -0.5161317586898804, 2.2131364345550537, -0.5643671751022339, 3.617873191833496, -0.561689555644989, 0.8195316791534424, -1.7175698280334473, 0.4788716733455658, -0.2897576093673706, -0.4190170168876648, -0.18898139894008636, -0.379800945520401, 0.6039789319038391, 1.8817816972732544, -0.6057416200637817, -1.8434537649154663, -0.7798628211021423, -0.17622743546962738, -0.9512267708778381, 0.3613428771495819, -0.601119339466095, 1.055955171585083, 0.20671816170215607, 0.4858836233615875, -0.4160318970680237, 0.12643255293369293, -0.7333404421806335, 0.6877394318580627, 0.013960043899714947, -0.9890575408935547, 1.2985477447509766, -0.3965800404548645, 0.678178608417511, 0.07038760930299759, 1.4259159564971924, -0.271127313375473, -0.19536139070987701, 0.2807382047176361, -1.0791990756988525, -0.02909313514828682, 0.556387722492218, -0.3380493223667145, -0.16120320558547974, -0.34838491678237915, 0.5824636220932007, 0.38412150740623474, 0.9410961866378784, -0.41413015127182007, 1.3109204769134521, -0.248696431517601, 0.4257144629955292, -0.9732648730278015, -1.6047354936599731, 0.6277173757553101, -1.1855247020721436, 1.3107606172561646, -1.5092052221298218, -1.3901183605194092, -1.4466924667358398, 0.14863207936286926, -0.9997431635856628, -0.8981682658195496, 0.5943639874458313, -1.1557915210723877, 0.24186548590660095, 1.6556482315063477, -1.061637282371521, -1.947229027748108, 1.3198281526565552, 0.5648294687271118, -1.8524690866470337, 0.09495213627815247, -2.135723352432251, 0.7449948191642761, 1.0130314826965332, -1.2467787265777588, 1.0227848291397095, 0.31229597330093384, 0.8287360668182373, 0.869304358959198, -0.5010467171669006, 0.7314898371696472, 0.11618383228778839, 0.6331035494804382, -0.8872820138931274, -2.1816189289093018, 2.847456455230713, -0.6376803517341614, -0.5436767935752869, -1.8643200397491455, 1.4795540571212769, -1.3606352806091309, -1.7092037200927734, -0.7368733882904053, 0.2590121924877167, -0.045005008578300476, -2.1287693977355957, 1.2291628122329712, 1.5694530010223389, 0.6827043890953064, -0.06073024868965149, 0.25294455885887146, 2.343048334121704, -0.8262125253677368, 0.5778741836547852, 0.09936985373497009, -0.4799056649208069, -0.07204844802618027, -0.1815340667963028, 2.0979247093200684, 0.9116222262382507, -0.9310845136642456, -1.0728225708007812, -0.22311893105506897, 1.2046829462051392, 0.29120758175849915, 0.7720561623573303, 1.299361228942871, -0.2587794065475464, 0.9676297903060913, 0.46530330181121826, 1.3517353534698486, 0.4701649844646454, 2.103926658630371, -0.07820764183998108, 0.25165799260139465, -0.6706101298332214, 0.6535604000091553, 0.3622497022151947, -2.1651854515075684, -1.7984718084335327, -1.1980829238891602, -0.7233563661575317, 1.7027469873428345, -0.0914827436208725, -0.1040358915925026, 0.45658737421035767, 0.32175490260124207, 1.1911507844924927, -0.1504669040441513, -0.66143399477005, 0.4813661277294159, 1.0943506956100464, 0.12862247228622437, -0.6894225478172302, 0.25737378001213074, -0.24667055904865265, -0.7888938188552856, 1.1816858053207397, -0.713484525680542, -0.5105250477790833, -1.0709584951400757, -0.24351342022418976, 0.7582367658615112, -1.1521828174591064, 0.3951890170574188, -0.1934359073638916, -0.8139802813529968, 2.6455893516540527, 0.15765222907066345, 0.34584882855415344, -1.6212198734283447, 2.583956718444824, 0.29981714487075806, 0.2788316309452057, 1.2230578660964966, -0.06615910679101944, 0.10732388496398926, -1.216593861579895, -0.526778519153595, -0.5234711766242981, 0.9531343579292297, 1.5962116718292236, -1.1392242908477783, 0.9921915531158447, 0.7484623193740845, 0.14001351594924927, -0.007008926942944527, 0.33101874589920044, -0.1395292580127716, 2.0166733264923096, -1.3429607152938843, 0.2218216210603714, 1.1999106407165527, 0.6842368245124817, 0.27845504879951477, 1.2761235237121582, 0.6665983200073242, 1.4464629888534546, 1.0942972898483276, 0.2779129147529602, 0.37819918990135193, 0.620904266834259, -1.2492504119873047, -0.1176629438996315, 0.9295563101768494, -0.2547684907913208, 0.2927302420139313, 0.12601767480373383, 0.3411708474159241, -1.0211094617843628, 0.1789572387933731, 0.9166514873504639, -1.8188682794570923, -2.2543981075286865, -0.2182319313287735, 0.052606768906116486, 0.3442743718624115, 0.9438307881355286, -0.8919368982315063, 0.3714592754840851, 0.7509414553642273, 0.5888989567756653, -0.8491208553314209, -0.6716272234916687, -0.10490830987691879, 0.42124173045158386, 0.040392324328422546, -1.0746991634368896, 1.1169662475585938, 1.4004160165786743, 2.137277126312256, -0.4014057517051697, -0.09029492735862732, 0.5702542066574097, -1.2843129634857178, -0.9498639702796936, -1.1392040252685547, 1.0507268905639648, 0.44576865434646606, -0.6033627986907959, -1.4130123853683472, -0.45610833168029785, 1.1777112483978271, 0.056299880146980286, 1.0940742492675781, -0.022508740425109863, 1.5057331323623657, -0.4706030488014221, -0.1752714365720749, 0.08053717017173767, -0.6894954442977905, -0.25457197427749634, -0.25530993938446045, -0.7735774517059326, 1.384568691253662, 0.3870811462402344, -0.9692542552947998, -0.8951451182365417, 0.022542783990502357, -0.9149006009101868, 1.3319790363311768, 0.9940914511680603, -0.3212610185146332, 1.0183223485946655, -0.736950695514679, -0.03486640006303787, 0.1251656413078308, -1.9594815969467163, 0.5400290489196777, 0.72115558385849, -0.6441042423248291, 1.5347176790237427, 0.2005930095911026, -0.462041974067688, 1.766411542892456, 1.0364612340927124, -0.19988200068473816, 2.012890338897705, 9.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": 9.0, "dtype": "float32"}, {"__nd__": 1500, "dtype": "int64"}]}}, {"a": [{"__nd__": [-1.1631582975387573, -1.8446221351623535, -3.145524501800537, -3.637233257293701, -2.6449384689331055, -1.5355254411697388, -2.3741273880004883, -2.6219611167907715, -1.667623519897461, -2.8569278717041016, -1.7939419746398926, -2.545532464981079, -1.2035541534423828, -1.2990210056304932, -1.3497811555862427, -1.1426409482955933, -2.1843996047973633, -1.2632062435150146, -1.039025902748108, -1.366032600402832, -1.2857334613800049, -1.6292117834091187, -1.6567264795303345, -1.0569356679916382, -1.1451817750930786, -3.9924747943878174, -1.729029655456543, -1.5224430561065674, -2.0161447525024414, -1.498373031616211, -2.6760964393615723, -1.3519493341445923, -1.2391058206558228, -2.0533952713012695, -1.3317797183990479, -1.4022023677825928, -3.2156448364257812, -1.8254802227020264, -2.2732009887695312, -1.4511277675628662, -1.5958306789398193, -1.3618793487548828, -2.8548197746276855, -1.6254689693450928, -2.1137001514434814, -1.6263411045074463, -2.1782655715942383, -1.7492213249206543, -2.1458804607391357, -2.666266918182373, -2.4924607276916504, -1.4496300220489502, -1.1712884902954102, -1.0428972244262695, -1.1016923189163208, -1.995511770248413, -2.181213140487671, -1.6352386474609375, -1.2256194353103638, -1.4102094173431396, -1.447646141052246, -1.0462310314178467, -1.581345796585083, -1.7898948192596436, -1.7319570779800415, -1.1466364860534668, -1.57281494140625, -1.2684814929962158, -2.5409927368164062, -1.8706685304641724, -1.040557861328125, -2.1043217182159424, -1.0977704524993896, -2.738351821899414, -1.5673534870147705, -1.8199975490570068, -1.226260781288147, -1.1418590545654297, -2.752601146697998, -1.2124437093734741, -1.7672882080078125, -1.1664836406707764, -1.002694845199585, -1.193575382232666, -2.740300416946411, -1.755839228630066, -1.1901615858078003, -2.610877513885498, -1.7727174758911133, -3.213728904724121, -2.146179437637329, -1.9119393825531006, -1.6082279682159424, -2.415931224822998, -1.1540849208831787, -2.0654749870300293, -1.5004220008850098, -1.437696099281311, -1.574915885925293, -1.958164095878601, -2.0747733116149902, -2.115297317504883, -1.7483985424041748, -2.726844072341919, -2.2796437740325928, -2.19130277633667, -2.523611068725586, -1.7030588388442993, -2.5483880043029785, -2.1927008628845215, -1.6313591003417969, -1.1461849212646484, -1.1803392171859741, -1.4011125564575195, -1.291699767112732, -1.0304490327835083, -2.3480560779571533, -1.624004602432251, -1.6844165325164795, -1.362945318222046, -1.691846489906311, -1.148721694946289, -2.265455722808838, -1.6145803928375244, -1.0199034214019775, -1.870405673980713, -2.0785250663757324, -2.0510106086730957, -2.1122632026672363, -1.4836342334747314, -2.334880828857422, -1.9024758338928223, -2.146932363510132, -1.8102033138275146, -1.875953197479248, -2.631227493286133, -1.1854956150054932, -4.027256011962891, -1.734924077987671, -1.1885557174682617, -1.2242822647094727, -1.1015000343322754, -1.9249446392059326, -1.7878048419952393, -1.4511449337005615, -1.197849154472351, -2.3833792209625244, -1.61067795753479, -1.560983419418335, -1.4859380722045898, -2.3715438842773438, -1.3523259162902832, -1.042140245437622, -2.966573715209961, -2.9087462425231934, -1.7891534566879272, -1.0859992504119873, -2.56341552734375, -1.1232848167419434, -1.8290953636169434, -1.3301162719726562, -2.0579075813293457, -1.3630661964416504, -1.596187710762024, -1.65641188621521, -3.425122022628784, -1.2551480531692505, -3.4147047996520996, -1.1697641611099243, -1.7481776475906372, -3.0419721603393555, -1.4978129863739014, -1.7099937200546265, -2.619020462036133, -1.1871658563613892, -2.3470544815063477, -1.1038572788238525, -2.5316123962402344, -1.803997278213501, -1.8155581951141357, -1.3187576532363892, -1.5529935359954834, -1.7471282482147217, -1.489633321762085, -2.4581551551818848, -1.0057419538497925, -1.3329821825027466, -2.8809313774108887, -1.5651054382324219, -1.888805866241455, -1.2234236001968384, -2.1907358169555664, -3.0103976726531982, -3.2760183811187744, -1.5761830806732178, -1.1866424083709717, -1.8314604759216309, -2.5702295303344727, -1.492814302444458, -1.7990374565124512, -1.9752883911132812, -1.8201603889465332, -1.689378261566162, -1.015544056892395, -1.1659142971038818, -1.2580904960632324, -1.4329253435134888, -1.1020071506500244, -1.4004708528518677, -1.8009693622589111, -1.2448203563690186, -1.5203973054885864, -1.5639407634735107, -1.5080910921096802, -1.3088245391845703, -1.6957114934921265, -1.3096344470977783, -1.3726550340652466, -1.0681449174880981, -1.8540568351745605, -1.4053256511688232, -1.1709033250808716, -1.9091529846191406, -1.2520167827606201, -1.2283992767333984, -1.6350665092468262, -1.7168762683868408, -1.66847825050354, -1.5582324266433716, -1.189872145652771, -2.0886569023132324, -2.91987681388855, -1.0996938943862915, -1.1344693899154663, -2.217587471008301, -2.7879552841186523, -1.830755591392517, -1.206022024154663, -1.5377345085144043, -2.0837135314941406, -2.384003162384033, -2.0351734161376953, -2.3239147663116455, -3.5554778575897217, -1.4083982706069946, -1.8524658679962158, -1.2101582288742065, -1.4261345863342285, -1.5297448635101318, -1.6536226272583008, -1.0369080305099487, -1.1327316761016846, -2.4957971572875977, -1.5504039525985718, -1.9368478059768677, -1.5876648426055908, -1.5780733823776245, -1.2036159038543701, -1.1817278861999512, -2.554147243499756, -1.9526662826538086, -1.7175402641296387, -1.4007606506347656, -1.3376448154449463, -2.355468273162842, -1.494567632675171, -1.0428205728530884, -1.8819429874420166, -2.4406490325927734, -1.7092434167861938, -1.5418733358383179, -1.4618191719055176, -2.90065336227417, -1.1410919427871704, -1.5263333320617676, -1.7269055843353271, -1.6202865839004517, -1.4432708024978638, -1.146788477897644, -1.2101584672927856, -1.5384913682937622, -1.0973221063613892, -2.3125476837158203, -1.0307188034057617, -1.2532336711883545, -2.265007972717285, -1.0854718685150146, -1.712368130683899, -1.7198748588562012, -2.227259635925293, -2.1241507530212402, -1.964642882347107, -2.0494861602783203, -1.1766960620880127, -1.881856083869934, -2.1914777755737305, -2.391425371170044, -1.555433750152588, -2.2183985710144043, -1.0027923583984375, -2.5219883918762207, -1.8089613914489746, -1.3986808061599731, -2.0731003284454346, -1.7425498962402344, -1.806961178779602, -1.5472874641418457, -1.073305606842041, -1.4018080234527588, -2.4235050678253174, -2.5097103118896484, -2.331050395965576, -1.2563058137893677, -1.1158665418624878, -2.293917179107666, -1.243664264678955, -1.6269984245300293, -1.9014822244644165, -1.6828205585479736, -1.2113806009292603, -1.0819320678710938, -1.8785409927368164, -1.3256590366363525, -2.8104076385498047, -1.5372207164764404, -1.8479681015014648, -2.7387614250183105, -1.9950168132781982, -2.3451569080352783, -1.5170801877975464, -1.6570487022399902, -1.174196481704712, -2.669926643371582, -1.9145394563674927, -1.4825093746185303, -1.4835771322250366, -1.6467783451080322, -2.5073020458221436, -2.231135845184326, -1.038949728012085, -1.6524791717529297, -1.115713357925415, -1.6388046741485596, -2.1065313816070557, -1.0745575428009033, -2.503034830093384, -2.4034886360168457, -3.528357982635498, -1.2186793088912964, -1.7359967231750488, -2.1100900173187256, -2.1382508277893066, -1.406076431274414, -2.5385303497314453, -2.0144925117492676, -1.0574201345443726, -2.5629992485046387, -1.722532033920288, -1.8224666118621826, -2.4294557571411133, -1.0531258583068848, -1.5107433795928955, -1.1984074115753174, -1.0490400791168213, -1.1106840372085571, -1.0778390169143677, -1.3504630327224731, -1.4211405515670776, -2.240482807159424, -3.080173969268799, -1.0463323593139648, -1.7602394819259644, -1.166717767715454, -1.2316668033599854, -1.738248586654663, -2.399656057357788, -1.2441424131393433, -1.9564194679260254, -1.3501777648925781, -1.734273910522461, -1.8801674842834473, -2.074516773223877, -1.071756362915039, -1.5626503229141235, -1.9936256408691406, -2.163853645324707, -1.4815068244934082, -1.7788150310516357, -2.5401525497436523, -2.1958539485931396, -1.0036098957061768, -1.2775193452835083, -1.2221275568008423, -1.6136157512664795, -2.1511030197143555, -1.3809471130371094, -1.476417064666748, -1.8394609689712524, -3.338149309158325, -1.9313840866088867, -1.2832223176956177, -3.310587167739868, -2.0691771507263184, -2.0491671562194824, -1.2477226257324219, -1.203925371170044, -1.8702722787857056, -2.350886821746826, -1.1327731609344482, -2.7644033432006836, -1.082228183746338, -2.2058544158935547, -1.8524081707000732, -1.2709743976593018, -2.0364608764648438, -1.2316406965255737, -2.782499074935913, -1.738345980644226, -1.6450932025909424, -2.66947078704834, -1.3490424156188965, -1.4720767736434937, -1.1043378114700317, -1.4624714851379395, -1.1194376945495605, -3.095903158187866, -2.9872093200683594, -1.4683107137680054, -1.1122186183929443, -2.4502623081207275, -1.184514045715332, -1.9878344535827637, -1.52586829662323, -3.1292724609375, -1.9072712659835815, -2.0933456420898438, -2.231783866882324, -2.999368667602539, -1.6008870601654053, -2.584876298904419, -1.3646972179412842, -1.309730052947998, -1.1097272634506226, -2.746692657470703, -1.8369088172912598, -1.9085218906402588, -2.107677698135376, -2.015024185180664, -1.873511552810669, -1.9536858797073364, -2.5221285820007324, -2.209245443344116, -1.191096544265747, -1.4772754907608032, -1.0241531133651733, -1.445375680923462, -2.3756957054138184, -1.6707720756530762, -2.474956512451172, -1.5343096256256104, -2.4384632110595703, -2.0029819011688232, -1.252190113067627, -2.765709400177002, -1.0489692687988281, -1.1192489862442017, -2.5868372917175293, -2.2044124603271484, -1.5147416591644287, -1.7687819004058838, -1.0548582077026367, -2.2456612586975098, -1.0165002346038818, -1.3699101209640503, -1.0706162452697754, -1.6232044696807861, -2.216815233230591, -1.356659173965454, -2.073784828186035, -1.7343038320541382, -2.3542518615722656, -2.066121816635132, -1.2749608755111694, -1.5098029375076294, -2.7844438552856445, -1.0663495063781738, -1.252305030822754, -2.2099967002868652, -1.1573336124420166, -2.2833192348480225, -1.690248727798462, -3.084921360015869, -2.0482277870178223, -1.4428672790527344, -2.3480749130249023, -1.4918675422668457, -2.4137277603149414, -1.0216939449310303, -3.0036094188690186, -1.4303354024887085, -1.7727954387664795, -2.1742331981658936, -1.1102794408798218, -2.234287738800049, -2.6832754611968994, -2.8002071380615234, -1.7314825057983398, -2.2323169708251953, -1.619781255722046, -2.2166800498962402, -1.0160382986068726, -1.8806564807891846, -1.887545108795166, -2.1319820880889893, -1.1705907583236694, -1.8797552585601807, -1.700284481048584, -2.5127034187316895, -3.096468448638916, -1.7654056549072266, -2.6771535873413086, -1.8195013999938965, -1.480898380279541, -1.5999610424041748, -2.668064594268799, -1.6157522201538086, -1.0239337682724, -1.0509706735610962, -1.3341429233551025, -1.2609753608703613, -2.34210467338562, -1.3878858089447021, -1.2995140552520752, -1.575547456741333, -2.9811410903930664, -1.6145212650299072, -1.260528326034546, -1.0799070596694946, -1.3617839813232422, -1.5870182514190674, -1.1317627429962158, -1.8764897584915161, -1.425520420074463, -1.5321483612060547, -1.0027730464935303, -1.5408167839050293, -2.127263069152832, -1.1726547479629517, -1.3873963356018066, -1.902667760848999, -2.1372017860412598, -1.100175142288208, -2.3637595176696777, -2.438852310180664, -1.7569129467010498, -1.4352571964263916, -2.3971381187438965, -1.3441314697265625, -1.6483099460601807, -1.343944787979126, -1.5193192958831787, -1.7133324146270752, -1.449349045753479, -2.8323779106140137, -1.3649399280548096, -1.6199082136154175, -2.5124990940093994, -3.2475028038024902, -2.108752489089966, -1.1302073001861572, -1.3772046566009521, -1.368686556816101, -2.336527109146118, -2.8621835708618164, -1.305262565612793, -1.5225443840026855, -1.377794861793518, -1.771727204322815, -1.7128245830535889, -2.079235792160034, -1.043890118598938, -2.071640968322754, -1.6879627704620361, -1.7241122722625732, -3.850414752960205, -2.4695751667022705, -2.7098090648651123, -1.9679112434387207, -2.5565900802612305, -1.4639310836791992, -1.39952552318573, -1.6675329208374023, -2.1601552963256836, -2.4184823036193848, -1.1628334522247314, -2.091623067855835, -2.7886624336242676, -1.1067733764648438, -1.916390299797058, -1.7873317003250122, -1.0747267007827759, -2.28531551361084, -3.6781833171844482, -2.47719669342041, -1.6620218753814697, -1.409505844116211, -1.6539597511291504, -2.159663438796997, -2.544344425201416, -2.131195068359375, -1.0399149656295776, -2.91973614692688, -1.281145453453064, -1.6180477142333984, -1.244532823562622, -2.177851676940918, -1.1518635749816895, -1.5231578350067139, -2.387746572494507, -1.6487038135528564, -1.2596125602722168, -2.5252466201782227, -1.0587661266326904, -1.8414210081100464, -1.4895751476287842, -1.4009647369384766, -2.1994872093200684, -2.0295307636260986, -1.3684251308441162, -4.145540237426758, -2.3236398696899414, -1.4241929054260254, -1.849395751953125, -2.0540871620178223, -3.063054084777832, -2.8966662883758545, -1.1687777042388916, -2.179762840270996, -2.2875568866729736, -2.801218032836914, -1.8445205688476562, -1.2494679689407349, -1.3043508529663086, -1.5901086330413818, -1.149742841720581, -3.137575149536133, -1.8588180541992188, -2.0109763145446777, -1.3615076541900635, -3.5238735675811768, -1.169500708580017, -1.2836110591888428, -1.327160358428955, -1.0729666948318481, -1.5317718982696533, -2.507030963897705, -1.53800630569458, -1.5408658981323242, -1.0480281114578247, -1.9607408046722412, -2.8663182258605957, -1.0459480285644531, -1.5949289798736572, -1.3604152202606201, -1.1948087215423584, -1.3916913270950317, -2.9480812549591064, -1.2147260904312134, -1.2528327703475952, -1.455546498298645, -1.4219027757644653, -1.6047587394714355, -1.8868207931518555, -1.3418506383895874, -1.2208784818649292, -3.301022529602051, -3.3975977897644043, -1.4680156707763672, -2.254528045654297, -1.212552785873413, -1.6398634910583496, -1.2557564973831177, -1.0231926441192627, -2.4549598693847656, -1.756821870803833, -1.2457003593444824, -1.1919012069702148, -1.666973352432251, -2.748140335083008, -1.263383388519287, -1.9387791156768799, -1.846954107284546, -1.9916934967041016, -2.748623847961426, -1.4175761938095093, -1.8613938093185425, -1.898261308670044, -2.109004497528076, -1.0376147031784058, -1.2476109266281128, -1.6115686893463135, -2.0511550903320312, -1.6712470054626465, -1.3757007122039795, -2.6713767051696777, -1.5875210762023926, -1.2668007612228394, -1.957311749458313, -1.3410618305206299, -1.4809746742248535, -1.5416975021362305, -2.1007518768310547, -1.8063528537750244, -2.156069278717041, -1.6894251108169556, -1.3746919631958008, -1.3561683893203735, -2.3956775665283203, -2.04801607131958, -1.0032106637954712, -1.4736138582229614, -1.4699068069458008, -2.6332435607910156, -1.7383520603179932, -2.313713312149048, -2.0161595344543457, -1.2032219171524048, -1.4360947608947754, -2.734429121017456, -1.6191565990447998, -2.016906976699829, -1.5701050758361816, -2.1714401245117188, -2.597764015197754, -4.322911739349365, -1.2652901411056519, -2.3811817169189453, -1.739303708076477, -1.1381009817123413, -1.1056241989135742, -2.2062504291534424, -1.832594633102417, -1.7310869693756104, -2.4828147888183594, -1.0392062664031982, -1.149219274520874, -2.2796742916107178, -1.7633013725280762, -1.5845062732696533, -3.0154526233673096, -1.8725343942642212, -1.6902490854263306, -2.562277317047119, -1.3324450254440308, -3.886570930480957, -1.4659807682037354, -2.03012752532959, -1.3529460430145264, -1.0272789001464844, -1.895572543144226, -1.3960299491882324, -1.9216349124908447, -1.0504395961761475, -1.5901072025299072, -1.1250606775283813, -2.0352494716644287, -3.601038694381714, -1.3738272190093994, -1.673570990562439, -1.6876741647720337, -1.570871353149414, -2.1159491539001465, -1.6957894563674927, -1.2688015699386597, -1.3940284252166748, -1.2230929136276245, -1.1336257457733154, -2.8357160091400146, -3.823967218399048, -2.3093581199645996, -1.2629313468933105, -1.6916086673736572, -1.2696092128753662, -3.0249197483062744, -3.3034257888793945, -1.0940748453140259, -2.0508828163146973, -1.5993876457214355, -1.5955342054367065, -1.7421592473983765, -2.038386344909668, -1.1750943660736084, -2.394587993621826, -1.63818359375, -2.297130823135376, -2.3334245681762695, -1.1080653667449951, -1.3293002843856812, -1.9296338558197021, -1.63142991065979, -1.2400341033935547, -1.536224126815796, -2.917811870574951, -2.6733808517456055, -2.5882532596588135, -1.203189492225647, -1.013725757598877, -1.3177199363708496, -1.052762746810913, -1.7282079458236694, -1.8835692405700684, -3.013465404510498, -2.7078137397766113, -2.3315255641937256, -1.0420570373535156, -1.9836997985839844, -1.2982079982757568, -1.5545103549957275, -3.0603721141815186, -3.5943477153778076, -1.4028462171554565, -1.9205328226089478, -1.136194109916687, -1.995168685913086, -1.3902873992919922, -1.7530808448791504, -1.9898908138275146, -2.383565902709961, -1.4675276279449463, -2.4899442195892334, -3.659061908721924, -1.2627582550048828, -1.2846494913101196, -1.8619515895843506, -1.0395539999008179, -1.674769401550293, -1.8798162937164307, -1.0269935131072998, -1.7249982357025146, -1.6033308506011963, -1.8997807502746582, -1.4268234968185425, -1.6406899690628052, -1.819490671157837, -2.1101160049438477, -1.975502610206604, -1.1489346027374268, -1.942676067352295, -2.621896743774414, -1.563279390335083, -2.2911505699157715, -1.4539424180984497, -1.7929160594940186, -1.414302110671997, -1.85103178024292, -1.452828049659729, -1.5979093313217163, -1.5922260284423828, -1.0386781692504883, -2.550281047821045, -4.291444778442383, -2.3752474784851074, -1.1060519218444824, -1.5639567375183105, -1.0900384187698364, -1.2666888236999512, -2.5727152824401855, -1.7062933444976807, -2.0920982360839844, -1.2358590364456177, -1.7572951316833496, -2.242155075073242, -1.288589596748352, -2.319702625274658, -1.2043306827545166, -1.2412197589874268, -2.118537425994873, -1.8666378259658813, -1.047204613685608, -3.5863289833068848, -1.9176738262176514, -1.8477537631988525, -2.1351771354675293, -2.537755250930786, -2.9747447967529297, -2.3561882972717285, -1.5567820072174072, -2.043490171432495, -1.2808425426483154, -1.061234474182129, -2.777494430541992, -1.0516563653945923, -2.0120742321014404, -1.6880345344543457, -1.640157699584961, -1.1777610778808594, -2.8164639472961426, -2.1340131759643555, -2.8184256553649902, -1.0399646759033203, -2.0121772289276123, -1.4802285432815552, -2.0625667572021484, -1.2883782386779785, -2.6583259105682373, -1.3267649412155151, -1.8711659908294678, -1.4646236896514893, -2.5211048126220703, -1.4993361234664917, -2.3981826305389404, -1.9227325916290283, -1.5334393978118896, -1.5878775119781494, -1.2049033641815186, -1.7534337043762207, -1.9388680458068848, -1.6050450801849365, -1.945614218711853, -1.5631697177886963, -2.645498275756836, -1.2051489353179932, -2.471951961517334, -2.42690372467041, -1.4094152450561523, -2.816854953765869, -2.2870595455169678, -1.461353063583374, -1.1865750551223755, -1.0217081308364868, -2.0664901733398438, -1.5308946371078491, -1.2338824272155762, -2.348529100418091, -2.5655856132507324, -2.165917158126831, -1.1633274555206299, -1.9634904861450195, -1.4311678409576416, -1.2974538803100586, -2.1793293952941895, -1.0422002077102661, -1.03047776222229, -1.8594343662261963, -2.429015874862671, -1.2372993230819702, -2.286342144012451, -2.507617950439453, -1.0616261959075928, -1.6960034370422363, -1.3282407522201538, -1.76460862159729, -1.01316499710083, -2.545280694961548, -2.7162375450134277, -1.1453351974487305, -2.855992078781128, -1.515812635421753, -2.3153672218322754, -2.8328099250793457, -2.6252074241638184, -1.7448396682739258, -1.9027695655822754, -1.4903310537338257, -1.501509189605713, -2.2273366451263428, -1.7116049528121948, -2.0007853507995605, -1.3057446479797363, -1.0749316215515137, -1.731142282485962, -1.8453395366668701, -2.1768245697021484, -2.1911392211914062, -1.1280804872512817, -1.0781570672988892, -2.337299346923828, -1.268779993057251, -2.4268455505371094, -1.1211689710617065, -1.4813519716262817, -1.9844189882278442, -1.3534536361694336, -1.3123583793640137, -1.5319623947143555, -2.5371241569519043, -2.490431070327759, -1.3727370500564575, -1.436295509338379, -2.959954261779785, -1.29793381690979, -3.123568296432495, -1.7590460777282715, -1.611409306526184, -1.4439330101013184, -1.2191295623779297, -1.371255874633789, -1.496105670928955, -1.6401209831237793, -1.7345709800720215, -1.0067148208618164, -1.2444779872894287, -1.4715458154678345, -1.6928231716156006, -1.5739905834197998, -1.718332052230835, -2.9687020778656006, -1.0693178176879883, -2.3685569763183594, -1.2976707220077515, -2.232556104660034, -1.380491852760315, -2.1712100505828857, -1.7002294063568115, -2.149435043334961, -1.3868495225906372, -1.407283902168274, -1.738842248916626, -1.7063555717468262, -1.247436285018921, -2.1743836402893066, -1.1923412084579468, -2.443523406982422, -1.1016006469726562, -1.8434884548187256, -1.814450979232788, -1.4042598009109497, -1.6635806560516357, -4.3789215087890625, -1.145736813545227, -1.8491551876068115, -1.151078224182129, -1.3036712408065796, -2.0531725883483887, -1.5738189220428467, -1.3505560159683228, -1.0706264972686768, -1.6606895923614502, -1.2294843196868896, -2.8885765075683594, -2.1936044692993164, -2.1593642234802246, -1.192527413368225, -3.1718590259552, -1.5921354293823242, -1.4438154697418213, -1.8258113861083984, -1.1355957984924316, -1.716277837753296, -2.181497573852539, -1.3401142358779907, -2.2093818187713623, -1.2901782989501953, -1.2101505994796753, -2.4184188842773438, -1.1167833805084229, -1.8506126403808594, -1.6002583503723145, -2.0999765396118164, -1.5869717597961426, -1.3931859731674194, -1.2886793613433838, -2.0715365409851074, -1.3699986934661865, -3.024440050125122, -2.831205368041992, -2.058903694152832, -1.3774211406707764, -1.3664116859436035, -2.0258989334106445, -2.5754380226135254, -2.0964951515197754, -2.4916458129882812, -1.1681280136108398, -1.4051775932312012, -1.2801990509033203, -1.6904079914093018, -1.1753214597702026, -1.052037239074707, -1.1407041549682617, -2.0254392623901367, -2.2070724964141846, -1.8647416830062866, -1.657752275466919, -1.1175884008407593, -1.8674798011779785, -2.6912994384765625, -1.725443720817566, -1.3437790870666504, -1.0913758277893066, -2.5470380783081055, -1.4002628326416016, -1.374598503112793, -2.237008571624756, -1.8088295459747314, -2.7659173011779785, -1.1451137065887451, -2.650574207305908, -1.3472318649291992, -1.845710039138794, -1.3842929601669312, -1.4105479717254639, -1.3801567554473877, -2.023564338684082, -1.727158546447754, -1.0769535303115845, -2.002440929412842, -1.8791556358337402, -1.653049349784851, -1.3223673105239868, -2.268139600753784, -1.6782476902008057, -2.1187918186187744, -1.3510384559631348, -2.00239896774292, -1.5084879398345947, -1.6251983642578125, -2.045330047607422, -1.4390640258789062, -1.3462563753128052, -1.5012738704681396, -2.341635227203369, -1.6553781032562256, -2.0576467514038086, -1.1108607053756714, -1.1755279302597046, -1.1958938837051392, -1.1493690013885498, -1.7173376083374023, -1.0794028043746948, -1.4356671571731567, -3.2081351280212402, -1.5530357360839844, -2.0079004764556885, -1.723050594329834, -2.4468014240264893, -1.640876293182373, -1.3012230396270752, -1.3992112874984741, -1.8828389644622803, -1.0345276594161987, -1.4848856925964355, -2.4023852348327637, -2.1793415546417236, -1.3424447774887085, -1.1749284267425537, -1.7147023677825928, -2.393075942993164, -1.6788952350616455, -2.591519832611084, -1.0580159425735474, -1.6698434352874756, -1.2346726655960083, -1.515052318572998, -1.3002865314483643, -1.5351618528366089, -1.564100742340088, -1.4743629693984985, -1.6414897441864014, -1.8867824077606201, -1.362458348274231, -1.172734260559082, -1.5069684982299805, -2.1192946434020996, -1.396043300628662, -1.5669193267822266, -1.487028956413269, -1.6878890991210938, -2.118532180786133, -1.326542854309082, -1.193795919418335, -1.3280552625656128, -1.8876479864120483, -1.5138442516326904, -1.2261872291564941, -1.1424942016601562, -1.3684853315353394, -1.2557992935180664, -1.1891932487487793, -2.096630811691284, -1.5365216732025146, -1.6785888671875, -2.397385597229004, -1.0088380575180054, -2.684304714202881, -1.788190245628357, -2.778268814086914, -2.475403308868408, -1.000153660774231, -1.8489902019500732, -1.9076523780822754, -2.5672144889831543, -1.5376945734024048, -2.552943468093872, -1.0228530168533325, -2.4203996658325195, -1.302204966545105, -1.2995574474334717, -1.6446865797042847, -1.4986897706985474, -1.1889889240264893, -1.0401368141174316, -1.7478257417678833, -2.692574977874756, -1.7482922077178955, -1.5166951417922974, -2.017979383468628, -1.568099021911621, -2.4672439098358154, -2.1301345825195312, -1.7793810367584229, -1.89467453956604, -3.2418124675750732, -1.2363793849945068, -2.0178613662719727, -1.3144875764846802, -2.440626859664917, -1.2928967475891113, -2.7543530464172363, -2.3302345275878906, -1.090376615524292, -1.0680654048919678, -3.0514824390411377, -1.6248342990875244, -2.5183115005493164, -1.2091286182403564, -2.0054430961608887, -1.8869681358337402, -1.704134464263916, -1.756988763809204, -1.3869144916534424, -2.124810218811035, -1.2429804801940918, -1.1849735975265503, -2.4880270957946777, -1.2389174699783325, -1.7445138692855835, -1.3515576124191284, -1.5734846591949463, -1.662755012512207, -1.6155731678009033, -2.96995210647583, -2.203634262084961, -1.5983068943023682, -1.2103197574615479, -1.430537223815918, -2.3360495567321777, -2.152176856994629, -1.7623164653778076, -1.9586882591247559, -1.7557692527770996, -2.375950813293457, -1.5664572715759277, -2.271742820739746, -1.8639922142028809, -1.1564793586730957, -1.5297579765319824, -1.5893536806106567, -2.171471118927002, -1.1252827644348145, -1.2644647359848022, -1.4321552515029907, -1.634612798690796, -1.2087130546569824, -1.8222097158432007, -2.961379289627075, -1.926682949066162, -2.8759469985961914, -1.827415108680725, -1.1618574857711792, -3.983997344970703, -2.8715081214904785, -2.3115386962890625, -1.3667025566101074, -1.347815752029419, -1.2124515771865845, -1.037176489830017, -2.5408363342285156, -1.4826381206512451, -2.3374361991882324, -2.526191234588623, -1.3942257165908813, -1.9958164691925049, -2.1328697204589844, -1.3099145889282227, -1.4230306148529053, -1.8122097253799438, -1.561645746231079, -1.533856749534607, -1.4988139867782593, -1.2817885875701904, -2.5033276081085205, -2.449709892272949, -1.5222761631011963, -1.3984360694885254, -1.2189877033233643, -1.948194146156311, -1.1133298873901367, -3.147308826446533, -2.4966659545898438, -2.053905963897705, -2.270582675933838, -2.4410111904144287, -2.384855270385742, -1.792802333831787, -1.3174636363983154, -1.191346287727356, -1.1038397550582886, -2.0856785774230957, -1.1784671545028687, -2.3369154930114746, -2.1569571495056152, -2.4132168292999268, -2.1220123767852783, -1.8156604766845703, -2.3840298652648926, -2.634650230407715, -2.2785348892211914, -1.2992738485336304, -2.219393253326416, -1.4688584804534912, -2.0727243423461914, -1.2530932426452637, -3.198368549346924, -1.323622465133667, -1.6102523803710938, -1.1080955266952515, -2.487886667251587, -2.239398956298828, -1.9960432052612305, -2.743905782699585, -2.8355507850646973, -1.9718737602233887, -1.5403797626495361, -1.6767733097076416, -2.2064335346221924, -1.020331859588623, -1.179598331451416, -1.4858195781707764, -2.01959228515625, -1.9487366676330566, -1.9765560626983643, -1.4455132484436035, -1.3157521486282349, -1.0915175676345825, -1.8494197130203247, -1.5591135025024414, -1.6865787506103516, -1.585153341293335, -1.6768181324005127, -1.9509122371673584, -1.1652250289916992, -1.8902205228805542, -1.0788151025772095, -1.2140679359436035, -1.5894049406051636, -1.0197383165359497, -1.281038522720337, -1.5302597284317017, -1.5592671632766724, -1.4687496423721313, -1.3958717584609985, -1.7276796102523804, -1.97483491897583, -1.5254302024841309, -1.1834943294525146, -3.063734531402588, -2.473351001739502, -1.1254056692123413, -1.2696019411087036, -2.403412103652954, -1.688276767730713, -1.530303716659546, -1.1624407768249512, -2.3643338680267334, -1.4635154008865356, -1.2677592039108276, -1.1411601305007935, -1.8808921575546265, -2.2462711334228516, -1.5877771377563477, -1.2404056787490845, -1.7406731843948364, -1.0189515352249146, -1.35215163230896, -1.4534214735031128, -1.211042881011963, -1.2007694244384766, -1.5475819110870361, -1.701138973236084, -2.1423070430755615, -1.320324420928955, -1.1609742641448975, -1.1176071166992188, -1.17599618434906, -1.7447352409362793, -1.4329031705856323, -2.168797016143799, -1.7183761596679688, -3.072873592376709, -1.688622236251831, -1.4389642477035522, -1.8703631162643433, -1.4473271369934082, -2.6735012531280518, -1.656507968902588, -1.2243574857711792, -2.029618978500366, -2.0877132415771484, -1.8249919414520264, -1.3364276885986328, -2.6654696464538574, -2.240391492843628, -1.1250689029693604, -2.060260772705078, -1.4593708515167236, -1.7498977184295654, -1.6183390617370605, -1.4510588645935059, -2.252656936645508, -1.123660922050476, -1.13874089717865, -1.310981035232544, -1.8943641185760498, -3.3056626319885254, -1.4895199537277222, -1.3385438919067383, -1.1866739988327026, -2.548031806945801, -2.7848598957061768, -2.34443998336792, -1.1431348323822021, -3.2917816638946533, -1.7502379417419434, -1.1361980438232422, -1.2392388582229614, -1.1596426963806152, -1.2690725326538086, -1.8256328105926514, -3.310859203338623, -1.177565097808838, -1.2256264686584473, -1.3830499649047852, -1.3602657318115234, -1.3981752395629883, -1.3271417617797852, -1.8912945985794067, -2.688478469848633, -1.4465821981430054, -1.990344762802124, -1.5352356433868408, -1.7627241611480713, -1.0784366130828857, -1.0049599409103394, -1.3061387538909912, -1.5048482418060303, -1.3688093423843384, -1.6070692539215088, -1.2824863195419312, -2.9864068031311035, -1.7566697597503662, -2.302854061126709, -2.6225576400756836, -1.6167421340942383, -1.7625641822814941, -1.083307147026062, -1.1632431745529175, -1.735951542854309, -1.021988868713379, -1.096409559249878, -1.346228003501892, -1.4379826784133911, -1.6999619007110596, -1.444539189338684, -1.016872763633728, -1.9945180416107178, -1.717822790145874, -1.9638688564300537, -1.5957772731781006, -1.9702646732330322, -1.104931354522705, -1.1957805156707764, -1.5073989629745483, -2.2562179565429688, -1.0360335111618042, -2.542031764984131, -2.129375457763672, -1.5886125564575195, -2.064863681793213, -1.5718235969543457, -1.8960354328155518, -2.0577402114868164, -1.4559422731399536, -1.5647900104522705, -1.020453929901123, -1.6529884338378906, -2.1082711219787598, -2.031665563583374, -1.456629753112793, -1.2219661474227905], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": -1.000153660774231, "dtype": "float32"}, {"__nd__": 1188, "dtype": "int64"}]}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0, 93.0, 94.0, 95.0, 96.0, 97.0, 98.0, 99.0, 100.0, 101.0, 102.0, 103.0, 104.0, 105.0, 106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0, 114.0, 115.0, 116.0, 117.0, 118.0, 119.0, 120.0, 121.0, 122.0, 123.0, 124.0, 125.0, 126.0, 127.0, 128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 137.0, 138.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 157.0, 158.0, 159.0, 160.0, 161.0, 162.0, 163.0, 164.0, 165.0, 166.0, 167.0, 168.0, 169.0, 170.0, 171.0, 172.0, 173.0, 174.0, 175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 190.0, 191.0, 192.0, 193.0, 194.0, 195.0, 196.0, 197.0, 198.0, 199.0, 200.0, 201.0, 202.0, 203.0, 204.0, 205.0, 206.0, 207.0, 208.0, 209.0, 210.0, 211.0, 212.0, 213.0, 214.0, 215.0, 216.0, 217.0, 218.0, 219.0, 220.0, 221.0, 222.0, 223.0, 224.0, 225.0, 226.0, 227.0, 228.0, 229.0, 230.0, 231.0, 232.0, 233.0, 234.0, 235.0, 236.0, 237.0, 238.0, 239.0, 240.0, 241.0, 242.0, 243.0, 244.0, 245.0, 246.0, 247.0, 248.0, 249.0, 250.0, 251.0, 252.0, 253.0, 254.0, 255.0, 256.0, 257.0, 258.0, 259.0, 260.0, 261.0, 262.0, 263.0, 264.0, 265.0, 266.0, 267.0, 268.0, 269.0, 270.0, 271.0, 272.0, 273.0, 274.0, 275.0, 276.0, 277.0, 278.0, 279.0, 280.0, 281.0, 282.0, 283.0, 284.0, 285.0, 286.0, 287.0, 288.0, 289.0, 290.0, 291.0, 292.0, 293.0, 294.0, 295.0, 296.0, 297.0, 298.0, 299.0, 300.0, 301.0, 302.0, 303.0, 304.0, 305.0, 306.0, 307.0, 308.0, 309.0, 310.0, 311.0, 312.0, 313.0, 314.0, 315.0, 316.0, 317.0, 318.0, 319.0, 320.0, 321.0, 322.0, 323.0, 324.0, 325.0, 326.0, 327.0, 328.0, 329.0, 330.0, 331.0, 332.0, 333.0, 334.0, 335.0, 336.0, 337.0, 338.0, 339.0, 340.0, 341.0, 342.0, 343.0, 344.0, 345.0, 346.0, 347.0, 348.0, 349.0, 350.0, 351.0, 352.0, 353.0, 354.0, 355.0, 356.0, 357.0, 358.0, 359.0, 360.0, 361.0, 362.0, 363.0, 364.0, 365.0, 366.0, 367.0, 368.0, 369.0, 370.0, 371.0, 372.0, 373.0, 374.0, 375.0, 376.0, 377.0, 378.0, 379.0, 380.0, 381.0, 382.0, 383.0, 384.0, 385.0, 386.0, 387.0, 388.0, 389.0, 390.0, 391.0, 392.0, 393.0, 394.0, 395.0, 396.0, 397.0, 398.0, 399.0, 400.0, 401.0, 402.0, 403.0, 404.0, 405.0, 406.0, 407.0, 408.0, 409.0, 410.0, 411.0, 412.0, 413.0, 414.0, 415.0, 416.0, 417.0, 418.0, 419.0, 420.0, 421.0, 422.0, 423.0, 424.0, 425.0, 426.0, 427.0, 428.0, 429.0, 430.0, 431.0, 432.0, 433.0, 434.0, 435.0, 436.0, 437.0, 438.0, 439.0, 440.0, 441.0, 442.0, 443.0, 444.0, 445.0, 446.0, 447.0, 448.0, 449.0, 450.0, 451.0, 452.0, 453.0, 454.0, 455.0, 456.0, 457.0, 458.0, 459.0, 460.0, 461.0, 462.0, 463.0, 464.0, 465.0, 466.0, 467.0, 468.0, 469.0, 470.0, 471.0, 472.0, 473.0, 474.0, 475.0, 476.0, 477.0, 478.0, 479.0, 480.0, 481.0, 482.0, 483.0, 484.0, 485.0, 486.0, 487.0, 488.0, 489.0, 490.0, 491.0, 492.0, 493.0, 494.0, 495.0, 496.0, 497.0, 498.0, 499.0, 500.0, 501.0, 502.0, 503.0, 504.0, 505.0, 506.0, 507.0, 508.0, 509.0, 510.0, 511.0, 512.0, 513.0, 514.0, 515.0, 516.0, 517.0, 518.0, 519.0, 520.0, 521.0, 522.0, 523.0, 524.0, 525.0, 526.0, 527.0, 528.0, 529.0, 530.0, 531.0, 532.0, 533.0, 534.0, 535.0, 536.0, 537.0, 538.0, 539.0, 540.0, 541.0, 542.0, 543.0, 544.0, 545.0, 546.0, 547.0, 548.0, 549.0, 550.0, 551.0, 552.0, 553.0, 554.0, 555.0, 556.0, 557.0, 558.0, 559.0, 560.0, 561.0, 562.0, 563.0, 564.0, 565.0, 566.0, 567.0, 568.0, 569.0, 570.0, 571.0, 572.0, 573.0, 574.0, 575.0, 576.0, 577.0, 578.0, 579.0, 580.0, 581.0, 582.0, 583.0, 584.0, 585.0, 586.0, 587.0, 588.0, 589.0, 590.0, 591.0, 592.0, 593.0, 594.0, 595.0, 596.0, 597.0, 598.0, 599.0, 600.0, 601.0, 602.0, 603.0, 604.0, 605.0, 606.0, 607.0, 608.0, 609.0, 610.0, 611.0, 612.0, 613.0, 614.0, 615.0, 616.0, 617.0, 618.0, 619.0, 620.0, 621.0, 622.0, 623.0, 624.0, 625.0, 626.0, 627.0, 628.0, 629.0, 630.0, 631.0, 632.0, 633.0, 634.0, 635.0, 636.0, 637.0, 638.0, 639.0, 640.0, 641.0, 642.0, 643.0, 644.0, 645.0, 646.0, 647.0, 648.0, 649.0, 650.0, 651.0, 652.0, 653.0, 654.0, 655.0, 656.0, 657.0, 658.0, 659.0, 660.0, 661.0, 662.0, 663.0, 664.0, 665.0, 666.0, 667.0, 668.0, 669.0, 670.0, 671.0, 672.0, 673.0, 674.0, 675.0, 676.0, 677.0, 678.0, 679.0, 680.0, 681.0, 682.0, 683.0, 684.0, 685.0, 686.0, 687.0, 688.0, 689.0, 690.0, 691.0, 692.0, 693.0, 694.0, 695.0, 696.0, 697.0, 698.0, 699.0, 700.0, 701.0, 702.0, 703.0, 704.0, 705.0, 706.0, 707.0, 708.0, 709.0, 710.0, 711.0, 712.0, 713.0, 714.0, 715.0, 716.0, 717.0, 718.0, 719.0, 720.0, 721.0, 722.0, 723.0, 724.0, 725.0, 726.0, 727.0, 728.0, 729.0, 730.0, 731.0, 732.0, 733.0, 734.0, 735.0, 736.0, 737.0, 738.0, 739.0, 740.0, 741.0, 742.0, 743.0, 744.0, 745.0, 746.0, 747.0, 748.0, 749.0, 750.0, 751.0, 752.0, 753.0, 754.0, 755.0, 756.0, 757.0, 758.0, 759.0, 760.0, 761.0, 762.0, 763.0, 764.0, 765.0, 766.0, 767.0, 768.0, 769.0, 770.0, 771.0, 772.0, 773.0, 774.0, 775.0, 776.0, 777.0, 778.0, 779.0, 780.0, 781.0, 782.0, 783.0, 784.0, 785.0, 786.0, 787.0, 788.0, 789.0, 790.0, 791.0, 792.0, 793.0, 794.0, 795.0, 796.0, 797.0, 798.0, 799.0, 800.0, 801.0, 802.0, 803.0, 804.0, 805.0, 806.0, 807.0, 808.0, 809.0, 810.0, 811.0, 812.0, 813.0, 814.0, 815.0, 816.0, 817.0, 818.0, 819.0, 820.0, 821.0, 822.0, 823.0, 824.0, 825.0, 826.0, 827.0, 828.0, 829.0, 830.0, 831.0, 832.0, 833.0, 834.0, 835.0, 836.0, 837.0, 838.0, 839.0, 840.0, 841.0, 842.0, 843.0, 844.0, 845.0, 846.0, 847.0, 848.0, 849.0, 850.0, 851.0, 852.0, 853.0, 854.0, 855.0, 856.0, 857.0, 858.0, 859.0, 860.0, 861.0, 862.0, 863.0, 864.0, 865.0, 866.0, 867.0, 868.0, 869.0, 870.0, 871.0, 872.0, 873.0, 874.0, 875.0, 876.0, 877.0, 878.0, 879.0, 880.0, 881.0, 882.0, 883.0, 884.0, 885.0, 886.0, 887.0, 888.0, 889.0, 890.0, 891.0, 892.0, 893.0, 894.0, 895.0, 896.0, 897.0, 898.0, 899.0, 900.0, 901.0, 902.0, 903.0, 904.0, 905.0, 906.0, 907.0, 908.0, 909.0, 910.0, 911.0, 912.0, 913.0, 914.0, 915.0, 916.0, 917.0, 918.0, 919.0, 920.0, 921.0, 922.0, 923.0, 924.0, 925.0, 926.0, 927.0, 928.0, 929.0, 930.0, 931.0, 932.0, 933.0, 934.0, 935.0, 936.0, 937.0, 938.0, 939.0, 940.0, 941.0, 942.0, 943.0, 944.0, 945.0, 946.0, 947.0, 948.0, 949.0, 950.0, 951.0, 952.0, 953.0, 954.0, 955.0, 956.0, 957.0, 958.0, 959.0, 960.0, 961.0, 962.0, 963.0, 964.0, 965.0, 966.0, 967.0, 968.0, 969.0, 970.0, 971.0, 972.0, 973.0, 974.0, 975.0, 976.0, 977.0, 978.0, 979.0, 980.0, 981.0, 982.0, 983.0, 984.0, 985.0, 986.0, 987.0, 988.0, 989.0, 990.0, 991.0, 992.0, 993.0, 994.0, 995.0, 996.0, 997.0, 998.0, 999.0, 1000.0, 1001.0, 1002.0, 1003.0, 1004.0, 1005.0, 1006.0, 1007.0, 1008.0, 1009.0, 1010.0, 1011.0, 1012.0, 1013.0, 1014.0, 1015.0, 1016.0, 1017.0, 1018.0, 1019.0, 1020.0, 1021.0, 1022.0, 1023.0, 1024.0, 1025.0, 1026.0, 1027.0, 1028.0, 1029.0, 1030.0, 1031.0, 1032.0, 1033.0, 1034.0, 1035.0, 1036.0, 1037.0, 1038.0, 1039.0, 1040.0, 1041.0, 1042.0, 1043.0, 1044.0, 1045.0, 1046.0, 1047.0, 1048.0, 1049.0, 1050.0, 1051.0, 1052.0, 1053.0, 1054.0, 1055.0, 1056.0, 1057.0, 1058.0, 1059.0, 1060.0, 1061.0, 1062.0, 1063.0, 1064.0, 1065.0, 1066.0, 1067.0, 1068.0, 1069.0, 1070.0, 1071.0, 1072.0, 1073.0, 1074.0, 1075.0, 1076.0, 1077.0, 1078.0, 1079.0, 1080.0, 1081.0, 1082.0, 1083.0, 1084.0, 1085.0, 1086.0, 1087.0, 1088.0, 1089.0, 1090.0, 1091.0, 1092.0, 1093.0, 1094.0, 1095.0, 1096.0, 1097.0, 1098.0, 1099.0, 1100.0, 1101.0, 1102.0, 1103.0, 1104.0, 1105.0, 1106.0, 1107.0, 1108.0, 1109.0, 1110.0, 1111.0, 1112.0, 1113.0, 1114.0, 1115.0, 1116.0, 1117.0, 1118.0, 1119.0, 1120.0, 1121.0, 1122.0, 1123.0, 1124.0, 1125.0, 1126.0, 1127.0, 1128.0, 1129.0, 1130.0, 1131.0, 1132.0, 1133.0, 1134.0, 1135.0, 1136.0, 1137.0, 1138.0, 1139.0, 1140.0, 1141.0, 1142.0, 1143.0, 1144.0, 1145.0, 1146.0, 1147.0, 1148.0, 1149.0, 1150.0, 1151.0, 1152.0, 1153.0, 1154.0, 1155.0, 1156.0, 1157.0, 1158.0, 1159.0, 1160.0, 1161.0, 1162.0, 1163.0, 1164.0, 1165.0, 1166.0, 1167.0, 1168.0, 1169.0, 1170.0, 1171.0, 1172.0, 1173.0, 1174.0, 1175.0, 1176.0, 1177.0, 1178.0, 1179.0, 1180.0, 1181.0, 1182.0, 1183.0, 1184.0, 1185.0, 1186.0, 1187.0, 1188.0, 1189.0, 1190.0, 1191.0, 1192.0, 1193.0, 1194.0, 1195.0, 1196.0, 1197.0, 1198.0, 1199.0, 1200.0, 1201.0, 1202.0, 1203.0, 1204.0, 1205.0, 1206.0, 1207.0, 1208.0, 1209.0, 1210.0, 1211.0, 1212.0, 1213.0, 1214.0, 1215.0, 1216.0, 1217.0, 1218.0, 1219.0, 1220.0, 1221.0, 1222.0, 1223.0, 1224.0, 1225.0, 1226.0, 1227.0, 1228.0, 1229.0, 1230.0, 1231.0, 1232.0, 1233.0, 1234.0, 1235.0, 1236.0, 1237.0, 1238.0, 1239.0, 1240.0, 1241.0, 1242.0, 1243.0, 1244.0, 1245.0, 1246.0, 1247.0, 1248.0, 1249.0, 1250.0, 1251.0, 1252.0, 1253.0, 1254.0, 1255.0, 1256.0, 1257.0, 1258.0, 1259.0, 1260.0, 1261.0, 1262.0, 1263.0, 1264.0, 1265.0, 1266.0, 1267.0, 1268.0, 1269.0, 1270.0, 1271.0, 1272.0, 1273.0, 1274.0, 1275.0, 1276.0, 1277.0, 1278.0, 1279.0, 1280.0, 1281.0, 1282.0, 1283.0, 1284.0, 1285.0, 1286.0, 1287.0, 1288.0, 1289.0, 1290.0, 1291.0, 1292.0, 1293.0, 1294.0, 1295.0, 1296.0, 1297.0, 1298.0, 1299.0, 1300.0, 1301.0, 1302.0, 1303.0, 1304.0, 1305.0, 1306.0, 1307.0, 1308.0, 1309.0, 1310.0, 1311.0, 1312.0, 1313.0, 1314.0, 1315.0, 1316.0, 1317.0, 1318.0, 1319.0, 1320.0, 1321.0, 1322.0, 1323.0, 1324.0, 1325.0, 1326.0, 1327.0, 1328.0, 1329.0, 1330.0, 1331.0, 1332.0, 1333.0, 1334.0, 1335.0, 1336.0, 1337.0, 1338.0, 1339.0, 1340.0, 1341.0, 1342.0, 1343.0, 1344.0, 1345.0, 1346.0, 1347.0, 1348.0, 1349.0, 1350.0, 1351.0, 1352.0, 1353.0, 1354.0, 1355.0, 1356.0, 1357.0, 1358.0, 1359.0, 1360.0, 1361.0, 1362.0, 1363.0, 1364.0, 1365.0, 1366.0, 1367.0, 1368.0, 1369.0, 1370.0, 1371.0, 1372.0, 1373.0, 1374.0, 1375.0, 1376.0, 1377.0, 1378.0, 1379.0, 1380.0, 1381.0, 1382.0, 1383.0, 1384.0, 1385.0, 1386.0, 1387.0, 1388.0, 1389.0, 1390.0, 1391.0, 1392.0, 1393.0, 1394.0, 1395.0, 1396.0, 1397.0, 1398.0, 1399.0, 1400.0, 1401.0, 1402.0, 1403.0, 1404.0, 1405.0, 1406.0, 1407.0, 1408.0, 1409.0, 1410.0, 1411.0, 1412.0, 1413.0, 1414.0, 1415.0, 1416.0, 1417.0, 1418.0, 1419.0, 1420.0, 1421.0, 1422.0, 1423.0, 1424.0, 1425.0, 1426.0, 1427.0, 1428.0, 1429.0, 1430.0, 1431.0, 1432.0, 1433.0, 1434.0, 1435.0, 1436.0, 1437.0, 1438.0, 1439.0, 1440.0, 1441.0, 1442.0, 1443.0, 1444.0, 1445.0, 1446.0, 1447.0, 1448.0, 1449.0, 1450.0, 1451.0, 1452.0, 1453.0, 1454.0, 1455.0, 1456.0, 1457.0, 1458.0, 1459.0, 1460.0, 1461.0, 1462.0, 1463.0, 1464.0, 1465.0, 1466.0, 1467.0, 1468.0, 1469.0, 1470.0, 1471.0, 1472.0, 1473.0, 1474.0, 1475.0, 1476.0, 1477.0, 1478.0, 1479.0, 1480.0, 1481.0, 1482.0, 1483.0, 1484.0, 1485.0, 1486.0, 1487.0, 1488.0, 1489.0, 1490.0, 1491.0, 1492.0, 1493.0, 1494.0, 1495.0, 1496.0, 1497.0, 1498.0, 1499.0, 1500.0, 1501.0, 1502.0, 1503.0, 1504.0, 1505.0, 1506.0, 1507.0, 1508.0, 1509.0, 1510.0, 1511.0, 1512.0, 1513.0, 1514.0, 1515.0, 1516.0, 1517.0, 1518.0, 1519.0, 1520.0, 1521.0, 1522.0, 1523.0, 1524.0, 1525.0, 1526.0, 1527.0, 1528.0, 1529.0, 1530.0, 1531.0, 1532.0, 1533.0, 1534.0, 1535.0, 1536.0, 1537.0, 1538.0, 1539.0, 1540.0, 1541.0, 1542.0, 1543.0, 1544.0, 1545.0, 1546.0, 1547.0, 1548.0, 1549.0, 1550.0, 1551.0, 1552.0, 1553.0, 1554.0, 1555.0, 1556.0, 1557.0, 1558.0, 1559.0, 1560.0, 1561.0, 1562.0, 1563.0, 1564.0, 1565.0, 1566.0, 1567.0, 1568.0, 1569.0, 1570.0, 1571.0, 1572.0, 1573.0, 1574.0, 1575.0, 1576.0, 1577.0, 1578.0, 1579.0, 1580.0, 1581.0, 1582.0, 1583.0, 1584.0, 1585.0, 1586.0, 1587.0, 1588.0, 1589.0, 1590.0, 1591.0, 1592.0, 1593.0, 1594.0, 1595.0, 1596.0, 1597.0, 1598.0, 1599.0, 1600.0, 1601.0, 1602.0, 1603.0, 1604.0, 1605.0, 1606.0, 1607.0, 1608.0, 1609.0, 1610.0, 1611.0, 1612.0, 1613.0, 1614.0, 1615.0, 1616.0, 1617.0, 1618.0, 1619.0, 1620.0, 1621.0, 1622.0, 1623.0, 1624.0, 1625.0, 1626.0, 1627.0, 1628.0, 1629.0, 1630.0, 1631.0, 1632.0, 1633.0, 1634.0, 1635.0, 1636.0, 1637.0, 1638.0, 1639.0, 1640.0, 1641.0, 1642.0, 1643.0, 1644.0, 1645.0, 1646.0, 1647.0, 1648.0, 1649.0, 1650.0, 1651.0, 1652.0, 1653.0, 1654.0, 1655.0, 1656.0, 1657.0, 1658.0, 1659.0, 1660.0, 1661.0, 1662.0, 1663.0, 1664.0, 1665.0, 1666.0, 1667.0, 1668.0, 1669.0, 1670.0, 1671.0, 1672.0, 1673.0, 1674.0, 1675.0, 1676.0, 1677.0, 1678.0, 1679.0, 1680.0, 1681.0, 1682.0, 1683.0, 1684.0, 1685.0, 1686.0, 1687.0, 1688.0, 1689.0, 1690.0, 1691.0, 1692.0, 1693.0, 1694.0, 1695.0, 1696.0, 1697.0, 1698.0, 1699.0, 1700.0, 1701.0, 1702.0, 1703.0, 1704.0, 1705.0, 1706.0, 1707.0, 1708.0, 1709.0, 1710.0, 1711.0, 1712.0, 1713.0, 1714.0, 1715.0, 1716.0, 1717.0, 1718.0, 1719.0, 1720.0, 1721.0, 1722.0, 1723.0, 1724.0, 1725.0, 1726.0, 1727.0, 1728.0, 1729.0, 1730.0, 1731.0, 1732.0, 1733.0, 1734.0, 1735.0, 1736.0, 1737.0, 1738.0, 1739.0, 1740.0, 1741.0, 1742.0, 1743.0, 1744.0, 1745.0, 1746.0, 1747.0, 1748.0, 1749.0, 1750.0, 1751.0, 1752.0, 1753.0, 1754.0, 1755.0, 1756.0, 1757.0, 1758.0, 1759.0, 1760.0, 1761.0, 1762.0, 1763.0, 1764.0, 1765.0, 1766.0, 1767.0, 1768.0, 1769.0, 1770.0, 1771.0, 1772.0, 1773.0, 1774.0, 1775.0, 1776.0, 1777.0, 1778.0, 1779.0, 1780.0, 1781.0, 1782.0, 1783.0, 1784.0, 1785.0, 1786.0, 1787.0, 1788.0, 1789.0, 1790.0, 1791.0, 1792.0, 1793.0, 1794.0, 1795.0, 1796.0, 1797.0, 1798.0, 1799.0, 1800.0, 1801.0, 1802.0, 1803.0, 1804.0, 1805.0, 1806.0, 1807.0, 1808.0, 1809.0, 1810.0, 1811.0, 1812.0, 1813.0, 1814.0, 1815.0, 1816.0, 1817.0, 1818.0, 1819.0, 1820.0, 1821.0, 1822.0, 1823.0, 1824.0, 1825.0, 1826.0, 1827.0, 1828.0, 1829.0, 1830.0, 1831.0, 1832.0, 1833.0, 1834.0, 1835.0, 1836.0, 1837.0, 1838.0, 1839.0, 1840.0, 1841.0, 1842.0, 1843.0, 1844.0, 1845.0, 1846.0, 1847.0, 1848.0, 1849.0, 1850.0, 1851.0, 1852.0, 1853.0, 1854.0, 1855.0, 1856.0, 1857.0, 1858.0, 1859.0, 1860.0, 1861.0, 1862.0, 1863.0, 1864.0, 1865.0, 1866.0, 1867.0, 1868.0, 1869.0, 1870.0, 1871.0, 1872.0, 1873.0, 1874.0, 1875.0, 1876.0, 1877.0, 1878.0, 1879.0, 1880.0, 1881.0, 1882.0, 1883.0, 1884.0, 1885.0, 1886.0, 1887.0, 1888.0, 1889.0, 1890.0, 1891.0, 1892.0, 1893.0, 1894.0, 1895.0, 1896.0, 1897.0, 1898.0, 1899.0, 1900.0, 1901.0, 1902.0, 1903.0, 1904.0, 1905.0, 1906.0, 1907.0, 1908.0, 1909.0, 1910.0, 1911.0, 1912.0, 1913.0, 1914.0, 1915.0, 1916.0, 1917.0, 1918.0, 1919.0, 1920.0, 1921.0, 1922.0, 1923.0, 1924.0, 1925.0, 1926.0, 1927.0, 1928.0, 1929.0, 1930.0, 1931.0, 1932.0, 1933.0, 1934.0, 1935.0, 1936.0, 1937.0, 1938.0, 1939.0, 1940.0, 1941.0, 1942.0, 1943.0, 1944.0, 1945.0, 1946.0, 1947.0, 1948.0, 1949.0, 1950.0, 1951.0, 1952.0, 1953.0, 1954.0, 1955.0, 1956.0, 1957.0, 1958.0, 1959.0, 1960.0, 1961.0, 1962.0, 1963.0, 1964.0, 1965.0, 1966.0, 1967.0, 1968.0, 1969.0, 1970.0, 1971.0, 1972.0, 1973.0, 1974.0, 1975.0, 1976.0, 1977.0, 1978.0, 1979.0, 1980.0, 1981.0, 1982.0, 1983.0, 1984.0, 1985.0, 1986.0, 1987.0, 1988.0, 1989.0, 1990.0, 1991.0, 1992.0, 1993.0, 1994.0, 1995.0, 1996.0, 1997.0, 1998.0, 1999.0, 2000.0, 2001.0, 2002.0, 2003.0, 2004.0, 2005.0, 2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, 2024.0, 2025.0, 2026.0, 2027.0, 2028.0, 2029.0, 2030.0, 2031.0, 2032.0, 2033.0, 2034.0, 2035.0, 2036.0, 2037.0, 2038.0, 2039.0, 2040.0, 2041.0, 2042.0, 2043.0, 2044.0, 2045.0, 2046.0, 2047.0, 2048.0, 2049.0, 2050.0, 2051.0, 2052.0, 2053.0, 2054.0, 2055.0, 2056.0, 2057.0, 2058.0, 2059.0, 2060.0, 2061.0, 2062.0, 2063.0, 2064.0, 2065.0, 2066.0, 2067.0, 2068.0, 2069.0, 2070.0, 2071.0, 2072.0, 2073.0, 2074.0, 2075.0, 2076.0, 2077.0, 2078.0, 2079.0, 2080.0, 2081.0, 2082.0, 2083.0, 2084.0, 2085.0, 2086.0, 2087.0, 2088.0, 2089.0, 2090.0, 2091.0, 2092.0, 2093.0, 2094.0, 2095.0, 2096.0, 2097.0, 2098.0, 2099.0, 2100.0, 2101.0, 2102.0, 2103.0, 2104.0, 2105.0, 2106.0, 2107.0, 2108.0, 2109.0, 2110.0, 2111.0, 2112.0, 2113.0, 2114.0, 2115.0, 2116.0, 2117.0, 2118.0, 2119.0, 2120.0, 2121.0, 2122.0, 2123.0, 2124.0, 2125.0, 2126.0, 2127.0, 2128.0, 2129.0, 2130.0, 2131.0, 2132.0, 2133.0, 2134.0, 2135.0, 2136.0, 2137.0, 2138.0, 2139.0, 2140.0, 2141.0, 2142.0, 2143.0, 2144.0, 2145.0, 2146.0, 2147.0, 2148.0, 2149.0, 2150.0, 2151.0, 2152.0, 2153.0, 2154.0, 2155.0, 2156.0, 2157.0, 2158.0, 2159.0, 2160.0, 2161.0, 2162.0, 2163.0, 2164.0, 2165.0, 2166.0, 2167.0, 2168.0, 2169.0, 2170.0, 2171.0, 2172.0, 2173.0, 2174.0, 2175.0, 2176.0, 2177.0, 2178.0, 2179.0, 2180.0, 2181.0, 2182.0, 2183.0, 2184.0, 2185.0, 2186.0, 2187.0, 2188.0, 2189.0, 2190.0, 2191.0, 2192.0, 2193.0, 2194.0, 2195.0, 2196.0, 2197.0, 2198.0, 2199.0, 2200.0, 2201.0, 2202.0, 2203.0, 2204.0, 2205.0, 2206.0, 2207.0, 2208.0, 2209.0, 2210.0, 2211.0, 2212.0, 2213.0, 2214.0, 2215.0, 2216.0, 2217.0, 2218.0, 2219.0, 2220.0, 2221.0, 2222.0, 2223.0, 2224.0, 2225.0, 2226.0, 2227.0, 2228.0, 2229.0, 2230.0, 2231.0, 2232.0, 2233.0, 2234.0, 2235.0, 2236.0, 2237.0, 2238.0, 2239.0, 2240.0, 2241.0, 2242.0, 2243.0, 2244.0, 2245.0, 2246.0, 2247.0, 2248.0, 2249.0, 2250.0, 2251.0, 2252.0, 2253.0, 2254.0, 2255.0, 2256.0, 2257.0, 2258.0, 2259.0, 2260.0, 2261.0, 2262.0, 2263.0, 2264.0, 2265.0, 2266.0, 2267.0, 2268.0, 2269.0, 2270.0, 2271.0, 2272.0, 2273.0, 2274.0, 2275.0, 2276.0, 2277.0, 2278.0, 2279.0, 2280.0, 2281.0, 2282.0, 2283.0, 2284.0, 2285.0, 2286.0, 2287.0, 2288.0, 2289.0, 2290.0, 2291.0, 2292.0, 2293.0, 2294.0, 2295.0, 2296.0, 2297.0, 2298.0, 2299.0, 2300.0, 2301.0, 2302.0, 2303.0, 2304.0, 2305.0, 2306.0, 2307.0, 2308.0, 2309.0, 2310.0, 2311.0, 2312.0, 2313.0, 2314.0, 2315.0, 2316.0, 2317.0, 2318.0, 2319.0, 2320.0, 2321.0, 2322.0, 2323.0, 2324.0, 2325.0, 2326.0, 2327.0, 2328.0, 2329.0, 2330.0, 2331.0, 2332.0, 2333.0, 2334.0, 2335.0, 2336.0, 2337.0, 2338.0, 2339.0, 2340.0, 2341.0, 2342.0, 2343.0, 2344.0, 2345.0, 2346.0, 2347.0, 2348.0, 2349.0, 2350.0, 2351.0, 2352.0, 2353.0, 2354.0, 2355.0, 2356.0, 2357.0, 2358.0, 2359.0, 2360.0, 2361.0, 2362.0, 2363.0, 2364.0, 2365.0, 2366.0, 2367.0, 2368.0, 2369.0, 2370.0, 2371.0, 2372.0, 2373.0, 2374.0, 2375.0, 2376.0, 2377.0, 2378.0, 2379.0, 2380.0, 2381.0, 2382.0, 2383.0, 2384.0, 2385.0, 2386.0, 2387.0, 2388.0, 2389.0, 2390.0, 2391.0, 2392.0, 2393.0, 2394.0, 2395.0, 2396.0, 2397.0, 2398.0, 2399.0, 2400.0, 2401.0, 2402.0, 2403.0, 2404.0, 2405.0, 2406.0, 2407.0, 2408.0, 2409.0, 2410.0, 2411.0, 2412.0, 2413.0, 2414.0, 2415.0, 2416.0, 2417.0, 2418.0, 2419.0, 2420.0, 2421.0, 2422.0, 2423.0, 2424.0, 2425.0, 2426.0, 2427.0, 2428.0, 2429.0, 2430.0, 2431.0, 2432.0, 2433.0, 2434.0, 2435.0, 2436.0, 2437.0, 2438.0, 2439.0, 2440.0, 2441.0, 2442.0, 2443.0, 2444.0, 2445.0, 2446.0, 2447.0, 2448.0, 2449.0, 2450.0, 2451.0, 2452.0, 2453.0, 2454.0, 2455.0, 2456.0, 2457.0, 2458.0, 2459.0, 2460.0, 2461.0, 2462.0, 2463.0, 2464.0, 2465.0, 2466.0, 2467.0, 2468.0, 2469.0, 2470.0, 2471.0, 2472.0, 2473.0, 2474.0, 2475.0, 2476.0, 2477.0, 2478.0, 2479.0, 2480.0, 2481.0, 2482.0, 2483.0, 2484.0, 2485.0, 2486.0, 2487.0, 2488.0, 2489.0, 2490.0, 2491.0, 2492.0, 2493.0, 2494.0, 2495.0, 2496.0, 2497.0, 2498.0, 2499.0, 2500.0, 2501.0, 2502.0, 2503.0, 2504.0, 2505.0, 2506.0, 2507.0, 2508.0, 2509.0, 2510.0, 2511.0, 2512.0, 2513.0, 2514.0, 2515.0, 2516.0, 2517.0, 2518.0, 2519.0, 2520.0, 2521.0, 2522.0, 2523.0, 2524.0, 2525.0, 2526.0, 2527.0, 2528.0, 2529.0, 2530.0, 2531.0, 2532.0, 2533.0, 2534.0, 2535.0, 2536.0, 2537.0, 2538.0, 2539.0, 2540.0, 2541.0, 2542.0, 2543.0, 2544.0, 2545.0, 2546.0, 2547.0, 2548.0, 2549.0, 2550.0, 2551.0, 2552.0, 2553.0, 2554.0, 2555.0, 2556.0, 2557.0, 2558.0, 2559.0, 2560.0, 2561.0, 2562.0, 2563.0, 2564.0, 2565.0, 2566.0, 2567.0, 2568.0, 2569.0, 2570.0, 2571.0, 2572.0, 2573.0, 2574.0, 2575.0, 2576.0, 2577.0, 2578.0, 2579.0, 2580.0, 2581.0, 2582.0, 2583.0, 2584.0, 2585.0, 2586.0, 2587.0, 2588.0, 2589.0, 2590.0, 2591.0, 2592.0, 2593.0, 2594.0, 2595.0, 2596.0, 2597.0, 2598.0, 2599.0, 2600.0, 2601.0, 2602.0, 2603.0, 2604.0, 2605.0, 2606.0, 2607.0, 2608.0, 2609.0, 2610.0, 2611.0, 2612.0, 2613.0, 2614.0, 2615.0, 2616.0, 2617.0, 2618.0, 2619.0, 2620.0, 2621.0, 2622.0, 2623.0, 2624.0, 2625.0, 2626.0, 2627.0, 2628.0, 2629.0, 2630.0, 2631.0, 2632.0, 2633.0, 2634.0, 2635.0, 2636.0, 2637.0, 2638.0, 2639.0, 2640.0, 2641.0, 2642.0, 2643.0, 2644.0, 2645.0, 2646.0, 2647.0, 2648.0, 2649.0, 2650.0, 2651.0, 2652.0, 2653.0, 2654.0, 2655.0, 2656.0, 2657.0, 2658.0, 2659.0, 2660.0, 2661.0, 2662.0, 2663.0, 2664.0, 2665.0, 2666.0, 2667.0, 2668.0, 2669.0, 2670.0, 2671.0, 2672.0, 2673.0, 2674.0, 2675.0, 2676.0, 2677.0, 2678.0, 2679.0, 2680.0, 2681.0, 2682.0, 2683.0, 2684.0, 2685.0, 2686.0, 2687.0, 2688.0, 2689.0, 2690.0, 2691.0, 2692.0, 2693.0, 2694.0, 2695.0, 2696.0, 2697.0, 2698.0, 2699.0, 2700.0, 2701.0, 2702.0, 2703.0, 2704.0, 2705.0, 2706.0, 2707.0, 2708.0, 2709.0, 2710.0, 2711.0, 2712.0, 2713.0, 2714.0, 2715.0, 2716.0, 2717.0, 2718.0, 2719.0, 2720.0, 2721.0, 2722.0, 2723.0, 2724.0, 2725.0, 2726.0, 2727.0, 2728.0, 2729.0, 2730.0, 2731.0, 2732.0, 2733.0, 2734.0, 2735.0, 2736.0, 2737.0, 2738.0, 2739.0, 2740.0, 2741.0, 2742.0, 2743.0, 2744.0, 2745.0, 2746.0, 2747.0, 2748.0, 2749.0, 2750.0, 2751.0, 2752.0, 2753.0, 2754.0, 2755.0, 2756.0, 2757.0, 2758.0, 2759.0, 2760.0, 2761.0, 2762.0, 2763.0, 2764.0, 2765.0, 2766.0, 2767.0, 2768.0, 2769.0, 2770.0, 2771.0, 2772.0, 2773.0, 2774.0, 2775.0, 2776.0, 2777.0, 2778.0, 2779.0, 2780.0, 2781.0, 2782.0, 2783.0, 2784.0, 2785.0, 2786.0, 2787.0, 2788.0, 2789.0, 2790.0, 2791.0, 2792.0, 2793.0, 2794.0, 2795.0, 2796.0, 2797.0, 2798.0, 2799.0, 2800.0, 2801.0, 2802.0, 2803.0, 2804.0, 2805.0, 2806.0, 2807.0, 2808.0, 2809.0, 2810.0, 2811.0, 2812.0, 2813.0, 2814.0, 2815.0, 2816.0, 2817.0, 2818.0, 2819.0, 2820.0, 2821.0, 2822.0, 2823.0, 2824.0, 2825.0, 2826.0, 2827.0, 2828.0, 2829.0, 2830.0, 2831.0, 2832.0, 2833.0, 2834.0, 2835.0, 2836.0, 2837.0, 2838.0, 2839.0, 2840.0, 2841.0, 2842.0, 2843.0, 2844.0, 2845.0, 2846.0, 2847.0, 2848.0, 2849.0, 2850.0, 2851.0, 2852.0, 2853.0, 2854.0, 2855.0, 2856.0, 2857.0, 2858.0, 2859.0, 2860.0, 2861.0, 2862.0, 2863.0, 2864.0, 2865.0, 2866.0, 2867.0, 2868.0, 2869.0, 2870.0, 2871.0, 2872.0, 2873.0, 2874.0, 2875.0, 2876.0, 2877.0, 2878.0, 2879.0, 2880.0, 2881.0, 2882.0, 2883.0, 2884.0, 2885.0, 2886.0, 2887.0, 2888.0, 2889.0, 2890.0, 2891.0, 2892.0, 2893.0, 2894.0, 2895.0, 2896.0, 2897.0, 2898.0, 2899.0, 2900.0, 2901.0, 2902.0, 2903.0, 2904.0, 2905.0, 2906.0, 2907.0, 2908.0, 2909.0, 2910.0, 2911.0, 2912.0, 2913.0, 2914.0, 2915.0, 2916.0, 2917.0, 2918.0, 2919.0, 2920.0, 2921.0, 2922.0, 2923.0, 2924.0, 2925.0, 2926.0, 2927.0, 2928.0, 2929.0, 2930.0, 2931.0, 2932.0, 2933.0, 2934.0, 2935.0, 2936.0, 2937.0, 2938.0, 2939.0, 2940.0, 2941.0, 2942.0, 2943.0, 2944.0, 2945.0, 2946.0, 2947.0, 2948.0, 2949.0, 2950.0, 2951.0, 2952.0, 2953.0, 2954.0, 2955.0, 2956.0, 2957.0, 2958.0, 2959.0, 2960.0, 2961.0, 2962.0, 2963.0, 2964.0, 2965.0, 2966.0, 2967.0, 2968.0, 2969.0, 2970.0, 2971.0, 2972.0, 2973.0, 2974.0, 2975.0, 2976.0, 2977.0, 2978.0, 2979.0, 2980.0, 2981.0, 2982.0, 2983.0, 2984.0, 2985.0, 2986.0, 2987.0, 2988.0, 2989.0, 2990.0, 2991.0, 2992.0, 2993.0, 2994.0, 2995.0, 2996.0, 2997.0, 2998.0, 2999.0, 3000.0, 3001.0, 3002.0, 3003.0, 3004.0, 3005.0, 3006.0, 3007.0, 3008.0, 3009.0, 3010.0, 3011.0, 3012.0, 3013.0, 3014.0, 3015.0, 3016.0, 3017.0, 3018.0, 3019.0, 3020.0, 3021.0, 3022.0, 3023.0, 3024.0, 3025.0, 3026.0, 3027.0, 3028.0, 3029.0, 3030.0, 3031.0, 3032.0, 3033.0, 3034.0, 3035.0, 3036.0, 3037.0, 3038.0, 3039.0, 3040.0, 3041.0, 3042.0, 3043.0, 3044.0, 3045.0, 3046.0, 3047.0, 3048.0, 3049.0, 3050.0, 3051.0, 3052.0, 3053.0, 3054.0, 3055.0, 3056.0, 3057.0, 3058.0, 3059.0, 3060.0, 3061.0, 3062.0, 3063.0, 3064.0, 3065.0, 3066.0, 3067.0, 3068.0, 3069.0, 3070.0, 3071.0, 3072.0, 3073.0, 3074.0, 3075.0, 3076.0, 3077.0, 3078.0, 3079.0, 3080.0, 3081.0, 3082.0, 3083.0, 3084.0, 3085.0, 3086.0, 3087.0, 3088.0, 3089.0, 3090.0, 3091.0, 3092.0, 3093.0, 3094.0, 3095.0, 3096.0, 3097.0, 3098.0, 3099.0, 3100.0, 3101.0, 3102.0, 3103.0, 3104.0, 3105.0, 3106.0, 3107.0, 3108.0, 3109.0, 3110.0, 3111.0, 3112.0, 3113.0, 3114.0, 3115.0, 3116.0, 3117.0, 3118.0, 3119.0, 3120.0, 3121.0, 3122.0, 3123.0, 3124.0, 3125.0, 3126.0, 3127.0, 3128.0, 3129.0, 3130.0, 3131.0, 3132.0, 3133.0, 3134.0, 3135.0, 3136.0, 3137.0, 3138.0, 3139.0, 3140.0, 3141.0, 3142.0, 3143.0, 3144.0, 3145.0, 3146.0, 3147.0, 3148.0, 3149.0, 3150.0, 3151.0, 3152.0, 3153.0, 3154.0, 3155.0, 3156.0, 3157.0, 3158.0, 3159.0, 3160.0, 3161.0, 3162.0, 3163.0, 3164.0, 3165.0, 3166.0, 3167.0, 3168.0, 3169.0, 3170.0, 3171.0, 3172.0, 3173.0, 3174.0, 3175.0, 3176.0, 3177.0, 3178.0, 3179.0, 3180.0, 3181.0, 3182.0, 3183.0, 3184.0, 3185.0, 3186.0, 3187.0, 3188.0, 3189.0, 3190.0, 3191.0, 3192.0, 3193.0, 3194.0, 3195.0, 3196.0, 3197.0, 3198.0, 3199.0, 3200.0, 3201.0, 3202.0, 3203.0, 3204.0, 3205.0, 3206.0, 3207.0, 3208.0, 3209.0, 3210.0, 3211.0, 3212.0, 3213.0, 3214.0, 3215.0, 3216.0, 3217.0, 3218.0, 3219.0, 3220.0, 3221.0, 3222.0, 3223.0, 3224.0, 3225.0, 3226.0, 3227.0, 3228.0, 3229.0, 3230.0, 3231.0, 3232.0, 3233.0, 3234.0, 3235.0, 3236.0, 3237.0, 3238.0, 3239.0, 3240.0, 3241.0, 3242.0, 3243.0, 3244.0, 3245.0, 3246.0, 3247.0, 3248.0, 3249.0, 3250.0, 3251.0, 3252.0, 3253.0, 3254.0, 3255.0, 3256.0, 3257.0, 3258.0, 3259.0, 3260.0, 3261.0, 3262.0, 3263.0, 3264.0, 3265.0, 3266.0, 3267.0, 3268.0, 3269.0, 3270.0, 3271.0, 3272.0, 3273.0, 3274.0, 3275.0, 3276.0, 3277.0, 3278.0, 3279.0, 3280.0, 3281.0, 3282.0, 3283.0, 3284.0, 3285.0, 3286.0, 3287.0, 3288.0, 3289.0, 3290.0, 3291.0, 3292.0, 3293.0, 3294.0, 3295.0, 3296.0, 3297.0, 3298.0, 3299.0, 3300.0, 3301.0, 3302.0, 3303.0, 3304.0, 3305.0, 3306.0, 3307.0, 3308.0, 3309.0, 3310.0, 3311.0, 3312.0, 3313.0, 3314.0, 3315.0, 3316.0, 3317.0, 3318.0, 3319.0, 3320.0, 3321.0, 3322.0, 3323.0, 3324.0, 3325.0, 3326.0, 3327.0, 3328.0, 3329.0, 3330.0, 3331.0, 3332.0, 3333.0, 3334.0, 3335.0, 3336.0, 3337.0, 3338.0, 3339.0, 3340.0, 3341.0, 3342.0, 3343.0, 3344.0, 3345.0, 3346.0, 3347.0, 3348.0, 3349.0, 3350.0, 3351.0, 3352.0, 3353.0, 3354.0, 3355.0, 3356.0, 3357.0, 3358.0, 3359.0, 3360.0, 3361.0, 3362.0, 3363.0, 3364.0, 3365.0, 3366.0, 3367.0, 3368.0, 3369.0, 3370.0, 3371.0, 3372.0, 3373.0, 3374.0, 3375.0, 3376.0, 3377.0, 3378.0, 3379.0, 3380.0, 3381.0, 3382.0, 3383.0, 3384.0, 3385.0, 3386.0, 3387.0, 3388.0, 3389.0, 3390.0, 3391.0, 3392.0, 3393.0, 3394.0, 3395.0, 3396.0, 3397.0, 3398.0, 3399.0, 3400.0, 3401.0, 3402.0, 3403.0, 3404.0, 3405.0, 3406.0, 3407.0, 3408.0, 3409.0, 3410.0, 3411.0, 3412.0, 3413.0, 3414.0, 3415.0, 3416.0, 3417.0, 3418.0, 3419.0, 3420.0, 3421.0, 3422.0, 3423.0, 3424.0, 3425.0, 3426.0, 3427.0, 3428.0, 3429.0, 3430.0, 3431.0, 3432.0, 3433.0, 3434.0, 3435.0, 3436.0, 3437.0, 3438.0, 3439.0, 3440.0, 3441.0, 3442.0, 3443.0, 3444.0, 3445.0, 3446.0, 3447.0, 3448.0, 3449.0, 3450.0, 3451.0, 3452.0, 3453.0, 3454.0, 3455.0, 3456.0, 3457.0, 3458.0, 3459.0, 3460.0, 3461.0, 3462.0, 3463.0, 3464.0, 3465.0, 3466.0, 3467.0, 3468.0, 3469.0, 3470.0, 3471.0, 3472.0, 3473.0, 3474.0, 3475.0, 3476.0, 3477.0, 3478.0, 3479.0, 3480.0, 3481.0, 3482.0, 3483.0, 3484.0, 3485.0, 3486.0, 3487.0, 3488.0, 3489.0, 3490.0, 3491.0, 3492.0, 3493.0, 3494.0, 3495.0, 3496.0, 3497.0, 3498.0, 3499.0, 3500.0, 3501.0, 3502.0, 3503.0, 3504.0, 3505.0, 3506.0, 3507.0, 3508.0, 3509.0, 3510.0, 3511.0, 3512.0, 3513.0, 3514.0, 3515.0, 3516.0, 3517.0, 3518.0, 3519.0, 3520.0, 3521.0, 3522.0, 3523.0, 3524.0, 3525.0, 3526.0, 3527.0, 3528.0, 3529.0, 3530.0, 3531.0, 3532.0, 3533.0, 3534.0, 3535.0, 3536.0, 3537.0, 3538.0, 3539.0, 3540.0, 3541.0, 3542.0, 3543.0, 3544.0, 3545.0, 3546.0, 3547.0, 3548.0, 3549.0, 3550.0, 3551.0, 3552.0, 3553.0, 3554.0, 3555.0, 3556.0, 3557.0, 3558.0, 3559.0, 3560.0, 3561.0, 3562.0, 3563.0, 3564.0, 3565.0, 3566.0, 3567.0, 3568.0, 3569.0, 3570.0, 3571.0, 3572.0, 3573.0, 3574.0, 3575.0, 3576.0, 3577.0, 3578.0, 3579.0, 3580.0, 3581.0, 3582.0, 3583.0, 3584.0, 3585.0, 3586.0, 3587.0, 3588.0, 3589.0, 3590.0, 3591.0, 3592.0, 3593.0, 3594.0, 3595.0, 3596.0, 3597.0, 3598.0, 3599.0, 3600.0, 3601.0, 3602.0, 3603.0, 3604.0, 3605.0, 3606.0, 3607.0, 3608.0, 3609.0, 3610.0, 3611.0, 3612.0, 3613.0, 3614.0, 3615.0, 3616.0, 3617.0, 3618.0, 3619.0, 3620.0, 3621.0, 3622.0, 3623.0, 3624.0, 3625.0, 3626.0, 3627.0, 3628.0, 3629.0, 3630.0, 3631.0, 3632.0, 3633.0, 3634.0, 3635.0, 3636.0, 3637.0, 3638.0, 3639.0, 3640.0, 3641.0, 3642.0, 3643.0, 3644.0, 3645.0, 3646.0, 3647.0, 3648.0, 3649.0, 3650.0, 3651.0, 3652.0, 3653.0, 3654.0, 3655.0, 3656.0, 3657.0, 3658.0, 3659.0, 3660.0, 3661.0, 3662.0, 3663.0, 3664.0, 3665.0, 3666.0, 3667.0, 3668.0, 3669.0, 3670.0, 3671.0, 3672.0, 3673.0, 3674.0, 3675.0, 3676.0, 3677.0, 3678.0, 3679.0, 3680.0, 3681.0, 3682.0, 3683.0, 3684.0, 3685.0, 3686.0, 3687.0, 3688.0, 3689.0, 3690.0, 3691.0, 3692.0, 3693.0, 3694.0, 3695.0, 3696.0, 3697.0, 3698.0, 3699.0, 3700.0, 3701.0, 3702.0, 3703.0, 3704.0, 3705.0, 3706.0, 3707.0, 3708.0, 3709.0, 3710.0, 3711.0, 3712.0, 3713.0, 3714.0, 3715.0, 3716.0, 3717.0, 3718.0, 3719.0, 3720.0, 3721.0, 3722.0, 3723.0, 3724.0, 3725.0, 3726.0, 3727.0, 3728.0, 3729.0, 3730.0, 3731.0, 3732.0, 3733.0, 3734.0, 3735.0, 3736.0, 3737.0, 3738.0, 3739.0, 3740.0, 3741.0, 3742.0, 3743.0, 3744.0, 3745.0, 3746.0, 3747.0, 3748.0, 3749.0, 3750.0, 3751.0, 3752.0, 3753.0, 3754.0, 3755.0, 3756.0, 3757.0, 3758.0, 3759.0, 3760.0, 3761.0, 3762.0, 3763.0, 3764.0, 3765.0, 3766.0, 3767.0, 3768.0, 3769.0, 3770.0, 3771.0, 3772.0, 3773.0, 3774.0, 3775.0, 3776.0, 3777.0, 3778.0, 3779.0, 3780.0, 3781.0, 3782.0, 3783.0, 3784.0, 3785.0, 3786.0, 3787.0, 3788.0, 3789.0, 3790.0, 3791.0, 3792.0, 3793.0, 3794.0, 3795.0, 3796.0, 3797.0, 3798.0, 3799.0, 3800.0, 3801.0, 3802.0, 3803.0, 3804.0, 3805.0, 3806.0, 3807.0, 3808.0, 3809.0, 3810.0, 3811.0, 3812.0, 3813.0, 3814.0, 3815.0, 3816.0, 3817.0, 3818.0, 3819.0, 3820.0, 3821.0, 3822.0, 3823.0, 3824.0, 3825.0, 3826.0, 3827.0, 3828.0, 3829.0, 3830.0, 3831.0, 3832.0, 3833.0, 3834.0, 3835.0, 3836.0, 3837.0, 3838.0, 3839.0, 3840.0, 3841.0, 3842.0, 3843.0, 3844.0, 3845.0, 3846.0, 3847.0, 3848.0, 3849.0, 3850.0, 3851.0, 3852.0, 3853.0, 3854.0, 3855.0, 3856.0, 3857.0, 3858.0, 3859.0, 3860.0, 3861.0, 3862.0, 3863.0, 3864.0, 3865.0, 3866.0, 3867.0, 3868.0, 3869.0, 3870.0, 3871.0, 3872.0, 3873.0, 3874.0, 3875.0, 3876.0, 3877.0, 3878.0, 3879.0, 3880.0, 3881.0, 3882.0, 3883.0, 3884.0, 3885.0, 3886.0, 3887.0, 3888.0, 3889.0, 3890.0, 3891.0, 3892.0, 3893.0, 3894.0, 3895.0, 3896.0, 3897.0, 3898.0, 3899.0, 3900.0, 3901.0, 3902.0, 3903.0, 3904.0, 3905.0, 3906.0, 3907.0, 3908.0, 3909.0, 3910.0, 3911.0, 3912.0, 3913.0, 3914.0, 3915.0, 3916.0, 3917.0, 3918.0, 3919.0, 3920.0, 3921.0, 3922.0, 3923.0, 3924.0, 3925.0, 3926.0, 3927.0, 3928.0, 3929.0, 3930.0, 3931.0, 3932.0, 3933.0, 3934.0, 3935.0, 3936.0, 3937.0, 3938.0, 3939.0, 3940.0, 3941.0, 3942.0, 3943.0, 3944.0, 3945.0, 3946.0, 3947.0, 3948.0, 3949.0, 3950.0, 3951.0, 3952.0, 3953.0, 3954.0, 3955.0, 3956.0, 3957.0, 3958.0, 3959.0, 3960.0, 3961.0, 3962.0, 3963.0, 3964.0, 3965.0, 3966.0, 3967.0, 3968.0, 3969.0, 3970.0, 3971.0, 3972.0, 3973.0, 3974.0, 3975.0, 3976.0, 3977.0, 3978.0, 3979.0, 3980.0, 3981.0, 3982.0, 3983.0, 3984.0, 3985.0, 3986.0, 3987.0, 3988.0, 3989.0, 3990.0, 3991.0, 3992.0, 3993.0, 3994.0, 3995.0, 3996.0, 3997.0, 3998.0, 3999.0, 4000.0, 4001.0, 4002.0, 4003.0, 4004.0, 4005.0, 4006.0, 4007.0, 4008.0, 4009.0, 4010.0, 4011.0, 4012.0, 4013.0, 4014.0, 4015.0, 4016.0, 4017.0, 4018.0, 4019.0, 4020.0, 4021.0, 4022.0, 4023.0, 4024.0, 4025.0, 4026.0, 4027.0, 4028.0, 4029.0, 4030.0, 4031.0, 4032.0, 4033.0, 4034.0, 4035.0, 4036.0, 4037.0, 4038.0, 4039.0, 4040.0, 4041.0, 4042.0, 4043.0, 4044.0, 4045.0, 4046.0, 4047.0, 4048.0, 4049.0, 4050.0, 4051.0, 4052.0, 4053.0, 4054.0, 4055.0, 4056.0, 4057.0, 4058.0, 4059.0, 4060.0, 4061.0, 4062.0, 4063.0, 4064.0, 4065.0, 4066.0, 4067.0, 4068.0, 4069.0, 4070.0, 4071.0, 4072.0, 4073.0, 4074.0, 4075.0, 4076.0, 4077.0, 4078.0, 4079.0, 4080.0, 4081.0, 4082.0, 4083.0, 4084.0, 4085.0, 4086.0, 4087.0, 4088.0, 4089.0, 4090.0, 4091.0, 4092.0, 4093.0, 4094.0, 4095.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": 4095.0, "dtype": "float32"}, {"__nd__": 4095, "dtype": "int64"}]}}, {"a": [{"__nd__": [4095.0, 4094.0, 4093.0, 4092.0, 4091.0, 4090.0, 4089.0, 4088.0, 4087.0, 4086.0, 4085.0, 4084.0, 4083.0, 4082.0, 4081.0, 4080.0, 4079.0, 4078.0, 4077.0, 4076.0, 4075.0, 4074.0, 4073.0, 4072.0, 4071.0, 4070.0, 4069.0, 4068.0, 4067.0, 4066.0, 4065.0, 4064.0, 4063.0, 4062.0, 4061.0, 4060.0, 4059.0, 4058.0, 4057.0, 4056.0, 4055.0, 4054.0, 4053.0, 4052.0, 4051.0, 4050.0, 4049.0, 4048.0, 4047.0, 4046.0, 4045.0, 4044.0, 4043.0, 4042.0, 4041.0, 4040.0, 4039.0, 4038.0, 4037.0, 4036.0, 4035.0, 4034.0, 4033.0, 4032.0, 4031.0, 4030.0, 4029.0, 4028.0, 4027.0, 4026.0, 4025.0, 4024.0, 4023.0, 4022.0, 4021.0, 4020.0, 4019.0, 4018.0, 4017.0, 4016.0, 4015.0, 4014.0, 4013.0, 4012.0, 4011.0, 4010.0, 4009.0, 4008.0, 4007.0, 4006.0, 4005.0, 4004.0, 4003.0, 4002.0, 4001.0, 4000.0, 3999.0, 3998.0, 3997.0, 3996.0, 3995.0, 3994.0, 3993.0, 3992.0, 3991.0, 3990.0, 3989.0, 3988.0, 3987.0, 3986.0, 3985.0, 3984.0, 3983.0, 3982.0, 3981.0, 3980.0, 3979.0, 3978.0, 3977.0, 3976.0, 3975.0, 3974.0, 3973.0, 3972.0, 3971.0, 3970.0, 3969.0, 3968.0, 3967.0, 3966.0, 3965.0, 3964.0, 3963.0, 3962.0, 3961.0, 3960.0, 3959.0, 3958.0, 3957.0, 3956.0, 3955.0, 3954.0, 3953.0, 3952.0, 3951.0, 3950.0, 3949.0, 3948.0, 3947.0, 3946.0, 3945.0, 3944.0, 3943.0, 3942.0, 3941.0, 3940.0, 3939.0, 3938.0, 3937.0, 3936.0, 3935.0, 3934.0, 3933.0, 3932.0, 3931.0, 3930.0, 3929.0, 3928.0, 3927.0, 3926.0, 3925.0, 3924.0, 3923.0, 3922.0, 3921.0, 3920.0, 3919.0, 3918.0, 3917.0, 3916.0, 3915.0, 3914.0, 3913.0, 3912.0, 3911.0, 3910.0, 3909.0, 3908.0, 3907.0, 3906.0, 3905.0, 3904.0, 3903.0, 3902.0, 3901.0, 3900.0, 3899.0, 3898.0, 3897.0, 3896.0, 3895.0, 3894.0, 3893.0, 3892.0, 3891.0, 3890.0, 3889.0, 3888.0, 3887.0, 3886.0, 3885.0, 3884.0, 3883.0, 3882.0, 3881.0, 3880.0, 3879.0, 3878.0, 3877.0, 3876.0, 3875.0, 3874.0, 3873.0, 3872.0, 3871.0, 3870.0, 3869.0, 3868.0, 3867.0, 3866.0, 3865.0, 3864.0, 3863.0, 3862.0, 3861.0, 3860.0, 3859.0, 3858.0, 3857.0, 3856.0, 3855.0, 3854.0, 3853.0, 3852.0, 3851.0, 3850.0, 3849.0, 3848.0, 3847.0, 3846.0, 3845.0, 3844.0, 3843.0, 3842.0, 3841.0, 3840.0, 3839.0, 3838.0, 3837.0, 3836.0, 3835.0, 3834.0, 3833.0, 3832.0, 3831.0, 3830.0, 3829.0, 3828.0, 3827.0, 3826.0, 3825.0, 3824.0, 3823.0, 3822.0, 3821.0, 3820.0, 3819.0, 3818.0, 3817.0, 3816.0, 3815.0, 3814.0, 3813.0, 3812.0, 3811.0, 3810.0, 3809.0, 3808.0, 3807.0, 3806.0, 3805.0, 3804.0, 3803.0, 3802.0, 3801.0, 3800.0, 3799.0, 3798.0, 3797.0, 3796.0, 3795.0, 3794.0, 3793.0, 3792.0, 3791.0, 3790.0, 3789.0, 3788.0, 3787.0, 3786.0, 3785.0, 3784.0, 3783.0, 3782.0, 3781.0, 3780.0, 3779.0, 3778.0, 3777.0, 3776.0, 3775.0, 3774.0, 3773.0, 3772.0, 3771.0, 3770.0, 3769.0, 3768.0, 3767.0, 3766.0, 3765.0, 3764.0, 3763.0, 3762.0, 3761.0, 3760.0, 3759.0, 3758.0, 3757.0, 3756.0, 3755.0, 3754.0, 3753.0, 3752.0, 3751.0, 3750.0, 3749.0, 3748.0, 3747.0, 3746.0, 3745.0, 3744.0, 3743.0, 3742.0, 3741.0, 3740.0, 3739.0, 3738.0, 3737.0, 3736.0, 3735.0, 3734.0, 3733.0, 3732.0, 3731.0, 3730.0, 3729.0, 3728.0, 3727.0, 3726.0, 3725.0, 3724.0, 3723.0, 3722.0, 3721.0, 3720.0, 3719.0, 3718.0, 3717.0, 3716.0, 3715.0, 3714.0, 3713.0, 3712.0, 3711.0, 3710.0, 3709.0, 3708.0, 3707.0, 3706.0, 3705.0, 3704.0, 3703.0, 3702.0, 3701.0, 3700.0, 3699.0, 3698.0, 3697.0, 3696.0, 3695.0, 3694.0, 3693.0, 3692.0, 3691.0, 3690.0, 3689.0, 3688.0, 3687.0, 3686.0, 3685.0, 3684.0, 3683.0, 3682.0, 3681.0, 3680.0, 3679.0, 3678.0, 3677.0, 3676.0, 3675.0, 3674.0, 3673.0, 3672.0, 3671.0, 3670.0, 3669.0, 3668.0, 3667.0, 3666.0, 3665.0, 3664.0, 3663.0, 3662.0, 3661.0, 3660.0, 3659.0, 3658.0, 3657.0, 3656.0, 3655.0, 3654.0, 3653.0, 3652.0, 3651.0, 3650.0, 3649.0, 3648.0, 3647.0, 3646.0, 3645.0, 3644.0, 3643.0, 3642.0, 3641.0, 3640.0, 3639.0, 3638.0, 3637.0, 3636.0, 3635.0, 3634.0, 3633.0, 3632.0, 3631.0, 3630.0, 3629.0, 3628.0, 3627.0, 3626.0, 3625.0, 3624.0, 3623.0, 3622.0, 3621.0, 3620.0, 3619.0, 3618.0, 3617.0, 3616.0, 3615.0, 3614.0, 3613.0, 3612.0, 3611.0, 3610.0, 3609.0, 3608.0, 3607.0, 3606.0, 3605.0, 3604.0, 3603.0, 3602.0, 3601.0, 3600.0, 3599.0, 3598.0, 3597.0, 3596.0, 3595.0, 3594.0, 3593.0, 3592.0, 3591.0, 3590.0, 3589.0, 3588.0, 3587.0, 3586.0, 3585.0, 3584.0, 3583.0, 3582.0, 3581.0, 3580.0, 3579.0, 3578.0, 3577.0, 3576.0, 3575.0, 3574.0, 3573.0, 3572.0, 3571.0, 3570.0, 3569.0, 3568.0, 3567.0, 3566.0, 3565.0, 3564.0, 3563.0, 3562.0, 3561.0, 3560.0, 3559.0, 3558.0, 3557.0, 3556.0, 3555.0, 3554.0, 3553.0, 3552.0, 3551.0, 3550.0, 3549.0, 3548.0, 3547.0, 3546.0, 3545.0, 3544.0, 3543.0, 3542.0, 3541.0, 3540.0, 3539.0, 3538.0, 3537.0, 3536.0, 3535.0, 3534.0, 3533.0, 3532.0, 3531.0, 3530.0, 3529.0, 3528.0, 3527.0, 3526.0, 3525.0, 3524.0, 3523.0, 3522.0, 3521.0, 3520.0, 3519.0, 3518.0, 3517.0, 3516.0, 3515.0, 3514.0, 3513.0, 3512.0, 3511.0, 3510.0, 3509.0, 3508.0, 3507.0, 3506.0, 3505.0, 3504.0, 3503.0, 3502.0, 3501.0, 3500.0, 3499.0, 3498.0, 3497.0, 3496.0, 3495.0, 3494.0, 3493.0, 3492.0, 3491.0, 3490.0, 3489.0, 3488.0, 3487.0, 3486.0, 3485.0, 3484.0, 3483.0, 3482.0, 3481.0, 3480.0, 3479.0, 3478.0, 3477.0, 3476.0, 3475.0, 3474.0, 3473.0, 3472.0, 3471.0, 3470.0, 3469.0, 3468.0, 3467.0, 3466.0, 3465.0, 3464.0, 3463.0, 3462.0, 3461.0, 3460.0, 3459.0, 3458.0, 3457.0, 3456.0, 3455.0, 3454.0, 3453.0, 3452.0, 3451.0, 3450.0, 3449.0, 3448.0, 3447.0, 3446.0, 3445.0, 3444.0, 3443.0, 3442.0, 3441.0, 3440.0, 3439.0, 3438.0, 3437.0, 3436.0, 3435.0, 3434.0, 3433.0, 3432.0, 3431.0, 3430.0, 3429.0, 3428.0, 3427.0, 3426.0, 3425.0, 3424.0, 3423.0, 3422.0, 3421.0, 3420.0, 3419.0, 3418.0, 3417.0, 3416.0, 3415.0, 3414.0, 3413.0, 3412.0, 3411.0, 3410.0, 3409.0, 3408.0, 3407.0, 3406.0, 3405.0, 3404.0, 3403.0, 3402.0, 3401.0, 3400.0, 3399.0, 3398.0, 3397.0, 3396.0, 3395.0, 3394.0, 3393.0, 3392.0, 3391.0, 3390.0, 3389.0, 3388.0, 3387.0, 3386.0, 3385.0, 3384.0, 3383.0, 3382.0, 3381.0, 3380.0, 3379.0, 3378.0, 3377.0, 3376.0, 3375.0, 3374.0, 3373.0, 3372.0, 3371.0, 3370.0, 3369.0, 3368.0, 3367.0, 3366.0, 3365.0, 3364.0, 3363.0, 3362.0, 3361.0, 3360.0, 3359.0, 3358.0, 3357.0, 3356.0, 3355.0, 3354.0, 3353.0, 3352.0, 3351.0, 3350.0, 3349.0, 3348.0, 3347.0, 3346.0, 3345.0, 3344.0, 3343.0, 3342.0, 3341.0, 3340.0, 3339.0, 3338.0, 3337.0, 3336.0, 3335.0, 3334.0, 3333.0, 3332.0, 3331.0, 3330.0, 3329.0, 3328.0, 3327.0, 3326.0, 3325.0, 3324.0, 3323.0, 3322.0, 3321.0, 3320.0, 3319.0, 3318.0, 3317.0, 3316.0, 3315.0, 3314.0, 3313.0, 3312.0, 3311.0, 3310.0, 3309.0, 3308.0, 3307.0, 3306.0, 3305.0, 3304.0, 3303.0, 3302.0, 3301.0, 3300.0, 3299.0, 3298.0, 3297.0, 3296.0, 3295.0, 3294.0, 3293.0, 3292.0, 3291.0, 3290.0, 3289.0, 3288.0, 3287.0, 3286.0, 3285.0, 3284.0, 3283.0, 3282.0, 3281.0, 3280.0, 3279.0, 3278.0, 3277.0, 3276.0, 3275.0, 3274.0, 3273.0, 3272.0, 3271.0, 3270.0, 3269.0, 3268.0, 3267.0, 3266.0, 3265.0, 3264.0, 3263.0, 3262.0, 3261.0, 3260.0, 3259.0, 3258.0, 3257.0, 3256.0, 3255.0, 3254.0, 3253.0, 3252.0, 3251.0, 3250.0, 3249.0, 3248.0, 3247.0, 3246.0, 3245.0, 3244.0, 3243.0, 3242.0, 3241.0, 3240.0, 3239.0, 3238.0, 3237.0, 3236.0, 3235.0, 3234.0, 3233.0, 3232.0, 3231.0, 3230.0, 3229.0, 3228.0, 3227.0, 3226.0, 3225.0, 3224.0, 3223.0, 3222.0, 3221.0, 3220.0, 3219.0, 3218.0, 3217.0, 3216.0, 3215.0, 3214.0, 3213.0, 3212.0, 3211.0, 3210.0, 3209.0, 3208.0, 3207.0, 3206.0, 3205.0, 3204.0, 3203.0, 3202.0, 3201.0, 3200.0, 3199.0, 3198.0, 3197.0, 3196.0, 3195.0, 3194.0, 3193.0, 3192.0, 3191.0, 3190.0, 3189.0, 3188.0, 3187.0, 3186.0, 3185.0, 3184.0, 3183.0, 3182.0, 3181.0, 3180.0, 3179.0, 3178.0, 3177.0, 3176.0, 3175.0, 3174.0, 3173.0, 3172.0, 3171.0, 3170.0, 3169.0, 3168.0, 3167.0, 3166.0, 3165.0, 3164.0, 3163.0, 3162.0, 3161.0, 3160.0, 3159.0, 3158.0, 3157.0, 3156.0, 3155.0, 3154.0, 3153.0, 3152.0, 3151.0, 3150.0, 3149.0, 3148.0, 3147.0, 3146.0, 3145.0, 3144.0, 3143.0, 3142.0, 3141.0, 3140.0, 3139.0, 3138.0, 3137.0, 3136.0, 3135.0, 3134.0, 3133.0, 3132.0, 3131.0, 3130.0, 3129.0, 3128.0, 3127.0, 3126.0, 3125.0, 3124.0, 3123.0, 3122.0, 3121.0, 3120.0, 3119.0, 3118.0, 3117.0, 3116.0, 3115.0, 3114.0, 3113.0, 3112.0, 3111.0, 3110.0, 3109.0, 3108.0, 3107.0, 3106.0, 3105.0, 3104.0, 3103.0, 3102.0, 3101.0, 3100.0, 3099.0, 3098.0, 3097.0, 3096.0, 3095.0, 3094.0, 3093.0, 3092.0, 3091.0, 3090.0, 3089.0, 3088.0, 3087.0, 3086.0, 3085.0, 3084.0, 3083.0, 3082.0, 3081.0, 3080.0, 3079.0, 3078.0, 3077.0, 3076.0, 3075.0, 3074.0, 3073.0, 3072.0, 3071.0, 3070.0, 3069.0, 3068.0, 3067.0, 3066.0, 3065.0, 3064.0, 3063.0, 3062.0, 3061.0, 3060.0, 3059.0, 3058.0, 3057.0, 3056.0, 3055.0, 3054.0, 3053.0, 3052.0, 3051.0, 3050.0, 3049.0, 3048.0, 3047.0, 3046.0, 3045.0, 3044.0, 3043.0, 3042.0, 3041.0, 3040.0, 3039.0, 3038.0, 3037.0, 3036.0, 3035.0, 3034.0, 3033.0, 3032.0, 3031.0, 3030.0, 3029.0, 3028.0, 3027.0, 3026.0, 3025.0, 3024.0, 3023.0, 3022.0, 3021.0, 3020.0, 3019.0, 3018.0, 3017.0, 3016.0, 3015.0, 3014.0, 3013.0, 3012.0, 3011.0, 3010.0, 3009.0, 3008.0, 3007.0, 3006.0, 3005.0, 3004.0, 3003.0, 3002.0, 3001.0, 3000.0, 2999.0, 2998.0, 2997.0, 2996.0, 2995.0, 2994.0, 2993.0, 2992.0, 2991.0, 2990.0, 2989.0, 2988.0, 2987.0, 2986.0, 2985.0, 2984.0, 2983.0, 2982.0, 2981.0, 2980.0, 2979.0, 2978.0, 2977.0, 2976.0, 2975.0, 2974.0, 2973.0, 2972.0, 2971.0, 2970.0, 2969.0, 2968.0, 2967.0, 2966.0, 2965.0, 2964.0, 2963.0, 2962.0, 2961.0, 2960.0, 2959.0, 2958.0, 2957.0, 2956.0, 2955.0, 2954.0, 2953.0, 2952.0, 2951.0, 2950.0, 2949.0, 2948.0, 2947.0, 2946.0, 2945.0, 2944.0, 2943.0, 2942.0, 2941.0, 2940.0, 2939.0, 2938.0, 2937.0, 2936.0, 2935.0, 2934.0, 2933.0, 2932.0, 2931.0, 2930.0, 2929.0, 2928.0, 2927.0, 2926.0, 2925.0, 2924.0, 2923.0, 2922.0, 2921.0, 2920.0, 2919.0, 2918.0, 2917.0, 2916.0, 2915.0, 2914.0, 2913.0, 2912.0, 2911.0, 2910.0, 2909.0, 2908.0, 2907.0, 2906.0, 2905.0, 2904.0, 2903.0, 2902.0, 2901.0, 2900.0, 2899.0, 2898.0, 2897.0, 2896.0, 2895.0, 2894.0, 2893.0, 2892.0, 2891.0, 2890.0, 2889.0, 2888.0, 2887.0, 2886.0, 2885.0, 2884.0, 2883.0, 2882.0, 2881.0, 2880.0, 2879.0, 2878.0, 2877.0, 2876.0, 2875.0, 2874.0, 2873.0, 2872.0, 2871.0, 2870.0, 2869.0, 2868.0, 2867.0, 2866.0, 2865.0, 2864.0, 2863.0, 2862.0, 2861.0, 2860.0, 2859.0, 2858.0, 2857.0, 2856.0, 2855.0, 2854.0, 2853.0, 2852.0, 2851.0, 2850.0, 2849.0, 2848.0, 2847.0, 2846.0, 2845.0, 2844.0, 2843.0, 2842.0, 2841.0, 2840.0, 2839.0, 2838.0, 2837.0, 2836.0, 2835.0, 2834.0, 2833.0, 2832.0, 2831.0, 2830.0, 2829.0, 2828.0, 2827.0, 2826.0, 2825.0, 2824.0, 2823.0, 2822.0, 2821.0, 2820.0, 2819.0, 2818.0, 2817.0, 2816.0, 2815.0, 2814.0, 2813.0, 2812.0, 2811.0, 2810.0, 2809.0, 2808.0, 2807.0, 2806.0, 2805.0, 2804.0, 2803.0, 2802.0, 2801.0, 2800.0, 2799.0, 2798.0, 2797.0, 2796.0, 2795.0, 2794.0, 2793.0, 2792.0, 2791.0, 2790.0, 2789.0, 2788.0, 2787.0, 2786.0, 2785.0, 2784.0, 2783.0, 2782.0, 2781.0, 2780.0, 2779.0, 2778.0, 2777.0, 2776.0, 2775.0, 2774.0, 2773.0, 2772.0, 2771.0, 2770.0, 2769.0, 2768.0, 2767.0, 2766.0, 2765.0, 2764.0, 2763.0, 2762.0, 2761.0, 2760.0, 2759.0, 2758.0, 2757.0, 2756.0, 2755.0, 2754.0, 2753.0, 2752.0, 2751.0, 2750.0, 2749.0, 2748.0, 2747.0, 2746.0, 2745.0, 2744.0, 2743.0, 2742.0, 2741.0, 2740.0, 2739.0, 2738.0, 2737.0, 2736.0, 2735.0, 2734.0, 2733.0, 2732.0, 2731.0, 2730.0, 2729.0, 2728.0, 2727.0, 2726.0, 2725.0, 2724.0, 2723.0, 2722.0, 2721.0, 2720.0, 2719.0, 2718.0, 2717.0, 2716.0, 2715.0, 2714.0, 2713.0, 2712.0, 2711.0, 2710.0, 2709.0, 2708.0, 2707.0, 2706.0, 2705.0, 2704.0, 2703.0, 2702.0, 2701.0, 2700.0, 2699.0, 2698.0, 2697.0, 2696.0, 2695.0, 2694.0, 2693.0, 2692.0, 2691.0, 2690.0, 2689.0, 2688.0, 2687.0, 2686.0, 2685.0, 2684.0, 2683.0, 2682.0, 2681.0, 2680.0, 2679.0, 2678.0, 2677.0, 2676.0, 2675.0, 2674.0, 2673.0, 2672.0, 2671.0, 2670.0, 2669.0, 2668.0, 2667.0, 2666.0, 2665.0, 2664.0, 2663.0, 2662.0, 2661.0, 2660.0, 2659.0, 2658.0, 2657.0, 2656.0, 2655.0, 2654.0, 2653.0, 2652.0, 2651.0, 2650.0, 2649.0, 2648.0, 2647.0, 2646.0, 2645.0, 2644.0, 2643.0, 2642.0, 2641.0, 2640.0, 2639.0, 2638.0, 2637.0, 2636.0, 2635.0, 2634.0, 2633.0, 2632.0, 2631.0, 2630.0, 2629.0, 2628.0, 2627.0, 2626.0, 2625.0, 2624.0, 2623.0, 2622.0, 2621.0, 2620.0, 2619.0, 2618.0, 2617.0, 2616.0, 2615.0, 2614.0, 2613.0, 2612.0, 2611.0, 2610.0, 2609.0, 2608.0, 2607.0, 2606.0, 2605.0, 2604.0, 2603.0, 2602.0, 2601.0, 2600.0, 2599.0, 2598.0, 2597.0, 2596.0, 2595.0, 2594.0, 2593.0, 2592.0, 2591.0, 2590.0, 2589.0, 2588.0, 2587.0, 2586.0, 2585.0, 2584.0, 2583.0, 2582.0, 2581.0, 2580.0, 2579.0, 2578.0, 2577.0, 2576.0, 2575.0, 2574.0, 2573.0, 2572.0, 2571.0, 2570.0, 2569.0, 2568.0, 2567.0, 2566.0, 2565.0, 2564.0, 2563.0, 2562.0, 2561.0, 2560.0, 2559.0, 2558.0, 2557.0, 2556.0, 2555.0, 2554.0, 2553.0, 2552.0, 2551.0, 2550.0, 2549.0, 2548.0, 2547.0, 2546.0, 2545.0, 2544.0, 2543.0, 2542.0, 2541.0, 2540.0, 2539.0, 2538.0, 2537.0, 2536.0, 2535.0, 2534.0, 2533.0, 2532.0, 2531.0, 2530.0, 2529.0, 2528.0, 2527.0, 2526.0, 2525.0, 2524.0, 2523.0, 2522.0, 2521.0, 2520.0, 2519.0, 2518.0, 2517.0, 2516.0, 2515.0, 2514.0, 2513.0, 2512.0, 2511.0, 2510.0, 2509.0, 2508.0, 2507.0, 2506.0, 2505.0, 2504.0, 2503.0, 2502.0, 2501.0, 2500.0, 2499.0, 2498.0, 2497.0, 2496.0, 2495.0, 2494.0, 2493.0, 2492.0, 2491.0, 2490.0, 2489.0, 2488.0, 2487.0, 2486.0, 2485.0, 2484.0, 2483.0, 2482.0, 2481.0, 2480.0, 2479.0, 2478.0, 2477.0, 2476.0, 2475.0, 2474.0, 2473.0, 2472.0, 2471.0, 2470.0, 2469.0, 2468.0, 2467.0, 2466.0, 2465.0, 2464.0, 2463.0, 2462.0, 2461.0, 2460.0, 2459.0, 2458.0, 2457.0, 2456.0, 2455.0, 2454.0, 2453.0, 2452.0, 2451.0, 2450.0, 2449.0, 2448.0, 2447.0, 2446.0, 2445.0, 2444.0, 2443.0, 2442.0, 2441.0, 2440.0, 2439.0, 2438.0, 2437.0, 2436.0, 2435.0, 2434.0, 2433.0, 2432.0, 2431.0, 2430.0, 2429.0, 2428.0, 2427.0, 2426.0, 2425.0, 2424.0, 2423.0, 2422.0, 2421.0, 2420.0, 2419.0, 2418.0, 2417.0, 2416.0, 2415.0, 2414.0, 2413.0, 2412.0, 2411.0, 2410.0, 2409.0, 2408.0, 2407.0, 2406.0, 2405.0, 2404.0, 2403.0, 2402.0, 2401.0, 2400.0, 2399.0, 2398.0, 2397.0, 2396.0, 2395.0, 2394.0, 2393.0, 2392.0, 2391.0, 2390.0, 2389.0, 2388.0, 2387.0, 2386.0, 2385.0, 2384.0, 2383.0, 2382.0, 2381.0, 2380.0, 2379.0, 2378.0, 2377.0, 2376.0, 2375.0, 2374.0, 2373.0, 2372.0, 2371.0, 2370.0, 2369.0, 2368.0, 2367.0, 2366.0, 2365.0, 2364.0, 2363.0, 2362.0, 2361.0, 2360.0, 2359.0, 2358.0, 2357.0, 2356.0, 2355.0, 2354.0, 2353.0, 2352.0, 2351.0, 2350.0, 2349.0, 2348.0, 2347.0, 2346.0, 2345.0, 2344.0, 2343.0, 2342.0, 2341.0, 2340.0, 2339.0, 2338.0, 2337.0, 2336.0, 2335.0, 2334.0, 2333.0, 2332.0, 2331.0, 2330.0, 2329.0, 2328.0, 2327.0, 2326.0, 2325.0, 2324.0, 2323.0, 2322.0, 2321.0, 2320.0, 2319.0, 2318.0, 2317.0, 2316.0, 2315.0, 2314.0, 2313.0, 2312.0, 2311.0, 2310.0, 2309.0, 2308.0, 2307.0, 2306.0, 2305.0, 2304.0, 2303.0, 2302.0, 2301.0, 2300.0, 2299.0, 2298.0, 2297.0, 2296.0, 2295.0, 2294.0, 2293.0, 2292.0, 2291.0, 2290.0, 2289.0, 2288.0, 2287.0, 2286.0, 2285.0, 2284.0, 2283.0, 2282.0, 2281.0, 2280.0, 2279.0, 2278.0, 2277.0, 2276.0, 2275.0, 2274.0, 2273.0, 2272.0, 2271.0, 2270.0, 2269.0, 2268.0, 2267.0, 2266.0, 2265.0, 2264.0, 2263.0, 2262.0, 2261.0, 2260.0, 2259.0, 2258.0, 2257.0, 2256.0, 2255.0, 2254.0, 2253.0, 2252.0, 2251.0, 2250.0, 2249.0, 2248.0, 2247.0, 2246.0, 2245.0, 2244.0, 2243.0, 2242.0, 2241.0, 2240.0, 2239.0, 2238.0, 2237.0, 2236.0, 2235.0, 2234.0, 2233.0, 2232.0, 2231.0, 2230.0, 2229.0, 2228.0, 2227.0, 2226.0, 2225.0, 2224.0, 2223.0, 2222.0, 2221.0, 2220.0, 2219.0, 2218.0, 2217.0, 2216.0, 2215.0, 2214.0, 2213.0, 2212.0, 2211.0, 2210.0, 2209.0, 2208.0, 2207.0, 2206.0, 2205.0, 2204.0, 2203.0, 2202.0, 2201.0, 2200.0, 2199.0, 2198.0, 2197.0, 2196.0, 2195.0, 2194.0, 2193.0, 2192.0, 2191.0, 2190.0, 2189.0, 2188.0, 2187.0, 2186.0, 2185.0, 2184.0, 2183.0, 2182.0, 2181.0, 2180.0, 2179.0, 2178.0, 2177.0, 2176.0, 2175.0, 2174.0, 2173.0, 2172.0, 2171.0, 2170.0, 2169.0, 2168.0, 2167.0, 2166.0, 2165.0, 2164.0, 2163.0, 2162.0, 2161.0, 2160.0, 2159.0, 2158.0, 2157.0, 2156.0, 2155.0, 2154.0, 2153.0, 2152.0, 2151.0, 2150.0, 2149.0, 2148.0, 2147.0, 2146.0, 2145.0, 2144.0, 2143.0, 2142.0, 2141.0, 2140.0, 2139.0, 2138.0, 2137.0, 2136.0, 2135.0, 2134.0, 2133.0, 2132.0, 2131.0, 2130.0, 2129.0, 2128.0, 2127.0, 2126.0, 2125.0, 2124.0, 2123.0, 2122.0, 2121.0, 2120.0, 2119.0, 2118.0, 2117.0, 2116.0, 2115.0, 2114.0, 2113.0, 2112.0, 2111.0, 2110.0, 2109.0, 2108.0, 2107.0, 2106.0, 2105.0, 2104.0, 2103.0, 2102.0, 2101.0, 2100.0, 2099.0, 2098.0, 2097.0, 2096.0, 2095.0, 2094.0, 2093.0, 2092.0, 2091.0, 2090.0, 2089.0, 2088.0, 2087.0, 2086.0, 2085.0, 2084.0, 2083.0, 2082.0, 2081.0, 2080.0, 2079.0, 2078.0, 2077.0, 2076.0, 2075.0, 2074.0, 2073.0, 2072.0, 2071.0, 2070.0, 2069.0, 2068.0, 2067.0, 2066.0, 2065.0, 2064.0, 2063.0, 2062.0, 2061.0, 2060.0, 2059.0, 2058.0, 2057.0, 2056.0, 2055.0, 2054.0, 2053.0, 2052.0, 2051.0, 2050.0, 2049.0, 2048.0, 2047.0, 2046.0, 2045.0, 2044.0, 2043.0, 2042.0, 2041.0, 2040.0, 2039.0, 2038.0, 2037.0, 2036.0, 2035.0, 2034.0, 2033.0, 2032.0, 2031.0, 2030.0, 2029.0, 2028.0, 2027.0, 2026.0, 2025.0, 2024.0, 2023.0, 2022.0, 2021.0, 2020.0, 2019.0, 2018.0, 2017.0, 2016.0, 2015.0, 2014.0, 2013.0, 2012.0, 2011.0, 2010.0, 2009.0, 2008.0, 2007.0, 2006.0, 2005.0, 2004.0, 2003.0, 2002.0, 2001.0, 2000.0, 1999.0, 1998.0, 1997.0, 1996.0, 1995.0, 1994.0, 1993.0, 1992.0, 1991.0, 1990.0, 1989.0, 1988.0, 1987.0, 1986.0, 1985.0, 1984.0, 1983.0, 1982.0, 1981.0, 1980.0, 1979.0, 1978.0, 1977.0, 1976.0, 1975.0, 1974.0, 1973.0, 1972.0, 1971.0, 1970.0, 1969.0, 1968.0, 1967.0, 1966.0, 1965.0, 1964.0, 1963.0, 1962.0, 1961.0, 1960.0, 1959.0, 1958.0, 1957.0, 1956.0, 1955.0, 1954.0, 1953.0, 1952.0, 1951.0, 1950.0, 1949.0, 1948.0, 1947.0, 1946.0, 1945.0, 1944.0, 1943.0, 1942.0, 1941.0, 1940.0, 1939.0, 1938.0, 1937.0, 1936.0, 1935.0, 1934.0, 1933.0, 1932.0, 1931.0, 1930.0, 1929.0, 1928.0, 1927.0, 1926.0, 1925.0, 1924.0, 1923.0, 1922.0, 1921.0, 1920.0, 1919.0, 1918.0, 1917.0, 1916.0, 1915.0, 1914.0, 1913.0, 1912.0, 1911.0, 1910.0, 1909.0, 1908.0, 1907.0, 1906.0, 1905.0, 1904.0, 1903.0, 1902.0, 1901.0, 1900.0, 1899.0, 1898.0, 1897.0, 1896.0, 1895.0, 1894.0, 1893.0, 1892.0, 1891.0, 1890.0, 1889.0, 1888.0, 1887.0, 1886.0, 1885.0, 1884.0, 1883.0, 1882.0, 1881.0, 1880.0, 1879.0, 1878.0, 1877.0, 1876.0, 1875.0, 1874.0, 1873.0, 1872.0, 1871.0, 1870.0, 1869.0, 1868.0, 1867.0, 1866.0, 1865.0, 1864.0, 1863.0, 1862.0, 1861.0, 1860.0, 1859.0, 1858.0, 1857.0, 1856.0, 1855.0, 1854.0, 1853.0, 1852.0, 1851.0, 1850.0, 1849.0, 1848.0, 1847.0, 1846.0, 1845.0, 1844.0, 1843.0, 1842.0, 1841.0, 1840.0, 1839.0, 1838.0, 1837.0, 1836.0, 1835.0, 1834.0, 1833.0, 1832.0, 1831.0, 1830.0, 1829.0, 1828.0, 1827.0, 1826.0, 1825.0, 1824.0, 1823.0, 1822.0, 1821.0, 1820.0, 1819.0, 1818.0, 1817.0, 1816.0, 1815.0, 1814.0, 1813.0, 1812.0, 1811.0, 1810.0, 1809.0, 1808.0, 1807.0, 1806.0, 1805.0, 1804.0, 1803.0, 1802.0, 1801.0, 1800.0, 1799.0, 1798.0, 1797.0, 1796.0, 1795.0, 1794.0, 1793.0, 1792.0, 1791.0, 1790.0, 1789.0, 1788.0, 1787.0, 1786.0, 1785.0, 1784.0, 1783.0, 1782.0, 1781.0, 1780.0, 1779.0, 1778.0, 1777.0, 1776.0, 1775.0, 1774.0, 1773.0, 1772.0, 1771.0, 1770.0, 1769.0, 1768.0, 1767.0, 1766.0, 1765.0, 1764.0, 1763.0, 1762.0, 1761.0, 1760.0, 1759.0, 1758.0, 1757.0, 1756.0, 1755.0, 1754.0, 1753.0, 1752.0, 1751.0, 1750.0, 1749.0, 1748.0, 1747.0, 1746.0, 1745.0, 1744.0, 1743.0, 1742.0, 1741.0, 1740.0, 1739.0, 1738.0, 1737.0, 1736.0, 1735.0, 1734.0, 1733.0, 1732.0, 1731.0, 1730.0, 1729.0, 1728.0, 1727.0, 1726.0, 1725.0, 1724.0, 1723.0, 1722.0, 1721.0, 1720.0, 1719.0, 1718.0, 1717.0, 1716.0, 1715.0, 1714.0, 1713.0, 1712.0, 1711.0, 1710.0, 1709.0, 1708.0, 1707.0, 1706.0, 1705.0, 1704.0, 1703.0, 1702.0, 1701.0, 1700.0, 1699.0, 1698.0, 1697.0, 1696.0, 1695.0, 1694.0, 1693.0, 1692.0, 1691.0, 1690.0, 1689.0, 1688.0, 1687.0, 1686.0, 1685.0, 1684.0, 1683.0, 1682.0, 1681.0, 1680.0, 1679.0, 1678.0, 1677.0, 1676.0, 1675.0, 1674.0, 1673.0, 1672.0, 1671.0, 1670.0, 1669.0, 1668.0, 1667.0, 1666.0, 1665.0, 1664.0, 1663.0, 1662.0, 1661.0, 1660.0, 1659.0, 1658.0, 1657.0, 1656.0, 1655.0, 1654.0, 1653.0, 1652.0, 1651.0, 1650.0, 1649.0, 1648.0, 1647.0, 1646.0, 1645.0, 1644.0, 1643.0, 1642.0, 1641.0, 1640.0, 1639.0, 1638.0, 1637.0, 1636.0, 1635.0, 1634.0, 1633.0, 1632.0, 1631.0, 1630.0, 1629.0, 1628.0, 1627.0, 1626.0, 1625.0, 1624.0, 1623.0, 1622.0, 1621.0, 1620.0, 1619.0, 1618.0, 1617.0, 1616.0, 1615.0, 1614.0, 1613.0, 1612.0, 1611.0, 1610.0, 1609.0, 1608.0, 1607.0, 1606.0, 1605.0, 1604.0, 1603.0, 1602.0, 1601.0, 1600.0, 1599.0, 1598.0, 1597.0, 1596.0, 1595.0, 1594.0, 1593.0, 1592.0, 1591.0, 1590.0, 1589.0, 1588.0, 1587.0, 1586.0, 1585.0, 1584.0, 1583.0, 1582.0, 1581.0, 1580.0, 1579.0, 1578.0, 1577.0, 1576.0, 1575.0, 1574.0, 1573.0, 1572.0, 1571.0, 1570.0, 1569.0, 1568.0, 1567.0, 1566.0, 1565.0, 1564.0, 1563.0, 1562.0, 1561.0, 1560.0, 1559.0, 1558.0, 1557.0, 1556.0, 1555.0, 1554.0, 1553.0, 1552.0, 1551.0, 1550.0, 1549.0, 1548.0, 1547.0, 1546.0, 1545.0, 1544.0, 1543.0, 1542.0, 1541.0, 1540.0, 1539.0, 1538.0, 1537.0, 1536.0, 1535.0, 1534.0, 1533.0, 1532.0, 1531.0, 1530.0, 1529.0, 1528.0, 1527.0, 1526.0, 1525.0, 1524.0, 1523.0, 1522.0, 1521.0, 1520.0, 1519.0, 1518.0, 1517.0, 1516.0, 1515.0, 1514.0, 1513.0, 1512.0, 1511.0, 1510.0, 1509.0, 1508.0, 1507.0, 1506.0, 1505.0, 1504.0, 1503.0, 1502.0, 1501.0, 1500.0, 1499.0, 1498.0, 1497.0, 1496.0, 1495.0, 1494.0, 1493.0, 1492.0, 1491.0, 1490.0, 1489.0, 1488.0, 1487.0, 1486.0, 1485.0, 1484.0, 1483.0, 1482.0, 1481.0, 1480.0, 1479.0, 1478.0, 1477.0, 1476.0, 1475.0, 1474.0, 1473.0, 1472.0, 1471.0, 1470.0, 1469.0, 1468.0, 1467.0, 1466.0, 1465.0, 1464.0, 1463.0, 1462.0, 1461.0, 1460.0, 1459.0, 1458.0, 1457.0, 1456.0, 1455.0, 1454.0, 1453.0, 1452.0, 1451.0, 1450.0, 1449.0, 1448.0, 1447.0, 1446.0, 1445.0, 1444.0, 1443.0, 1442.0, 1441.0, 1440.0, 1439.0, 1438.0, 1437.0, 1436.0, 1435.0, 1434.0, 1433.0, 1432.0, 1431.0, 1430.0, 1429.0, 1428.0, 1427.0, 1426.0, 1425.0, 1424.0, 1423.0, 1422.0, 1421.0, 1420.0, 1419.0, 1418.0, 1417.0, 1416.0, 1415.0, 1414.0, 1413.0, 1412.0, 1411.0, 1410.0, 1409.0, 1408.0, 1407.0, 1406.0, 1405.0, 1404.0, 1403.0, 1402.0, 1401.0, 1400.0, 1399.0, 1398.0, 1397.0, 1396.0, 1395.0, 1394.0, 1393.0, 1392.0, 1391.0, 1390.0, 1389.0, 1388.0, 1387.0, 1386.0, 1385.0, 1384.0, 1383.0, 1382.0, 1381.0, 1380.0, 1379.0, 1378.0, 1377.0, 1376.0, 1375.0, 1374.0, 1373.0, 1372.0, 1371.0, 1370.0, 1369.0, 1368.0, 1367.0, 1366.0, 1365.0, 1364.0, 1363.0, 1362.0, 1361.0, 1360.0, 1359.0, 1358.0, 1357.0, 1356.0, 1355.0, 1354.0, 1353.0, 1352.0, 1351.0, 1350.0, 1349.0, 1348.0, 1347.0, 1346.0, 1345.0, 1344.0, 1343.0, 1342.0, 1341.0, 1340.0, 1339.0, 1338.0, 1337.0, 1336.0, 1335.0, 1334.0, 1333.0, 1332.0, 1331.0, 1330.0, 1329.0, 1328.0, 1327.0, 1326.0, 1325.0, 1324.0, 1323.0, 1322.0, 1321.0, 1320.0, 1319.0, 1318.0, 1317.0, 1316.0, 1315.0, 1314.0, 1313.0, 1312.0, 1311.0, 1310.0, 1309.0, 1308.0, 1307.0, 1306.0, 1305.0, 1304.0, 1303.0, 1302.0, 1301.0, 1300.0, 1299.0, 1298.0, 1297.0, 1296.0, 1295.0, 1294.0, 1293.0, 1292.0, 1291.0, 1290.0, 1289.0, 1288.0, 1287.0, 1286.0, 1285.0, 1284.0, 1283.0, 1282.0, 1281.0, 1280.0, 1279.0, 1278.0, 1277.0, 1276.0, 1275.0, 1274.0, 1273.0, 1272.0, 1271.0, 1270.0, 1269.0, 1268.0, 1267.0, 1266.0, 1265.0, 1264.0, 1263.0, 1262.0, 1261.0, 1260.0, 1259.0, 1258.0, 1257.0, 1256.0, 1255.0, 1254.0, 1253.0, 1252.0, 1251.0, 1250.0, 1249.0, 1248.0, 1247.0, 1246.0, 1245.0, 1244.0, 1243.0, 1242.0, 1241.0, 1240.0, 1239.0, 1238.0, 1237.0, 1236.0, 1235.0, 1234.0, 1233.0, 1232.0, 1231.0, 1230.0, 1229.0, 1228.0, 1227.0, 1226.0, 1225.0, 1224.0, 1223.0, 1222.0, 1221.0, 1220.0, 1219.0, 1218.0, 1217.0, 1216.0, 1215.0, 1214.0, 1213.0, 1212.0, 1211.0, 1210.0, 1209.0, 1208.0, 1207.0, 1206.0, 1205.0, 1204.0, 1203.0, 1202.0, 1201.0, 1200.0, 1199.0, 1198.0, 1197.0, 1196.0, 1195.0, 1194.0, 1193.0, 1192.0, 1191.0, 1190.0, 1189.0, 1188.0, 1187.0, 1186.0, 1185.0, 1184.0, 1183.0, 1182.0, 1181.0, 1180.0, 1179.0, 1178.0, 1177.0, 1176.0, 1175.0, 1174.0, 1173.0, 1172.0, 1171.0, 1170.0, 1169.0, 1168.0, 1167.0, 1166.0, 1165.0, 1164.0, 1163.0, 1162.0, 1161.0, 1160.0, 1159.0, 1158.0, 1157.0, 1156.0, 1155.0, 1154.0, 1153.0, 1152.0, 1151.0, 1150.0, 1149.0, 1148.0, 1147.0, 1146.0, 1145.0, 1144.0, 1143.0, 1142.0, 1141.0, 1140.0, 1139.0, 1138.0, 1137.0, 1136.0, 1135.0, 1134.0, 1133.0, 1132.0, 1131.0, 1130.0, 1129.0, 1128.0, 1127.0, 1126.0, 1125.0, 1124.0, 1123.0, 1122.0, 1121.0, 1120.0, 1119.0, 1118.0, 1117.0, 1116.0, 1115.0, 1114.0, 1113.0, 1112.0, 1111.0, 1110.0, 1109.0, 1108.0, 1107.0, 1106.0, 1105.0, 1104.0, 1103.0, 1102.0, 1101.0, 1100.0, 1099.0, 1098.0, 1097.0, 1096.0, 1095.0, 1094.0, 1093.0, 1092.0, 1091.0, 1090.0, 1089.0, 1088.0, 1087.0, 1086.0, 1085.0, 1084.0, 1083.0, 1082.0, 1081.0, 1080.0, 1079.0, 1078.0, 1077.0, 1076.0, 1075.0, 1074.0, 1073.0, 1072.0, 1071.0, 1070.0, 1069.0, 1068.0, 1067.0, 1066.0, 1065.0, 1064.0, 1063.0, 1062.0, 1061.0, 1060.0, 1059.0, 1058.0, 1057.0, 1056.0, 1055.0, 1054.0, 1053.0, 1052.0, 1051.0, 1050.0, 1049.0, 1048.0, 1047.0, 1046.0, 1045.0, 1044.0, 1043.0, 1042.0, 1041.0, 1040.0, 1039.0, 1038.0, 1037.0, 1036.0, 1035.0, 1034.0, 1033.0, 1032.0, 1031.0, 1030.0, 1029.0, 1028.0, 1027.0, 1026.0, 1025.0, 1024.0, 1023.0, 1022.0, 1021.0, 1020.0, 1019.0, 1018.0, 1017.0, 1016.0, 1015.0, 1014.0, 1013.0, 1012.0, 1011.0, 1010.0, 1009.0, 1008.0, 1007.0, 1006.0, 1005.0, 1004.0, 1003.0, 1002.0, 1001.0, 1000.0, 999.0, 998.0, 997.0, 996.0, 995.0, 994.0, 993.0, 992.0, 991.0, 990.0, 989.0, 988.0, 987.0, 986.0, 985.0, 984.0, 983.0, 982.0, 981.0, 980.0, 979.0, 978.0, 977.0, 976.0, 975.0, 974.0, 973.0, 972.0, 971.0, 970.0, 969.0, 968.0, 967.0, 966.0, 965.0, 964.0, 963.0, 962.0, 961.0, 960.0, 959.0, 958.0, 957.0, 956.0, 955.0, 954.0, 953.0, 952.0, 951.0, 950.0, 949.0, 948.0, 947.0, 946.0, 945.0, 944.0, 943.0, 942.0, 941.0, 940.0, 939.0, 938.0, 937.0, 936.0, 935.0, 934.0, 933.0, 932.0, 931.0, 930.0, 929.0, 928.0, 927.0, 926.0, 925.0, 924.0, 923.0, 922.0, 921.0, 920.0, 919.0, 918.0, 917.0, 916.0, 915.0, 914.0, 913.0, 912.0, 911.0, 910.0, 909.0, 908.0, 907.0, 906.0, 905.0, 904.0, 903.0, 902.0, 901.0, 900.0, 899.0, 898.0, 897.0, 896.0, 895.0, 894.0, 893.0, 892.0, 891.0, 890.0, 889.0, 888.0, 887.0, 886.0, 885.0, 884.0, 883.0, 882.0, 881.0, 880.0, 879.0, 878.0, 877.0, 876.0, 875.0, 874.0, 873.0, 872.0, 871.0, 870.0, 869.0, 868.0, 867.0, 866.0, 865.0, 864.0, 863.0, 862.0, 861.0, 860.0, 859.0, 858.0, 857.0, 856.0, 855.0, 854.0, 853.0, 852.0, 851.0, 850.0, 849.0, 848.0, 847.0, 846.0, 845.0, 844.0, 843.0, 842.0, 841.0, 840.0, 839.0, 838.0, 837.0, 836.0, 835.0, 834.0, 833.0, 832.0, 831.0, 830.0, 829.0, 828.0, 827.0, 826.0, 825.0, 824.0, 823.0, 822.0, 821.0, 820.0, 819.0, 818.0, 817.0, 816.0, 815.0, 814.0, 813.0, 812.0, 811.0, 810.0, 809.0, 808.0, 807.0, 806.0, 805.0, 804.0, 803.0, 802.0, 801.0, 800.0, 799.0, 798.0, 797.0, 796.0, 795.0, 794.0, 793.0, 792.0, 791.0, 790.0, 789.0, 788.0, 787.0, 786.0, 785.0, 784.0, 783.0, 782.0, 781.0, 780.0, 779.0, 778.0, 777.0, 776.0, 775.0, 774.0, 773.0, 772.0, 771.0, 770.0, 769.0, 768.0, 767.0, 766.0, 765.0, 764.0, 763.0, 762.0, 761.0, 760.0, 759.0, 758.0, 757.0, 756.0, 755.0, 754.0, 753.0, 752.0, 751.0, 750.0, 749.0, 748.0, 747.0, 746.0, 745.0, 744.0, 743.0, 742.0, 741.0, 740.0, 739.0, 738.0, 737.0, 736.0, 735.0, 734.0, 733.0, 732.0, 731.0, 730.0, 729.0, 728.0, 727.0, 726.0, 725.0, 724.0, 723.0, 722.0, 721.0, 720.0, 719.0, 718.0, 717.0, 716.0, 715.0, 714.0, 713.0, 712.0, 711.0, 710.0, 709.0, 708.0, 707.0, 706.0, 705.0, 704.0, 703.0, 702.0, 701.0, 700.0, 699.0, 698.0, 697.0, 696.0, 695.0, 694.0, 693.0, 692.0, 691.0, 690.0, 689.0, 688.0, 687.0, 686.0, 685.0, 684.0, 683.0, 682.0, 681.0, 680.0, 679.0, 678.0, 677.0, 676.0, 675.0, 674.0, 673.0, 672.0, 671.0, 670.0, 669.0, 668.0, 667.0, 666.0, 665.0, 664.0, 663.0, 662.0, 661.0, 660.0, 659.0, 658.0, 657.0, 656.0, 655.0, 654.0, 653.0, 652.0, 651.0, 650.0, 649.0, 648.0, 647.0, 646.0, 645.0, 644.0, 643.0, 642.0, 641.0, 640.0, 639.0, 638.0, 637.0, 636.0, 635.0, 634.0, 633.0, 632.0, 631.0, 630.0, 629.0, 628.0, 627.0, 626.0, 625.0, 624.0, 623.0, 622.0, 621.0, 620.0, 619.0, 618.0, 617.0, 616.0, 615.0, 614.0, 613.0, 612.0, 611.0, 610.0, 609.0, 608.0, 607.0, 606.0, 605.0, 604.0, 603.0, 602.0, 601.0, 600.0, 599.0, 598.0, 597.0, 596.0, 595.0, 594.0, 593.0, 592.0, 591.0, 590.0, 589.0, 588.0, 587.0, 586.0, 585.0, 584.0, 583.0, 582.0, 581.0, 580.0, 579.0, 578.0, 577.0, 576.0, 575.0, 574.0, 573.0, 572.0, 571.0, 570.0, 569.0, 568.0, 567.0, 566.0, 565.0, 564.0, 563.0, 562.0, 561.0, 560.0, 559.0, 558.0, 557.0, 556.0, 555.0, 554.0, 553.0, 552.0, 551.0, 550.0, 549.0, 548.0, 547.0, 546.0, 545.0, 544.0, 543.0, 542.0, 541.0, 540.0, 539.0, 538.0, 537.0, 536.0, 535.0, 534.0, 533.0, 532.0, 531.0, 530.0, 529.0, 528.0, 527.0, 526.0, 525.0, 524.0, 523.0, 522.0, 521.0, 520.0, 519.0, 518.0, 517.0, 516.0, 515.0, 514.0, 513.0, 512.0, 511.0, 510.0, 509.0, 508.0, 507.0, 506.0, 505.0, 504.0, 503.0, 502.0, 501.0, 500.0, 499.0, 498.0, 497.0, 496.0, 495.0, 494.0, 493.0, 492.0, 491.0, 490.0, 489.0, 488.0, 487.0, 486.0, 485.0, 484.0, 483.0, 482.0, 481.0, 480.0, 479.0, 478.0, 477.0, 476.0, 475.0, 474.0, 473.0, 472.0, 471.0, 470.0, 469.0, 468.0, 467.0, 466.0, 465.0, 464.0, 463.0, 462.0, 461.0, 460.0, 459.0, 458.0, 457.0, 456.0, 455.0, 454.0, 453.0, 452.0, 451.0, 450.0, 449.0, 448.0, 447.0, 446.0, 445.0, 444.0, 443.0, 442.0, 441.0, 440.0, 439.0, 438.0, 437.0, 436.0, 435.0, 434.0, 433.0, 432.0, 431.0, 430.0, 429.0, 428.0, 427.0, 426.0, 425.0, 424.0, 423.0, 422.0, 421.0, 420.0, 419.0, 418.0, 417.0, 416.0, 415.0, 414.0, 413.0, 412.0, 411.0, 410.0, 409.0, 408.0, 407.0, 406.0, 405.0, 404.0, 403.0, 402.0, 401.0, 400.0, 399.0, 398.0, 397.0, 396.0, 395.0, 394.0, 393.0, 392.0, 391.0, 390.0, 389.0, 388.0, 387.0, 386.0, 385.0, 384.0, 383.0, 382.0, 381.0, 380.0, 379.0, 378.0, 377.0, 376.0, 375.0, 374.0, 373.0, 372.0, 371.0, 370.0, 369.0, 368.0, 367.0, 366.0, 365.0, 364.0, 363.0, 362.0, 361.0, 360.0, 359.0, 358.0, 357.0, 356.0, 355.0, 354.0, 353.0, 352.0, 351.0, 350.0, 349.0, 348.0, 347.0, 346.0, 345.0, 344.0, 343.0, 342.0, 341.0, 340.0, 339.0, 338.0, 337.0, 336.0, 335.0, 334.0, 333.0, 332.0, 331.0, 330.0, 329.0, 328.0, 327.0, 326.0, 325.0, 324.0, 323.0, 322.0, 321.0, 320.0, 319.0, 318.0, 317.0, 316.0, 315.0, 314.0, 313.0, 312.0, 311.0, 310.0, 309.0, 308.0, 307.0, 306.0, 305.0, 304.0, 303.0, 302.0, 301.0, 300.0, 299.0, 298.0, 297.0, 296.0, 295.0, 294.0, 293.0, 292.0, 291.0, 290.0, 289.0, 288.0, 287.0, 286.0, 285.0, 284.0, 283.0, 282.0, 281.0, 280.0, 279.0, 278.0, 277.0, 276.0, 275.0, 274.0, 273.0, 272.0, 271.0, 270.0, 269.0, 268.0, 267.0, 266.0, 265.0, 264.0, 263.0, 262.0, 261.0, 260.0, 259.0, 258.0, 257.0, 256.0, 255.0, 254.0, 253.0, 252.0, 251.0, 250.0, 249.0, 248.0, 247.0, 246.0, 245.0, 244.0, 243.0, 242.0, 241.0, 240.0, 239.0, 238.0, 237.0, 236.0, 235.0, 234.0, 233.0, 232.0, 231.0, 230.0, 229.0, 228.0, 227.0, 226.0, 225.0, 224.0, 223.0, 222.0, 221.0, 220.0, 219.0, 218.0, 217.0, 216.0, 215.0, 214.0, 213.0, 212.0, 211.0, 210.0, 209.0, 208.0, 207.0, 206.0, 205.0, 204.0, 203.0, 202.0, 201.0, 200.0, 199.0, 198.0, 197.0, 196.0, 195.0, 194.0, 193.0, 192.0, 191.0, 190.0, 189.0, 188.0, 187.0, 186.0, 185.0, 184.0, 183.0, 182.0, 181.0, 180.0, 179.0, 178.0, 177.0, 176.0, 175.0, 174.0, 173.0, 172.0, 171.0, 170.0, 169.0, 168.0, 167.0, 166.0, 165.0, 164.0, 163.0, 162.0, 161.0, 160.0, 159.0, 158.0, 157.0, 156.0, 155.0, 154.0, 153.0, 152.0, 151.0, 150.0, 149.0, 148.0, 147.0, 146.0, 145.0, 144.0, 143.0, 142.0, 141.0, 140.0, 139.0, 138.0, 137.0, 136.0, 135.0, 134.0, 133.0, 132.0, 131.0, 130.0, 129.0, 128.0, 127.0, 126.0, 125.0, 124.0, 123.0, 122.0, 121.0, 120.0, 119.0, 118.0, 117.0, 116.0, 115.0, 114.0, 113.0, 112.0, 111.0, 110.0, 109.0, 108.0, 107.0, 106.0, 105.0, 104.0, 103.0, 102.0, 101.0, 100.0, 99.0, 98.0, 97.0, 96.0, 95.0, 94.0, 93.0, 92.0, 91.0, 90.0, 89.0, 88.0, 87.0, 86.0, 85.0, 84.0, 83.0, 82.0, 81.0, 80.0, 79.0, 78.0, 77.0, 76.0, 75.0, 74.0, 73.0, 72.0, 71.0, 70.0, 69.0, 68.0, 67.0, 66.0, 65.0, 64.0, 63.0, 62.0, 61.0, 60.0, 59.0, 58.0, 57.0, 56.0, 55.0, 54.0, 53.0, 52.0, 51.0, 50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43.0, 42.0, 41.0, 40.0, 39.0, 38.0, 37.0, 36.0, 35.0, 34.0, 33.0, 32.0, 31.0, 30.0, 29.0, 28.0, 27.0, 26.0, 25.0, 24.0, 23.0, 22.0, 21.0, 20.0, 19.0, 18.0, 17.0, 16.0, 15.0, 14.0, 13.0, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0, 0.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": 4095.0, "dtype": "float32"}, {"__nd__": 0, "dtype": "int64"}]}}, {"a": [{"__nd__": [-0.1465129256248474, 1.1678411960601807, 1.069018006324768, 0.025955215096473694, 0.6430160403251648, 0.5633448362350464, -0.5654096603393555, 0.6160897016525269, 1.8629858493804932, 0.33721327781677246, -2.0401742458343506, 0.9687879085540771, -0.22129103541374207, -0.3345159888267517, -0.9348636865615845, 0.43387240171432495, -0.2745921313762665, 2.03490948677063, -1.3222376108169556, -0.4558241367340088, 1.7164943218231201, -0.44557175040245056, 0.3522481620311737, -0.05433575436472893, 0.7049575448036194, 0.6248272657394409, 0.9875816702842712, -0.14864279329776764, 0.9091802835464478, 0.15141893923282623, 0.6500976085662842, -0.5402806401252747, -2.3223907947540283, -1.5651110410690308, 0.6289846897125244, -0.3036331832408905, -0.9177509546279907, 0.5566563010215759, 0.6929896473884583, 0.08576474338769913, -1.766640067100525, 0.831169843673706, -0.6488447189331055, -0.5914568305015564, -0.393378347158432, 0.03611239418387413, -1.527266025543213, -0.16739888489246368, -1.3768593072891235, -1.070616602897644, -1.5528496503829956, -0.10114818811416626, -0.42840418219566345, 0.6891545653343201, -0.8837705254554749, 0.4139728546142578, -0.12075632810592651, 0.8964173793792725, -1.0445936918258667, -1.1694343090057373, 0.06879021227359772, 0.4263031482696533, 0.5857365131378174, 1.2354596853256226, 0.6784664988517761, 0.6674093008041382, 2.1801841259002686, -0.06367740035057068, 1.337904453277588, -0.19596879184246063, 0.6326837539672852, 0.11797448247671127, -0.02042214572429657, 1.4519237279891968, 1.5624710321426392, 0.024712178856134415, -0.147637739777565, -0.25406864285469055, -1.568618893623352, 1.2591828107833862, 0.10057483613491058, 0.9698046445846558, 1.189836025238037, -0.2670767307281494, 0.6482747793197632, -0.7329568266868591, -1.542348027229309, -0.6630795001983643, -0.31927284598350525, 0.0543411560356617, -0.3162418305873871, -0.9139438271522522, -0.8788609504699707, 0.165882408618927, -1.3295246362686157, 0.05143829435110092, -0.152074933052063, 0.18230289220809937, -2.312277317047119, 1.2659318447113037, -2.2063868045806885, 0.26508691906929016, -0.5461316108703613, -0.2439919412136078, -0.4528268873691559, -0.17764706909656525, -0.6218001842498779, 1.2389315366744995, 1.0793508291244507, 1.1993660926818848, 0.39311400055885315, -0.39464521408081055, 2.4870615005493164, 0.2073325514793396, -1.5553009510040283, 1.7773717641830444, 2.019190549850464, -1.2613857984542847, 0.29326507449150085, -0.33660197257995605, 1.2655199766159058, -1.8079127073287964, -0.2970994710922241, 0.052219316363334656, 0.33001652359962463, -1.0119976997375488, 1.0757765769958496, -1.2721247673034668, 0.6349689960479736, 2.781644821166992, -2.2451488971710205, 0.45952364802360535, 1.0877341032028198, -1.6089708805084229, 0.32728663086891174, -0.11740971356630325, -0.44772279262542725, -1.34689199924469, -0.5802748799324036, -0.32920295000076294, -1.7494330406188965, -2.0733859539031982, -1.185316801071167, -1.043778419494629, -0.4589448869228363, -0.015432941727340221, 0.355061799287796, -0.5551515221595764, 0.7207797169685364, 0.32503455877304077, -0.44959133863449097, 0.7024005055427551, 0.06937690079212189, 0.2754620611667633, 1.702691674232483, -1.3865123987197876, -0.5322197079658508, -0.3433443307876587, 0.5647196173667908, -1.3302758932113647, -1.0241732597351074, -1.6858294010162354, -2.4147937297821045, -1.1032838821411133, -1.5467967987060547, -0.20815476775169373, 0.8844970464706421, 0.1370319128036499, -1.3289501667022705, 1.0342795848846436, 1.4050315618515015, 0.348197877407074, 0.30691710114479065, -0.5565518736839294, 0.9304146766662598, 1.376700758934021, -0.8380635380744934, 1.0970717668533325, -0.6934836506843567, -0.5066977739334106, 2.6485002040863037, 0.9193989038467407, 1.803907036781311, -1.470134973526001, 1.4912749528884888, 2.3400113582611084, -0.3640883266925812, 0.7409713864326477, -0.30537086725234985, 0.06214619055390358, -0.9601098299026489, 1.0462056398391724, 0.025805503129959106, -0.9768009781837463, -0.10079807788133621, -1.161220908164978, -0.12543530762195587, -0.1892433613538742, -1.0793850421905518, -0.7223497033119202, 2.812156915664673, 0.9771953225135803, -0.06573456525802612, -2.0575356483459473, -0.17216190695762634, -2.1394684314727783, 1.339544415473938, 0.14554880559444427, 2.0280678272247314, 0.7276510000228882, 0.2743328809738159, 0.07686734944581985, -0.9338719844818115, 0.15708963572978973, -2.1205663681030273, -0.3716147840023041, 2.359271764755249, -0.9361292123794556, 0.5857166647911072, 0.9583808779716492, -1.262454867362976, 1.0754423141479492, -0.6758100986480713, -0.04186435788869858, -1.7719669342041016, -1.545809268951416, -0.3126964271068573, -0.6356315612792969, -0.9685235023498535, 1.201590657234192, -0.13091567158699036, -1.3461920022964478, -1.2403548955917358, -1.1196069717407227, 0.8424199819564819, -0.03600060194730759, 0.386471688747406, -0.5047882795333862, -0.15911492705345154, -0.0727926716208458, 0.7997040748596191, 0.10802490264177322, 0.04469086974859238, 1.0275052785873413, -0.15898674726486206, 0.8874837160110474, -0.16545508801937103, 0.6228925585746765, -0.8605572581291199, 0.1860063076019287, 0.5560944080352783, -0.6591723561286926, 0.09095431119203568, 0.18903623521327972, -0.8453273177146912, -0.6163061857223511, -0.9480747580528259, 1.050165057182312, 0.16066834330558777, 0.66617351770401, -0.7385390400886536, -0.41533708572387695, -0.07235192507505417, 0.735099732875824, -1.1535427570343018, -0.9863194227218628, 0.955028772354126, -0.8455226421356201, -0.05023425072431564, -0.3915899991989136, 1.3936518430709839, -0.00839394610375166, -0.9259684681892395, 1.167689561843872, -0.41068747639656067, 0.21143412590026855, -0.791053831577301, -1.3313615322113037, 0.32137852907180786, -0.19086892902851105, 1.3244704008102417, -0.1982288360595703, -0.7706194519996643, 1.3794127702713013, 0.28198403120040894, -0.5602211952209473, 0.8517439365386963, -0.34966030716896057, 1.4728492498397827, 1.073204755783081, 0.013715553097426891, -0.04071285203099251, -0.31333988904953003, -0.19074185192584991, -0.7777328491210938, 0.4246765971183777, -2.1483709812164307, 0.8529468774795532, -0.5553184747695923, 0.25520631670951843, 0.14360040426254272, 0.2235269546508789, 2.9123830795288086, -1.961380958557129, -1.1546390056610107, -0.16203808784484863, 0.6847468614578247, -1.3004450798034668, 1.5566762685775757, -1.473598837852478, -0.36962389945983887, -0.09415338933467865, -0.34064459800720215, 0.3024207353591919, 1.0417006015777588, -0.20361053943634033, 0.6965591311454773, 1.0960979461669922, -0.4809739887714386, 0.13211360573768616, -1.76317298412323, -0.39483770728111267, 0.22178921103477478, 1.5403971672058105, -2.265955686569214, 0.1723007708787918, -0.45570826530456543, -1.2751108407974243, -0.9523504972457886, -0.022280853241682053, 0.9083728194236755, -0.49064141511917114, 0.8611583113670349, -0.9291286468505859, -0.635684609413147, 0.1588674932718277, 0.5072785019874573, 0.634030282497406, -0.9488804936408997, 0.43273216485977173, -0.10592352598905563, 1.0925768613815308, -0.2454070746898651, 0.6874104142189026, 0.7279759049415588, 0.13760629296302795, -1.690413475036621, -1.9091044664382935, -0.6024359464645386, -0.13032859563827515, 0.04271196573972702, -0.6476093530654907, -1.532421350479126, 0.3218275308609009, -1.4346137046813965, 0.19782455265522003, 1.7856464385986328, -0.04251231625676155, 1.5370464324951172, -0.43224164843559265, -0.2339017242193222, 0.9757406711578369, 0.05525112897157669, 0.654983401298523, -1.0453187227249146, -0.459977924823761, 0.20478598773479462, -0.6441556811332703, 0.5361231565475464, 2.0168449878692627, -0.33112969994544983, -0.16587606072425842, 0.8731471300125122, -0.7805851697921753, 0.47092190384864807, -0.14156942069530487, 0.2859507203102112, 1.4174929857254028, -1.2957054376602173, 0.45108088850975037, 0.30379602313041687, 0.15724492073059082, 0.17954376339912415, 1.6986747980117798, 1.7433209419250488, 0.6887901425361633, 0.12546859681606293, -0.687954843044281, -0.149275004863739, -0.45037755370140076, 0.7292315363883972, -0.6674418449401855, 0.7447224855422974, -0.04358697682619095, -1.4161604642868042, 0.6007182598114014, 0.3852611184120178, 0.7957123517990112, -0.8627877831459045, 0.7626836895942688, -0.13512910902500153, -1.0363028049468994, 1.0948679447174072, 0.5192405581474304, -1.3956669569015503, 2.379556179046631, 1.0798773765563965, -0.22951263189315796, -0.8802441358566284, -0.47178494930267334, 0.05485481396317482, 1.8258118629455566, -0.5394783020019531, -1.2379820346832275, -0.821552574634552, -0.07342880219221115, -1.8070265054702759, -0.5883966088294983, -0.40481463074684143, 0.4851504862308502, -0.7233153581619263, -1.1978782415390015, -0.14874519407749176, -0.06062382087111473, 0.7937554717063904, 0.8908074498176575, 0.17494294047355652, -0.005025245249271393, -0.7958250045776367, 1.6835815906524658, 0.8227246999740601, -0.7805101871490479, 1.2777734994888306, 0.09435157477855682, -0.7454646229743958, -0.8015854358673096, -0.19152957201004028, 1.2328869104385376, 0.2762615382671356, -0.3686352074146271, -0.5729662179946899, -1.4412190914154053, -0.12785941362380981, 0.450327068567276, 0.09675095230340958, 2.2870304584503174, -0.11404263228178024, -0.6229072213172913, -1.5183298587799072, 1.132287621498108, -0.9590154886245728, -0.628150463104248, 0.02675246074795723, 0.6145373582839966, 2.2935996055603027, -0.6219443082809448, -0.27239206433296204, -0.6766296029090881, -0.9511699676513672, 1.0281267166137695, 0.46464911103248596, -0.599749743938446, -0.07142192870378494, 1.9076225757598877, 0.14751026034355164, 2.691720724105835, -0.566128671169281, -1.672229528427124, -0.6716355085372925, -0.3065318763256073, -0.4416399598121643, -0.6718713045120239, -0.03432866558432579, -0.5873064398765564, -1.4166139364242554, 1.462344765663147, -0.7521493434906006, 0.2669927179813385, -1.180866003036499, 1.3484963178634644, 0.5844550132751465, -1.6020935773849487, -1.1196398735046387, -0.07192979753017426, -0.8357444405555725, -0.6319939494132996, -0.3232903778553009, 0.6488624215126038, 0.5705512762069702, 0.31538429856300354, -0.4302879869937897, -1.7550475597381592, 0.23047706484794617, -0.12202855199575424, 0.019801542162895203, 2.137035846710205, 0.0462220162153244, 0.10122118890285492, 1.0101919174194336, 0.05807347968220711, 0.03231165558099747, 1.1585723161697388, 1.0769749879837036, -0.8150655031204224, 0.07317466288805008, 0.006342063192278147, -0.5443562269210815, -1.3368157148361206, -1.7473008632659912, -0.009717132896184921, 0.19527554512023926, 1.486375331878662, 0.4581347405910492, -0.10144123435020447, 0.14678844809532166, -0.0042790137231349945, -1.483075737953186, -0.5469691157341003, 0.14294883608818054, -0.8087210059165955, -0.9201705455780029, -0.8952125906944275, -0.44900578260421753, 0.10392783582210541, 0.6462673544883728, -0.8167273998260498, 0.39659571647644043, -0.26894229650497437, 0.8256574273109436, 0.9894672632217407, -0.09102536737918854, -0.4419296979904175, 0.040347497910261154, 0.20667394995689392, -1.097464680671692, -0.3999280035495758, 0.8799586296081543, 1.42122220993042, -0.17980311810970306, 0.2694430351257324, 0.3728710412979126, -0.0775873064994812, 0.37659958004951477, 0.5597180128097534, -0.4302752912044525, 0.5586689710617065, 0.42365172505378723, -0.8866589069366455, 0.2725863456726074, 2.252363443374634, -0.5054945945739746, 0.2724876403808594, 0.9508709907531738, 0.5998725891113281, -1.172206997871399, 0.4398645758628845, 0.0695999264717102, -0.47251096367836, 2.346949577331543, -0.22475379705429077, 1.0021721124649048, 0.3354659676551819, -0.660369336605072, -1.1282069683074951, -0.6399996876716614, -0.18113471567630768, -0.34763142466545105, 1.3925397396087646, 1.0691910982131958, -1.4795531034469604, 0.3330705165863037, -0.9437338709831238, 1.457106351852417, 0.33667245507240295, -1.8047385215759277, 1.564968228340149, 2.0512242317199707, 1.5264631509780884, -1.0897088050842285, -0.8070045113563538, -0.8506485819816589, 0.500241219997406, 0.06470637768507004, -0.6261348128318787, 2.3654725551605225, -0.22790196537971497, -0.3009200394153595, -0.9919068217277527, -0.020666180178523064, -2.434272050857544, -0.48698899149894714, 0.6537313461303711, 0.5145083665847778, 0.7066047787666321, -0.24726459383964539, 1.204667329788208, -0.2822778522968292, -0.28474029898643494, 1.122909665107727, 1.2455445528030396, 1.2048194408416748, -0.712911069393158, -0.5961993336677551, -0.0751362293958664, 0.8826425075531006, 1.2884753942489624, -0.8575204610824585, 1.41060471534729, -0.4890815019607544, -0.4801752269268036, 0.8544436097145081, -0.7608116865158081, 1.9634816646575928, -0.15787644684314728, -0.07142454385757446, 0.9674112796783447, 0.1918104887008667, -0.4263208210468292, 1.1035888195037842, 0.6737043857574463, -1.0198476314544678, -0.8882031440734863, 0.9616126418113708, 2.157968759536743, -0.5032060146331787, 1.5800354480743408, -1.2106763124465942, 0.06594356894493103, 0.2606947720050812, 0.12834006547927856, -0.17515882849693298, -0.40013355016708374, -0.8876675367355347, 1.6378787755966187, -0.8424682021141052, -0.28015682101249695, 1.193546175956726, -0.5774219036102295, -1.0209261178970337, -0.5744476318359375, 0.11573289334774017, -0.11275041103363037, 0.11523271352052689, 1.59493088722229, 0.026334743946790695, 0.14419332146644592, 0.0899486169219017, 0.17027446627616882, 0.16088832914829254, -0.3279463052749634, -1.516017198562622, -1.5295909643173218, 1.6705080270767212, -1.2174161672592163, 1.260019063949585, -1.3866381645202637, 0.12152714282274246, -0.7605618834495544, 0.08484578132629395, 1.2153533697128296, -0.1801876723766327, 2.455732583999634, -1.9501477479934692, -1.1609923839569092, 0.17285136878490448, -1.2997528314590454, 0.6501748561859131, 0.5207368731498718, -0.5991829633712769, -0.812389075756073, 0.5454397201538086, 0.09717626124620438, 1.167908787727356, 1.097262978553772, -0.14167316257953644, -0.732205867767334, 0.3376268446445465, -0.1290547400712967, 1.4455745220184326, -0.45575717091560364, 1.2441333532333374, -0.2643759548664093, -0.2325674444437027, 1.856110692024231, -0.9585363268852234, -0.5052881836891174, -1.2750455141067505, 2.29838228225708, 1.2664943933486938, 0.14959916472434998, 1.4194782972335815, -0.5726728439331055, 1.477664828300476, 0.15623065829277039, -0.7854467034339905, 1.1191438436508179, 1.627914309501648, -0.10107535868883133, 1.0865023136138916, -0.44616568088531494, 1.1106598377227783, 1.9776774644851685, 0.8444734215736389, 0.02218446135520935, -0.5346204042434692, 0.9296509027481079, 2.111016273498535, -2.0565707683563232, 1.00995671749115, 1.6026955842971802, -0.9008191823959351, -1.2762573957443237, 0.26972195506095886, -0.7115046977996826, -2.9125654697418213, -1.231142520904541, -0.01546103972941637, 0.3913188874721527, -0.07612843811511993, -0.3599497377872467, 0.6933631300926208, -0.6562718749046326, 1.9092798233032227, -0.03448193892836571, -1.0618089437484741, -1.1200642585754395, -0.9288992285728455, -2.098825693130493, 0.4716288447380066, 0.27288195490837097, 0.384491890668869, 0.12939070165157318, 0.9462470412254333, 0.09299258142709732, -1.2025576829910278, 0.8176062107086182, -0.9660419821739197, 0.5208093523979187, 1.2514406442642212, 0.6809446811676025, -2.6813294887542725, -0.39221444725990295, 0.21803389489650726, -0.15247221291065216, -2.120842933654785, -0.3491981625556946, -0.9786456227302551, 0.34916943311691284, 1.165908932685852, 1.4126514196395874, 0.21361422538757324, -1.496204137802124, -1.0304325819015503, 0.27261996269226074, -0.10939941555261612, 1.7328113317489624, 0.03346463665366173, 0.22476477921009064, -0.33422476053237915, 1.9763836860656738, 0.14514245092868805, 0.5297684669494629, 0.43577173352241516, -1.7146755456924438, -1.735609769821167, 0.5037487745285034, -1.2797996997833252, -1.1813527345657349, 0.6759909391403198, -0.6259008646011353, -1.4221543073654175, -0.5481910705566406, -0.5919389724731445, -0.6071280241012573, 0.16709880530834198, 0.9722825884819031, -0.033643607050180435, 0.11707114428281784, 0.38260355591773987, -0.10731860995292664, -1.3769694566726685, 1.310438632965088, -0.22343960404396057, 0.5809895396232605, -1.410555362701416, 0.31211596727371216, -0.38782867789268494, 0.2904950678348541, 1.2347023487091064, 1.4377073049545288, -1.3944844007492065, 0.5618236064910889, 0.09444880485534668, -0.5637743473052979, 2.1476762294769287, -0.11984452605247498, -1.3314828872680664, -1.947302222251892, -1.360428810119629, -2.7852418422698975, -0.023777781054377556, 1.1450824737548828, 0.22070977091789246, -1.0487494468688965, -1.4377521276474, 1.2169628143310547, -2.840297222137451, 1.9229769706726074, -0.7950506210327148, -0.033856574445962906, 0.1585719734430313, 0.8555062413215637, 0.5689223408699036, -0.4987095594406128, 0.051500964909791946, -0.909177839756012, 0.11174455285072327, 0.5870484709739685, -1.119929552078247, 1.3695920705795288, -1.1968657970428467, 2.038407564163208, 0.9977051615715027, -1.1502374410629272, 0.4724642336368561, 0.932634711265564, 1.0170496702194214, 0.06777545064687729, -0.6163236498832703, 0.07157981395721436, -0.6791893839836121, -2.370518684387207, 1.9704711437225342, -0.30975979566574097, 1.0980366468429565, 1.2344895601272583, 0.2432108223438263, -0.26731181144714355, -1.4025461673736572, -1.3441386222839355, -1.663427710533142, -0.17819392681121826, -0.979232132434845, -1.015912652015686, -0.07480023801326752, -0.273186057806015, -1.2443853616714478, -1.4012030363082886, 1.902814507484436, -1.6567188501358032, -0.6303154826164246, 0.22532305121421814, 0.20803868770599365, 1.21152663230896, 0.0272745992988348, -0.6156520247459412, 1.6212869882583618, 0.1660933941602707, 0.12111202627420425, 0.6125574707984924, 0.10274061560630798, 2.1030149459838867, -1.3199024200439453, 0.26016250252723694, 1.6411093473434448, -1.4356516599655151, -0.48201489448547363, 0.12025847285985947, -0.1630699187517166, -0.3429754078388214, -1.3501673936843872, -0.2327958196401596, -0.3711262345314026, -1.139940857887268, -0.05684211477637291, -0.8959363102912903, 0.10206244885921478, -0.9193685054779053, 0.5327492952346802, -1.6887774467468262, 0.20314332842826843, 0.4124559164047241, -0.6852786540985107, -0.47489044070243835, 0.4764949381351471, 1.4592479467391968, 0.0797383263707161, -0.12940019369125366, 0.5450571179389954, -1.83254873752594, -0.8871037364006042, 0.3418619632720947, 1.8854548931121826, 0.4480072557926178, 1.1131763458251953, -0.6984474062919617, -0.6141859292984009, 2.3063411712646484, -0.2993997633457184, -1.9564136266708374, -0.5376923084259033, 0.4134065806865692, -0.6898128986358643, 0.2185380756855011, 0.7340222597122192, 1.8144609928131104, 2.4603271484375, 1.2994412183761597, 1.1507176160812378, 0.20053216814994812, -0.2068520039319992, 1.4095478057861328, 0.3410889804363251, 0.5998988151550293, 0.465741902589798, 1.5100735425949097, -0.8071244359016418, -0.9580514430999756, -0.4965592920780182, -1.8816845417022705, 0.4103142321109772, -0.5133334398269653, 0.7067367434501648, 0.7525767087936401, -0.8059310913085938, 0.38462144136428833, -1.0695905685424805, -0.31652843952178955, -0.9201200604438782, -1.6279078722000122, 0.8524020910263062, -0.15126700699329376, -0.24890625476837158, 0.5410378575325012, 0.7849671244621277, -0.5532781481742859, 0.7967484593391418, 0.010755816474556923, -0.4946092367172241, -0.1271737664937973, -0.02192295342683792, 1.9157335758209229, 0.12255140393972397, 1.414806842803955, 0.834406316280365, -1.9394303560256958, 0.00303271459415555, 0.4285230040550232, -0.5470498204231262, -0.5065386891365051, -0.3375832438468933, 0.7919520735740662, 0.42304620146751404, -1.9197871685028076, -0.43864965438842773, -2.1391196250915527, 0.3203762471675873, 0.9703764915466309, -0.8222227692604065, 1.7181018590927124, -1.4877885580062866, 0.45106735825538635, 0.6820639967918396, -0.8809123039245605, 0.01204802468419075, 0.3081975281238556, 1.4526190757751465, -0.9775869846343994, 0.2881941795349121, -0.7116654515266418, 1.1943777799606323, 0.36891525983810425, -0.29476767778396606, 1.1509007215499878, 0.3551044762134552, -0.7182834148406982, -1.1178656816482544, -1.1307011842727661, 0.1006905809044838, -1.1737713813781738, -1.2031810283660889, -1.0596823692321777, 0.4345146417617798, -0.937016487121582, 0.6179141402244568, 0.6976513862609863, -1.500389814376831, 0.13451360166072845, 0.33615684509277344, -0.5661912560462952, 1.0792890787124634, -0.16090135276317596, -0.569697916507721, -1.2173434495925903, 0.08970638364553452, 0.5187451243400574, -1.0998142957687378, -1.672001838684082, -0.9689987897872925, -0.72344970703125, 0.06590551137924194, -0.03334644064307213, -1.6077210903167725, -1.321839451789856, -0.42064905166625977, -0.11387234181165695, 0.04208212345838547, 0.29511934518814087, -1.25816810131073, 1.0369465351104736, -0.8724868893623352, -2.0740973949432373, -0.5855668187141418, 0.1719290316104889, 1.6338073015213013, 1.4870426654815674, 1.7697837352752686, 0.21636484563350677, -1.2414510250091553, 1.6574469804763794, 1.1886403560638428, 0.26033955812454224, 1.1220717430114746, -0.5070635080337524, -1.936415195465088, -0.4926310181617737, -0.11527522653341293, -0.5994893908500671, -0.20542380213737488, -0.7577935457229614, 0.893332839012146, 0.9705971479415894, 0.0759744942188263, 1.6561864614486694, -0.2606973946094513, 0.7649878859519958, 0.31629905104637146, 0.5486652851104736, 1.1394636631011963, 0.7768304944038391, 0.6177241802215576, -1.2962944507598877, 0.888857364654541, 0.7362582683563232, 0.7222470045089722, 0.9949324727058411, -1.0733776092529297, 0.3492511808872223, 0.5404483079910278, -1.0869171619415283, -0.6600819230079651, 1.4631634950637817, -0.05358956754207611, -0.6485345959663391, 0.35378533601760864, -0.35315510630607605, -0.35687491297721863, -0.335823118686676, 0.009178739972412586, -0.6624298095703125, 0.4171225428581238, 0.6902503967285156, -0.15165314078330994, -0.3178444504737854, -1.9608142375946045, 1.2843583822250366, 0.05180663242936134, 0.1421806961297989, -0.6990678906440735, 1.8194093704223633, 1.0901992321014404, 0.07673926651477814, 0.3643399178981781, 1.0012927055358887, 0.0006898621795699, -0.9127792716026306, -0.0798460990190506, 0.5687468647956848, 0.05347563698887825, 0.07734572887420654, 1.187011480331421, 0.3822144865989685, -1.2891219854354858, 0.031048759818077087, 1.0163849592208862, -2.0951364040374756, 0.8304958939552307, -2.209493637084961, 1.6014046669006348, 0.1592811495065689, -0.34268441796302795, 0.125128835439682, 0.31169024109840393, -0.8097766637802124, 0.07217641919851303, -0.6349175572395325, 0.7110079526901245, 0.23831166326999664, 1.564334511756897, 1.0752166509628296, 1.9281307458877563, -1.6543455123901367, 1.5416069030761719, -0.677635908126831, -0.21825820207595825, 0.44617822766304016, -0.9261150360107422, 0.008933842182159424, -0.16051878035068512, -1.3048264980316162, 0.434052050113678, -1.2864834070205688, 0.14407745003700256, 0.445511132478714, 0.12240155786275864, -1.2093521356582642, -1.479943037033081, 0.21483977138996124, 1.6491115093231201, 0.8754264116287231, -0.5754443407058716, 0.5600642561912537, 1.3158167600631714, 0.5248540639877319, 0.955647885799408, 0.5780502557754517, -1.064024567604065, 0.3775802254676819, 1.0488510131835938, 0.3508029580116272, -3.092430353164673, -2.9422659873962402, 1.7708817720413208, 1.4706813097000122, 0.41517627239227295, -0.756851077079773, 2.4977240562438965, -1.0401114225387573, -2.262428045272827, 1.3473384380340576, 0.47457030415534973, 1.0450236797332764, -0.0935901626944542, -1.1326051950454712, -1.0006788969039917, -2.0156805515289307, -0.2101474553346634, -0.5982625484466553, -0.23772041499614716, 0.810307502746582, 0.26374801993370056, -1.2127811908721924, 0.002594607649371028, 0.2554352283477783, 1.5539469718933105, -0.5333946347236633, 0.8438970446586609, -0.7290744185447693, 0.6606563329696655, 0.27156010270118713, -0.515816867351532, -1.3125154972076416, -0.431397020816803, -0.15999308228492737, -0.223871111869812, -0.22330987453460693, -0.765612781047821, 0.9451131820678711, -0.8927428126335144, 0.07626373320817947, 1.7302511930465698, -0.5409947037696838, -0.8444340229034424, -1.5130983591079712, 2.158696174621582, -0.20244605839252472, -0.9582399725914001, -0.6002036333084106, -0.1095639318227768, 1.087215542793274, 0.3538345396518707, -1.9190990924835205, 0.05492871627211571, -2.5655558109283447, -1.454961895942688, -0.77541583776474, -0.3537224233150482, 1.595401644706726, 0.14844970405101776, 0.24055494368076324, -0.8279566168785095, 0.729473352432251, -0.8492095470428467, -0.9997211694717407, 1.854377269744873, -0.4647083580493927, -1.8982771635055542, 0.9825212359428406, -1.2426657676696777, 0.3312174379825592, 1.664425253868103, 0.3014734089374542, -0.07484684139490128, 0.2774486541748047, -0.528702437877655, -0.2773982584476471, 1.381244421005249, -1.1261529922485352, -0.004942432511597872, -0.1439448893070221, -0.9122046232223511, -0.6600449085235596, -2.0462217330932617, -0.2636720538139343, 0.4120689034461975, -0.8291932344436646, 0.5928794145584106, -0.29160428047180176, 0.05534553900361061, -1.1626158952713013, -1.1424925327301025, -0.7155175805091858, -0.26658645272254944, -0.7974466681480408, 1.05715811252594, 0.8055992722511292, -0.595491349697113, -0.5415669083595276, 0.059010181576013565, 0.09295891225337982, 1.013858675956726, 1.3370245695114136, 0.8249227404594421, -0.007433631457388401, -0.04062676057219505, 0.6589247584342957, -0.6266583800315857, 1.167506217956543, 0.29115748405456543, -1.5087988376617432, 1.3840034008026123, 0.708053469657898, 1.2598363161087036, -0.3210723102092743, -0.9740583896636963, -1.6345268487930298, 2.2151851654052734, 0.6784058809280396, -1.204068899154663, -0.808479905128479, -0.8701777458190918, -0.06563110649585724, -0.19847044348716736, -0.6113373041152954, -0.3917880952358246, -0.6303110122680664, 0.6292930841445923, -0.6320870518684387, -0.9831038117408752, 0.749101459980011, -0.14240209758281708, 0.007177102379500866, -0.5359223484992981, 0.6541337370872498, -1.4614990949630737, -0.35788726806640625, 0.7711495161056519, 0.3279038071632385, -1.077961802482605, 0.05260394513607025, 0.6339763402938843, 1.4415104389190674, -0.6731073260307312, -0.9287607073783875, -0.05397341772913933, -0.4173729121685028, 0.07681051641702652, -1.2650833129882812, -1.1028703451156616, -1.0072951316833496, 0.7581683397293091, 0.2770853340625763, -0.16435204446315765, 0.7277158498764038, 1.5858123302459717, -1.1536635160446167, 1.0751773118972778, -0.9042295813560486, -0.03737013414502144, 0.7244485020637512, -1.1976823806762695, 1.2274513244628906, 0.8347015976905823, -0.13638781011104584, 1.249420166015625, -0.794639527797699, 0.5988419055938721, -1.1068731546401978, -0.47696688771247864, -0.9943954348564148, -0.6910601258277893, 0.9323159456253052, -1.7502262592315674, -0.48358312249183655, 1.1931272745132446, 0.7373778820037842, -0.3512556850910187, 0.04747426137328148, 0.33907556533813477, -1.0212557315826416, -0.36142754554748535, 0.663722038269043, 1.6447513103485107, -0.02724778652191162, 1.3245317935943604, 0.8335222005844116, -1.3665637969970703, 0.6252038478851318, -1.3394041061401367, -0.8601189851760864, 0.5079915523529053, 0.1306052803993225, 0.33554667234420776, 0.9778162837028503, -0.3121163249015808, -0.9532654285430908, 0.0402613990008831, -0.338124543428421, -1.3184515237808228, 0.5683559775352478, 0.12173541635274887, -0.042602021247148514, -0.09523911029100418, 0.46126243472099304, -1.5081706047058105, 0.8723866939544678, -0.36834225058555603, -0.6910415887832642, -2.8651602268218994, -0.08849141746759415, 0.8523532152175903, -1.5434626340866089, -0.7861654162406921, -0.39743462204933167, -0.643764853477478, 2.711294651031494, 0.07476393133401871, 0.20963425934314728, -0.8408118486404419, -1.2280982732772827, 0.09955523908138275, 0.17921806871891022, -1.907212734222412, 0.14537791907787323, 1.2390434741973877, -1.383772373199463, -0.5860925912857056, 1.7974120378494263, 0.09860371053218842, 1.1936256885528564, -0.6363248229026794, 0.3217476010322571, -0.6588026285171509, -0.17408892512321472, 1.3964753150939941, 1.336533784866333, -0.7695855498313904, 1.678215742111206, 0.43470028042793274, 0.18960410356521606, 0.2402130514383316, -0.01962907984852791, -0.24853293597698212, -0.8949085474014282, -1.5748363733291626, -1.1374008655548096, -0.26825210452079773, -0.05308561772108078, -0.8304575681686401, -0.9779455661773682, -1.0202653408050537, -0.7960190773010254, 0.6755275130271912, 2.0260207653045654, 0.9124929308891296, 0.6175938844680786, 0.01997442916035652, -0.6657221913337708, -0.07204896211624146, 0.3360206186771393, -1.6930742263793945, -0.26178330183029175, 0.45359787344932556, 2.5739729404449463, -0.7878545522689819, 1.5144585371017456, 1.038094401359558, 1.2639493942260742, 0.8579569458961487, 1.8587524890899658, -1.0287127494812012, -0.21683847904205322, 0.4913327693939209, 0.22136203944683075, -0.5027967691421509, -1.289656639099121, -0.17734846472740173, 0.8318199515342712, 0.8094161748886108, -1.5741959810256958, 1.0361584424972534, 1.0896939039230347, -3.1225693225860596, -2.188483238220215, -0.9333641529083252, -0.2609352767467499, 1.6743663549423218, -0.26666343212127686, -0.27769985795021057, -0.8414695858955383, 1.1578761339187622, 1.915151596069336, 0.0515609085559845, 0.2487153708934784, 0.1504552662372589, -0.6261126399040222, -0.6029256582260132, 0.23968948423862457, -0.4253436326980591, 0.7957304120063782, 0.6373009085655212, -1.6575281620025635, 0.6850303411483765, 0.704883337020874, 1.2237154245376587, -0.3399159014225006, -0.24270252883434296, -0.7088571190834045, 1.050930380821228, 0.28743287920951843, 0.7263102531433105, -0.01580040529370308, 0.8257396221160889, 0.7145417928695679, 1.2482692003250122, 1.156992793083191, -1.07321298122406, 0.31048476696014404, -0.04321977496147156, -0.8551256656646729, -0.8431098461151123, 0.9489958882331848, 0.1545276790857315, 0.9093021154403687, -0.7294021248817444, 1.3836243152618408, -0.8873167037963867, 0.542067289352417, -0.6921228766441345, 0.5102480053901672, -0.3222011923789978, 0.05159996449947357, -0.2424166202545166, -0.09790067374706268, -3.635979652404785, -0.31172725558280945, -0.33538901805877686, 1.5120155811309814, 1.5567724704742432, -0.297200083732605, -1.090454339981079, 0.24493899941444397, 0.8790926933288574, 1.3399776220321655, 0.5444004535675049, 0.04855216667056084, 1.627663493156433, -2.6157827377319336, 0.5706624984741211, -1.1966639757156372, -0.22784696519374847, -0.9181400537490845, 0.043589916080236435, -0.18972982466220856, -2.0655875205993652, 2.3775038719177246, -0.07440903037786484, -1.2972661256790161, 1.3096327781677246, -0.48151788115501404, -0.8421651721000671, 0.8580347299575806, 2.3462777137756348, 0.0793982446193695, 0.21918903291225433, 0.5852020978927612, 0.8324118852615356, 0.04626678302884102, 1.0997178554534912, 0.642056941986084, -1.5892263650894165, 0.3641984164714813, 1.2551043033599854, 0.111564502120018, 0.2269589602947235, -0.9789666533470154, 1.0851892232894897, -0.20999786257743835, -0.007688167504966259, 1.079216480255127, 0.31962117552757263, -1.1820377111434937, 0.02857673354446888, 0.2699681222438812, -1.9784451723098755, -0.15494482219219208, -1.3597160577774048, 0.7987897992134094, -2.1466312408447266, -0.0251321941614151, 0.27133235335350037, 0.21556349098682404, 2.5288279056549072, 0.36235207319259644, 0.14715884625911713, -0.4109005331993103, 1.3317406177520752, -1.4582513570785522, 0.06936691701412201, 0.6304970383644104, -0.043220680207014084, -0.7307950854301453, -1.0262089967727661, -0.1654251515865326, 0.25847139954566956, -0.7584323883056641, 0.031934335827827454, -0.22852982580661774, -0.16935844719409943, 1.4467264413833618, -0.24529917538166046, -0.3255930244922638, -0.9851142764091492, 0.5579245090484619, 0.904286801815033, 0.31706702709198, -0.5616661906242371, 0.09267356246709824, 1.6921814680099487, 0.9187958836555481, 0.8937054872512817, 0.5302789211273193, 0.18347495794296265, 2.0117788314819336, 0.30597952008247375, -0.40647587180137634, -1.214117407798767, -1.0302395820617676, -1.5651791095733643, 0.1403387039899826, 0.4360390901565552, 1.255416750907898, 0.5889699459075928, -0.9713955521583557, 1.8284865617752075, 0.2125425636768341, 0.32244980335235596, 0.041587699204683304, -0.12732163071632385, -0.22354358434677124, -0.15475597977638245, 1.0092052221298218, -0.7313056588172913, -0.14505243301391602, -0.047902513295412064, -1.5525476932525635, -0.6085938811302185, -1.7463202476501465, 0.3091265559196472, 0.665117084980011, -0.972096860408783, 1.7778550386428833, 0.4972030818462372, -0.24603964388370514, -0.7515352964401245, 1.5243979692459106, -0.0831352174282074, -0.2566775381565094, -0.8480284214019775, 0.20204448699951172, -0.6013410091400146, -0.12714388966560364, -0.7141582369804382, 0.21539156138896942, 0.31219562888145447, 0.7236117720603943, -0.7636702656745911, 2.3430073261260986, -3.0203990936279297, 0.643162190914154, 0.3654218018054962, 0.8312746286392212, -0.9320136904716492, -0.5221703052520752, -1.811182975769043, 1.4353175163269043, -0.4689534306526184, -1.6854496002197266, -2.7655715942382812, -0.31329262256622314, -1.2032982110977173, -1.4171547889709473, 0.07099435478448868, -0.6566516160964966, 0.6847817897796631, -0.04097297042608261, -0.7625723481178284, -0.07064568251371384, 0.4001000225543976, 0.9299691915512085, 0.162095308303833, 0.20001472532749176, -0.7292721271514893, 0.30619367957115173, -1.575653314590454, -0.15494690835475922, 0.22370094060897827, -0.9642778038978577, -1.09567129611969, 1.3980662822723389, -0.37971067428588867, -0.10550697147846222, -0.6635350584983826, -0.26590681076049805, -1.1927446126937866, -1.0728232860565186, -0.986977219581604, 0.0672304555773735, -0.17520879209041595, 1.7650518417358398, 0.036435727030038834, -1.5051312446594238, 0.7407616972923279, 0.0002678480523172766, 1.3932056427001953, 0.3620975911617279, 0.7494853734970093, 0.4404088258743286, 1.022295355796814, -0.43482211232185364, 0.3893442749977112, 0.6174993515014648, -1.695980191230774, 0.26430949568748474, -0.32681846618652344, -2.1589629650115967, -1.8275821208953857, 1.477421522140503, -0.53171306848526, -1.2648042440414429, 0.801551878452301, 0.8567665219306946, -0.5013779401779175, -0.9276146292686462, -1.118847131729126, -0.38329383730888367, -1.1597970724105835, 1.2716699838638306, 0.515579104423523, 1.3639439344406128, -0.3301791548728943, 1.9360663890838623, -0.9513817429542542, -1.918299913406372, 0.20960770547389984, 0.5284480452537537, -0.7340314388275146, -0.5368945598602295, 0.4999282956123352, -0.6108031868934631, -0.787977933883667, 0.509682834148407, -1.2528399229049683, -1.0079591274261475, -0.09500215202569962, 0.21244321763515472, 1.366352915763855, -1.6244704723358154, -0.22120390832424164, -0.20862089097499847, -0.4026421308517456, 0.2585790753364563, -0.041415583342313766, -1.3115135431289673, 0.6940938234329224, -0.808227002620697, 1.1474729776382446, -1.1498998403549194, 0.5678364038467407, -0.5239316821098328, -0.5951014757156372, 1.19132661819458, 0.345375120639801, -0.3847472369670868, 1.2344553470611572, -0.6545172929763794, -0.11412224918603897, 0.5360187292098999, -0.0061203655786812305, -0.35531139373779297, 0.1944085955619812, -0.4348510503768921, -0.48692643642425537, -2.1600582599639893, 1.8466373682022095, -0.26439711451530457, -0.4182625710964203, 1.844856858253479, 0.10173636674880981, -0.2312965840101242, 1.3057632446289062, 1.0534461736679077, -0.8676161766052246, 0.4308728575706482, 1.104541301727295, 0.9595789909362793, 0.6686514616012573, -1.8738961219787598, -0.7979387044906616, -0.22958140075206757, 0.2964116930961609, -0.4951779246330261, -0.8595313429832458, -0.008616993203759193, 2.884861707687378, -0.03671087697148323, 0.6481195688247681, 0.7650135159492493, 0.7735133171081543, -0.07390545308589935, 1.9739885330200195, -1.518143653869629, -0.9867831468582153, 0.19036024808883667, 0.4781574606895447, 0.9657761454582214, 0.4467650055885315, -0.9362844824790955, -0.3987509608268738, -0.21634796261787415, 0.03997253626585007, 0.6882867813110352, -1.0070444345474243, -0.8114350438117981, -0.011153966188430786, -0.8893332481384277, 1.077082872390747, -0.9667632579803467, -0.6302760243415833, 0.15390163660049438, -1.9542616605758667, 2.3727402687072754, -1.3316895961761475, -0.21497483551502228, -0.5966410636901855, -1.2159435749053955, -1.4458638429641724, -0.49900516867637634, 1.0661166906356812, 1.0755096673965454, -0.9524266719818115, 0.09455692023038864, -2.476172685623169, -0.37660372257232666, -0.6054270267486572, -0.6611365079879761, 1.1228727102279663, 0.07883478701114655, -0.016426127403974533, 0.20970991253852844, 0.030779583379626274, 0.6379454731941223, -0.08386329561471939, 0.17979420721530914, -0.7917037010192871, -0.17942780256271362, 0.03724076598882675, -0.7883783578872681, 1.9838781356811523, 0.6599135994911194, -0.049609847366809845, -0.7181717753410339, 0.6401717066764832, 0.016761772334575653, -0.5785699486732483, -0.9051294922828674, -0.17795461416244507, 1.9729243516921997, -1.266807198524475, 0.17688508331775665, 0.9930357933044434, 0.45667916536331177, 0.7065322995185852, 1.4075554609298706, 0.2309424728155136, -0.18409404158592224, 0.11296728253364563, 1.6015651226043701, 0.6485881209373474, 0.15862111747264862, -1.397217035293579, 0.5376701354980469, 0.11512569338083267, 0.25215989351272583, -0.16838817298412323, 0.43945202231407166, 0.40706294775009155, -0.15243040025234222, 0.41570329666137695, -0.7859949469566345, -0.7020151019096375, 0.055083226412534714, -0.11686769872903824, -0.35428768396377563, 1.4613268375396729, 0.4476563632488251, 0.5310111045837402, -0.6385650634765625, -0.8790587782859802, 2.881258249282837, -0.6324810981750488, -1.2796639204025269, -0.4604792296886444, 0.7555500864982605, 1.7447506189346313, 1.3142180442810059, 0.3219481110572815, -2.1455698013305664, -0.8962457776069641, -0.30201634764671326, -0.13756339251995087, -0.4508652091026306, 1.590047001838684, 0.7203897833824158, 0.4751594066619873, 0.7532845735549927, 0.3691869378089905, -1.2362290620803833, -1.2702491283416748, 0.7000258564949036, 0.29226747155189514, 0.32618534564971924, 1.1144944429397583, -0.7157509922981262, -0.8299599289894104, 0.2847777009010315, 0.7292436957359314, 0.21022237837314606, -0.7852243781089783, -0.9205405116081238, 1.5362458229064941, 1.066415548324585, -0.2207316756248474, 0.578027606010437, -1.5484744310379028, -0.4773135483264923, -0.5945985913276672, -0.3516724705696106, 0.8569364547729492, -0.3867272734642029, 1.3125945329666138, -2.1881256103515625, -2.241708278656006, 1.1588337421417236, 0.7942612171173096, -0.45221078395843506, 2.9687256813049316, -0.5241288542747498, 0.36679887771606445, -1.1872645616531372, -1.0672215223312378, 0.0632353276014328, -0.3510282039642334, 1.1928859949111938, -0.12662550806999207, 0.30198973417282104, 0.11562146246433258, 1.442215919494629, 1.1449719667434692, 1.506108283996582, -0.18884481489658356, -1.3058325052261353, 1.0903568267822266, -1.710155725479126, -0.05963096767663956, -0.8905760049819946, -0.5416309833526611, -0.290608674287796, -0.23397204279899597, 0.1263146698474884, 0.8190229535102844, 0.2792523503303528, -0.04250601679086685, 0.011806548573076725, -1.171470284461975, -0.44337230920791626, 1.347272276878357, 0.9716435074806213, -1.5561416149139404, -1.2366647720336914, -0.44842860102653503, -0.009495762176811695, -0.6560651659965515, -0.14406180381774902, -0.24616584181785583, -1.3582265377044678, 0.08476575464010239, -0.42227527499198914, -1.1142246723175049, 0.6735072135925293, -0.1814485788345337, -0.03596661612391472, -0.5792794227600098, 0.7624384164810181, -0.5884326696395874, -0.5540897250175476, 0.1947241872549057, -0.5671886801719666, -0.5789904594421387, 0.311024934053421, 0.32486453652381897, 0.11402879655361176, 1.1396338939666748, -1.4010282754898071, -0.8184146881103516, -1.1212455034255981, -1.9619214534759521, -0.6175872087478638, -0.5885635018348694, -1.133472204208374, 1.2218265533447266, -0.779951810836792, 0.47212356328964233, -0.747626543045044, -0.7511493563652039, 0.22559329867362976, 0.5885072350502014, 1.3483352661132812, 0.4744530916213989, -0.9742071032524109, 0.10094738006591797, 1.208253264427185, -0.5669984817504883, 0.9593216180801392, 0.513510525226593, -0.003326443023979664, 0.9736272692680359, -0.5268038511276245, 0.856902539730072, 0.48116928339004517, 0.3698679804801941, 0.6608101725578308, -0.03775116801261902, 1.5668257474899292, 2.59820818901062, 1.5773382186889648, -0.9401339888572693, -1.0829529762268066, 0.18612775206565857, -0.8680365085601807, 0.22269241511821747, 0.3469025194644928, -0.30494529008865356, -0.544960081577301, -0.3145875632762909, -0.7113867998123169, -0.8253321647644043, 0.9688743948936462, -0.928411066532135, -0.6061386466026306, 0.06949662417173386, 0.9530140161514282, 0.8303685784339905, 1.3656680583953857, -0.4256393611431122, 0.6095540523529053, 0.785727322101593, 0.043032873421907425, 1.5239086151123047, -0.9396125674247742, 0.5750485062599182, -0.2573421597480774, -0.2983167767524719, -0.6456636190414429, 1.1896767616271973, -0.2518823444843292, -0.05641418322920799, -0.3427625596523285, 0.848481297492981, -1.7839601039886475, -0.2517310380935669, 0.4883098006248474, -0.06738303601741791, -0.6575825214385986, -0.9059109091758728, 0.7563784122467041, -1.4916878938674927, -0.1765558123588562, -1.4716116189956665, -0.49736684560775757, 0.40807321667671204, 0.16292613744735718, -0.976984441280365, -0.12129084020853043, 0.32119807600975037, 0.0662667527794838, 0.35276639461517334, -0.26866963505744934, 0.6500458717346191, -0.3760552406311035, -0.936575710773468, -0.47394582629203796, 0.6179261803627014, -1.1503163576126099, -0.9452036023139954, -0.1709655374288559, 0.4178001284599304, 0.716456413269043, -1.558434009552002, -1.9234979152679443, 0.11331237107515335, 0.26229438185691833, -0.3560790419578552, 1.969923496246338, -3.544626235961914, -0.05472631752490997, 2.0486068725585938, 0.5997329950332642, 0.37834349274635315, 0.026978349313139915, 0.11608780175447464, 0.43781381845474243, 0.016067180782556534, -1.0862330198287964, 0.2641189694404602, -0.05741394683718681, 0.22465820610523224, 1.2390156984329224, -0.5388384461402893, -1.2617353200912476, -0.6937210559844971, 0.8309839367866516, -1.4077801704406738, 0.22303113341331482, 1.1046361923217773, 0.43708306550979614, 0.9196650385856628, -0.3134000301361084, 0.45723605155944824, 0.48376861214637756, -1.0417747497558594, -0.6707676649093628, -0.4885745048522949, -0.5936102271080017, -0.14423926174640656, 0.35654574632644653, 0.6407240629196167, -0.1683533489704132, 0.730009913444519, -1.7442744970321655, 1.594322681427002, -0.29655611515045166, 1.3069909811019897, 0.5619632005691528, -0.42258837819099426, 0.8022511005401611, 0.25381046533584595, 0.8774797916412354, 2.2069404125213623, -0.8789482712745667, -0.44923314452171326, -1.1389291286468506, 0.6933778524398804, -1.4636285305023193, -0.35210785269737244, -0.3309536278247833, -1.9065712690353394, 0.30287671089172363, 1.943698763847351, 1.3739652633666992, -1.2121493816375732, 0.05581168457865715, -1.8958479166030884, -0.6974084973335266, -0.5103871822357178, -0.9388452768325806, -0.37743037939071655, 1.1623337268829346, -1.2668588161468506, -0.9443643689155579, -0.7956732511520386, -0.5256133675575256, 1.142842411994934, -1.4586764574050903, 0.6383507251739502, 0.8348177075386047, 1.137266755104065, -3.2230474948883057, -0.7979417443275452, -0.6648803949356079, -0.049717869609594345, -0.5992863774299622, -0.014384748414158821, -0.4694920778274536, -0.6276771426200867, 1.5483351945877075, -0.47395890951156616, 0.5525376796722412, -1.1224920749664307, -0.024855365976691246, 0.6735324859619141, -0.46003052592277527, 0.6308733224868774, -0.7103488445281982, 0.16843508183956146, -1.0530359745025635, -2.0461394786834717, -0.788442850112915, 1.9369184970855713, -0.1158548891544342, 0.28362971544265747, 1.2911595106124878, -0.020690107718110085, -0.2589307427406311, -0.811339795589447, 0.37573686242103577, 0.21627800166606903, -0.4544565975666046, -0.8172018527984619, 0.5807042121887207, -0.6432366371154785, -2.1568944454193115, 0.8393545746803284, -2.182745933532715, -0.466484934091568, 0.4943245053291321, -1.3526506423950195, 0.2708428204059601, -0.5405209064483643, 1.054618239402771, 1.68352210521698, 1.7022531032562256, 0.8825649619102478, -0.04883032292127609, -0.34609800577163696, -0.6566052436828613, -2.8860349655151367, -1.9179904460906982, -2.0732312202453613, 0.13278189301490784, 1.22693932056427, -1.1368417739868164, 0.4828025996685028, 0.008246758952736855, -0.4503660202026367, 0.18041598796844482, 0.07121898233890533, 0.9770748615264893, 0.7359277009963989, 0.2362183928489685, -0.7901506423950195, 1.1029562950134277, 0.2666805684566498, -1.6621019840240479, 0.3868040144443512, -0.2817554175853729, 1.649679183959961, 1.9894378185272217, 0.08293108642101288, -0.4251766502857208, 0.6111185550689697, 0.38941511511802673, -0.7140770554542542, -0.008412811905145645, 0.12464282661676407, -0.27680107951164246, 0.409760981798172, -0.7188069820404053, 0.7436686754226685, 1.5959727764129639, 0.5498817563056946, 0.5123971104621887, 1.5493026971817017, -1.2723751068115234, 0.2187122255563736, 0.5608343482017517, 0.10392691195011139, 1.0500696897506714, -0.23333585262298584, -0.07590538263320923, 0.5165153741836548, 2.350496292114258, -1.608527421951294, -0.4357113242149353, 0.5317275524139404, -0.6121248006820679, 0.022296350449323654, -0.43697404861450195, 0.3910118341445923, 0.8713048696517944, 0.31437310576438904, 0.865812361240387, 0.16802750527858734, 0.7842757701873779, 0.6240136027336121, 0.13093388080596924, 0.21065428853034973, -1.635752558708191, 1.4061001539230347, 0.14581623673439026, -0.8655638098716736, 1.0170094966888428, 1.4809842109680176, -0.7025901079177856, 1.7957969903945923, 0.26752138137817383, 0.35922038555145264, 2.328995704650879, 1.2876285314559937, -0.6299574375152588, -0.1450234204530716, 1.0672446489334106, -0.24993140995502472, 0.9872216582298279, 0.0845269039273262, 0.6043787598609924, 0.8250135779380798, -1.0496571063995361, 2.0995407104492188, -1.717862606048584, 1.16368567943573, 0.6505601406097412, 1.708299994468689, 0.2107183188199997, -0.05485017970204353, -0.5275579690933228, 1.049821138381958, 0.361613005399704, -1.52788245677948, 0.9937394261360168, -0.48019397258758545, 0.5947405695915222, 0.3527531623840332, 0.8114743828773499, -0.6817756295204163, -0.4029329717159271, 1.4421590566635132, -1.2595213651657104, 0.6529570817947388, 1.315295934677124, -1.203762412071228, 1.070177435874939, -1.5765243768692017, -0.15711471438407898, -0.500515341758728, -0.5751904845237732, 1.0421960353851318, -1.9348899126052856, -0.6456044912338257, -0.0009968457743525505, 0.23984913527965546, -0.1075940877199173, -0.15174318850040436, -0.4747638404369354, -1.5236743688583374, 1.449308156967163, 0.36740946769714355, -2.5816831588745117, -0.0898008644580841, -0.06936024129390717, -0.06065061688423157, 0.020688379183411598, 0.08554951846599579, 1.1825482845306396, -0.01320725493133068, 2.085050106048584, -0.7787649035453796, -0.5774646997451782, -0.12895016372203827, 0.24995635449886322, -2.114187717437744, -0.5165248513221741, 0.07524750381708145, 0.04056364670395851, 0.5613576173782349, 1.5686019659042358, 0.5603671669960022, -0.035749003291130066, -0.8039867877960205, 2.1494622230529785, -0.916946530342102, -0.004386776126921177, -1.0846985578536987, 0.3475196063518524, -0.691965639591217, -0.9893920421600342, 0.746949315071106, 0.44346463680267334, 0.8619040250778198, -0.6444557309150696, 0.18827691674232483, -1.188631296157837, -2.800330877304077, -1.345633864402771, -0.8529878258705139, -1.6267437934875488, -0.48121732473373413, -1.3360531330108643, -0.27907678484916687, 0.9170916080474854, 1.019656777381897, 0.7344772815704346, 0.930533230304718, 1.2223551273345947, -1.0321518182754517, 0.12670612335205078, 0.4192850887775421, 1.3959455490112305, 2.0679562091827393, -0.6054084300994873, -0.4105089604854584, 0.4225146770477295, -1.4030873775482178, -0.7175387144088745, 0.7434641718864441, 0.9842773675918579, -0.26941725611686707, -0.9546786546707153, -1.3096765279769897, 0.9426047801971436, 1.6438190937042236, -0.6984351277351379, 0.18119145929813385, -0.012134646996855736, 0.22921057045459747, 0.057768020778894424, -0.08936791121959686, 0.3751940131187439, -0.5116539597511292, 0.2567017376422882, 0.37577924132347107, 1.8111554384231567, 1.8309723138809204, -0.07037626206874847, -0.5128790140151978, 0.3984209895133972, -0.580223023891449, -0.2186897248029709, 0.36321237683296204, 0.5755037069320679, 0.7451601624488831, 1.2839990854263306, -0.7575969696044922, -1.2979334592819214, 0.016044840216636658, 0.4701933264732361, 1.1469807624816895, -0.400526762008667, -0.8530271053314209, -0.9802868366241455, 0.14309009909629822, -1.0068117380142212, -0.4475257694721222, -0.649125337600708, 1.6347968578338623, 0.6702370047569275, -0.7456845045089722, 1.2674643993377686, 0.7814781069755554, -0.9025485515594482, 1.0568276643753052, -1.2001160383224487, -0.177345409989357, 0.9060616493225098, 0.07469571381807327, -0.011597856879234314, -0.4513586163520813, -0.4696330428123474, 0.1554683893918991, -2.38155198097229, 0.7063168287277222, 2.032177209854126, 0.8368192315101624, -1.3304948806762695, 0.6470208764076233, -0.8992756605148315, -0.6493910551071167, -0.06740645319223404, 0.9117671251296997, 0.7012119889259338, 0.2950532138347626, -0.6541872620582581, 0.4352925717830658, 1.3271152973175049, 0.7358352541923523, -0.004278330598026514, 0.5095322132110596, 0.0032799160107970238, -0.059887245297431946, -0.8696159720420837, -1.801820158958435, -2.3318490982055664, 1.139427900314331, 0.7107806205749512, 0.9595209360122681, -1.3838012218475342, 1.3236924409866333, -0.8225927352905273, -1.4990955591201782, 0.41577327251434326, -0.5297895073890686, 0.6600928902626038, -0.4921995997428894, 0.1672784835100174, 0.9929064512252808, -0.07918345183134079, 0.1753709614276886, -1.4327138662338257, 0.9295056462287903, -2.0974721908569336, -0.22572112083435059, 1.0648034811019897, -1.1850615739822388, 0.7169690728187561, 1.0637614727020264, -3.410465955734253, -0.4208851158618927, -0.15086746215820312, -0.6856571435928345, -1.3010867834091187, -0.8381485342979431, -1.051519751548767, 0.589644730091095, -1.6046205759048462, -0.41236621141433716, -1.214118480682373, -2.010620594024658, 0.7878071069717407, 2.029700517654419, 0.7739740610122681, 0.05720460042357445, -0.971328854560852, 0.14710091054439545, 0.32838863134384155, 0.2045457512140274, -2.113551378250122, 0.7872382998466492, 0.031185436993837357, 1.1473628282546997, 0.12460603564977646, 1.0210576057434082, -1.8503206968307495, -0.08828670531511307, 0.8224467039108276, 0.6423012018203735, 0.16051602363586426, -2.1788330078125, -0.32213032245635986, -0.16539959609508514, -1.2389975786209106, -0.10067438334226608, -1.1922239065170288, 1.145036220550537, -0.921622097492218, 1.7161221504211426, -1.7234525680541992, -0.06409227848052979, -1.1705505847930908, -0.03856546804308891, 0.5500549077987671, 1.6615699529647827, 1.0328950881958008, -0.718625009059906, -0.22522613406181335, -0.15189598500728607, 0.4429694712162018, 0.5036677122116089, 1.3443547487258911, 1.9329833984375, -0.42049044370651245, -0.722847044467926, 0.8090048432350159, 0.2807507812976837, 0.9871594309806824, -0.03797600790858269, -0.3435569405555725, -1.6322603225708008, 0.5365217924118042, -1.537100076675415, -0.23803049325942993, 0.6035187244415283, 0.08042387664318085, 0.35926660895347595, 0.19571177661418915, 0.580134391784668, -1.113867998123169, 0.5418685674667358, -0.4847507178783417, -2.070335865020752, 0.1408691257238388, 0.7778407335281372, 1.7135998010635376, 0.5395567417144775, 0.19964557886123657, -0.47692686319351196, -1.0854970216751099, -0.3570585250854492, -0.6473658680915833, -0.032053641974925995, 0.9617687463760376, -0.481163889169693, 0.6243658065795898, 1.2697876691818237, -0.15409989655017853, -0.19494038820266724, 0.9790448546409607, -0.9818094968795776, 0.28352612257003784, 1.7019405364990234, 0.26371991634368896, -2.248173952102661, -0.15871649980545044, -0.33107420802116394, 0.8349628448486328, 1.1655526161193848, -1.148234486579895, -0.9762342572212219, 0.6384503841400146, -0.2902930974960327, -0.22261971235275269, -0.3744509518146515, 0.1881781667470932, 0.8122258186340332, 0.9087969660758972, -0.5620021224021912, 3.853172779083252, 2.3106677532196045, -1.246610164642334, -0.9538910984992981, -0.9409288167953491, -0.4803691506385803, -0.1312805712223053, -0.5242692828178406, -0.10493119806051254, -0.1178400069475174, -1.9832783937454224, -0.5560013651847839, 0.16465729475021362, 0.10722644627094269, -0.4290260970592499, -0.6161109805107117, -0.9817923903465271, -0.6169371008872986, -0.8141735196113586, 1.371262788772583, -2.205240488052368, 0.3341478407382965, 1.593806266784668, 1.0228545665740967, -0.04339814558625221, 0.37981483340263367, 1.0802128314971924, 1.6285954713821411, -0.7541069984436035, 1.2065415382385254, 0.3023412525653839, 0.024543605744838715, 1.2406409978866577, 0.03716195747256279, -1.7704682350158691, 0.3567439615726471, -0.5963974595069885, -0.6514174938201904, -0.06989067792892456, -1.374620795249939, -0.5604099035263062, -0.7417452931404114, -0.26631656289100647, 0.35860350728034973, -0.48633402585983276, -0.2795662581920624, 1.7581872940063477, -1.1539462804794312, -2.0947043895721436, -0.7834043502807617, -0.5666383504867554, -0.4877033531665802, 0.6301906704902649, 0.0473514124751091, -1.7402760982513428, 0.8092696070671082, 0.9653051495552063, 0.21756203472614288, 1.2688201665878296, -0.6845306754112244, -1.2148444652557373, -0.649613082408905, 1.201717734336853, -0.2174227386713028, 0.5637305974960327, -0.6851748824119568, 1.3250858783721924, -1.0978245735168457, 0.07751087844371796, -0.22483742237091064, -0.18811236321926117, 0.3187953531742096, 0.38056427240371704, -0.7776209115982056, -0.5462185144424438, 0.7900540828704834, 0.2839656174182892, 0.22923938930034637, -0.30406734347343445, -0.6441850662231445, -0.800622820854187, -0.4183030128479004, 0.8825048804283142, 0.2244337946176529, 0.2539399564266205, -1.9839292764663696, 1.2752530574798584, -1.039567470550537, 0.18085910379886627, 0.5341216921806335, -0.9626757502555847, 0.8779507875442505, -0.2584335207939148, 0.5668654441833496, -2.39829158782959, 2.504359722137451, -0.21056245267391205, 0.39839839935302734, 0.023283900693058968, 0.051016706973314285, -1.5412243604660034, -1.4107508659362793, 0.2827094495296478, -1.1623121500015259, 0.46017563343048096, 2.0124266147613525, -0.3223891258239746, -0.8348045349121094, -1.8913742303848267, -1.3156400918960571, -0.03338393568992615, -1.6144474744796753, -0.5029605031013489, 1.6322507858276367, -0.16100172698497772, -0.27645352482795715, -0.7857736349105835, 0.23483122885227203, 0.23355473577976227, 0.9945840835571289, 0.3710889220237732, 0.22919729351997375, -0.991595447063446, -0.6034455895423889, -0.2004406750202179, -0.09125816076993942, -0.46240267157554626, 0.5055311918258667, -1.3535493612289429, -1.3602674007415771, -0.6209896206855774, -0.13528986275196075, -0.12042803317308426, -0.1908620297908783, 0.7385768294334412, -0.6684699058532715, -0.8343085646629333, 0.24563685059547424, -0.5884382724761963, -1.3413417339324951, -0.7107842564582825, 1.0618425607681274, -0.295251727104187, 0.11979968100786209, 0.6847839951515198, -2.2744140625, -0.29721298813819885, -0.05065252631902695, -1.4549825191497803, -2.1119260787963867, -1.1517319679260254, 0.14732353389263153, 1.6662708520889282, -1.2952752113342285, 2.084810972213745, 1.2532365322113037, -0.6898733973503113, 0.6676188707351685, 0.964176595211029, 1.6443371772766113, -0.6740845441818237, -1.5011582374572754, 0.4780416786670685, -0.13967376947402954, -0.8152077198028564, -0.09907028079032898, -0.27907276153564453, 1.6774884462356567, -0.3662697970867157, 0.3056519627571106, 0.4455406367778778, 1.1703431606292725, -0.536393940448761, 0.36765146255493164, 1.4830996990203857, -0.9238807559013367, -0.2011190950870514, -1.0217396020889282, 0.10339321196079254, -0.8494876027107239, -1.5865976810455322, -0.7277357578277588, 0.4194428622722626, 0.4052654802799225, -0.40352529287338257, 1.178613543510437, 2.3068857192993164, -0.09106718003749847, -0.985657811164856, 0.3554609417915344, 0.729077935218811, -0.2641288638114929, -0.5669454336166382, -0.08634907752275467, -0.7396325469017029, -0.3940966725349426, 1.0016769170761108, 0.8731083273887634, -0.9225242733955383, -0.1906970590353012, -1.2247722148895264, -0.6289511919021606, 1.5942621231079102, -0.9490097761154175, 0.8101673126220703, 1.165317177772522, -0.9318199753761292, 0.5637364983558655, 0.8445448875427246, 0.37145668268203735, -1.4827810525894165, 0.1200145035982132, 1.1999375820159912, 0.9906467795372009, -0.7892269492149353, 0.5168681144714355, 0.6154690384864807, -1.0152031183242798, -0.7615413665771484, -0.06890857964754105, 0.07625231891870499, 0.4517275393009186, 0.7264356017112732, 1.138973355293274, 0.15640662610530853, 2.9214015007019043, -1.0287309885025024, 1.226204514503479, 1.5460470914840698, 1.2397539615631104, -1.90122652053833, 0.26421523094177246, 0.4880335330963135, -1.0623137950897217, -1.1713720560073853, 0.31168103218078613, 0.2735522985458374, -0.7268894910812378, 0.6430729031562805, -0.5606750845909119, -0.5782909393310547, -0.2000425010919571, 1.003242015838623, -1.6615091562271118, -0.42632925510406494, 0.3662688434123993, 2.3943610191345215, 0.1357438713312149, 0.24943877756595612, 1.0426955223083496, -0.7585471868515015, -0.22962132096290588, 0.10349065065383911, 0.5984975695610046, 1.1073346138000488, -1.0425984859466553, -1.6522221565246582, -0.10873185098171234, 0.6281079053878784, -0.7295113205909729, 0.16284476220607758, 0.07756302505731583, -0.30673420429229736, 1.105756163597107, -0.5558783411979675, -0.014653084799647331, -2.098027467727661, -0.49281007051467896, 1.2450257539749146, 0.37470877170562744, -0.4922417998313904, -0.23932644724845886, -0.5008963942527771, 0.586027204990387, -1.0374479293823242, 0.8122448325157166, 0.16304269433021545, -0.19188624620437622, 1.856473445892334, -0.2717841565608978, 0.3501591384410858, -0.8772896528244019, 0.6677753329277039, -1.105391502380371, 0.45225223898887634, 0.8088122606277466, 1.4615015983581543, -1.3000377416610718, -0.0250470582395792, -2.6570725440979004, -0.740199089050293, 0.9615038633346558, -0.9394940733909607, -1.255120038986206, -1.1840651035308838, 0.8005383610725403, -0.5128953456878662, 0.6604140400886536, -1.3773552179336548, -1.9962761402130127, -0.3108553886413574, -0.24812567234039307, 0.5784192681312561, 1.4715763330459595, 0.5566285252571106, 0.22743099927902222, -0.3722812831401825, -1.9983359575271606, 1.9615644216537476, 0.33934128284454346, 0.5078783631324768, -0.30200040340423584, 0.27422332763671875, -0.33503249287605286, 1.574392557144165, -0.6154419779777527, -0.5766595005989075, -2.0947487354278564, 0.7919525504112244, 0.8572530746459961, -0.6734119653701782, -0.6393594741821289, 0.2252281904220581, 0.1590525358915329, -0.21995504200458527, -1.1638973951339722, -1.3583524227142334, -0.07592929899692535, -0.9633423089981079, 1.122525930404663, -1.0554817914962769, -0.05355304479598999, 0.8976271748542786, -0.34395354986190796, 2.247817277908325, 0.9804812073707581, -1.882352352142334, -1.1005381345748901, -1.077446699142456, -0.5467404723167419, -0.3603664040565491, -0.969728946685791, 0.1763858199119568, -1.0512876510620117, -1.544321060180664, -0.9182855486869812, 1.5043858289718628, -1.2141497135162354, -1.4485199451446533, -1.3317182064056396, 0.7254724502563477, 2.7228200435638428, 1.526543378829956, -0.6290730834007263, 2.0415830612182617, -0.4348674714565277, 1.6711058616638184, 1.1253763437271118, 1.2869242429733276, 0.4808858036994934, -0.6445958018302917, -1.0656501054763794, -0.39590299129486084, 0.6596335768699646, -1.1193366050720215, 0.6502612829208374, 0.30475831031799316, 0.4099486768245697, 0.769751250743866, -2.095573663711548, 1.8823370933532715, -0.2938609719276428, -0.6910684108734131, -0.13498905301094055, -0.3057962656021118, 0.28750038146972656, -0.9682759642601013, 0.0720156654715538, 1.0049090385437012, 0.060195259749889374, -0.7413488626480103, -1.237535834312439, 0.0952879935503006, -0.2522079646587372, 0.5523615479469299, -1.2122323513031006, 0.7226475477218628, -0.27600768208503723, -0.1903638243675232, 0.3592756390571594, -0.9801970720291138, -0.43361619114875793, 1.1434940099716187, 0.9141033291816711, -2.347257614135742, 3.5067434310913086, 0.3881900906562805, 1.0845584869384766, 0.5020681023597717, 0.38381320238113403, 0.207609161734581, 0.6183347105979919, 0.2604966461658478, 0.9840909838676453, 0.496927946805954, -1.1561222076416016, 2.0599021911621094, -0.0293127354234457, -0.7457401752471924, 1.0342782735824585, -0.9689308404922485, -0.5599164962768555, -0.7345978021621704, 1.2612024545669556, 1.0614361763000488, 0.6810181736946106, 0.8286541104316711, -0.6221224069595337, 2.146491050720215, -2.52006196975708, 1.2821887731552124, 0.18460291624069214, 0.25944045186042786, -0.7441421747207642, -0.3226240277290344, 0.3413853347301483, -0.6206790804862976, -0.48073333501815796, -0.7437593340873718, 1.0900294780731201, 0.14510180056095123, -0.7550498843193054, -1.1816154718399048, 0.7706760168075562, -1.0248472690582275, -0.2657201290130615, 0.17313794791698456, -1.3874064683914185, -0.1382233053445816, -0.5382676124572754, -0.46138492226600647, -0.10746919363737106, 0.8058739304542542, 0.23391997814178467, -3.004061698913574, -1.909010648727417, -1.3282946348190308, -0.6294066905975342, 0.9325178265571594, 0.6149961352348328, 0.2133631706237793, -1.1482961177825928, -1.0782884359359741, 1.1239242553710938, 1.4414349794387817, -1.4791407585144043, -2.5610172748565674, 0.7666175365447998, -0.816606342792511, -0.8897199034690857, -0.38822948932647705, 1.3194818496704102, -0.15267768502235413, 1.1156740188598633, 1.0784951448440552, -0.1807054728269577, 1.2331883907318115, -0.04592551290988922, 0.9417546391487122, -2.073843240737915, -1.1266945600509644, 0.5114506483078003, -0.1633533090353012, 0.006856574211269617, 0.9166994690895081, -0.27837124466896057, -0.687818706035614, -0.7588691115379333, -0.09648280590772629, 0.6152974367141724, 0.3944930136203766, 0.5003985166549683, 0.9987335801124573, 0.18778999149799347, 0.8239503502845764, -0.3254714906215668, -2.0288262367248535, -0.8335221409797668, 0.9953257441520691, -1.4573347568511963, 0.719179630279541, -0.09414570778608322, -1.0088739395141602, -0.006362996995449066, -1.812350869178772, -0.6563689112663269, 0.20682716369628906, -0.2139747142791748, 0.4640401303768158, -0.7461966276168823, 0.6183528304100037, 0.9765370488166809, 0.5414838194847107, 0.635646402835846, 1.4958425760269165, -1.413179636001587, 1.2426961660385132, -0.45839419960975647, 0.6995235085487366, -1.0618191957473755, -0.7127797603607178, -0.47062015533447266, 0.04246991127729416, 0.5586544871330261, 0.4513170123100281, -0.1529710739850998, -0.9317210912704468, -0.9463033676147461, -0.3313369154930115, 0.4092901945114136, 0.7980374693870544, 1.3570822477340698, -1.8545992374420166, 1.7019970417022705, 0.1827053725719452, -0.3494410812854767, 1.0013400316238403, -0.5977384448051453, -0.05253629386425018, -0.398082971572876, 0.3011479079723358, -0.7717856168746948, -0.025168390944600105, -0.47725385427474976, -1.0113401412963867, 1.0663397312164307, 0.21641843020915985, -0.2472197562456131, 0.5580289959907532, -0.18866343796253204, -0.7014133334159851, -0.6039889454841614, 0.6785123348236084, 0.46743619441986084, -0.3742365837097168, 0.5982981324195862, -1.4451900720596313, -1.3835461139678955, 0.2428474873304367, -1.0210179090499878, 1.9066082239151, -1.035483479499817, 0.7608011364936829, 0.30748251080513, 0.8555040955543518, 0.8259621262550354, 0.8305763006210327, 1.2855143547058105, 0.472220242023468, 0.6342541575431824, -0.019378677010536194, 0.6717411875724792, -0.2769903838634491, -0.6958063840866089, -2.06679630279541, 0.6529546976089478, -0.2857981324195862, -0.07128563523292542, -0.9607613682746887, 0.25770965218544006, 0.6841186285018921, -0.0392208993434906, 0.42759788036346436, -0.7307628393173218, 0.164819598197937, -0.9797143340110779, -0.05878709256649017, 0.5142172574996948, -0.40407615900039673, -1.1679922342300415, 0.7447258830070496, -0.12386764585971832, 0.13818062841892242, -1.7146942615509033, -1.5612971782684326, 0.6139822602272034, -0.08591101318597794, 0.8536800742149353, 0.6849298477172852, -0.41466397047042847, 1.1314384937286377, -0.41640836000442505, -1.3435194492340088, 0.310538113117218, 0.5326902270317078, -1.4931068420410156, -0.19287075102329254, 1.7109578847885132, 1.170605182647705, -0.11826658993959427, -0.6285033226013184, 1.3605276346206665, 0.9146089553833008, 2.284961700439453, -1.8981972932815552, -0.0935775637626648, 0.8137035369873047, 0.06992308050394058, -0.008828259073197842, -0.8115270733833313, 0.21151742339134216, 0.108565554022789, 0.6003014445304871, -0.6921930313110352, 1.8396844863891602, 1.0705019235610962, 3.06632661819458, 0.1412135511636734, 0.42674389481544495, 0.6613987684249878, 0.37908777594566345, 0.14295338094234467, -0.2366369217634201, -1.1242963075637817, 0.5054648518562317, -1.0313737392425537, -0.9475265145301819, -1.2876192331314087, -0.11882930248975754, 0.11876779049634933, -0.8530666828155518, 0.9331003427505493, -2.539532423019409, 0.8797113299369812, -0.5515000224113464, 0.12439987808465958, 0.9935114979743958, 1.402933120727539, 1.1269198656082153, 1.024143099784851, -0.9479305744171143, -0.2562256455421448, 1.7879784107208252, -0.7710707783699036, -0.3276677131652832, 1.8923290967941284, 1.3961485624313354, 0.23500464856624603, 0.19838480651378632, 0.827409029006958, 1.1520359516143799, 0.4531618356704712, -0.3944395184516907, -1.2186925411224365, 0.28231924772262573, 1.290049433708191, -1.8410472869873047, -1.0343255996704102, 1.116537094116211, -0.9354957342147827, 0.2999037802219391, 0.5582346320152283, 0.3278636634349823, 0.36286887526512146, 0.031422339379787445, -0.23009131848812103, -0.7009581327438354, -0.45204296708106995, -1.062017560005188, -0.038609664887189865, -0.6067865490913391, -0.11624518036842346, 0.9941642880439758, 0.019797027111053467, 0.47959190607070923, -0.9822600483894348, -0.3252769708633423, 0.019161222502589226, 0.5836348533630371, 1.5392684936523438, 1.3476651906967163, -1.2298072576522827, -0.7032217383384705, -1.7032901048660278, -0.04824461787939072, -1.847801685333252, 0.431654691696167, 0.009145720861852169, 0.40199458599090576, -0.16711543500423431, -0.5355218052864075, -1.1876087188720703, -1.0381602048873901, 0.22324927151203156, 0.10840269178152084, -1.2173174619674683, -0.2720167934894562, -0.8839535713195801, -0.12912338972091675, 0.5416832566261292, 0.35634031891822815, -0.4085986614227295, 0.41052404046058655, -0.37079712748527527, -0.8549247980117798, 0.5493815541267395, -0.7776013612747192, -1.2050814628601074, -0.6466790437698364, 0.5658702850341797, 0.32980990409851074, 1.4002230167388916, -0.28806567192077637, 0.4950854480266571, 0.8644273281097412, 0.8270697593688965, -1.6425552368164062, 1.588901162147522, -1.1588375568389893, -0.90595543384552, 0.9716288447380066, -0.20160320401191711, 0.6488339900970459, 0.26567280292510986, 1.6894173622131348, -0.624500572681427, -0.7492074370384216, -0.3856031000614166, 0.6103528738021851, 0.35661807656288147, 0.449013352394104, -1.737679362297058, -0.924990177154541, -1.4376577138900757, 1.3761528730392456, 0.8726770877838135, -0.0036830417811870575, 0.4090684652328491, 0.8353579640388489, 0.09313803911209106, -1.9403672218322754, -1.1052792072296143, -1.8813413381576538, 0.3888351023197174, -1.2468900680541992, -2.3308215141296387, 0.27191463112831116, 2.047950267791748, 0.3155151307582855, -0.5940134525299072, 1.285852074623108, -1.555725336074829, 0.5256975293159485, 1.6540827751159668, 0.16073289513587952, -0.6502415537834167, 1.4949437379837036, -0.8801847696304321, 0.48531240224838257, 0.19908416271209717, 0.15415756404399872, -0.036789968609809875, -0.5935851335525513, 0.3178638219833374, -1.8512133359909058, 1.368367314338684, -0.5907061696052551, -1.3586369752883911, -0.017277047038078308, 0.23708698153495789, -0.3161534070968628, 0.9095339775085449, 0.045629095286130905, 1.9309680461883545, 0.2147509604692459, -0.7245169878005981, 0.24610324203968048, 0.028874684125185013, 0.10749823600053787, 0.5932549834251404, -1.5643359422683716, 1.254709005355835, -1.1470332145690918, 0.14758887887001038, -0.45838606357574463, -0.21859677135944366, -0.7305161952972412, 0.4049435555934906, 0.24680300056934357, -1.7142010927200317, 0.6630645394325256, -0.3527636229991913, 0.6312040090560913, -0.2365436851978302, 1.361175775527954, -0.7166818976402283, 0.3205086290836334, -1.0795648097991943, 0.7903712391853333, -0.1536477506160736, -0.24645106494426727, -1.4542577266693115, 0.38832369446754456, 0.4763713479042053, 0.25863924622535706, 1.1267478466033936, -0.4947706162929535, -2.265321969985962, -0.652883768081665, 0.43086379766464233, -0.6277335286140442, -0.38831743597984314, -0.24111521244049072, -1.3705366849899292, 0.6884472966194153, -0.40561047196388245, -1.7368571758270264, 0.9332265853881836, 1.3559235334396362, -0.00589227257296443, 0.2680352032184601, -0.9457000494003296, -0.4585567116737366, 0.2899310290813446, -0.8953811526298523, 2.012998580932617, -0.28583988547325134, -0.8485521674156189, -1.2153104543685913, 1.2409573793411255, -1.2097551822662354, -1.2315077781677246, -0.6448712348937988, 0.5255532264709473, 1.5894957780838013, 0.07524450123310089, 1.8262033462524414, 1.8664631843566895, 1.438368320465088, -0.3730855882167816, 0.43213123083114624, 0.4988994002342224, 0.01451244205236435, -0.6719830632209778, 0.7222238779067993, -0.5427094101905823, 0.17652280628681183, -0.6311925053596497, 0.7025659084320068, 0.6756588220596313, 0.6308064460754395, -0.3032310903072357, 0.02753942459821701, 0.7620500922203064, 2.204258680343628, 0.8330402374267578, 0.5145899057388306, -1.2706141471862793, -0.45108577609062195, 1.3575557470321655, 1.2866979837417603, -2.2046992778778076, -1.2379876375198364, 0.9755880832672119, 0.47349679470062256, 0.5654809474945068, -1.1127854585647583, 2.158400535583496, -0.5625787377357483, 0.0704011470079422, 0.7087703943252563, -0.8226674199104309, 1.345941185951233, -0.3172699511051178, -0.10289806872606277, -0.33831334114074707, -1.7561018466949463, 0.6543715596199036, -0.6387273669242859, 0.032137300819158554, -0.31066200137138367, -0.37668803334236145, 0.6094366312026978, 0.20994600653648376, 0.5649890303611755, 0.67626953125, -1.657480001449585, 0.354807585477829, -0.4901341199874878, 0.22930817306041718, 0.3124723434448242, -1.5187098979949951, 0.0050735981203615665, -0.28809496760368347, -1.308057188987732, -1.8360614776611328, -0.4708699584007263, 1.5317023992538452, -0.6987653970718384, 0.3385624885559082, 0.44457828998565674, -0.9625272750854492, -1.1771467924118042, -0.5988845825195312, -0.39640846848487854, -0.22619879245758057, 0.020614465698599815, 0.5227397680282593, 0.27131596207618713, -1.3792707920074463, -1.2367138862609863, 0.8243280649185181, -0.14109298586845398, 0.46137112379074097, -0.07648700475692749, -0.4445091187953949, 0.6982089877128601, 1.4676374197006226, -1.418204426765442, -0.8811590075492859, -0.32800784707069397, -0.5444716215133667, 0.6289920806884766, 1.3405181169509888, 0.2638777494430542, -0.2798379361629486, 0.5736916065216064, -0.04862474650144577, 1.3471581935882568, 0.7955727577209473, -1.2554223537445068, -0.9513801336288452, -1.5955183506011963, -1.4558160305023193, -1.0342891216278076, -0.1480867564678192, -0.7611863613128662, 0.10458120703697205, 0.46443888545036316, 0.021788939833641052, 0.00489488011226058, -0.5323482751846313, -0.04522484913468361, -0.23390191793441772, 1.610052466392517, 0.07490812987089157, 0.7892206311225891, 0.06789866834878922, -0.9246426224708557, -0.005535186734050512, 1.0819088220596313, 0.3363839089870453, -0.8214411735534668, -2.2350001335144043, 1.4251067638397217, -0.8869402408599854, 0.3749339282512665, -0.4975878894329071, 0.4347648024559021, 1.0121387243270874, 0.08073045313358307, 0.14017511904239655, 0.2329036295413971, 0.7939010858535767, 1.607167363166809, 0.7424218058586121, 0.7680485248565674, 1.3565486669540405, -0.33013924956321716, 1.637760043144226, -0.460814893245697, 2.2988064289093018, -1.0372145175933838, -1.2718307971954346, 0.09918967634439468, 0.6039066314697266, -1.1685348749160767, 0.6647961735725403, -0.7886253595352173, -1.023342490196228, -0.44395798444747925, 0.21886186301708221, -0.7821235060691833, -1.036847710609436, 0.21540653705596924, -0.4870677590370178, 2.031343936920166, 1.4018168449401855, 0.08926335722208023, -0.24175284802913666, -1.1104631423950195, 0.6665999889373779, 0.053682927042245865, -0.22789724171161652, 0.17070424556732178, -0.4382932782173157, -0.5026736259460449, 1.252068281173706, 0.7575317621231079, -1.3939932584762573, -0.12958163022994995, 0.9661084413528442, 0.37675508856773376, -1.7701084613800049, 1.7764896154403687, 0.1887882947921753, -0.8772169947624207, 0.07266247272491455, 1.2568104267120361, -1.0531284809112549, -0.6333810091018677, -1.3023159503936768, -0.41243547201156616, 1.0268585681915283, -0.3502750098705292, -1.9927184581756592, -0.4179689288139343, -1.3252893686294556, 0.27227863669395447, -0.07045163214206696, 0.9313197135925293, -0.07122183591127396, 0.05388609319925308, -2.2060537338256836, 0.006733572576195002, 0.9236592650413513, 0.16674618422985077, 0.15651297569274902, -0.2418178915977478, -0.7630370259284973, -0.06923685222864151, 0.5880495309829712, -1.0995895862579346, -0.7933077216148376, 0.49461299180984497, -0.10905236750841141, -0.5317444801330566, 0.564913272857666, 0.042870573699474335, 0.8735154271125793, -0.06298256665468216, -0.5448997616767883, 1.0999730825424194, 1.096401333808899, -1.010001301765442, -0.16123688220977783, -2.153834104537964, 1.1389902830123901, -0.77204430103302, 0.33761805295944214, 0.8111441135406494, -0.0637015700340271, 0.5654730200767517, 0.8396682739257812, -0.5460565686225891, -0.9318146705627441, -2.7071163654327393, 0.791663408279419, 0.9883626699447632, -0.48668694496154785, -0.9300618171691895, 0.30688121914863586, 0.8924604654312134, -0.6356443762779236, 0.8775266408920288, -0.4614395797252655, -1.4980560541152954, -0.9417358636856079, 1.8747427463531494, -0.05337822437286377, -0.14476299285888672, 0.8040357828140259, -1.3335319757461548, -0.5575018525123596, 0.5259311199188232, -1.2883166074752808, 0.17022152245044708, 1.6204572916030884, -1.897939682006836, -1.148734450340271, 0.06409408152103424, -0.15359623730182648, -2.67077898979187, 0.9623280167579651, -0.6549420952796936, -1.0353567600250244, -0.7942191958427429, 1.012561559677124, 0.5801618099212646, -0.4910076856613159, -2.210683584213257, 0.8956952095031738, -0.4346141219139099, 0.9278141260147095, 0.22854448854923248, 0.2695060670375824, -1.560444712638855, -0.03303232789039612, 0.6214191913604736, -0.6679118275642395, 1.2683924436569214, -0.41696715354919434, 1.3297340869903564, 1.4500784873962402, -0.961087167263031, 0.478194922208786, -0.04427485540509224, -1.9684834480285645, 0.7942642569541931, 1.0301249027252197, -0.2483048141002655, -0.8636549711227417, 1.5727453231811523, 1.1539376974105835, -1.1814591884613037, 0.6819310784339905, 0.23147885501384735, 1.2052100896835327, 0.1305927038192749, 1.1981136798858643, -1.7490812540054321, 1.2241772413253784, 1.1891897916793823, 0.4131864607334137, 0.3231680393218994, 0.6411700248718262, 0.21638788282871246, -0.22406615316867828, -1.0982228517532349, -0.09954418241977692, 0.5656486749649048, 0.4731828272342682, -1.2856446504592896, -0.902380108833313, -0.5027372241020203, 0.8468296527862549, 0.19725462794303894, 0.5261826515197754, -0.42810145020484924, -0.818830668926239, 0.0958709567785263, -0.818252444267273, -0.03830217570066452, -0.46370238065719604, 0.7217236161231995, 0.2761536240577698, -0.7217591404914856, -0.04839707911014557, 0.7077552080154419, -0.6074419021606445, -0.02730477973818779, 1.7296909093856812, 0.13285745680332184, -0.501849889755249, 0.6030771136283875, -0.0710589736700058, -0.04994385316967964, -0.7493883967399597, 0.03531713783740997, 0.39626991748809814, 2.898932933807373, 0.5426191687583923, 0.5665017366409302, 0.8077332973480225, 0.5268546938896179, -0.7728347182273865, -0.007649250328540802, 0.47959235310554504, -0.9542331099510193, -0.9356405138969421, 0.7598323822021484, 0.330487459897995, 0.08108846098184586, 1.296260952949524, 0.36764195561408997, 0.9137659072875977, -0.3863031268119812, -0.309783399105072, -0.2585684061050415, -0.3225281238555908, -1.8425257205963135, -2.2349257469177246, 1.5791007280349731, 0.44581565260887146, 1.1164724826812744, 0.30089443922042847, 0.7888291478157043, 0.17284199595451355, 0.14080193638801575, -1.1752163171768188, -0.24711108207702637, -0.8206755518913269, -0.6030713319778442, -1.4614195823669434, -1.1806046962738037, -0.5550674200057983, 0.12609778344631195, 0.21532917022705078, 0.17382390797138214, 0.09412077069282532, 0.5842580795288086, -0.5520500540733337, -1.424906849861145, -0.9123080968856812, -0.02694239467382431, 1.102729082107544, -1.5042333602905273, -2.1947832107543945, -0.19145028293132782, 0.4408010244369507, -0.48429960012435913, -0.8364424705505371, 0.24252162873744965, -0.46757397055625916, 0.3962508738040924, -0.17644040286540985, -0.9371773600578308, 0.5769101977348328, -1.2225981950759888, -0.3891131579875946, -0.12522971630096436, 0.5864506363868713, -0.9200870394706726, 0.8122479319572449, -0.5871734619140625, 0.009585990570485592, 2.515878438949585, -2.1336629390716553, 0.26373782753944397, 0.005032722372561693, 1.6953035593032837, -1.8116158246994019, -0.9580706357955933, -0.4176056385040283, -0.3376721441745758, -0.4526717960834503, 0.4628989100456238, 0.5083950161933899, 1.1286906003952026, 0.5857502818107605, -0.3909144699573517, 0.332779198884964, 2.0238096714019775, 0.7695180773735046, 0.6316198110580444, 0.31225520372390747, 1.5293593406677246, -0.4600820541381836, -0.06741472333669662, 0.5348167419433594, 0.49237576127052307, -1.0151811838150024, 0.06511646509170532, -1.2025829553604126, -0.9280782341957092, -0.818185031414032, -1.1458849906921387, 1.6745625734329224, -1.2425748109817505, 0.6450739502906799, -0.47161078453063965, -2.7987782955169678, 1.014044165611267, -0.38719630241394043, 0.24331796169281006, 0.7422671914100647, -0.03035324439406395, -0.45248422026634216, -1.6278605461120605, -0.9838545918464661, -0.2563108205795288, 1.88840651512146, -0.6100130081176758, 0.8873732686042786, -1.196378469467163, -0.5430248379707336, -2.3212730884552, 2.084933280944824, 1.336409091949463, 0.5041806697845459, 0.4056452214717865, 0.8597508072853088, -0.9790003895759583, 0.9764407873153687, 1.3368415832519531, 0.6288847327232361, -2.1322484016418457, -0.6707000136375427, 0.5408825874328613, 0.22730885446071625, 0.46490463614463806, 0.47970351576805115, 0.2260783165693283, -0.7232218980789185, -3.0240349769592285, -0.43758127093315125, -2.325819492340088, 0.3492323160171509, 1.3335133790969849, 0.584359884262085, -0.7655421495437622, -1.7128862142562866, 0.014353765174746513, 1.8772294521331787, -0.08956052362918854, 1.60676109790802, 0.8251947164535522, 0.19854231178760529, 0.5293248891830444, -0.04641331732273102, -0.3394045829772949, -1.6391866207122803, -0.23913554847240448, -0.4280695617198944, 0.7297512292861938, -0.453220009803772, -0.23763656616210938, -0.4324919581413269, 0.5577869415283203, -0.5590158700942993, -0.6072654128074646, 0.2501366436481476, 0.5390980839729309, -0.8355875611305237, -0.8838621973991394, 1.6263707876205444, 1.8930145502090454, -1.5261774063110352, -0.19609493017196655, 0.057765889912843704, 1.3416335582733154, -0.6943002939224243, 0.21426694095134735, -0.43776175379753113, 0.03180431202054024, 1.105220913887024, 1.164859414100647, 1.208901286125183, -0.9675217270851135, 0.23821701109409332, 1.1383060216903687, 0.9673738479614258, -0.1381121724843979, 1.5354233980178833, -0.42398902773857117, 0.5659191012382507, 0.5587046146392822, -0.7560768723487854, -0.13718999922275543, 0.5435897707939148, -0.2647806704044342, -0.11108997464179993, 0.30261677503585815, 1.4745688438415527, -0.6440703868865967, 0.7724253535270691, -1.102553129196167, 1.541946291923523, -0.09415120631456375, 0.17300528287887573, 1.2337970733642578, 2.2168056964874268, -0.2587190270423889, -1.095316767692566, 0.5475514531135559, -1.1129214763641357, -0.33982130885124207, -0.4408271610736847, 0.9152253866195679, 0.1418406218290329, -0.5627671480178833, -0.7073978185653687, 1.9867902994155884, -0.7052819132804871, 2.249129295349121, -0.24000108242034912, 0.8876955509185791, 0.6597512364387512, -1.1695057153701782, -0.175083190202713, -1.1771899461746216, -0.4903719127178192, 0.4370226263999939, 0.941847026348114, 0.9879974722862244, 1.3066668510437012, -1.7224819660186768, 0.3194851279258728, -0.09061277657747269, -1.0998787879943848, -0.20854564011096954, -0.3723228871822357, -0.19394735991954803, -0.11355508118867874, -0.07147639244794846, 1.4913030862808228, -0.08119593560695648, -1.4917855262756348, 0.8587413430213928, -0.9330266118049622, -0.7958758473396301, -0.5254920125007629, -1.5824693441390991, -0.6662003993988037, 0.5225154161453247, 0.9264054894447327, 0.7150647640228271, 1.394167423248291, 0.2147071212530136, 0.868672788143158, -0.20766279101371765, -1.3491551876068115, -0.04639501869678497, 1.948311686515808, 0.5794997811317444, -0.41695815324783325, -0.5495222210884094, -0.3122146725654602, 1.437571406364441, -1.144945502281189, -0.844819188117981, -1.3183879852294922, -1.1222702264785767, 0.43317821621894836, -1.1882983446121216, -0.7590243816375732, -0.36015400290489197, 0.5918177962303162, -1.8927453756332397, 0.13269171118736267, 0.3736262917518616, -1.9399510622024536, -0.6850364804267883, -0.728230893611908, -1.2700955867767334, 0.13047516345977783, 1.249064326286316, 0.4820610582828522, 2.4783289432525635, -0.28277185559272766, 0.44774487614631653, 0.009773816913366318, 0.8625104427337646, -1.191432237625122, 1.2619043588638306, 1.5190787315368652, -0.9487411379814148, 0.7079964280128479, 0.24291124939918518, -0.9769552946090698, -0.7169265151023865, 1.0542643070220947, 0.11448093503713608, 0.8335694670677185, -0.2412806898355484, -0.3409562110900879, 0.35647261142730713, 0.3538459837436676, -0.42816683650016785, -1.5050864219665527, 0.4345153272151947, 0.7722880244255066, 0.26017746329307556, -1.1866408586502075, 1.1852035522460938, -0.5184198021888733, -1.585209608078003, 1.9632585048675537, -0.13798411190509796, -0.6651198863983154, -0.26144927740097046, 0.17570944130420685, -0.7818069458007812, -0.5003029704093933, 0.5828235149383545, -0.971138596534729, -1.72782564163208, -0.10995492339134216, -0.7207372784614563, 0.39904865622520447, 0.18548227846622467, -1.2402886152267456, 1.0352362394332886, -1.3367843627929688, 0.8461349010467529, -1.2189202308654785, 0.2864648103713989, -0.6771519184112549, 0.7714027166366577, 0.22738665342330933, -0.450119286775589, -1.4696937799453735, -0.3005019426345825, -0.11889311671257019, -0.015354260802268982, -0.31373924016952515, 0.2513362765312195, -0.03380925953388214, -0.6801010370254517, -0.6105643510818481, 0.38620230555534363, -0.08017284423112869, -1.8853338956832886, -1.3275892734527588, 1.2970372438430786, -0.37269356846809387, 0.48838376998901367, -0.899958610534668, 1.000901699066162, 0.8628365993499756, 0.7661195397377014, -0.5003901124000549, 1.4832106828689575, 0.6497620344161987, -0.13196368515491486, -0.2604855000972748, -0.5179682970046997, -0.6879278421401978, 0.9338691234588623, -0.15048924088478088, -1.2490137815475464, 0.8995329737663269, 0.08662281185388565, -1.3205795288085938, -1.989834189414978, 0.36205771565437317, 0.02658940479159355, -0.23509864509105682, -1.2547667026519775, 0.9429159164428711, -0.27752619981765747, 0.6836500763893127, 1.8931217193603516, 0.4003475606441498, -0.3222310543060303, -0.6167470812797546, 0.13343699276447296, -1.8594173192977905, 0.15680980682373047, -0.654076337814331, 1.0853487253189087, 0.030268274247646332, -1.183415174484253, -0.7923516631126404, 1.5477887392044067, 0.49363574385643005, 0.7067633271217346, -1.121411919593811, -0.4280991852283478, -0.2985585927963257, 0.08629584312438965, -0.6642416715621948, 0.17507120966911316, -0.5728009343147278, 1.628925085067749, 1.2004261016845703, -1.2623602151870728, 0.6060577034950256, 1.1411054134368896, 0.34687843918800354, 0.2697042226791382, -0.5390527844429016, 0.4031897187232971, 0.31812170147895813, 0.613641083240509, 0.9998199939727783, 1.5391595363616943, 0.787852942943573, -0.641733705997467, 0.49981027841567993, 0.9262551069259644, 0.4121066927909851, 0.16754655539989471, 1.7398098707199097, -1.1590715646743774, 0.7015254497528076, 0.006756481248885393, 1.135493516921997, 0.8530384302139282, -1.367218017578125, -0.18539342284202576, -1.0719910860061646, 0.00974903255701065, -0.5632526278495789, 1.1738204956054688, 2.0126876831054688, 0.41166311502456665, -1.2954176664352417, -0.2949138581752777, 1.0199964046478271, 0.6294395923614502, -0.042308855801820755, 0.14437568187713623, 0.3406634032726288, -1.223022222518921, 1.4382790327072144, 0.3619343340396881, -1.5844509601593018, 0.6739652752876282, 0.6574507355690002, -1.0263539552688599, 1.608052372932434, 0.4797717034816742, 1.128983974456787, 0.9819973707199097, -0.11288128793239594, 0.21505041420459747, 0.2941087484359741, 0.2693694531917572, 0.4028973877429962, -1.313827633857727, -0.11287995427846909, 0.20025266706943512, 0.17310795187950134, 1.384412169456482, -0.16379034519195557, 2.226029634475708, 0.7204821705818176, 0.5147479772567749, 0.5351588726043701, -1.1900407075881958, -1.7279845476150513, 0.02482634037733078, 1.3230652809143066, -0.6918241381645203, 0.11823160201311111, 0.26722076535224915, -0.05504423752427101, 1.3088304996490479, -1.057808518409729, 0.285715252161026, 0.7972227334976196, 2.1508126258850098, -0.01987091265618801, 1.4762094020843506, -0.5641084313392639, 0.9745569229125977, 0.43510255217552185, -0.2814268469810486, 1.1231153011322021, 0.462703675031662, -0.45186418294906616, 0.3834565579891205, 0.684037983417511, 2.6856250762939453, -0.5029521584510803, -0.5490518808364868, -0.302412748336792, -0.6470821499824524, 0.5486199259757996, -0.3515917658805847, 0.028728147968649864, 0.3204375207424164, -1.245872974395752, -0.7208077907562256, 0.5550339818000793, 0.5096344351768494, 1.0716582536697388, 0.0009297961951233447, -1.4381691217422485, -0.3355973958969116, 0.3307966887950897, 0.4460555911064148, 0.04593446105718613, 1.4578971862792969, -2.421569585800171, 0.051248934119939804, 0.821424663066864, 0.41348767280578613, -0.5226294994354248, -1.5097593069076538, 0.43979164958000183, 0.3354169726371765, -1.080122947692871, -0.3475618362426758, 0.4461410939693451, 0.03832678124308586, -0.26582658290863037, 0.27669501304626465, -0.8593589663505554, -0.43856358528137207, -0.5698514580726624, 0.3745771050453186, -0.6759450435638428, -1.0847914218902588, -0.5898351669311523, -0.4314868152141571, 0.19273266196250916, -1.9840648174285889, -1.1388911008834839, -0.8749516010284424, 0.6713029146194458, 0.861054539680481, -0.7889672517776489, 1.456148624420166, 0.19847580790519714, 0.4868431091308594, 0.011676134541630745, -1.3859773874282837, -0.15953591465950012, -1.6936572790145874, -0.38365620374679565, 0.46771323680877686, -1.766888976097107, 2.4338879585266113, -0.5926443934440613, -0.8383814692497253, 0.39344704151153564, -0.32076677680015564, -1.1133760213851929, 0.2385992854833603, 0.0357787124812603, -0.30069220066070557, 0.4060920774936676, -0.5756245255470276, 0.39238855242729187, 0.09259262681007385, -0.03758052736520767, 0.012399567291140556, 0.13989399373531342, -2.0883727073669434, 0.1806715428829193, 2.1794683933258057, -0.362906277179718, 1.6590352058410645, -0.014995161443948746, -1.7612779140472412, -0.846227765083313, -1.0255147218704224, 0.8291288614273071, 0.8667795062065125, 0.4001487195491791, 0.38981547951698303, 0.3082503378391266, -0.744238018989563, -0.14137113094329834, -1.3787322044372559, -0.4648965299129486, 0.40457943081855774, -0.5884500741958618, 0.3849724531173706, 0.7281976342201233, 0.9031568169593811, 0.46848705410957336, 0.3452173173427582, 1.453490972518921, -0.561648964881897, -0.5541297793388367, 0.4107087254524231, 1.0234873294830322, -0.384461909532547, -0.9752941131591797, 0.26400095224380493, -0.5136206746101379, 1.1003021001815796, 0.1673322319984436, 0.8685016632080078, -0.34039855003356934, -0.5427650213241577, -0.2538627088069916, 2.8030223846435547, -0.5459134578704834, 0.02581568993628025, 0.08565313369035721, -0.5244388580322266, 0.0754813626408577, -0.10121550410985947, -0.6658380627632141, 0.40570276975631714, 0.7179095149040222, 0.9049832224845886, -1.168927788734436, -1.5923489332199097, -0.28469139337539673, -1.8867675065994263, 0.255206435918808, 3.010559320449829, 0.1533256471157074, 1.433184266090393, -0.4322563409805298, 1.100187063217163, 1.1323554515838623, -1.3498653173446655, -0.5995092391967773, 0.5326626896858215, -0.09033716470003128, -1.639296054840088, 0.8787994980812073, 0.09046981483697891, 1.0545903444290161, -0.9434770941734314, -0.7531329989433289, -0.8213309645652771, -0.45390838384628296, 0.8346058130264282, 0.5604162216186523, 2.1060073375701904, -0.42623332142829895, 0.458449125289917, 1.1004499197006226, -0.18492797017097473, 1.9529989957809448, -0.5721084475517273, 0.11271889507770538, -0.8520908951759338, 1.7934749126434326, 1.1988122463226318, 1.7167396545410156, 0.07185199856758118, 0.8122825026512146, 0.4312351942062378, -0.7874371409416199, 1.6678768396377563, 0.320265531539917, -0.41328099370002747, -0.10299431532621384, -1.407566785812378, -0.5148429274559021, -0.586494505405426, 0.3136083483695984, 0.9242100715637207, 1.822468638420105, -0.5179224610328674, 0.5739771127700806, 0.12790361046791077, -0.9143892526626587, 1.216520071029663, 1.6175320148468018, -0.7822462916374207, -0.6389184594154358, 1.1036735773086548, 0.5652435421943665, 0.32221928238868713, -0.010480018332600594, 0.6922886967658997, -1.032059669494629, -1.6003729104995728, 0.6964869499206543, 0.845181405544281, 1.3103305101394653, 1.773449420928955, 1.7334965467453003, -0.8265306353569031, -1.5701731443405151, -1.1168259382247925, 0.5119765400886536, 0.3434160649776459, 0.669951319694519, 1.0200626850128174, 0.3763500452041626, -0.04463094100356102, -0.5858578681945801, 1.0379060506820679, 0.15065549314022064, 0.5488662719726562, -0.8244935870170593, 1.1698637008666992, -1.9563335180282593, 1.5951790809631348, 1.0140331983566284, -0.0152716850861907, 0.07799091190099716, -0.7451507449150085, 0.13771814107894897, -1.1595203876495361, 1.4427456855773926, 1.2421338558197021, 0.8796611428260803, 0.054591983556747437, -0.9660632610321045, -1.6760987043380737, -1.0473917722702026, 0.5423360466957092, 0.4745519459247589, 1.6433565616607666, 0.2984767556190491, -0.6877743601799011, -0.12981732189655304, -0.349982887506485, 0.2720036804676056, 0.9431922435760498, -0.9503282308578491, 0.1545543521642685, 0.5111836791038513, -0.07861678302288055, -1.0922486782073975, 0.8121462464332581, -0.3776750862598419, -1.0854387283325195, -0.7057990431785583, 0.24874797463417053, 0.008182008750736713, -1.6464834213256836, -1.3766051530838013, 0.5522724986076355, -1.0365010499954224, -0.9374538064002991, 0.9454986453056335, -0.6022329330444336, -2.2160284519195557, -0.38432416319847107, -1.2016336917877197, 0.854647696018219, 1.607572078704834, 1.2542701959609985, 1.1198838949203491, 2.089385986328125, -0.1841634064912796, 0.0942661464214325, 2.4070117473602295, -0.321399986743927, -0.10957711935043335, 0.19603262841701508, -1.113008975982666, 0.42902278900146484, 0.8253970146179199, 0.9280596971511841, 0.3407863974571228, -1.0331296920776367, 0.7424044013023376, -0.15987752377986908, -0.23962338268756866, -1.477081060409546, -0.425557404756546, -0.8220103979110718, 0.2972680628299713, 1.6987240314483643, -0.21351206302642822, 1.114486813545227, 0.4606947600841522, 0.0831431970000267, 1.1719903945922852, -0.840552031993866, 0.42347782850265503, -0.6485004425048828, 0.5254724025726318, 0.4912686347961426, -0.09741856157779694, 1.0347498655319214, 0.3650232255458832, -0.2552551329135895, -1.325518250465393, -0.7163083553314209, -1.4423195123672485, -1.4674042463302612, 0.1444985717535019, -0.06160099059343338, -1.3921501636505127, -1.4474471807479858, -0.885199248790741, 1.3449296951293945, 1.9916964769363403, 0.5679666996002197, -0.5175540447235107, 1.8687118291854858, 0.19649788737297058, 0.4467198848724365, -1.2043910026550293, 0.24033106863498688, 0.24626487493515015, -0.3067223131656647, 0.5549069046974182, 0.013138321228325367, -0.5475417971611023, -0.11049434542655945, -0.8128234148025513, 1.5994162559509277, -0.4818631708621979, 0.15050941705703735, -0.2494201958179474, -0.5779113173484802, 0.30752524733543396, 1.0408183336257935, 0.4121651351451874, 0.04975656047463417, 1.1031304597854614, 0.5224616527557373, -0.974204957485199, 2.3260414600372314, -0.7082099914550781, -0.35935863852500916, 0.04292643070220947, 0.21465188264846802, 0.09089181572198868, -0.25327032804489136, 1.5128787755966187, 1.315346360206604, -0.5674542188644409, 1.5195815563201904, -1.2330940961837769, 1.801949381828308, 0.2734619379043579, -0.8331323862075806, -0.8302077651023865, 0.27345913648605347, -0.24187636375427246, -0.7948116064071655, -1.0990228652954102, 0.5934977531433105, -0.49770253896713257, -0.29844385385513306, -0.6378234028816223, 0.2682766318321228, 0.5157456994056702, -0.7965753078460693, -0.9077872633934021, -0.7744400501251221, -1.203687071800232, -2.363773822784424, -0.3978292942047119, -1.320414662361145, -0.11372686177492142, 0.4168449640274048, -0.7918010354042053, -0.3566824197769165, 0.1438530683517456, -0.7941506505012512, 0.3226872384548187, -0.035499632358551025, 0.5776521563529968, 0.40791305899620056, -0.26029038429260254, 0.48880842328071594, -0.010150928981602192, -0.23384681344032288, 0.18849945068359375, -1.5335930585861206, -2.1001389026641846, 0.37314656376838684, -0.03255719691514969, 0.25039002299308777, -0.016804682090878487, -1.305127501487732, -0.8236685395240784, 0.01919659413397312, -1.1316871643066406, -0.9674388766288757, 1.2757682800292969, 1.1831437349319458, -0.4188343286514282, -0.9922102093696594, 0.03904639929533005, 0.5227648615837097, 1.1125030517578125, -0.15619665384292603, 0.0022693094797432423, 0.45859643816947937, -0.13564395904541016, 0.24859628081321716, -0.022670025005936623, -0.00860816240310669, -0.30334922671318054, -0.5049682259559631, 1.2775604724884033, -0.45305508375167847, 0.11965955048799515, 0.5469644665718079, -0.24669137597084045, 1.5309321880340576, -0.20912623405456543, 1.004441499710083, 0.9204617738723755, 0.15589968860149384, -0.1674233376979828, -1.022823452949524, 1.8358691930770874, 0.8202552199363708, 1.1516062021255493, -0.9307600855827332, -0.6299139261245728, -0.5710991024971008, 1.085970401763916, -0.052661359310150146, 0.9678477644920349, 0.20321188867092133, 0.5790550708770752, -0.3994821012020111, -0.5413965582847595, 1.2278650999069214, 0.40458470582962036, -1.0286171436309814, 2.104412317276001, 1.5220487117767334, -0.13187521696090698, 0.6195330619812012, 1.7568039894104004, 0.4802211821079254, 0.8314377069473267, -0.8722640872001648, 1.491105556488037, -1.7588629722595215, -0.4128212630748749, 0.24765165150165558, 0.7500747442245483, -1.6792359352111816, 0.8637334704399109, -0.6709825396537781, -0.7120693325996399, 0.810177206993103, -0.40125957131385803, -0.24173115193843842, -3.2513105869293213, 0.5991786122322083, 0.017878925427794456, 0.2726915180683136, 2.09879732131958, -0.04676280915737152, -1.2759947776794434, -1.4612600803375244, -1.5058790445327759, 0.6751089692115784, -0.37254151701927185, -1.2152940034866333, -0.8051884174346924, 0.8523520827293396, 0.9950281977653503, 0.10219746828079224, 0.17409424483776093, 0.05503133684396744, -1.1123769283294678, -0.8602327108383179, -1.106877326965332, -0.6283416152000427, -1.007644534111023, 0.23686033487319946, -0.3746475577354431, -0.2695848345756531, 0.3064720034599304, -1.3160576820373535, 1.095953345298767, -0.7459849119186401, -0.403435617685318, -0.4912060797214508, 0.5979064106941223, -1.0355521440505981, -2.4447247982025146, -0.2254965752363205, -0.37655118107795715, -1.6900382041931152, 0.4552208185195923, -0.06940115988254547, -0.2468821108341217, -1.6009294986724854, 0.7161424160003662, -0.5707809329032898, -2.108621120452881, -0.16915519535541534, 1.1627957820892334, -0.16889289021492004, 1.5600961446762085, 0.009777060709893703, 0.3846699297428131, -0.03347478434443474, 0.48802629113197327, 1.4085227251052856, -0.21855607628822327, -0.21976394951343536, -0.030087579041719437, -0.9646179676055908, 0.3287217617034912, -0.7221289873123169, 1.417055606842041, 0.7726306915283203, -1.6715010404586792, -0.13378186523914337, 1.910237193107605, 0.659550666809082, -0.39828047156333923, 0.5477818846702576, 0.24058905243873596, 1.549557089805603, -0.019357966259121895, 0.8143489956855774, 0.6805453896522522, -0.2800244688987732, -0.2559061348438263, 1.1982280015945435, -0.6941373348236084, 0.05265028774738312, -1.4141607284545898, 0.8841238021850586, 0.5417199730873108, -1.2497210502624512, 0.43734845519065857, -0.3679625391960144, -0.03227357566356659, -0.7810323238372803, 0.33259883522987366, 0.15810418128967285, 0.48925989866256714, 0.23805738985538483, -0.9227950572967529, -1.5974332094192505, -1.060019850730896, 1.3126894235610962, -0.1520194262266159, 0.2958841025829315, -0.25773534178733826, 0.19739076495170593, 1.2670906782150269, -1.9966517686843872, 1.1772849559783936, 0.633946418762207, -0.6359748840332031, 0.05299589782953262, 0.6323438882827759, -0.03591945394873619, 0.8677900433540344, 1.1782245635986328, -0.8221883773803711, 1.1162723302841187, 0.029969248920679092, 0.7635352611541748, -0.4870319664478302, -0.18027564883232117, 0.037670060992240906, 1.0513745546340942, 0.03374495357275009, -1.9433937072753906, 1.0050053596496582, -1.3821560144424438, -0.9938209652900696, -0.03037240356206894, 1.2654420137405396, 0.9631879329681396, -0.04659689590334892, -1.1042687892913818, 0.40734702348709106, -0.31545811891555786, -0.3618033826351166, -0.27210986614227295, 0.41890400648117065, 0.27961790561676025, -0.8448777794837952, -1.0831037759780884, -0.4925590455532074, 0.49750956892967224, -1.9632534980773926, -1.5125328302383423, -1.1514521837234497, -0.4493233263492584, -0.305048406124115, -1.1540316343307495, -0.6241243481636047, 1.9671746492385864, 1.8909988403320312, -0.1595730036497116, 1.4966977834701538, -0.9379517436027527, -1.5515080690383911, -0.19930976629257202, 1.3171831369400024, -0.48591098189353943, 1.895274043083191, 1.5272669792175293, -1.6539082527160645, 0.8370599746704102, -1.1641231775283813, -0.07198391854763031, 0.3004002869129181, 0.4756389260292053, 0.6690922379493713, -0.4570986032485962, 1.5354819297790527, 0.5559056997299194, -2.448901891708374, -0.7216454744338989, -0.005828737746924162, -0.17497922480106354, 0.20783330500125885, -0.5302484035491943, 0.8224595785140991, 0.15076999366283417, -1.312289834022522, -0.7303452491760254, -1.2380321025848389, -0.6722974181175232, 0.20779253542423248, -0.1498621702194214, -0.10861318558454514, -1.7137376070022583, -0.23457284271717072, -0.5855923891067505, 1.4706451892852783, -0.652387261390686, -1.6029366254806519, 0.002690725727006793, -1.2469362020492554, 0.7028587460517883, -1.209355115890503, -0.09971954673528671, -0.6087470054626465, 1.016869068145752, -1.3391209840774536, -0.22336862981319427, 0.7034115195274353, -0.23270052671432495, -0.21825183928012848, 0.24790941178798676, 1.3308687210083008, 0.19938485324382782, 0.003575490787625313, -0.6506752967834473, -1.1084622144699097, -0.761280357837677, 1.1655439138412476, -1.4635571241378784, 0.18738007545471191, 0.3479992747306824, 0.8885111212730408, 1.1181548833847046, -2.799760580062866, -0.0674448013305664, -1.3712199926376343, 1.58390474319458, 0.7198745608329773, -1.0034009218215942, 1.814987063407898, 0.5041716694831848, -0.9458604454994202, 0.8744986653327942, 0.34962764382362366, 0.838264524936676, -0.5110317468643188, -0.6509175896644592, -1.4923120737075806, 0.4619807004928589, 0.5049580931663513, 0.5947310924530029, 0.5997491478919983, -0.3544721305370331, -0.6663848161697388, -0.15070179104804993, -2.4019622802734375, -0.24128106236457825, -0.8017566204071045, -0.40704381465911865, -0.26138633489608765, -0.22778935730457306, 0.7068972587585449, 1.0318681001663208, 1.5013638734817505, 0.24991093575954437, -0.5960135459899902, -0.9653267860412598, 0.7998942732810974, -0.6397131681442261, 0.7495999932289124, -0.8978883624076843, -1.2113537788391113, 0.9387512803077698, -1.1738020181655884, 1.0207070112228394, -0.20517458021640778, -0.7499009966850281, -1.125233769416809, 0.7093797922134399, 0.5786211490631104, 1.0789228677749634, 0.5342543125152588, -0.5739648342132568, 0.1271674484014511, -1.5079153776168823, -0.4566887617111206, -0.15615150332450867, -0.5908833146095276, 1.3621281385421753, -0.014656227082014084, -0.3378453850746155, -1.7928358316421509, -1.0834319591522217, 1.7260756492614746, 0.8654354214668274, 0.7733364105224609, -0.14907528460025787, 0.7437669634819031, -1.5681277513504028, 0.6755828261375427, -0.7067725658416748, 1.272018313407898, 0.45376476645469666, -0.2965344786643982, 0.25234678387641907, 0.43825310468673706, 0.4691679775714874, 2.2395846843719482, 0.25734809041023254, -0.840913712978363, 0.189191535115242, -0.8821954727172852, -0.6910596489906311, 0.139762282371521, -0.1349957138299942, -0.8496917486190796, -2.6311795711517334, 0.06576056778430939, -0.8098745942115784, -1.0189613103866577, -0.49735936522483826, -1.1932793855667114, 0.43997722864151, 0.2752952575683594, 1.2254295349121094, 0.2974289655685425, 1.159718632698059, 0.31678593158721924, -0.32112154364585876, 1.0017714500427246, 0.38780269026756287, 0.622922956943512, -0.5575805902481079, -1.928133487701416, 0.33759942650794983, 0.35301536321640015, 0.41961991786956787, 0.5207749605178833, 0.09849795699119568, 0.044235508888959885, 0.009851448237895966, 0.2257218360900879, -0.42108434438705444, -1.0536590814590454, -0.7575437426567078, -0.4360155761241913, 0.9352340698242188, 0.9467415809631348, 0.6375636458396912, 0.2719738483428955, 1.1506543159484863, -1.7157326936721802, 0.32870393991470337, -0.21641868352890015, 0.9971279501914978, 0.21238946914672852, 0.488029807806015, 1.2706694602966309, 0.4604952931404114, -0.7370007038116455, -0.3294580280780792, 0.5002402067184448, -0.17391730844974518, -1.0442754030227661, -0.3034708797931671, -1.0392495393753052, -0.7330630421638489, -0.6438800692558289, -0.43657931685447693, 1.421016812324524, 0.1665685623884201, 0.32201310992240906, 1.613654613494873, 1.1011006832122803, -0.016519401222467422, -0.21740181744098663, -1.0266882181167603, -0.01708744838833809, 1.2232085466384888, -0.05291231349110603, 0.5681234002113342, 1.03202486038208, -1.2753376960754395, -0.1478290855884552, 1.9626710414886475, -0.4542612135410309, 0.3287316560745239, 0.22443249821662903, 0.6633840799331665, -2.369450807571411, 0.09146823734045029, 1.365917682647705, -1.123094081878662, -0.8831155300140381, -1.4704241752624512, -1.03133225440979, 0.9361397624015808, 0.33455321192741394, -1.6592167615890503, -1.1259702444076538, 0.007487560156732798, -0.7529054284095764, 0.4494430720806122, 0.7794495224952698, -0.9922283887863159, -0.26887524127960205, -0.5376635193824768, -0.7743681073188782, 0.9050180912017822, 0.8124548196792603, 0.3854810893535614, 1.3648380041122437, 1.348524808883667, 1.524449110031128, 0.9459726214408875, 0.31613773107528687, -0.8323194980621338, 1.476285696029663, 0.39396750926971436, -0.33446425199508667, 1.468807578086853, -0.25977951288223267, 0.1471717655658722, 0.2940492331981659, 0.7137904167175293, -0.3042866587638855, 0.4685514569282532, 0.4851530194282532, 1.2525453567504883, -0.6426026225090027, -0.9852120280265808, 1.3218517303466797, 0.7603615522384644, -1.6892513036727905, -0.36773380637168884, 0.5438005924224854, 0.5839466452598572, -0.9533949494361877, -0.6349422335624695, 0.5238623023033142, -0.8969577550888062, -1.5601627826690674, 1.3924857378005981, 1.8067594766616821, 0.0676414966583252, 0.36200645565986633, -0.028421487659215927, 0.02884332835674286, 0.4526180922985077, 0.5482913851737976, 0.6245789527893066, -0.227716326713562, -0.3151523470878601, -0.743888258934021, 1.2792437076568604, 0.013455570675432682, -0.528557538986206, -0.8309544920921326, 0.0590396523475647, -1.1865657567977905, 0.37258750200271606, -0.9563500285148621, -1.870444416999817, -0.5062074065208435, 0.6886506676673889, -0.646697998046875, -1.286611795425415, -0.9961366057395935, 1.8102829456329346, 0.30367279052734375, 1.2602993249893188, -0.5523778796195984, -0.9497605562210083, -0.5567189455032349, -0.1315665990114212, 0.1402779221534729, 0.9609730839729309, 1.2805564403533936, 0.278623104095459, 0.634824812412262, -1.1208317279815674, -0.1494860202074051, 1.3182792663574219, 0.3190155625343323, 0.24429664015769958, -1.816085696220398, -0.7941017746925354, 0.5210686922073364, -1.334107518196106, 0.07966592162847519, 1.3051884174346924, -1.0789793729782104, 0.3832630515098572, 0.9280487895011902, 0.8759913444519043, -0.11238756775856018, -0.4319300055503845, 0.09444793313741684, -1.1449213027954102, 0.4480929970741272, 2.0111236572265625, 0.650047779083252, 0.9242440462112427, -0.6487299203872681, 0.34023624658584595, -1.1157869100570679, 0.530576229095459, -0.44429489970207214, 1.2851115465164185, 0.4604812264442444, -0.6235973834991455, 1.2735295295715332, 0.02747240476310253, -0.6645020842552185, 0.860571563243866, 0.4320763349533081, -0.8155452609062195, 1.0609960556030273, 0.8374505043029785, 1.4735413789749146, -0.8064562082290649, 1.0390479564666748, 0.18671976029872894, 0.17240525782108307, -1.2204084396362305, 0.8574661612510681, 0.624074399471283, -0.46106797456741333, -0.31680840253829956, 0.6550437211990356, 1.399507999420166, 0.08658873289823532, -2.8215713500976562, -1.1048485040664673, 1.012298822402954, 1.0943559408187866, -1.2772966623306274, -0.7547557950019836, 1.5162346363067627, -0.425363153219223, -2.490825653076172, 0.7522313594818115, 1.0078842639923096, -0.2683396637439728, -0.4510491192340851, -0.3149511218070984, 0.828798770904541, -0.42560338973999023, -1.7877824306488037, 0.7593929171562195, -1.6468009948730469, 0.04786236211657524, 0.27321040630340576, -1.037887692451477, -0.8845516443252563, -1.3685715198516846, -0.7394288778305054, -1.1675852537155151, 0.25462111830711365, -0.4047307074069977, -0.233853280544281, 0.6881635785102844, -0.4914022386074066, 0.10638963431119919, -0.8848087191581726, -1.1356456279754639, -0.057339299470186234, -2.495713710784912, 1.442488670349121, 1.2702871561050415, -0.7566893696784973, 0.818313479423523, 0.6937487721443176, -0.1685183346271515, -1.4968293905258179, 0.6905404925346375, 1.5746862888336182, 1.3061888217926025, 1.2815510034561157, -0.029273590072989464, -1.8643759489059448, 0.3901834189891815, -0.29043132066726685, -0.7617989778518677, -0.6579321622848511, 0.24335993826389313, -0.24538905918598175, 1.936415433883667, -1.0673257112503052, 0.25215137004852295, -0.655974268913269, 0.5078191757202148, -0.6288440227508545, -1.0291595458984375, -0.49639761447906494, -1.1462081670761108, 0.5800314545631409, -0.0746864303946495, 0.4785606861114502, 1.9617674350738525, -0.8469038009643555, 2.3509268760681152, -0.18099445104599, 0.5924412608146667, 0.0312795452773571, 0.10045170038938522, 0.3808302581310272, 1.5640333890914917, 0.25816810131073, 2.8630194664001465, 1.2933193445205688, 0.48596274852752686, -0.08751808851957321, -0.9364731311798096, 0.08444467931985855, -0.239646777510643, 0.03966594114899635, -1.1817171573638916, 0.7679516077041626, 0.007396007422357798, 0.8736262321472168, -0.7047140598297119, 1.6158063411712646, 1.4961575269699097, -0.03905586898326874, 0.6752638816833496, 1.4720737934112549, 0.378196656703949, -1.0328444242477417, -0.2628093957901001, -0.2026643455028534, -0.2721955180168152, 0.8589606285095215, -0.31661567091941833, 0.39670082926750183, 0.9209827780723572, -0.34236055612564087, 0.9378927946090698, 2.1440696716308594, 0.7974784970283508, 1.3479012250900269, 0.277011901140213, 0.5924471616744995, -1.153941035270691, -0.36161887645721436, 0.06953012198209763, 0.2499106526374817, -1.00950026512146, 1.7352666854858398, 0.7184128761291504, -1.3947049379348755, 0.00691738398745656, -0.077808678150177, -0.23363786935806274, -1.871998906135559, 0.30580541491508484, -0.4610138237476349, 0.5773630142211914, -0.2282562404870987, -1.078405499458313, 0.7449312806129456, 0.31629177927970886, 0.043530579656362534, 1.472130537033081, -0.8555764555931091, -0.6592500805854797, -0.205238938331604, 0.09058967977762222, 0.5142411589622498, -0.38000714778900146, -1.950827956199646, -0.7285274267196655, -0.7978166937828064, 1.7727994918823242, 0.8693720698356628, 0.7979190349578857, -0.3010465204715729, -0.6003270149230957, 0.029227713122963905, 1.2430052757263184, 1.2372347116470337, 1.6989434957504272, 0.8648763298988342, 0.697405219078064, -0.9784984588623047, -0.10514869540929794, -0.37890690565109253, -1.0838533639907837, -0.5158373713493347, -0.22877366840839386, 0.9988166093826294, -0.1494653820991516, 0.4150381088256836, -0.5700567960739136, -0.9596397876739502, -0.5368364453315735, -0.9925081729888916, 0.8772537708282471, -0.5710824131965637, 0.18520374596118927, -0.21551772952079773, -0.6871706247329712, -0.4571740925312042, 0.014513510279357433, 1.0983617305755615, -0.8043427467346191, -0.09659666568040848, 0.5635698437690735, 0.10779072344303131, -1.486507773399353, -1.7521427869796753, -1.0774235725402832, -1.6012951135635376, -1.5421208143234253, -0.5576903820037842, 1.0149555206298828, 0.18867884576320648, -0.6436660289764404, 0.827019214630127, 0.6367542743682861, 0.35101932287216187, 0.7193306684494019, 0.2202223688364029, -2.9151954650878906, -0.5013976097106934, -0.1187192052602768, 0.28359028697013855, 0.1520194411277771, -0.2726173400878906, 0.46701890230178833, 0.50582355260849, -0.020946143195033073, 0.9752414226531982, -1.0097087621688843, -0.3102486729621887, -0.9912031888961792, -0.0757506862282753, 1.2953791618347168, 0.4637284576892853, -0.7703434228897095, -1.0574532747268677, 0.3801995515823364, -1.8369314670562744, 2.8476223945617676, 0.9196426272392273, 0.36699095368385315, -0.10583757609128952, -1.1847357749938965, 0.289705753326416, -0.1614149659872055, 1.0676895380020142, 1.2442350387573242, -1.7095052003860474, 0.7310501933097839, 0.9351826906204224, -0.31475532054901123, -0.23296765983104706, 0.9675869941711426, 0.11444234848022461, -1.0721162557601929, -1.93048095703125, 0.9719443321228027, 0.29336264729499817, 0.1535579115152359, -0.2903558611869812, 0.9917047619819641, -1.5965960025787354, 0.25455600023269653, 0.3627563416957855, -1.310795783996582, 0.4167132079601288, 0.3653879463672638, -0.4688754677772522, -0.29919174313545227, 1.7514981031417847, -1.8616753816604614, 0.8157158493995667, 0.5636305809020996, -0.25916221737861633, -0.6493203043937683, 0.10288912057876587, 0.8133856058120728, -2.7425503730773926, -0.8127357959747314, 1.4847869873046875, -0.842797577381134, 0.31722965836524963, -0.9146356582641602, -0.19421765208244324, 0.11785433441400528, 0.7062261700630188, -0.7622106075286865, -0.9540178179740906, -0.9942550659179688, -0.043683260679244995, 0.5795016884803772, 0.8685894012451172, 0.7204439640045166, -0.14031367003917694, -0.7413926124572754, -1.9435456991195679, 1.3931738138198853, -0.9833840131759644, -1.7013819217681885, 0.8832634091377258, 0.4920444190502167, -0.18460819125175476, -0.7558088302612305, 0.11386385560035706, 0.902527928352356, -0.9743050336837769, 0.12498214840888977, -1.2750470638275146, -0.8776413202285767, -0.635463535785675, 0.7502011656761169, -0.8015512228012085, 0.6261470317840576, -0.5869160294532776, 0.19502104818820953, 1.3237121105194092, 0.3012714087963104, 1.5983896255493164, -0.7538639307022095, -0.3320384621620178, -1.0351253747940063, -2.3832125663757324, -0.8274124264717102, 0.381211519241333, 1.6712284088134766, 0.06874682009220123, -0.04606426879763603, -1.0194569826126099, 0.0626920685172081, 1.2320350408554077, -0.41202542185783386, 2.1441774368286133, 0.9008561968803406, -1.0492722988128662, -0.41144412755966187, -0.6376021504402161, -0.013313286937773228, -0.6019065380096436, -2.020941734313965, -0.0667809471487999, -0.8657394051551819, -0.9728299379348755, -0.7639464735984802, -0.9254224896430969, -0.4462190270423889, 1.3100769519805908, 0.9659468531608582, 0.08470088988542557, 0.3339715600013733, 0.6595956087112427, 0.8887209892272949, 1.2744660377502441, 1.191019058227539, 0.3286691904067993, -0.5660937428474426, 1.3155241012573242, -0.4660808742046356, 1.0619252920150757, -1.0142064094543457, -2.2022781372070312, -0.7701230049133301, -0.5377264618873596, -0.2252214550971985, 0.996369481086731, -0.7081665992736816, -1.5943833589553833, -0.2812863886356354, 0.5144272446632385, 0.5955758094787598, -0.3246544301509857, -1.0460965633392334, -0.7548823356628418, 0.6612681150436401, -1.7789273262023926, -3.204040050506592, -1.2642372846603394, -0.2620427906513214, 0.5105820298194885, 1.0923837423324585, 0.01895252987742424, 1.839231014251709, -0.5836306810379028, -0.07979702949523926, -0.22005906701087952, -0.017739703878760338, -1.8061453104019165, -0.5434327721595764, 0.527891993522644, 0.49748700857162476, 0.1895502656698227, -1.24994957447052, -0.894097626209259, 0.7567635178565979, 1.4191031455993652, 0.4761001169681549, 0.34823381900787354, 1.5900341272354126, 0.7357516884803772, -0.9716848731040955, 1.2568941116333008, -0.3910246193408966, -0.08048084378242493, 0.20040816068649292, 0.39004603028297424, -1.4690004587173462, 1.550034761428833, -1.2018370628356934, -0.5265956521034241, -2.1584694385528564, -0.146132692694664, -1.7812539339065552, 1.247369408607483, 0.02369452267885208, 1.304457664489746, -0.12887080013751984, 0.059796370565891266, 0.116101935505867, 0.818202793598175, 0.6971570253372192, 0.31576085090637207, -1.0557210445404053, -0.9649803042411804, -0.23499523103237152, -1.6627018451690674, 0.6339743733406067, -0.6582979559898376, -0.484619140625, 1.7540736198425293, 0.6171301603317261, -1.0846812725067139, -1.2610313892364502, -1.5978535413742065, -1.383542776107788, -2.000354528427124, -0.21096381545066833, -1.3503750562667847, -0.9496877193450928, -1.2126736640930176, -0.9328418970108032, -0.4488071799278259, -2.8380613327026367, 0.933262825012207, -1.1801261901855469, -0.6266403794288635, -0.09218133240938187, -0.3070421516895294, 0.8228433132171631, -1.5131114721298218, 0.9805783629417419, -0.7354813814163208, 0.5602074265480042, 0.3705451488494873, 1.5695765018463135, -1.3152669668197632, -0.6486666202545166, -1.4659475088119507, -0.8549842238426208, 2.5682482719421387, 1.0902565717697144, 0.012389760464429855, -0.6735564470291138, -0.40143024921417236, 2.274660110473633, 1.031343936920166, -0.2756199240684509, 0.03334296867251396, 0.9262698292732239, -0.5446651577949524, -0.48390862345695496, -0.7276790738105774, -0.2207830548286438, -1.279673457145691, 2.252415657043457, -0.4586547911167145, -0.19145266711711884, -0.9842627048492432, 0.6342012286186218, 0.11969812214374542, -1.1460477113723755, -1.006487488746643, -0.23288415372371674, -1.5085829496383667, -0.6995686888694763, 0.8180941343307495, -0.797196626663208, -1.0477254390716553, 0.5175772905349731, -0.3188830018043518, -0.42357292771339417, 1.0874712467193604, -1.0175453424453735, 1.1055446863174438, -0.011645842343568802, 0.3745291829109192, -0.23979033529758453, -0.010050035081803799, -0.20435409247875214, -1.1858019828796387, 0.07380624860525131, -1.3132095336914062, 0.13779649138450623, -0.9919589161872864, 1.3927805423736572, -0.4344348609447479, 1.6877856254577637, -0.23529663681983948, -0.06189318373799324, -0.7536725401878357, 0.8455984592437744, 1.784157395362854, 0.22879788279533386, 0.6874521374702454, 0.0437968447804451, 1.1300828456878662, -0.8007410168647766, 1.6525516510009766, -1.0701103210449219, 0.7650493383407593, 0.8522059321403503, 0.5190826654434204, -1.4869440793991089, 1.165144920349121, -1.5078341960906982, 1.541497826576233, 0.48991870880126953, 0.7369810342788696, -0.23452720046043396, 0.2615000307559967, 1.3679614067077637, 1.0964581966400146, 1.718037486076355, 0.43706369400024414, 0.18076984584331512, -0.7138955593109131, 0.47052377462387085, -0.37846270203590393, -0.028754720464348793, -0.8226162791252136, 0.23240722715854645, -1.1018595695495605, -0.14061598479747772, 0.9695355296134949, 0.28613418340682983, 0.7724601626396179, 0.36683374643325806, -0.5630865097045898, -1.072355031967163, -0.3999873697757721, 1.2813969850540161, 0.7170048356056213, 1.810742735862732, 0.41027477383613586, 0.4824664890766144, 3.012652635574341, 0.16619078814983368, 1.0000859498977661, 0.3614916503429413, -0.2023967057466507, 1.010495662689209, 0.7835162281990051, -0.24872905015945435, 1.3061931133270264, -1.4288885593414307, 0.7421146035194397, -0.7931568622589111, 0.4518876373767853, -1.0949623584747314, 0.34234824776649475, 0.18870611488819122, 1.2677758932113647, -0.8444974422454834, -2.265390157699585, -0.06185274198651314, 0.19336232542991638, -0.8680764436721802, -2.3074839115142822, -0.7354870438575745, -0.9809719324111938, -1.752357840538025, 0.5714545249938965, -0.01912108063697815, 0.664630115032196, 0.5728673934936523, 0.219833642244339, -0.8673185706138611, 0.9292516112327576, -0.1568598449230194, 1.3442926406860352, -0.7421299815177917, 0.07702786475419998, 1.4701039791107178, -0.0974046140909195, 1.5421550273895264, -0.679043710231781, 1.3613593578338623, 0.11367709189653397, -1.6094502210617065, -0.25752267241477966, 1.5613831281661987, -1.3944717645645142, -1.369608998298645, 0.1862574964761734, 0.018172604963183403, 0.27321988344192505, -0.9963157773017883, -1.579979658126831, 0.2737351357936859, -1.7574952840805054, -1.5949434041976929, -1.61472487449646, -1.1067607402801514, 0.7492899298667908, -0.08503745496273041, 0.7122204303741455, 0.2706715166568756, 0.007850063033401966, -0.5418627858161926, -0.64145827293396, 0.06797017902135849, 0.6411183476448059, -0.0272870771586895, -0.7127261757850647, -0.22095754742622375, 0.8442266583442688, -1.8663560152053833, 1.7446280717849731, 1.903730869293213, -2.4138479232788086, 1.5024933815002441, 0.6261755228042603, 0.09281141310930252, -0.13437549769878387, -2.55713152885437, -1.4938877820968628, -1.3819706439971924, 1.0322867631912231, -0.2749916613101959, -1.4042280912399292, -0.6833471655845642, -0.5268386006355286, -1.8543044328689575, 0.8262704610824585, -0.7854271531105042, 1.0495383739471436, -0.9639902114868164, -0.1504472941160202, 1.3198344707489014, -0.30269503593444824, -0.48509112000465393, -1.1852679252624512, 0.7443103194236755, -0.9075539708137512, -1.6515334844589233, 0.663058340549469, -0.5879126191139221, 0.182566300034523, -0.8680799007415771, 1.5786216259002686, 0.13445229828357697, -0.3979146182537079, 0.7192748785018921, 0.12995854020118713, -1.086821436882019, -0.14218497276306152, -1.1456854343414307, 2.6470375061035156, -0.3557305335998535, -1.1589295864105225, -0.5720265507698059, 0.7407583594322205, -1.0792360305786133, 1.3062212467193604, 0.18308022618293762, -0.16981345415115356, -0.04088132455945015, -0.6019595265388489, -0.44408291578292847, 0.859630286693573, 0.4853825271129608, -1.9402146339416504, -1.4100971221923828, -0.11524879187345505, -2.044867992401123, 0.8769840002059937, 0.6321463584899902, -0.38144004344940186, -1.5060125589370728, 0.020598987117409706, 0.6618422269821167, 0.16997788846492767, -0.3605930805206299, -1.7105200290679932, 0.4046857953071594, -1.967079758644104, 1.1657793521881104, 0.5112594962120056, -0.018038827925920486, -0.9288132190704346, 1.3483461141586304, 1.0485413074493408, -0.3827648162841797, -0.2701014578342438, -0.555234432220459, -0.41885098814964294, -0.6897352337837219, 1.896166205406189, -1.1344541311264038, -0.02382836863398552, -0.7994005680084229, 0.6801428198814392, 0.8909607529640198, 0.5922026634216309, 0.7965951561927795, 0.23771391808986664, -0.035846780985593796, -1.3027664422988892, 0.3452170193195343, -1.9882875680923462, -0.7369723916053772, 0.4720146059989929, -0.4733935296535492, -0.7788152694702148, -0.524040162563324, 0.5997705459594727, -0.5694648027420044, -0.9415677189826965, 0.3930005729198456, 1.296120524406433, -0.426168829202652, -1.3453551530838013, 1.3207142353057861, 1.9314824342727661, -0.10780905932188034, 0.22982773184776306, 0.20003139972686768, 0.12461425364017487, 0.6659035086631775, 0.12767578661441803, -0.2182074636220932, -0.43459489941596985, -0.5918613076210022, 0.9988341927528381, -0.7271607518196106, -0.37290138006210327, -0.3771292567253113, 0.32723280787467957, -0.5683664679527283, 0.22238336503505707, 1.2985702753067017, 0.258125364780426, 1.3371732234954834, 0.0071812705136835575, 0.09112011641263962, 1.2261885404586792, 0.3418480157852173, -0.04347100481390953, -0.12133342772722244, 0.21582455933094025, 0.713040828704834, 0.53923499584198, 3.068732738494873, -1.5785059928894043, -0.9352181553840637, -0.6362642645835876, -0.2978438436985016, 0.9512513875961304, 1.0418431758880615, 0.6786535978317261, -0.4083537459373474, 0.08347871154546738, -1.1580371856689453, 0.40410128235816956, -0.039874181151390076, 0.816232442855835, 1.224459171295166, 0.9759951829910278, 1.8384356498718262, -2.161680221557617, -0.6590954065322876, -1.6427313089370728, 0.16012068092823029, 0.04002951830625534, 0.13799330592155457, 0.8796966075897217, 0.9136374592781067, -0.5432272553443909, -0.2195831835269928, -0.9469507336616516, 0.5607894062995911, -1.4978930950164795, -0.31276771426200867, 1.6401700973510742, -0.28846392035484314, 0.5154373645782471, 0.7305457592010498, -0.6721327900886536, 0.6842384934425354, 0.1876046508550644, 1.3759702444076538, -0.6175143122673035, 0.3093477189540863, 0.5696824789047241, 1.3797318935394287, -0.8257003426551819, -0.9218422770500183, 1.5867254734039307, 1.2164596319198608, 1.461975336074829, 0.16237664222717285, 0.32539352774620056, 0.019395064562559128, -1.9096379280090332, 0.6159791350364685, 0.30452513694763184, 0.6634471416473389, 0.9174736142158508, -1.2567722797393799, 0.33368247747421265, 2.091384172439575, 1.6914136409759521, -0.6833393573760986, -2.0601184368133545, 2.00093674659729, -1.308437466621399, 0.6728641390800476, 2.1403822898864746, 0.21468330919742584, -0.6510823369026184, 0.6358733177185059, -0.17803598940372467, -1.4352484941482544, 0.7377220392227173, -0.7223212122917175, 0.3793250322341919, -0.02566125988960266, 0.47477784752845764, 1.2583253383636475, 1.4736955165863037, 0.19483451545238495, -0.4480207562446594, 0.16877381503582, -0.09826064854860306, -0.13761112093925476, 1.7309496402740479, -0.001768771093338728, 0.7532408237457275, 0.04729762673377991, 0.29977864027023315, 0.7949780225753784, -0.3526632785797119, -0.5248317718505859, 1.0981560945510864, -0.015276744961738586, -0.2044273018836975, -0.042265091091394424, -0.6541739106178284, 0.5740534067153931, -0.09851448237895966, 0.32112598419189453, -0.11383029818534851, 0.17015327513217926, 1.0762755870819092, 0.6053504347801208, 1.3053919076919556, -1.2308696508407593, -0.32480472326278687, -0.4632624387741089, -0.041389767080545425, -0.5728558897972107, -1.7223336696624756, 2.087956428527832, 2.5554330348968506, -1.1732819080352783, 0.2670508325099945, 0.7104794383049011, -0.18787603080272675, 0.5619415044784546, -0.40582913160324097, 1.0604286193847656, 0.28701308369636536, -0.06519634276628494, 1.6962676048278809, 1.3193559646606445, -2.2798736095428467, -0.5839396715164185, 1.7028425931930542, 0.10461696237325668, 1.9433493614196777, 0.8362424373626709, -0.34767937660217285, 0.006926398258656263, -0.3471296727657318, 0.6537227034568787, 0.16200143098831177, 0.2773280143737793, 0.7683096528053284, 0.894016444683075, -0.07654394954442978, 0.13805082440376282, -1.7436693906784058, 0.5354548096656799, 0.12138638645410538, -0.8611485958099365, 1.0695382356643677, 0.5924605131149292, -0.7788732051849365, -0.5590535402297974, -1.5035755634307861, -1.0672508478164673, -0.5374922156333923, 0.29881277680397034, -0.5612148642539978, -0.3703532814979553, 1.726162314414978, 0.3157268464565277, -0.06192810833454132, -0.7795865535736084, 1.557127833366394, -1.583722710609436, -0.2775581181049347, -1.0890235900878906, -0.7409013509750366, 0.4638313055038452, 1.996658444404602, 1.0929288864135742, -0.431491494178772, 0.6205231547355652, 0.17590928077697754, -0.5952318906784058, 0.11475231498479843, -2.036471128463745, 0.28040945529937744, 0.03413238748908043, 1.088208794593811, -0.012674630619585514, 0.9985243082046509, -0.9578949809074402, -0.5217080116271973, 0.5961291790008545, -0.7859772443771362, -0.9121127128601074, -0.4504487216472626, 0.4420032501220703, -0.6466359496116638, 1.2482227087020874, -1.0853031873703003, -0.9501426219940186, 0.7269812822341919, -1.5294103622436523, -0.07537299394607544, -1.6090097427368164, 0.751777708530426, 1.7055331468582153, -0.8001559376716614, -2.4682059288024902, 0.09215249121189117, 0.7547053694725037, 0.5485462546348572, -0.607594907283783, -0.9850910902023315, -0.48332276940345764, -0.6893576383590698, -0.8738300204277039, 0.38859838247299194, 1.7173824310302734, -0.8831259608268738, -1.164291501045227, -0.1882241666316986, 0.6509137749671936, -0.8867226243019104, 0.6653281450271606, 0.002030285308137536, -0.908387303352356, 1.6295623779296875, 1.3086249828338623, 0.009082441218197346, 0.7946656346321106, 0.18770098686218262, 1.203033685684204, 0.4713239073753357, 0.4092232286930084, -1.4494459629058838, -0.7270901799201965, -0.07256567478179932, -0.14241017401218414, 0.022803260013461113, -0.7500770688056946, -0.9516313076019287, 0.34297800064086914, 0.4458455443382263, 0.7661991715431213, 1.1349914073944092, 1.1306532621383667, 0.25928354263305664, -0.0002843410475179553, -0.4312203526496887, 0.10743999481201172, -0.8891229033470154, -1.6850247383117676, -1.584228754043579, 0.15456682443618774, -0.1546386480331421, -0.15783202648162842, -0.8777320384979248, -0.037032414227724075, 1.2546104192733765, -0.8321900963783264, -2.214268445968628, 0.4808541238307953, -1.229101538658142, -0.6504517793655396, 0.9095240235328674, 0.2299877405166626, 0.6322543025016785, 0.3284277319908142, -1.0316917896270752, -2.73276948928833, 0.7503261566162109, -0.346142053604126, -0.35697171092033386, -1.1910796165466309, 0.9255087971687317, 1.9260512590408325, 0.4002261459827423, -0.8917455077171326, 1.3115112781524658, -1.4666800498962402, -0.3649972677230835, -0.08881882578134537, -0.6410139799118042, -2.5335545539855957, 1.0657093524932861, -0.600972056388855, -1.0613300800323486, 0.5449081063270569, -0.4343937933444977, 0.18945351243019104, 0.29129886627197266, -0.2724437117576599, -0.38050195574760437, 0.3358308970928192, -1.339819073677063, -1.5262221097946167, -1.4525692462921143, -0.4510107934474945, -0.14587567746639252, 0.5283639430999756, 0.17468658089637756, 0.17933456599712372, 0.2221137285232544, -1.1266744136810303, -0.1051313504576683, -0.873723566532135, 0.4381626844406128, 1.080362319946289, 0.8627524971961975, 0.4496869146823883, 0.19981156289577484, 0.4317374527454376, 0.6895586252212524, 1.5775879621505737, -0.22006721794605255, -0.15730836987495422, -0.5110849142074585, -1.4276396036148071, -1.1681619882583618, -1.525071382522583, 1.8545840978622437, -0.9103800654411316, -0.04085448011755943, -0.5616403222084045, -1.5922490358352661, 0.986118733882904, -0.6918590068817139, -0.1419801115989685, -0.22656658291816711, 0.40204891562461853, -0.4699958562850952, 0.6191859245300293, -0.7287806868553162, 0.6839165091514587, -0.2743101418018341, -2.0915727615356445, -1.1028764247894287, 0.1731736660003662, -1.6006501913070679, -2.0198700428009033, -1.5102028846740723, -1.007319688796997, 2.4855995178222656, 0.8229401111602783, 1.8202025890350342, 0.7992207407951355, -0.03719019517302513, -0.49542444944381714, 0.15615765750408173, 0.6253565549850464, 0.012165190652012825, -0.6354703307151794, 2.2194294929504395, -0.046429019421339035, 0.02095888927578926, 1.7580666542053223, 0.8092913627624512, -1.51596200466156, -1.2852587699890137, -0.032599493861198425, 0.4423794448375702, -0.34033915400505066, 1.1180541515350342, -0.00841656606644392, 0.7156572341918945, -0.7633605003356934, -0.7186384201049805, 1.1095906496047974, 1.3292911052703857, 0.595118522644043, -0.5654356479644775, 0.3257119059562683, 1.4605708122253418, 1.4499213695526123, -1.352813720703125, -1.2016384601593018, 0.01355869323015213, 0.05516723915934563, 0.8783918023109436, 0.959931492805481, 0.13243193924427032, 0.28003352880477905, 0.7485523819923401, 0.18735739588737488, -1.2420198917388916, -1.291786551475525, 0.21709857881069183, 2.0204017162323, 0.7938749194145203, 1.0890958309173584, -0.29924118518829346, -2.308687925338745, -0.03626659885048866, -0.18261651694774628, 0.0032002802472561598, -1.2195217609405518, -0.1939060091972351, 0.8234671354293823, 1.6347445249557495, 0.4036041498184204, 0.39507609605789185, 0.17721039056777954, -0.9100268483161926, -0.3596411645412445, -0.26469936966896057, 1.5761511325836182, -3.9197092056274414, -0.6126022338867188, -0.8041141629219055, -1.061119556427002, 0.5522990822792053, 0.7825334668159485, 1.4085686206817627, 0.0024512542877346277, -1.6878787279129028, -0.8511314392089844, 0.18401537835597992, 0.8386707901954651, 0.2822418808937073, -0.5396996140480042, 0.1808014065027237, -0.4994218945503235, -0.22508060932159424, 1.1403872966766357, -2.2695140838623047, -0.6032655239105225, -0.7672266960144043, 0.5061785578727722, 2.1114001274108887, -2.3541340827941895, -1.013332486152649, 0.1625913679599762, -0.9595036506652832, 0.30614373087882996, 0.7945483326911926, -1.5974878072738647, 0.7165554761886597, -0.2257639616727829, -0.021571796387434006, -0.2929377257823944, -0.23988962173461914, 2.751227855682373, 0.3826630711555481, 1.0674245357513428, 0.28986674547195435, 0.518021821975708, 1.2723371982574463, -0.662909209728241, -1.2979726791381836, 0.7580407857894897, 0.6656416654586792, -1.6320675611495972, -0.8643480539321899, 0.18187911808490753, 0.4671042263507843, 0.4055338501930237, -1.543099284172058, -0.18788638710975647, -1.217750906944275, 0.3766601085662842, 2.2421860694885254, 0.03941980376839638, -0.3071788549423218, -0.23928077518939972, -0.8154377937316895, 0.01052848156541586, 1.3554741144180298, -0.8411851525306702, 2.331967353820801, -0.27824434638023376, 1.2044535875320435, -1.159482479095459, -1.9243143796920776, 0.06555535644292831, -0.03133418783545494, -1.0381734371185303, -0.6521029472351074, 0.508124828338623, 1.5517966747283936, -0.5541666746139526, -0.9897966384887695, -1.2487741708755493, -0.22753062844276428, 0.6494795680046082, -0.5320113301277161, -0.4870738387107849, 0.8156510591506958, 0.10003272444009781, 0.35004135966300964, 0.8651309609413147, 0.7539592981338501, 0.2385191023349762, -0.6552555561065674, 0.33073052763938904, -0.543036699295044, -0.2533207833766937, -0.42831969261169434, -0.14839515089988708, 1.6365742683410645, -0.19941161572933197, 0.23293784260749817, 0.6738090515136719, -0.7314875721931458, 0.9720303416252136, -1.4653974771499634, -0.6592350602149963, 0.1421511173248291, 1.910216212272644, -1.549742341041565, -0.4078516960144043, 1.5997711420059204, 1.053584337234497, 1.7859408855438232, -1.1313652992248535, 2.4087412357330322, -2.2173540592193604, -0.06847947835922241, -1.024049162864685, 0.593036413192749, 0.8236230611801147, -0.4069616198539734, 1.041324496269226, -0.21346847712993622, 0.9091182351112366, 0.5962917804718018, -0.8238800764083862, 0.4775530993938446, 0.6694663166999817, -0.36103498935699463, -0.03846059367060661, -0.40468671917915344, 0.5957928895950317, 1.3920360803604126, -1.4080992937088013, 0.21975332498550415, 0.8396186232566833, 0.15805433690547943, -1.4824342727661133, 2.2227730751037598, -0.20623640716075897, -1.005202293395996, -0.4450741708278656, 0.11670470237731934, -0.3691682517528534, 1.7850364446640015, 0.07792910188436508, -0.6856808662414551, -0.9176415801048279, 0.48352593183517456, -0.633413553237915, 0.1327086091041565, -0.8241153359413147, 0.2931019961833954, 0.7750788331031799, -0.5790925621986389, 0.4881472587585449, 0.7499807476997375, 0.7329269051551819, 0.7705988883972168, -1.6419456005096436, -1.7565340995788574, -0.23417878150939941, -0.5437119007110596, -0.08085945248603821, 1.5211244821548462, -0.5203851461410522, 0.814262866973877, -1.5638355016708374, -0.024723459035158157, -0.8162997364997864, -1.7707221508026123, -4.328641414642334, -1.3253412246704102, -2.444735527038574, -0.11109892278909683, -0.42090100049972534, -0.7292594909667969, -0.24485933780670166, 0.5901467204093933, -0.8077148795127869, -0.27761003375053406, -0.621180534362793, -0.4502420723438263, -1.7577877044677734, 0.5437026023864746, -0.5644928216934204, 0.6875435709953308, -0.1046062484383583, 0.26357486844062805, -0.017275162041187286, 0.6828693151473999, 1.233978271484375, -0.032850176095962524, -1.5730105638504028, 0.30548280477523804, -0.5656747817993164, -1.0911062955856323, -0.5926761031150818, 0.8139303922653198, -0.14492519199848175, -0.13352051377296448, -0.3265376091003418, -0.9742162227630615, -0.17184410989284515, -0.04363700747489929, -0.6923844814300537, -0.3963329792022705, -1.4472553730010986, -1.268234133720398, -1.454984188079834, 1.8062866926193237, -0.20145024359226227, -2.261786699295044, 0.776964008808136, 1.0869441032409668, 1.290859341621399, 0.9177363514900208, -0.7159184217453003, -0.7632575035095215, 0.27804797887802124, 1.5566818714141846, -0.7815806865692139, -1.8923548460006714, -1.789686918258667, 0.4748052954673767, 1.6410490274429321, 1.13319993019104, 0.29381483793258667, 0.7751927971839905, -0.9349760413169861, 0.9484042525291443, 1.6219003200531006, 2.459155797958374, 0.7489866018295288, 0.9005147814750671, 0.062035974115133286, 2.393781900405884, 0.2956370711326599, -0.16107864677906036, -0.07009933143854141, -1.3390686511993408, 1.6953961849212646, 2.534118890762329, -0.6189637780189514, 0.46932002902030945, -0.09840522706508636, -0.8212417364120483, 0.4202954173088074, -0.6147297024726868, 0.9842472672462463, 0.6720881462097168, -1.2241047620773315, 0.2463453710079193, 0.9443042278289795, -2.351207971572876, 0.502670407295227, 0.9215821623802185, 2.1151840686798096, 0.002632886404171586, -1.0384889841079712, 0.5408655405044556, -0.27970972657203674, 1.0150132179260254, 0.8692042231559753, 0.5634223818778992, 0.6003310084342957, 0.24807126820087433, 0.16399388015270233, 0.3506394028663635, 2.3679916858673096, -0.6162742376327515, -0.4349123239517212, 0.20908933877944946, 0.09119782596826553, 0.0019072374561801553, -1.3673983812332153, 2.5920917987823486, -0.39161789417266846, -1.3333693742752075, -3.4948008060455322, 0.41134512424468994, -0.9121713042259216, 0.16273453831672668, -1.1657832860946655, 0.48403316736221313, 2.7495179176330566, 1.7119132280349731, 0.4656729996204376, 0.0677957609295845, -2.4303925037384033, -0.07817843556404114, -0.34814488887786865, 2.44262433052063, -0.8837555050849915, 0.40336254239082336, 0.6288039088249207, 3.018580198287964, 1.8028852939605713, -1.538084626197815, 1.9526116847991943, -1.2156082391738892, 1.696553111076355, 0.7383123636245728, -0.3691689670085907, 0.034137554466724396, 1.3627146482467651, 0.729246973991394, 1.0733598470687866, -0.907513439655304, 1.500562310218811, -0.43677547574043274, -0.42654457688331604, -0.7134706974029541, 1.3276137113571167, 0.2687641680240631, -0.0069388519041240215, -0.016386277973651886, 0.48725399374961853, -0.8874114751815796, 0.9586286544799805, 0.3082674741744995, 1.230055332183838, -0.05215194448828697, 1.2911763191223145, -0.5173385739326477, 0.726052463054657, 0.742920994758606, -0.4578796625137329, 0.5670557022094727, -0.02127815969288349, 1.60392427444458, 0.3773151934146881, 1.2642745971679688, -0.12168988585472107, 0.6447136998176575, 1.4691718816757202, -0.6913796663284302, -0.11392999440431595, 0.45350348949432373, -0.08332592993974686, -0.35108429193496704, -0.5605409145355225, -0.17387111485004425, -0.8380880951881409, -0.8695660829544067, -0.6075604557991028, -0.2653767466545105, -0.13462921977043152, 0.6956328749656677, 0.11827696114778519, 1.3238677978515625, 0.5585618615150452, 0.13840794563293457, 2.602583169937134, -1.5549356937408447, 0.6377347111701965, 0.5960203409194946, 1.6375268697738647, 0.15513262152671814, 0.4567921459674835, -0.4292254149913788, 1.784136414527893, -0.12126609683036804, -1.0265613794326782, -0.4565986394882202, -0.4594695568084717, 0.7368979454040527, 0.9100158214569092, -1.276665210723877, 0.4148523211479187, -1.6968210935592651, -1.351560354232788, -0.18682131171226501, -0.5142074227333069, -0.3011290431022644, 2.2731571197509766, 0.8375484943389893, -0.32794392108917236, 0.205735445022583, -0.6906577348709106, 1.5737204551696777, 2.005091428756714, -1.2471239566802979, 1.5935062170028687, -1.3950804471969604, 0.9960892200469971, -0.7299835681915283, 2.3255534172058105, -0.5412567853927612, 0.5455570220947266, 2.2378029823303223, -1.6251262426376343, 1.7107445001602173, 0.42388826608657837, 0.33434736728668213, -0.3717971444129944, -0.8811413049697876, 1.529600739479065, 0.7243282198905945, 1.2925324440002441, -0.705159068107605, -0.421609103679657, 0.24309982359409332, -0.2622177004814148, 0.7753819823265076, 0.11738326400518417, 0.10458748787641525, 1.2036923170089722, -1.888039469718933, 1.6690229177474976, -0.11316104233264923, -0.916526198387146, -0.7359578609466553, -0.9281726479530334, -2.4768009185791016, 1.6946358680725098, -0.912028431892395, 0.8479046821594238, -0.7083905935287476, 0.037962645292282104, 1.3392080068588257, -0.7989212870597839, 0.7015058398246765, -0.45811334252357483, -0.2538397014141083, 0.15422123670578003, -0.14562644064426422, -0.32508164644241333, -1.1156513690948486, -0.2566262483596802, -0.5670806169509888, -0.10088489204645157, 0.019552262499928474, -0.5158993005752563, 0.014494321309030056, -2.5318214893341064, 0.33817094564437866, 1.6385278701782227, -0.2510499954223633, -0.5862126350402832, 1.2712299823760986, -1.0238384008407593, -0.434513658285141, 0.69855797290802, -0.5787273645401001, -1.5537508726119995, 0.08531889319419861, 0.40340062975883484, 1.2307913303375244, 0.5524911880493164, -0.6704019904136658, -3.353778600692749, -1.6925020217895508, -2.2290451526641846, 1.266496181488037, -0.08488302677869797, 1.6351675987243652, 1.4193686246871948, -1.723422884941101, -0.4568781852722168, -0.1516626477241516, -0.8626931309700012, 1.121511697769165, -1.0797679424285889, -0.20173379778862, 0.12145373970270157, 0.13038083910942078, -0.19413620233535767, -1.3621406555175781, -1.191152811050415, -0.24525471031665802, -0.0826239287853241, -0.9682611227035522, -1.0534688234329224, -0.3184824585914612, 0.48879748582839966, 0.10884618014097214, -2.1420981884002686, -0.7941411137580872, 0.839931070804596, 0.6152734756469727, -0.9745715856552124, -1.6879982948303223, 0.8175063133239746, -1.1270784139633179, 0.2919715344905853, -1.443670392036438, 0.16676326096057892, -0.33140015602111816, 0.003046596422791481, 1.093130111694336, -2.2142632007598877, -0.1906769573688507, 1.1449005603790283, 0.4799747169017792, 2.018815279006958, -2.2024946212768555, -1.5959937572479248, 0.43220359086990356, 0.034826021641492844, 0.6537448167800903, -1.8642925024032593, -0.26793399453163147, 2.2446348667144775, 2.0199649333953857, 0.5631294846534729, 0.2628641724586487, 1.2098603248596191, -1.5898354053497314, -0.28817543387413025, 0.2216469645500183, -0.850673258304596, 0.5260480642318726, -0.1098775565624237, -0.5599664449691772, -0.5032034516334534, 0.25935691595077515, -0.12529096007347107, 0.760617196559906, -0.40503308176994324, -0.6574072241783142, 0.16706927120685577, -0.6729052662849426, 1.393481731414795, 0.02709418535232544, -0.3127524256706238, 1.2121524810791016, 0.10799434036016464, -0.42861229181289673, 1.4603843688964844, 0.9763001799583435, -0.12166418880224228, -1.0453481674194336, -0.7821426391601562, -0.17259101569652557, -0.3402974307537079, 1.4981839656829834, 1.0540204048156738, 0.34608063101768494, -1.3115025758743286, 1.4501042366027832, -0.1960432082414627, -2.453322649002075, 0.5184078812599182, -0.9747669696807861, 1.3228412866592407, 1.1063660383224487, -2.0186164379119873, -0.013549691997468472, 0.40854862332344055, -0.19389723241329193, -0.5190427899360657, 0.5775927305221558, 0.8642734885215759, 0.8899497985839844, 0.9581559300422668, -0.15784284472465515, 1.0820409059524536, 0.025979798287153244, 0.8698654770851135, 0.7598260045051575, -0.11439201980829239, -0.38178959488868713, -0.9464532136917114, -0.039563797414302826, -0.7141174077987671, -0.11342233419418335, -1.8589919805526733, 0.6305682063102722, -1.07219660282135, -0.4079083502292633, -0.03030594438314438, -0.7060728073120117, 1.4185950756072998, 0.7092664837837219, -0.06203262880444527, 0.9870634078979492, -0.14188195765018463, 1.129599928855896, -1.090379238128662, 0.4801802933216095, -1.4893603324890137, 0.8262092471122742, 1.0613683462142944, 1.517349362373352, 0.975824773311615, 0.0016319642309099436, -0.20061299204826355, -0.5583778619766235, -1.842280626296997, -0.042002175003290176, -1.3025190830230713, 0.4153441786766052, -0.8490922451019287, -0.8070403337478638, -0.3633963465690613, 0.298326700925827, 0.2798169255256653, -0.2735624611377716, 1.0907340049743652, -0.4256773591041565, -0.9726285934448242, 0.6910850405693054, -0.8344722986221313, 0.05095108225941658, -0.889315664768219, -1.155208945274353, 0.8459396958351135, -0.4638659358024597, 1.1193922758102417, -0.9239888191223145, 1.966326355934143, -1.3129150867462158, 0.7826958298683167, -1.4553128480911255, -1.7625993490219116, -0.2428646832704544, 0.14186741411685944, 1.4449794292449951, 0.0086634187027812, 0.4130278527736664, -0.9637925624847412, 1.705841302871704, 1.516640305519104, -0.7900259494781494, 0.35094594955444336, 1.5532965660095215, -2.224153995513916, 1.0541266202926636, 0.591279149055481, 0.06175154820084572, 1.8245809078216553, -1.2255252599716187, -1.1726512908935547, -0.2027287632226944, -2.1949191093444824, 1.1719633340835571, 0.637516438961029, -0.024376550689339638, 1.6883078813552856, 0.8333138227462769, 0.3582194745540619, -0.6548186540603638, 0.12873528897762299, -0.8520596623420715, 0.9818164706230164, 0.4100232422351837, -0.20641107857227325, 0.08682047575712204, -1.115471601486206, -2.9892399311065674, 3.542114496231079, -1.7674076557159424, 0.8174816370010376, -1.0969910621643066, -2.701688289642334, -0.9752354025840759, -1.3955717086791992, 0.23236483335494995, 0.23697762191295624, -1.0851150751113892, 0.8614394068717957, 0.0677686259150505, 0.3034455478191376, -0.09442129731178284, 1.2930554151535034, -0.45367780327796936, -0.4962487518787384, -0.2916106581687927, -0.09200630336999893, -0.45783311128616333, 0.10924796015024185, -0.8550676107406616, -0.45592042803764343, -0.39981693029403687, -1.6421403884887695, 1.472041130065918, 0.5269061326980591, 1.5372964143753052, 0.8001623153686523, 0.2939601540565491, 0.7154644727706909, 1.5369254350662231, -1.2900270223617554, -0.5707549452781677, -1.1122547388076782, 0.21569503843784332, -0.3381659984588623, 0.7126126289367676, -1.3686927556991577, 0.7477473020553589, -0.5496833920478821, 1.3857892751693726, -1.1484578847885132, -0.5962045192718506, -0.9914392232894897, -0.10595346242189407, -1.1202574968338013, 1.7225362062454224, 0.33728155493736267, 0.9846854209899902, 2.406386137008667, 0.3399258553981781, 1.3836567401885986, 1.1372333765029907, 0.6396433115005493, -1.5297937393188477, 0.7172077298164368, -0.8186725974082947, -0.14087674021720886, 1.293399453163147, 0.4011270999908447, -1.068786859512329, -0.6772564649581909, 0.8706538081169128, 0.31057941913604736, 0.38905808329582214, 1.0892409086227417, 2.461498260498047, -1.0158271789550781, 0.8979408144950867, 2.447990894317627, 0.6054343581199646, -0.4270574450492859, -0.15836189687252045, -1.752620816230774, -2.0658040046691895, 0.9072744846343994, -0.6858612298965454, -1.2172393798828125, -1.2838134765625, -0.8961019515991211, 0.5102134346961975, -0.7511441111564636, 1.6890512704849243, -0.8881537318229675, -0.7394794821739197, 0.6264462471008301, -0.7902089357376099, -1.511704683303833, -0.4372626543045044, 0.27763062715530396, 0.08309608697891235, 0.5930123925209045, 0.17968180775642395, 1.2084969282150269, 0.7142012119293213, 0.47120359539985657, 0.48752301931381226, 1.1165107488632202, 0.6326587200164795, -0.32622501254081726, 0.9577146768569946, 0.4675089716911316, 0.46999549865722656, 0.17435772716999054, -0.15770037472248077, 1.600019931793213, -0.1851649135351181, 1.030769944190979, -0.1526680290699005, -0.5361921191215515, 0.455229789018631, -0.9443266987800598, -0.4250282347202301, -1.0897645950317383, -0.08395856618881226, 0.09546716511249542, -0.6550833582878113, -0.06921686232089996, -2.1521360874176025, 0.289096862077713, 0.39696168899536133, 1.1988903284072876, -1.614187240600586, 0.35427841544151306, 1.4556254148483276, 0.9941732287406921, 1.2994881868362427, 0.5422720313072205, -1.9488946199417114, -0.6224366426467896, 1.169036865234375, 0.9215577840805054, 0.5288643836975098, 0.32586386799812317, -0.39702484011650085, 0.8000515103340149, 0.36693274974823, 1.1219595670700073, -0.35844582319259644, 2.2555978298187256, 1.2800511121749878, -1.1237077713012695, 0.8193097710609436, -0.1625373214483261, -0.26082584261894226, 0.5632689595222473, -1.4943480491638184, 0.07031914591789246, 0.11703447997570038, -0.9114280939102173, -0.3730161488056183, -1.5562652349472046, 0.6893341541290283, -0.4355567693710327, 0.6919915080070496, 0.7277560234069824, -1.7995059490203857, 0.855806291103363, -0.9313811659812927, -0.7228935956954956, -2.2194149494171143, 0.29674476385116577, -0.5033010840415955, -0.6259127855300903, -0.2131485939025879, -0.01965133473277092, 0.2793060839176178, 2.051589250564575, 0.06123819202184677, 0.3564727008342743, -0.44689297676086426, -0.6235848665237427, -0.22517123818397522, -2.452202081680298, -0.22393636405467987, 0.4394991397857666, -1.4645214080810547, -0.8150963187217712, 0.31941282749176025, -0.5302253365516663, -1.794568657875061, 1.3119635581970215, -0.35986819863319397, -1.7413026094436646, -2.5262227058410645, -0.7567675113677979, -1.14871084690094, -0.4036608338356018, 1.4539659023284912, 0.28479060530662537, -0.51927250623703, -1.024119257926941, 1.879127025604248, -0.5479975938796997, 0.7678052186965942, 1.33633553981781, -0.2670019567012787, -0.529273509979248, -1.387054204940796, -0.30854329466819763, -0.75698322057724, 0.08590806275606155, -0.6079868674278259, -0.5004960894584656, 1.1167258024215698, 1.5138331651687622, 1.7130663394927979, -0.6911721229553223, -0.962782621383667, -1.632895827293396, -1.6687183380126953, 0.8809843063354492, -1.1979708671569824, 0.3370113968849182, 0.04213747754693031, 0.48272547125816345, -0.7160710692405701, -0.6606305837631226, -0.06157630681991577, 1.1543612480163574, -1.470465064048767, -1.2873470783233643, -0.7009984850883484, 0.2996892035007477, 1.0957826375961304, -1.4940627813339233, -0.05575411394238472, -1.0959182977676392, -0.32785114645957947, 0.6322129964828491, 0.26800858974456787, 0.582003116607666, 0.2698400616645813, 0.02512253448367119, 0.576443076133728, -2.8894777297973633, 0.6639431715011597, -0.1864013820886612, 0.4969746768474579, -0.13029706478118896, 0.5122206211090088, 0.7829774022102356, 1.2277448177337646, -1.6374313831329346, 1.8445013761520386, -0.41800037026405334, 0.649429202079773, 1.9261484146118164, 0.9263374209403992, -1.9122101068496704, 0.9655041694641113, -1.8622549772262573, 0.058327171951532364, -0.4504464566707611, -0.6925750970840454, 0.6346113681793213, 0.09840796887874603, -0.2505984902381897, -1.4944854974746704, 0.5331097841262817, 1.4931517839431763, -1.8126407861709595, 0.8528164029121399, 0.3650186359882355, -0.36006298661231995, -0.5648062825202942, -0.7366637587547302, -0.7840087413787842, 0.5124313235282898, -0.10063878446817398, 1.4790695905685425, -2.134111166000366, -0.8140696883201599, 0.2802533805370331, -1.436375379562378, 0.9878347516059875, 1.7879664897918701, -0.38988226652145386, -0.19425418972969055, 1.1128796339035034, 1.4376001358032227, 0.6895556449890137, 1.3115507364273071, 1.5705524682998657, 1.7698479890823364, 0.3270350694656372, 1.6058558225631714, -0.7104003429412842, 0.3732537627220154, -0.42398443818092346, 1.3072396516799927, -1.4475963115692139, 0.15191805362701416, 0.3104618489742279, -0.15462306141853333, -0.18241499364376068, 0.559843122959137, -0.11965017020702362, 1.1748017072677612, 1.487854242324829, 1.159572720527649, -0.8956865072250366, -0.08648327738046646, -2.045701265335083, -1.683245062828064, 0.01907728984951973, 0.5109286904335022, -0.7845107913017273, -1.2301349639892578, -0.6647868752479553, -1.5197031497955322, 0.017875129356980324, -0.5976621508598328, -2.0096116065979004, -0.929115891456604, -0.7754124999046326, -0.8629195094108582, -0.7864391207695007, -0.06939007341861725, 0.8035227060317993, 0.4919387996196747, -0.4462968409061432, -0.6404815316200256, 1.7891521453857422, 0.2658263146877289, 0.3202068507671356, 1.586545467376709, 0.6603503823280334, 1.9188226461410522, -1.6799966096878052, -1.8105132579803467, -0.8580234050750732, -1.380062460899353, 2.2442057132720947, -0.6897455453872681, -2.275033950805664, -0.8846672773361206, 0.365910142660141, -0.7145013809204102, 0.28189441561698914, -1.431562066078186, -1.2470133304595947, 0.41235029697418213, -0.06779691576957703, -0.5757434368133545, 0.02770994044840336, -0.06211000680923462, 0.5611353516578674, 0.4413285255432129, 1.4868718385696411, 0.41868722438812256, 0.6263659596443176, -1.5724990367889404, -1.663069486618042, -0.4902759790420532, 0.6150765419006348, 0.1042228639125824, -0.024373607710003853, -0.9237328767776489, 0.026173172518610954, 1.1245758533477783, -0.6921125650405884, -0.8195111751556396, -1.7504698038101196, -0.6515353322029114, -0.3805072605609894, -1.823409914970398, 0.8706273436546326, 0.4976659417152405, -0.8823206424713135, 0.07830753922462463, 0.49648135900497437, -0.7182066440582275, -0.1536441147327423, 0.11537772417068481, 0.523762047290802, 0.5173574686050415, 0.8527370691299438, 2.0866029262542725, -0.3363085985183716, 1.0401453971862793, -0.4581357538700104, -1.148136854171753, 0.08564889430999756, 0.07380958646535873, -1.5257829427719116, -0.9149093627929688, -0.1932615339756012, 0.38080692291259766, 2.0274851322174072, -0.7252616286277771, -1.5650551319122314, -0.32824817299842834, 0.8782321810722351, 2.182325601577759, 0.4193088412284851, -2.3614745140075684, 1.216450810432434, 1.6618993282318115, 1.7113730907440186, -0.4255218207836151, 0.030178196728229523, -1.8895165920257568, 1.2980972528457642, 0.6007267236709595, -0.1771262139081955, 0.833277702331543, -0.2722800076007843, -0.3595257103443146, -0.22903412580490112, 0.1702183336019516, -1.078304409980774, -2.1577274799346924, 0.22863705456256866, 1.3573588132858276, 1.5445433855056763, 0.5619054436683655, 1.1252737045288086, 1.606216549873352, 0.43573427200317383, 0.8407434225082397, 1.3004975318908691, 0.4935252368450165, -0.5949975848197937, 0.31335243582725525, 0.16096125543117523, 0.14204800128936768, 0.7478093504905701, -1.0863491296768188, 0.9596260786056519, 0.7867115139961243, 0.31901371479034424, -0.11877285689115524, 1.3378021717071533, 2.0137548446655273, -0.3712168037891388, -1.5321075916290283, -1.1430693864822388, -0.3314788341522217, -0.3842211365699768, 0.8765358924865723, -0.5745334029197693, 0.1885046660900116, 0.2837581932544708, 3.071958541870117, -0.11268796771764755, 0.14229917526245117, 1.069077730178833, -0.7510429620742798, -0.9265354871749878, -0.7113431692123413, -0.35925400257110596, 0.6778660416603088, 0.9426876306533813, 1.6766098737716675, 1.2779165506362915, -0.31710004806518555, -1.6616640090942383, -1.3089653253555298, -0.3008764684200287, -0.03444807603955269, 0.7240080833435059, 0.11197587847709656, -1.1138144731521606, -1.851888656616211, 1.3952076435089111, -0.024597207084298134, 0.07881337404251099, 0.5313788652420044, -1.5730940103530884, 1.0594286918640137, -1.2377936840057373, 1.2943023443222046, 1.4278265237808228, 0.3811141848564148, -0.5133825540542603, 1.8661160469055176, -0.5713313817977905, 1.516245722770691, 1.20426607131958, -0.10349559783935547, 0.12563441693782806, -2.276411533355713, -0.5636417865753174, -0.4104878306388855, 1.2810746431350708, 0.9084352850914001, 1.3650329113006592, 0.024146635085344315, -0.24422191083431244, -2.83095645904541, 0.7452163100242615, -0.22880879044532776, -1.0110176801681519, -0.3878837525844574, -0.08012175559997559, 1.1617624759674072, -0.8069004416465759, 0.2210971713066101, -2.3909196853637695, -0.27199143171310425, -0.3777148723602295, -1.4641474485397339, -2.8497610092163086, -0.891465961933136, -0.8234049677848816, -2.7338201999664307, 0.6009719967842102, -0.4058872163295746, -0.6787652373313904, 1.2369132041931152, 0.9295976161956787, -0.3186136782169342, -0.7934565544128418, 0.277022123336792, -1.046236515045166, 0.9054685831069946, -2.3325891494750977, -0.7044200301170349, -1.241936445236206, -0.10988456010818481, -0.31899333000183105, 0.9395198822021484, 0.38967394828796387, 0.6985544562339783, 0.23497556149959564, -1.0637973546981812, 1.8813871145248413, 2.0360217094421387, -0.4779520630836487, 0.6147656440734863, 1.836039662361145, 1.1437299251556396, -2.2029969692230225, 0.4792788326740265, 0.4805946946144104, 0.9303085803985596, 0.5027316212654114, 0.6575043797492981, 0.08897364884614944, 2.0436861515045166, 1.558633804321289, -0.9994270205497742, 0.07292052358388901, -1.7321908473968506, -1.3426542282104492, 0.711543083190918, -0.25130075216293335, 0.5291412472724915, 0.7461034655570984, -0.677126407623291, 1.3586736917495728, 2.3686087131500244, 0.43525081872940063, -0.3930087387561798, -1.1936806440353394, 0.06481349468231201, 0.5835043787956238, 0.1690351814031601, 0.09360035508871078, 0.2714221179485321, -0.2076287716627121, -0.5550230145454407, 0.13357216119766235, 0.8654305338859558, 0.6796875, 0.07086294889450073, 0.7319794297218323, 1.354130506515503, 0.6365587115287781, 1.2426296472549438, 1.2430466413497925, -0.8043546676635742, 1.1992543935775757, 1.5217297077178955, 0.12138134241104126, -0.01100131031125784, 0.38506069779396057, -0.7052360773086548, 0.4049408733844757, 0.5435400605201721, -0.028224563226103783, -0.5048221945762634, 0.22378285229206085, -0.9987981915473938, 0.5739569067955017, 0.8652845025062561, -0.7466812133789062, 0.5230038166046143, 0.40929195284843445, 0.5329094529151917, -2.8750531673431396, -0.5342400074005127, 1.022341251373291, 0.6901784539222717, -1.2459512948989868, 0.33921849727630615, -0.7978907823562622, -0.922450602054596, 0.36347314715385437, 0.7601443529129028, -0.7667099833488464, 1.9690033197402954, 1.3344306945800781, -1.2528735399246216, -1.5514906644821167, 0.19113977253437042, 1.3783434629440308, -0.7948081493377686, -1.4696946144104004, -0.35708191990852356, 0.41752779483795166, -1.1370303630828857, 0.22744375467300415, -0.05235331133008003, -0.7631418704986572, -1.3243123292922974, 0.7061370015144348, 0.22463706135749817, 1.4127808809280396, -0.8049895763397217, 0.7481067776679993, 0.8327959179878235, 0.63965904712677, 0.30908605456352234, -2.3195600509643555, -1.6609392166137695, 0.5796774625778198, -1.69089937210083, -0.7978219389915466, 0.24784743785858154, 0.33016276359558105, 1.7828028202056885, 2.3906383514404297, -0.07558751851320267, 2.1457836627960205, -2.5708439350128174, 0.649988055229187, -1.2595597505569458, -0.12548696994781494, 1.2241315841674805, 0.10741870850324631, 1.474074125289917, -0.8785473704338074, 0.04826860874891281, -1.1983346939086914, 0.03810315206646919, 0.17190426588058472, -0.6339312791824341, -0.7124321460723877, -0.39757513999938965, 1.3244916200637817, 0.47230157256126404, -1.433543086051941, -0.8559934496879578, 0.8857667446136475, -0.01964356005191803, -0.45831048488616943, -0.611143946647644, -1.8639463186264038, -0.38274550437927246, 1.239626407623291, 0.8711941242218018, 0.6508001089096069, -0.07715432345867157, 0.2712971866130829, 0.0012729136506095529, -0.23890163004398346, -0.2969866693019867, -0.9061903357505798, 2.3321170806884766, 0.005423336755484343, -1.3478167057037354, 2.2288522720336914, -0.6278090476989746, 1.145022988319397, 0.8980129957199097, -0.18021804094314575, -0.002360396320000291, -0.2697117030620575, -0.10871659964323044, -0.5237977504730225, -0.6644230484962463, 0.38805028796195984, 0.6678757667541504, 1.9821492433547974, 1.1307967901229858, 1.8188344240188599, -0.5785654783248901, -0.6075311899185181, 1.1917572021484375, 0.10379127413034439, -1.5501223802566528, 0.488920658826828, 0.3158118724822998, -0.7633405923843384, -0.1580633670091629, -0.9120591878890991, 1.425365924835205, 1.4112426042556763, 0.7406326532363892, -1.26083505153656, 0.7493844628334045, -1.774834156036377, -0.04476257786154747, 0.41102543473243713, 0.25022488832473755, -1.1771924495697021, -0.45836877822875977, 0.9682331681251526, 0.07705933600664139, -0.5635235905647278, 0.3127633035182953, -2.9543395042419434, -1.2150877714157104, -1.914202332496643, 0.7521518468856812, -1.1660996675491333, -0.1562400758266449, -1.3474326133728027, -2.245532512664795, -0.6146785020828247, -0.45428916811943054, -0.6638686656951904, -0.16613304615020752, -1.447970986366272, 0.7270640134811401, -2.2771692276000977, 0.5062506198883057, -1.1940605640411377, 1.171864628791809, 0.333212673664093, 1.3148733377456665, 0.23150084912776947, 1.6402690410614014, -0.1963205188512802, 0.2544623613357544, -1.4000369310379028, 0.8176432251930237, 1.6385384798049927, -0.22711476683616638, 1.4288902282714844, -0.014677307568490505, 0.060539938509464264, 0.9471951723098755, -0.20925067365169525, -0.5215248465538025, 2.071333408355713, 0.20018576085567474, 0.4423724114894867, 0.7524669170379639, 0.10890835523605347, -0.8992941379547119, 0.10279212892055511, 1.1435446739196777, 1.599992275238037, -0.7406899333000183, 0.033451080322265625, -0.3600332736968994, -1.45712149143219, 0.5810490846633911, -0.881488561630249, 1.193497657775879, -0.3505626618862152, -0.530656099319458, -1.9547420740127563, 0.4390125870704651, -0.3057994544506073, 0.7846255302429199, -0.19565926492214203, -0.8816593289375305, 1.5317647457122803, -0.42484334111213684, 0.7017638683319092, -1.0520825386047363, -0.21320411562919617, 0.7849265933036804, 0.25857850909233093, -0.32216429710388184, 0.19279679656028748, 0.8341132998466492, -0.5771589875221252, -2.0243799686431885, 2.209907293319702, 0.5452611446380615, -1.5623090267181396, 1.0317418575286865, -0.13233566284179688, -1.6201074123382568, 0.021075425669550896, 0.09959156066179276, 0.053855620324611664, -0.6144133806228638, 0.03387841209769249, 0.3109234869480133, -0.38092097640037537, -0.1675581932067871, -0.26603081822395325, 0.5928551554679871, 0.06597530841827393, 1.4964545965194702, -1.1346875429153442, 2.5548810958862305, 0.017905423417687416, 0.07391960918903351, -0.8925476670265198, 3.082714319229126, 0.91275554895401, 0.6355981826782227, -0.6322950124740601, 0.1789310872554779, -0.2750292420387268, 0.2771691083908081, -0.49981164932250977, 0.35771510004997253, 0.5280048847198486, -0.1656583994626999, -0.26183661818504333, -0.992017924785614, 0.24654518067836761, 0.006195396184921265, -0.6592616438865662, 1.446196436882019, -0.6471527814865112, -0.42345482110977173, -1.8300385475158691, 1.5853534936904907, 0.726023256778717, 0.5368596911430359, 0.5477771759033203, -0.7270913124084473, -0.03413643315434456, 0.343464195728302, -1.242448329925537, -0.552372932434082, 0.15372011065483093, 0.050260279327631, 1.2820734977722168, -0.40708088874816895, -0.2033417522907257, -0.16121496260166168, -1.9409775632084347e-05, -0.3785199522972107, -0.9340388178825378, -0.09074848890304565, -1.0517184734344482, 1.38197660446167, 1.2478612661361694, 0.6842648983001709, -3.206826686859131, -0.09141498059034348, 0.4569290280342102, -0.9379037618637085, 2.464280605316162, 2.5177197456359863, 0.9469106197357178, 0.07490888983011246, -0.2539083659648895, -1.1311869621276855, -1.102613925933838, -0.44140172004699707, -2.0424277782440186, -0.45451611280441284, 0.2757451832294464, -0.984900712966919, -1.0431756973266602, -0.9576442837715149, -0.525201141834259, 1.2582160234451294, 0.11305955797433853, 0.84386146068573, 1.3257386684417725, -2.094409465789795, 0.5507528185844421, 0.173580139875412, -0.12781284749507904, 0.7128366827964783, 0.2692088782787323, 0.053850140422582626, -0.3840373456478119, -1.5143693685531616, -0.7165938019752502, 2.759735107421875, 0.5848844051361084, 0.5047712326049805, 0.20487761497497559, -2.0482354164123535, -1.4689421653747559, 1.0078288316726685, 1.5304592847824097, 0.8907368779182434, 0.6914755702018738, 0.7372376322746277, -0.44688498973846436, -0.11991582810878754, 0.27526748180389404, 0.7608948945999146, -0.8194654583930969, -0.2908102869987488, -1.9036540985107422, 0.47102460265159607, -0.3551458716392517, 1.5431932210922241, -0.045227061957120895, -1.0207735300064087, -0.013034525327384472, -0.9503298997879028, -1.5156654119491577, -0.7479634881019592, -0.48290199041366577, 0.41120752692222595, 0.20702345669269562, 2.1561508178710938, -0.7000259160995483, 3.6145057678222656, -0.9287084937095642, -0.5239882469177246, -0.2786273956298828, -0.5378299355506897, 0.5589858293533325, 1.8841001987457275, 0.2163640707731247, -0.2714627683162689, 0.25839337706565857, -0.1835300326347351, -0.768202006816864, -0.5524200797080994, 0.23930707573890686, 0.23394668102264404, -1.246278166770935, 2.9852399826049805, -1.054690957069397, 0.26586776971817017, 0.19095396995544434, -0.8727969527244568, 0.5191833972930908, -0.14103399217128754, 0.11923760175704956, 0.12709805369377136, 0.7138258218765259, 0.8574008345603943, -0.3097093999385834, -0.26336002349853516, 0.07256048172712326, -1.078523874282837, 0.9060728549957275, -1.1914781332015991, 1.7345869541168213, 3.6336913108825684, 1.0263615846633911, 0.6148837208747864, 2.15921688079834, 0.9379905462265015, 0.7387180328369141, 0.9052087664604187, 0.9939028024673462, 0.17031674087047577, 0.4500845670700073, 0.2943623960018158, -0.22381357848644257, -0.27409234642982483, 0.8738242387771606, 1.1290748119354248, 2.5373997688293457, -0.729400634765625, -0.6114317178726196, 0.4149996340274811, 0.5330759286880493, -0.05413723364472389, 1.456863522529602, -1.5408519506454468, 0.7985996603965759, -0.2955106198787689, -1.371741771697998, -0.47724080085754395, 0.16250267624855042, -1.5101161003112793, 1.4620012044906616, -0.370998740196228, 0.4181317985057831, -0.49715059995651245, -0.333942174911499, -0.03430953621864319, -1.074343204498291, 0.1741010546684265, 1.3268556594848633, 2.0553791522979736, 0.046627987176179886, 0.19255690276622772, 0.10356263816356659, -1.3236169815063477, -0.6585029363632202, 0.057300012558698654, -0.6526979207992554, 0.948296070098877, -0.3975263237953186, 1.1018292903900146, 1.0079293251037598, -1.1607521772384644, -0.09827545285224915, 0.5610578060150146, 0.49958404898643494, -0.06589479744434357, -0.031309954822063446, -0.613370954990387, -1.3928651809692383, 1.5486094951629639, -1.3432735204696655, 0.724717378616333, -0.9081395268440247, -0.5369228720664978, 1.1296501159667969, -1.0249311923980713, -0.24988670647144318, 0.28946980834007263, -1.1306401491165161, -0.49732571840286255, 1.1037681102752686, 0.46996450424194336, 0.1695302426815033, -0.04932091757655144, 1.2772821187973022, 2.1461291313171387, 0.9974044561386108, 0.3104851245880127, 1.0860892534255981, 1.4941809177398682, -1.0529038906097412, -0.4948250353336334, 0.5643530488014221, -0.4677421748638153, -0.5684733390808105, -0.0811723917722702, 1.0869098901748657, 0.4719606637954712, -0.7129403948783875, 1.8869155645370483, -0.7413946390151978, 0.7385833859443665, -0.41407904028892517, 0.21907036006450653, 0.02591344714164734, 1.1670621633529663, 1.153888463973999, -0.2734474539756775, -0.8681073784828186, -0.26319360733032227, 1.6875927448272705, 1.038454294204712, -1.0153473615646362, -0.09882790595293045, 1.0144301652908325, -0.020261313766241074, -0.07132156938314438, 1.454309105873108, -0.2405928522348404, 0.8597536087036133, 0.4531022906303406, 0.7062154412269592, -0.8207395672798157, 1.0383332967758179, 0.4072713255882263, 0.18544210493564606, 1.3429316282272339, 1.303654432296753, 1.277427315711975, 1.9516372680664062, 0.9024850130081177, 0.9400877356529236, 0.7305089235305786, -1.011582612991333, 0.21652796864509583, 0.010094436816871166, -4.541427915683016e-05, -0.7461628913879395, -0.4169364273548126, 1.4475398063659668, 0.20795759558677673, 1.651172161102295, -1.1985132694244385, 0.7958818078041077, 0.9396705627441406, 0.3889884054660797, 0.8959430456161499, -0.17234477400779724, -0.3444300889968872, 0.7224564552307129, 0.3959786295890808, -0.33059191703796387, 0.4582497477531433, 0.26654213666915894, -0.7586833834648132, 1.0837960243225098, 0.8904561996459961, -0.7760881185531616, 0.4134708046913147, -0.5873234868049622, 0.5675181150436401, 0.48330673575401306, 0.25262951850891113, 0.06778981536626816, 0.18487656116485596, 0.45037370920181274, -0.7463269829750061, 0.6071828603744507, -3.0143935680389404, 0.7747248411178589, -0.7800133228302002, 0.3108677566051483, -1.3719125986099243, 0.2557727098464966, 0.28244462609291077, -1.0322186946868896, 0.33576101064682007, 1.2044909000396729, 0.18280839920043945, -0.10658520460128784, -0.9323208928108215, -1.6919788122177124, -1.8271125555038452, -1.852716088294983, -1.0546703338623047, 1.8708629608154297, -0.876189112663269, -1.6577496528625488, -1.200890302658081, -0.3003220558166504, -0.012895254418253899, -0.6078120470046997, 0.2340686023235321, 1.1348377466201782, 0.9068951606750488, 2.3013365268707275, 0.8911961317062378, 0.2863945960998535, -1.1999574899673462, 1.2285922765731812, -0.7999249696731567, 0.19724087417125702, -1.708537220954895, -0.8951764106750488, 1.9455748796463013, -0.23519329726696014, -0.063608318567276, -0.4364749789237976, -1.275040626525879, -0.7778228521347046, -0.7466776967048645, 0.9244717359542847, -0.07382120192050934, 0.6414586901664734, 0.7929149866104126, 0.019357554614543915, -1.06093168258667, -0.5797887444496155, -1.2282323837280273, -0.6962550282478333, 1.301551103591919, 0.7185831665992737, -0.6000313758850098, -1.1038542985916138, -0.5534628629684448, 0.1448344737291336, 0.753576397895813, -1.1117264032363892, -0.7497447729110718, -0.4135074019432068, -2.059647798538208, 0.6051328778266907, 0.632716715335846, -0.5562853217124939, -0.11056090146303177, -0.5425453782081604, 0.2828802466392517, 0.2997344136238098, 0.9906905293464661, 0.3550625443458557, 0.42572155594825745, 0.4444344639778137, -0.607484757900238, 0.8399993777275085, -0.23906023800373077, 1.141768217086792, 1.6720515489578247, -0.1136012002825737, 2.209204912185669, 1.6809600591659546, 0.8783764243125916, 1.954676628112793, 0.016652550548315048, 0.39044514298439026, -0.3296380937099457, -0.4903889298439026, 1.0376935005187988, 1.7753127813339233, 0.04679935798048973, -1.194633960723877, -0.16194403171539307, 0.2760772705078125, 0.1834121197462082, -0.6268108487129211, 0.367744117975235, -0.36871272325515747, 0.6932290196418762, -0.27944642305374146, 1.7163513898849487, 1.178396463394165, -0.4167686104774475, -0.23693422973155975, -0.4583720862865448, -0.5163871049880981, -0.8887181282043457, -0.9357059001922607, 1.489498257637024, 1.4880634546279907, -0.5852363705635071, -0.2191958725452423, 0.48232799768447876, 1.364087462425232, 0.9087681770324707, 0.9767018556594849, 0.49627214670181274, -0.6418389081954956, 1.7818971872329712, 1.0854620933532715, -1.1895071268081665, -1.0423212051391602, 0.6971032023429871, 0.014595058746635914, -0.9680918455123901, -0.40603435039520264, -0.9136708974838257, 0.3914196789264679, 0.723042905330658, 0.61902916431427, 0.3071935474872589, 0.39597925543785095, 1.3323103189468384, -0.8147703409194946, 1.0440887212753296, -0.21652057766914368, 0.39882686734199524, 0.8063446283340454, -0.7513128519058228, -1.1796026229858398, -0.49171388149261475, 1.1229175329208374, -0.4793940484523773, 2.1617860794067383, 1.847841501235962, -1.0432302951812744, -0.5380295515060425, 0.390949547290802, -2.7237372398376465, 0.4371238648891449, 0.6906930208206177, -0.5450290441513062, -1.090516448020935, -0.2314387410879135, 0.5172911286354065, -1.0803884267807007, 1.1416983604431152, -0.5121477842330933, -0.19328784942626953, -1.210349440574646, 0.8584443926811218, -0.2586890757083893, -0.4193342924118042, -0.9120378494262695, -0.6553449630737305, 0.6842817068099976, 0.924166738986969, 1.4367514848709106, 0.6108384728431702, -0.9490873217582703, 0.6287029981613159, -0.876912534236908, -0.8725404143333435, -0.9479855298995972, 0.731235146522522, -0.661499559879303, 0.025296445935964584, -1.7595288753509521, -1.0161187648773193, 1.272373080253601, 0.5995606780052185, 1.3040754795074463, -1.9470032453536987, 2.2425363063812256, 0.04758041724562645, 0.4101742208003998, 1.5774832963943481, -0.3449457287788391, -1.062954068183899, 0.41518038511276245, -0.01034373790025711, -0.3170545697212219, 0.3891659080982208, -0.38976505398750305, -0.30391725897789, -0.9383429288864136, -0.7931206822395325, 0.8806024789810181, 1.0077425241470337, 0.2994864881038666, 1.9898761510849, 0.7768609523773193, 0.5647236704826355, -0.07215362787246704, 0.22514279186725616, 1.8615435361862183, 1.1166003942489624, -0.4559200406074524, 0.15022921562194824, 0.654606282711029, 0.640990138053894, -0.040668193250894547, -0.3600841462612152, 0.12062083929777145, -0.9090858697891235, 1.4005169868469238, 0.01538962870836258, 0.009389354847371578, -0.7226347923278809, 0.5883203744888306, -2.105468988418579, 0.4920031726360321, 0.5205533504486084, -0.2017337679862976, -0.10179461538791656, -0.10721748322248459, 1.3681282997131348, -1.0050727128982544, 0.928026556968689, 0.5187077522277832, -0.6481916308403015, -1.4191946983337402, 0.7166833877563477, 0.5328443050384521, 0.09679782390594482, -0.5417715907096863, 0.40576082468032837, 0.6581357717514038, 0.9445217251777649, 0.499268114566803, 0.4534514546394348, -0.5667576789855957, -1.3963265419006348, -0.3611053228378296, 0.8739609718322754, 0.2123805731534958, -1.3734217882156372, -1.0109645128250122, 0.9702264070510864, 1.0346531867980957, 0.4572085738182068, -0.7179498672485352, 1.0811707973480225, 1.858211636543274, -1.2228646278381348, -0.7312139272689819, -0.5852924585342407, -2.5282156467437744, 1.5966180562973022, -0.534758448600769, 0.2658214271068573, -0.4409296214580536, 0.16033977270126343, 1.1511791944503784, -1.1926809549331665, -0.3742337226867676, -1.0742652416229248, 0.8077476024627686, 0.4528353214263916, -1.220189094543457, -0.7007871270179749, -0.3971858322620392, 0.029904481023550034, -1.084621548652649, 1.329503059387207, 0.3367212414741516, -2.2257120609283447, 1.3624824285507202, -1.279457688331604, 0.5933270454406738, 1.1093034744262695, 1.3843975067138672, -0.511844277381897, 0.0484563410282135, -1.9073772430419922, 0.028214309364557266, -0.2747340798377991, -2.3382740020751953, -0.32740744948387146, 1.0859538316726685, -1.7381304502487183, -0.9455175995826721, -0.7431446313858032, -0.20293495059013367, -0.04848439246416092, -0.11312935501337051, -0.18767522275447845, 0.920800507068634, -0.004125417210161686, -0.6262369155883789, 0.1166972890496254, 1.3832775354385376, 0.21383269131183624, 1.0250872373580933, 0.24354232847690582, -0.130006343126297, -0.634920597076416, 0.3621974587440491, -0.8625342845916748, -1.3680099248886108, 1.547731876373291, -0.8968673944473267, 1.1316038370132446, 2.594863176345825, 1.3016148805618286, 0.3240293860435486, -0.314552903175354, 2.822570323944092, -0.891745924949646, 1.2126091718673706, 0.053941767662763596, 1.3836718797683716, 0.4187243580818176, -1.2900805473327637, 1.635026454925537, -0.17565008997917175, -0.13933655619621277, -1.078523874282837, -0.6926593780517578, 0.9047142267227173, 1.0838186740875244, 0.15483008325099945, 1.9988898038864136, 0.3873518407344818, -1.197115421295166, 0.4081852436065674, -0.476260781288147, -1.8172551393508911, 0.3176676332950592, 0.1729333996772766, 2.004465341567993, 0.6168444752693176, -1.8648414611816406, 0.061507828533649445, 1.0285813808441162, 0.396099716424942, 0.19800405204296112, -1.1905725002288818, 1.456586480140686, -0.29742705821990967, 0.3070277273654938, 0.16349582374095917, 0.30375146865844727, -0.5107181072235107, 0.5511751770973206, 1.8151085376739502, 0.8255015015602112, 0.2855527997016907, 1.0942188501358032, -1.4954454898834229, 1.5593860149383545, -0.2136007398366928, 0.47817423939704895, 0.5479044318199158, -0.18845075368881226, 0.23911194503307343, 1.6136800050735474, 0.8046035766601562, -1.149467945098877, 0.32955223321914673, -0.13430848717689514, -0.7846680283546448, 0.9729747772216797, 0.6468357443809509, -0.49682608246803284, 0.9301677942276001, 1.191427230834961, -0.5779731869697571, 0.17570072412490845, -0.8317750692367554, -0.5491147637367249, 0.024155491963028908, 0.6311233639717102, -0.7672051787376404, 1.6455482244491577, -0.34180331230163574, 0.08782780170440674, 0.10366085171699524, 1.172202467918396, -0.048238735646009445, 0.04533825442194939, 0.9495142102241516, 1.2297316789627075, 1.6657443046569824, 1.121427297592163, 0.817280113697052, 0.1888226717710495, -0.3364579975605011, 0.3166542947292328, 0.26698628067970276, -0.149064838886261, 2.2653322219848633, -0.8655959367752075, 0.842736005783081, 1.16826593875885, -1.1751811504364014, 0.19036880135536194, -0.41368016600608826, -0.2831800580024719, 0.20575876533985138, 0.5387296676635742, 0.2539057433605194, 1.1726568937301636, -0.7445824146270752, -0.6048017144203186, 0.39640986919403076, 0.044302910566329956, -0.8953036069869995, 2.403810977935791, -0.9456416368484497, 1.0159661769866943, -1.2923375368118286, -0.09925992041826248, -0.1193312332034111, 0.005797285120934248, 0.8847730755805969, 0.7420285940170288, -0.7182583808898926, -1.2836940288543701, -0.692825436592102, -1.0103256702423096, -0.8472713232040405, -0.39242464303970337, 0.9727526307106018, 1.0717642307281494, 1.927626132965088, 0.7993406653404236, -0.3889230489730835, -0.3675250709056854, 0.4404642879962921, 0.9360978603363037, -0.8507593870162964, 1.4190325736999512, -2.1532399654388428, 0.3859311640262604, 0.20857921242713928, -0.5330602526664734, -0.4760594069957733, -1.0668437480926514, 1.8935914039611816, 0.8291118144989014, 1.4891595840454102, -0.3212387263774872, 1.1984081268310547, 0.0848098024725914, -0.49957606196403503, 0.6178213357925415, 0.3130073547363281, -0.728223979473114, 0.39720913767814636, 0.15603046119213104, -0.29796481132507324, -0.5703398585319519, 1.1037402153015137, -1.1930241584777832, -2.3563449382781982, -0.26857858896255493, 1.934687852859497, 1.7535029649734497, -1.947800874710083, -0.16713736951351166, 0.7822843194007874, 2.177816867828369, 0.3377101421356201, 0.016760818660259247, 1.0974714756011963, -0.6500186920166016, 1.0413202047348022, 0.42032894492149353, -1.3576372861862183, -0.7332433462142944, -0.7340573668479919, -0.16438449919223785, 0.8919683694839478, -0.8075669407844543, 1.1701704263687134, 0.8211274147033691, 0.8808073997497559, -1.1713340282440186, -0.6769407391548157, 0.8619283437728882, -0.47754889726638794, -0.2947264015674591, -2.9088070392608643, 0.14315177500247955, -0.3949507176876068, 0.8506980538368225, 0.2636856138706207, -0.3419758975505829, 1.2192935943603516, 0.06408926844596863, 0.34958139061927795, 0.7357967495918274, -0.010688715614378452, 0.07321663945913315, 2.023909330368042, 2.9006824493408203, -1.1026467084884644, -0.10105851292610168, 0.1458476185798645, -0.7331597805023193, -0.01666279509663582, -0.8931167721748352, 0.007818938232958317, -1.8948086500167847, 0.34137365221977234, -0.42481544613838196, 0.2368132323026657, -0.616724967956543, 0.13999076187610626, 0.426002562046051, -0.6316105127334595, 1.0688560009002686, -0.26572170853614807, 0.10912606120109558, -0.09612355381250381, -1.4537932872772217, -1.2291889190673828, -1.205308198928833, 0.691217303276062, 2.627027750015259, 1.0510495901107788, 1.2490431070327759, -1.8954566717147827, -0.14371494948863983, -2.3610215187072754, -1.471199870109558, -0.03582384064793587, -1.1644006967544556, -1.4531762599945068, -0.491312175989151, 0.5193790197372437, 1.1533833742141724, -0.6194742321968079, -0.2518763840198517, 1.8323527574539185, 1.5770597457885742, 0.8870401382446289, -0.7049821615219116, 0.41001057624816895, -3.1536595821380615, -0.8031795620918274, -0.006743126083165407, -0.3221738338470459, -1.2945114374160767, -1.1915507316589355, -0.8072038292884827, -0.18478178977966309, -0.5451794862747192, -0.158206507563591, 0.35191062092781067, -0.027535436674952507, 1.0437060594558716, -0.9909348487854004, -0.0024847567547112703, -0.1420152485370636, -0.3432825803756714, 1.1650022268295288, -0.7613492012023926, 0.7759162783622742, -0.008241772651672363, -1.8242017030715942, -0.44262731075286865, -0.6043025851249695, 0.6016241312026978, -0.0605776272714138, -0.10277563333511353, 0.6854991316795349, 1.3371200561523438, -0.17771606147289276, 0.7073427438735962, 1.1148834228515625, -0.8067194819450378, -3.1320154666900635, 1.2591331005096436, -1.8767914772033691, 1.2535947561264038, -2.655595541000366, -0.9521676301956177, -1.663461446762085, -0.8264299631118774, 1.044593095779419, -0.9653377532958984, 1.3764995336532593, -0.29707708954811096, 0.11010827124118805, -1.2250685691833496, -0.6674079298973083, 1.0597617626190186, 1.3541574478149414, 0.37143221497535706, 0.19320982694625854, 1.4007259607315063, 0.9269618988037109, 0.150202676653862, 1.3291997909545898, 0.5845542550086975, 0.2585030496120453, -0.0033391884062439203, 0.32685092091560364, 0.0814482793211937, -1.4039345979690552, 0.20813101530075073, 0.4215950071811676, 1.0665380954742432, 1.2714232206344604, 1.2043201923370361, -0.6801007390022278, 0.6584528684616089, -1.1793711185455322, -0.03833338990807533, -1.6730422973632812, 0.3781890273094177, 1.0930231809616089, 0.47262799739837646, 0.7361019849777222, 0.6576717495918274, 0.38747674226760864, 0.1856691688299179, 0.35803017020225525, -1.5932515859603882, -0.6856001019477844, 1.644248604774475, 0.06821756064891815, -1.158976435661316, -0.4964078664779663, 0.5784644484519958, -0.4017651081085205, 0.10679065436124802, 1.4888877868652344, -0.2908039689064026, -0.08316177874803543, 0.9589916467666626, 0.5826283693313599, 0.14492018520832062, -0.6227404475212097, -0.6116554737091064, 0.6039106845855713, -1.5913903713226318, 1.409538745880127, -0.3671262562274933, 1.3736116886138916, -0.08445706218481064, 1.4362006187438965, 0.336036741733551, 1.8240442276000977, -1.4023265838623047, 0.6136159300804138, 0.22806653380393982, 1.2778068780899048, -0.17018699645996094, -0.18266302347183228, 0.17070965468883514, -0.09202110022306442, -0.0627949982881546, 0.7378774285316467, 1.2692352533340454, 1.5547471046447754, 0.2533281445503235, -0.47059059143066406, 0.02983865514397621, 0.6499266028404236, -0.6334792971611023, 1.4634824991226196, -0.4915598928928375, -1.161095380783081, -0.9180606603622437, 0.443682461977005, 0.41007673740386963, 0.43215250968933105, -0.8106305003166199, -0.8988462686538696, 1.4711917638778687, 0.12153828889131546, 1.151321291923523, -0.13713912665843964, 0.21538138389587402, 0.20758837461471558, -1.6261965036392212, -0.3408290445804596, -0.9361302256584167, 0.5119431614875793, 0.8911941647529602, -0.2822056710720062, 0.9639800786972046, 0.26101553440093994, 0.20739752054214478, 0.7587900161743164, -0.2934977114200592, 1.4082728624343872, 2.220902919769287, -0.7120420932769775, -0.3778880536556244, 0.08744391798973083, 1.454362392425537, 1.0920345783233643, -1.9299336671829224, 1.738985300064087, -0.009785789996385574, 0.29591912031173706, -0.662041187286377, -1.4874467849731445, 1.1220896244049072, -0.5035367012023926, -0.46784722805023193, -1.8552467823028564, 0.9416373372077942, 1.3212908506393433, 1.2105952501296997, -2.208632230758667, -0.24284173548221588, 0.025436200201511383, -2.355081081390381, 0.6226022839546204, 0.8865846991539001, 2.0903568267822266, -0.7449997067451477, 0.595469057559967, 1.657084345817566, -1.6674883365631104, 0.649569571018219, -0.7260829210281372, 0.007429669611155987, -0.68570876121521, -1.2110666036605835, 0.9565336108207703, 0.1049182340502739, 0.2881513237953186, 1.9929319620132446, -0.9725357890129089, 0.49413949251174927, 0.36278998851776123, -1.0196082592010498, -0.38608863949775696, -0.05265558511018753, 0.31703421473503113, -1.6850112676620483, -1.0986639261245728, -0.4604150950908661, 0.08774244785308838, 1.564644455909729, -0.8636152148246765, -0.2771049737930298, 1.2365000247955322, -0.3750084340572357, 1.4584957361221313, 0.40063178539276123, -0.8353991508483887, -2.211986541748047, 0.33508190512657166, -0.8186373710632324, 0.6734200119972229, 0.29658541083335876, 0.6587652564048767, -1.2904268503189087, -0.41459521651268005, -0.3722250759601593, 2.2091310024261475, 1.2223572731018066, -0.029802318662405014, -0.8439365029335022, 0.10565975308418274, 1.6948115825653076, -0.0661420226097107, 0.3435294032096863, 0.31232303380966187, -0.41494429111480713, 1.088241457939148, -1.8321313858032227, -0.4076962172985077, -0.07305653393268585, 1.1639589071273804, -1.0522677898406982, -0.7802232503890991, -0.8857191205024719, 1.3168874979019165, 0.4286249279975891, -0.25345394015312195, -0.2147127091884613, -1.4803186655044556, -0.497818261384964, 0.3933697044849396, 1.436559796333313, 1.751309871673584, 0.3601432740688324, -1.1530580520629883, -1.0185283422470093, -0.16909471154212952, 0.08959218114614487, 1.5470608472824097, 0.15666764974594116, 0.7952808141708374, -0.29345420002937317, 2.1974587440490723, -0.37530025839805603, 0.724058985710144, -0.5077780485153198, 0.40715187788009644, 0.1441897451877594, 0.868428111076355, -0.4293300211429596, 0.21834976971149445, 0.3197110593318939, 0.13757015764713287, 0.9103989005088806, -0.30074837803840637, -0.05992869660258293, 0.4498594105243683, -0.012729422189295292, 0.29629719257354736, -0.8963031768798828, 0.3629361093044281, -0.289607971906662, 0.4594140946865082, 1.1654924154281616, 0.521544873714447, -0.3134472966194153, -0.6583505272865295, -0.534626841545105, -1.504373550415039, -1.570106029510498, -0.1566586047410965, 1.6485755443572998, 0.4443701207637787, 1.0889818668365479, 1.1029417514801025, -0.04283769801259041, -0.47493717074394226, -0.9250145554542542, 1.084717869758606, -0.24615122377872467, 1.3322679996490479, 0.7291269302368164, -0.030904922634363174, 0.10508911311626434, -0.3803665339946747, 0.6605349183082581, 1.187002182006836, -0.010866137221455574, 0.2330581545829773, 0.1901317834854126, 0.43831387162208557, -1.1694318056106567, 0.005668679252266884, 0.675430178642273, -0.38158196210861206, -0.7459980249404907, -0.36473169922828674, 0.11034520715475082, 1.040625810623169, -1.2160502672195435, 1.4336892366409302, -0.44489356875419617, 1.3736292123794556, -1.4864170551300049, 1.1892509460449219, 0.34206634759902954, -0.5831790566444397, -0.12495557218790054, -0.6643882989883423, -0.45109689235687256, 0.822182297706604, 1.2072808742523193, -0.07545195519924164, 0.17160089313983917, 0.49428853392601013, -0.5748097896575928, 0.2552911043167114, -0.8734938502311707, 1.7244279384613037, -0.4456796646118164, 0.13397014141082764, -0.6528723239898682, 0.02983427792787552, 1.5387226343154907, -1.372446894645691, 0.47808781266212463, 1.835163950920105, -0.38130033016204834, 0.07473919540643692, 0.5086766481399536, 1.2673087120056152, 0.6531730890274048, -0.8250741958618164, -0.714791476726532, 0.03264299035072327, 1.02349054813385, -2.119462251663208, 1.3208709955215454, -0.327995240688324, 2.5858068466186523, 1.1197123527526855, -0.3032117486000061, -0.4357437491416931, -0.07758309692144394, 0.36924368143081665, 0.2029256969690323, 0.19271261990070343, 0.05450863018631935, -1.4911597967147827, -0.04910175874829292, 0.21123413741588593, 0.0826302096247673, 0.5246560573577881, -1.2362216711044312, 1.9124078750610352, 0.2287599891424179, 1.2803138494491577, -0.32203835248947144, 1.0147510766983032, -0.003024277277290821, 1.6475096940994263, 0.4782828092575073, -0.39615604281425476, 1.0238326787948608, -0.9164842963218689, -0.6561025977134705, 1.0021538734436035, -0.6166428327560425, 1.015236258506775, 1.3976556062698364, -1.1135001182556152, 0.9272726774215698, -0.8417661190032959, 0.3135085701942444, -0.5334393382072449, 0.6108114123344421, -2.197930097579956, 0.7673611640930176, -0.822256326675415, -0.9506799578666687, -1.074390172958374, 0.4829091727733612, 0.5182559490203857, 0.6522886157035828, -0.3658376932144165, -0.30961763858795166, -0.09377773106098175, 0.47642800211906433, -1.1644229888916016, 0.40309637784957886, -0.014921183697879314, -0.3957989513874054, -0.8817269802093506, 0.33547285199165344, -1.1730800867080688, -1.6644234657287598, -0.11420157551765442, -0.6401525735855103, -0.08710135519504547, 0.7598263025283813, -1.417086124420166, -0.2972802519798279, 0.1267477422952652, -1.0899051427841187, 0.1399102509021759, -1.19057035446167, -0.429567813873291, -0.2540791928768158, 1.5212199687957764, 1.2056894302368164, 1.357181191444397, 0.9619932770729065, -0.41058072447776794, 1.0870468616485596, 1.8835550546646118, -1.1009498834609985, -0.6993513107299805, -0.9701242446899414, -0.07075987011194229, 0.33272308111190796, -0.9751748442649841, 1.2434033155441284, 1.5062077045440674, 0.595603346824646, 0.5821240544319153, 1.7225373983383179, 0.9323304891586304, -0.5512353181838989, -0.8328462839126587, 0.5825373530387878, 0.9312559366226196, -0.16509178280830383, 1.1956876516342163, -0.6689918637275696, -1.6033234596252441, 1.049066424369812, -0.1803825944662094, -0.845840573310852, 0.9016387462615967, 1.2273463010787964, 1.5081546306610107, 0.4336555004119873, 2.107823610305786, -1.1057318449020386, -0.5218760967254639, 0.3346320688724518, -0.9927066564559937, 0.9409134984016418, -0.544432520866394, -0.31954512000083923, -0.14766080677509308, -1.4344348907470703, -0.26119011640548706, 0.47316741943359375, -0.12850990891456604, 1.4401520490646362, -0.08188688009977341, 0.013078324496746063, 0.43041494488716125, -0.062032558023929596, -0.36806008219718933, -0.7855990529060364, 1.4160758256912231, 0.06217317655682564, -0.5737313032150269, -0.15759241580963135, 0.23890496790409088, 0.5859537124633789, -0.6484250426292419, 1.1227601766586304, 0.8208863139152527, 1.1309442520141602, 0.7698764801025391, 0.4157041609287262, -2.174699068069458, -0.9842402338981628, -0.4480404257774353, 1.0455191135406494, -0.36168769001960754, 2.219958543777466, 0.7450408339500427, 1.2893494367599487, -1.3749617338180542, 1.16220223903656, 0.07821815460920334, 2.478700876235962, -1.6628118753433228, -0.14254790544509888, -0.05652979388833046, 1.1431002616882324, 0.19135992228984833, 0.48268550634384155, -0.40900856256484985, 0.29934319853782654, 1.735607385635376, -0.3843916356563568, -0.20827323198318481, 0.0006182981305755675, -0.30829089879989624, 0.09055352210998535, -1.669772744178772, -2.4094629287719727, -0.425435870885849, -0.571012020111084, 0.49401146173477173, -1.156171441078186, 0.2092522233724594, -0.4407705068588257, -1.6374964714050293, 0.6426597237586975, -1.2780888080596924, 0.49814632534980774, -1.6575442552566528, 1.6327641010284424, -0.6476877927780151, -0.30804938077926636, -0.7071853876113892, 1.6565158367156982, -0.03566884994506836, 0.10830190032720566, 0.39582428336143494, 0.3772749602794647, 0.9064145088195801, -0.11669465154409409, -2.0524847507476807, 1.1339291334152222, 0.4978986084461212, 0.6338546872138977, 0.18280553817749023, -0.7800754904747009, 0.23318630456924438, -0.4905572831630707, -0.26465827226638794, 0.8768237233161926, -0.14938002824783325, -1.6384271383285522, 0.4646419584751129, 0.0950319916009903, 0.594191312789917, 0.8332240581512451, -0.17121070623397827, -1.3376156091690063, -0.12456043809652328, 1.7880167961120605, -1.2224377393722534, -0.17005780339241028, 0.15345989167690277, -0.3685859739780426, 0.15776488184928894, -0.4082251191139221, -0.013897128403186798, 1.0424436330795288, 1.0207970142364502, 0.18847738206386566, 0.8258077502250671, -0.22869807481765747, -0.5092213153839111, 0.2260415256023407, -0.17228251695632935, -0.06963503360748291, 1.886059045791626, -0.9163007736206055, 1.9185127019882202, -3.3646364212036133, -0.9944484829902649, 1.9267635345458984, -1.1425676345825195, 0.14937391877174377, 0.08337779343128204, -1.1862982511520386, 0.926992654800415, 0.22198748588562012, -1.0960267782211304, 1.8601914644241333, -0.6166306138038635, -0.5825413465499878, -1.4153587818145752, -0.6113758087158203, 0.44073641300201416, 1.5135736465454102, -0.288404256105423, 0.2683869004249573, 0.3351300060749054, -0.8373383283615112, 0.2873072028160095, 1.1880214214324951, 0.5657792687416077, 0.15503862500190735, -0.658586859703064, 1.003713607788086, 0.4480988681316376, -1.2758747339248657, 1.1637520790100098, 0.009805737994611263, 0.03052784688770771, 1.129168152809143, 0.755287766456604, -0.17886945605278015, 1.5969367027282715, -0.7953519821166992, -0.23416434228420258, 1.3703908920288086, -0.06793029606342316, 0.42590948939323425, -0.28366926312446594, -1.0452097654342651, -0.10514745861291885, 1.1582317352294922, -0.25535136461257935, 0.8361847996711731, 0.46465280652046204, 2.0366592407226562, -0.21243296563625336, 0.35896140336990356, -1.7543569803237915, 0.17555208504199982, 1.0777482986450195, -0.8885402679443359, -0.666917622089386, 1.503293514251709, 0.8316650986671448, -0.6153600811958313, -2.2501163482666016, -1.1583735942840576, -1.2456624507904053, 0.6836650371551514, -0.13695411384105682, -0.05812688544392586, -0.7352240085601807, 1.080659031867981, 0.10809433460235596, -0.20801806449890137, 0.7831398844718933, -0.35017073154449463, 0.13580502569675446, 0.4745815098285675, 2.33192777633667, 0.36708563566207886, 0.17603911459445953, -0.722550630569458, -0.4017113745212555, 0.9715498089790344, 1.2861024141311646, -1.4345532655715942, -1.5167990922927856, 0.20383581519126892, 0.6852976083755493, -0.6480380892753601, -1.9414860010147095, 0.0849028155207634, 1.0878915786743164, -1.6681056022644043, 0.307682603597641, -0.5687626004219055, 0.7283048033714294, 1.7343827486038208, 0.3163660764694214, -1.5194931030273438, 1.2334532737731934, -1.4734300374984741, 0.552800178527832, 0.6932564973831177, 0.07092790305614471, 1.093581199645996, -0.4872477054595947, -0.7474011778831482, 1.8789918422698975, 1.3239895105361938, -0.46563106775283813, 0.9355188012123108, -1.3608132600784302, 1.1472293138504028, 0.45044922828674316, 0.9749042391777039, 0.046444229781627655, -0.018018608912825584, 0.6601631045341492, 1.221616268157959, -0.2670120894908905, -1.3699740171432495, 0.24525798857212067, 2.0864908695220947, 0.3307042419910431, 0.498118132352829, -0.33123213052749634, -0.4611858129501343, -0.9114565253257751, -1.5904438495635986, 0.10207963734865189, 0.4931052625179291, 1.608577847480774, -0.3633401095867157, -0.3843020796775818, -1.2467695474624634, -0.7099357843399048, 0.6586170196533203, -0.73670893907547, 1.1857742071151733, -1.6938810348510742, -0.45792028307914734, 0.034815382212400436, 0.05998038128018379, 2.2659049034118652, -0.8847538232803345, -0.23211997747421265, 0.39338916540145874, -0.8481745719909668, 0.27817845344543457, -0.26181256771087646, 0.06205999478697777, 2.6127583980560303, 1.4609107971191406, -1.1659984588623047, -1.5442146062850952, -0.01070626825094223, -0.1317867785692215, 0.46209725737571716, -0.8560634255409241, -0.9096319079399109, 0.3003081679344177, 0.020580153912305832, 0.2835468053817749, 0.5785273313522339, -0.6113709807395935, 0.3318091928958893, -0.7163922190666199, -0.04295310378074646, 0.638629138469696, 0.23720473051071167, 0.44289377331733704, -0.25243720412254333, -0.8454131484031677, 0.8053551316261292, 0.6507126092910767, -0.1787891685962677, -1.1769503355026245, 0.23323331773281097, 0.5078330636024475, -0.466974675655365, 0.5870531797409058, 0.6302235722541809, 1.432516098022461, 0.12420079112052917, 0.03990805521607399, 0.22528918087482452, 0.6202045679092407, 1.2452731132507324, -2.014315605163574, 0.8509796857833862, -0.347933292388916, 1.1968034505844116, 0.28484907746315, -1.0680376291275024, 1.0946568250656128, -0.9476511478424072, -0.6597329378128052, 0.8587583899497986, -1.014306902885437, 1.6097999811172485, -0.5998000502586365, -0.9041963815689087, -1.0166692733764648, 1.1626222133636475, -1.2647438049316406, 0.33506911993026733, 2.4151511192321777, 0.15747296810150146, -0.45773887634277344, 0.10813603550195694, 1.8757680654525757, -0.06361625343561172, -1.5925260782241821, 1.30185067653656, -0.35130080580711365, 0.07965365797281265, 1.6067883968353271, -0.9915313124656677, 1.2772293090820312, -0.4133581221103668, -0.29793572425842285, 0.08366305381059647, -1.0406662225723267, -0.24096286296844482, 0.6269372701644897, -1.090692400932312, -0.20405785739421844, -0.9537315964698792, -0.05686509981751442, -1.4442613124847412, -1.9460700750350952, 0.30669453740119934, -0.03269589692354202, -0.5649961829185486, 0.669426441192627, 0.04120834544301033, 0.44315841794013977, -0.5107331871986389, 0.19651208817958832, 0.14476124942302704, 1.7321202754974365, -0.7482032179832458, -1.9238901138305664, -0.04747810587286949, 0.7093892693519592, 0.2359013706445694, -0.13026389479637146, 0.5251266956329346, 0.6590242981910706, 0.8126936554908752, -0.11519032716751099, 0.5300980806350708, -1.9625164270401, -0.42526984214782715, 0.9939126968383789, -0.3046782910823822, -2.394059896469116, -1.4586244821548462, -0.6210663318634033, -0.7456144094467163, 1.3509857654571533, -1.4166975021362305, -0.18001210689544678, 1.616572380065918, 0.6195815801620483, -0.29847806692123413, 1.3301243782043457, 1.224727749824524, -0.4827703535556793, -0.32864201068878174, 0.13785962760448456, -0.9841122031211853, 0.2874925136566162, 0.37582260370254517, 0.6519235372543335, -2.416128396987915, 0.49748316407203674, -1.1047477722167969, -0.7648351192474365, 0.9040063619613647, -0.7561582326889038, 0.8743934035301208, -1.148956298828125, -0.17425945401191711, 1.0752320289611816, 0.1708262413740158, -0.8659313321113586, 0.06721905618906021, -0.8226340413093567, 0.39534080028533936, 0.5015463829040527, -0.4666907489299774, 1.8608208894729614, -0.3324405252933502, -2.5741841793060303, 2.742109775543213, 0.32932835817337036, 0.8279139995574951, -0.44067248702049255, 0.3196837902069092, -0.75666344165802, 0.16725778579711914, 1.2891573905944824, 1.0909751653671265, -0.7666522264480591, -0.01945289596915245, -0.3065541684627533, -0.09104637801647186, -0.38769346475601196, 0.14861808717250824, 0.8128733038902283, -0.12632733583450317, -1.6004217863082886, 1.645908236503601, 1.7837072610855103, 1.4033000469207764, 1.4214950799942017, 1.8008767366409302, 0.8653982281684875, -2.142343044281006, 0.12403155118227005, 0.5931640267372131, 0.9983406662940979, 0.4273627698421478, 0.07953441143035889, -0.8432977795600891, -2.0179545879364014, 0.2306242287158966, -0.316629558801651, 1.4106791019439697, -2.103268623352051, -0.10210774838924408, -0.17363426089286804, -0.31855833530426025, 0.6935694813728333, -2.590825080871582, 2.5536904335021973, 0.32463011145591736, -0.0932457223534584, 0.4841940701007843, 1.127136468887329, -1.5872876644134521, -0.7731684446334839, 0.8387544751167297, -0.6981624960899353, -1.0661966800689697, 0.9563241004943848, -0.4644332528114319, 0.2507099211215973, -0.9497870802879333, 1.3087116479873657, -0.06487364321947098, -0.13439802825450897, -1.220680832862854, -0.4953975975513458, -0.7924453616142273, -0.5597563982009888, 0.4062531888484955, 2.037017583847046, -0.9996326565742493, -0.1777295619249344, -0.9569224119186401, -1.1136678457260132, -0.8844980001449585, -0.8742901086807251, -0.6621177792549133, -1.4286402463912964, -0.28644582629203796, -1.6424628496170044, -0.043713413178920746, -1.0467455387115479, -0.4675978720188141, 0.1317860335111618, 0.7708842754364014, -0.4569392800331116, 0.04762034863233566, -0.15609049797058105, -1.5118824243545532, 1.0180155038833618, -0.3798530101776123, 1.0101039409637451, 1.0569746494293213, 0.11316941678524017, 0.4614550769329071, -1.777876377105713, 0.8985317349433899, -1.3222954273223877, -0.4160963296890259, -1.3499393463134766, 0.3261690139770508, 0.343317449092865, 0.3788779675960541, 0.5415530204772949, -0.9016497731208801, -1.7961468696594238, -0.6192116141319275, -1.470455288887024, -0.8535870909690857, -0.2508414387702942, 1.9803705215454102, -1.264344573020935, -1.297116994857788, -0.11942300945520401, -1.0483053922653198, -0.2756580710411072, 0.16878627240657806, 0.047497935593128204, 0.16611814498901367, -0.9199780225753784, -0.6245589852333069, -0.30919942259788513, 0.5223357081413269, 0.9246833920478821, 1.4391117095947266, -1.1732511520385742, 0.5674333572387695, -0.565918505191803, 1.0102570056915283, -0.18696072697639465, -1.3047878742218018, -0.9735652208328247, -0.07676944136619568, -0.23995760083198547, 1.4814605712890625, 0.6132442355155945, 2.443566083908081, 0.15011903643608093, -1.4486507177352905, 0.7560192346572876, 0.2086166888475418, -0.14465805888175964, 0.1490168273448944, -0.6696124076843262, -0.6521797180175781, -0.4010654389858246, -0.5324425101280212, -2.4723477363586426, 1.2461315393447876, 0.1573592871427536, -1.245794415473938, 0.24138641357421875, -0.32894450426101685, 0.7327169179916382, 2.660305976867676, -1.564164161682129, 0.7846882343292236, 0.32495251297950745, 1.5454944372177124, -1.4946610927581787, -2.23258638381958, -0.19809164106845856, -1.2615582942962646, 0.11175933480262756, -0.7050332427024841, -1.5018105506896973, 0.6795401573181152, 0.05375360697507858, 0.8834218382835388, -0.13788746297359467, 0.7053031921386719, 0.19112913310527802, -0.5308814644813538, 0.034702133387327194, 2.017024278640747, -0.3195713460445404, 2.2786760330200195, 1.2433754205703735, 1.474707007408142, -0.14474433660507202, -0.7659927010536194, -1.611691951751709, -1.0921802520751953, -0.4939263164997101, 0.03104991838335991, 0.20860230922698975, -0.1564076691865921, -0.7541886568069458, -0.1734534353017807, 0.21200726926326752, -0.19520793855190277, 0.007907063700258732, 0.09638973325490952, 0.6474704146385193, -2.040651559829712, 0.3561450242996216, 1.9364453554153442, 0.1921617090702057, -0.827258288860321, -0.9998334646224976, 0.0939769521355629, -0.13990236818790436, 0.26227855682373047, -0.00311065511777997, 1.2163318395614624, -0.7495997548103333, -0.46926769614219666, -2.192521333694458, -0.4172387719154358, 1.0293556451797485, 1.537812352180481, -0.3083183765411377, -1.1050019264221191, 2.263357639312744, -0.5054017901420593, 0.9409804344177246, 1.7538299560546875, -0.3085627257823944, 0.5858830213546753, -0.8469865918159485, 0.3854147791862488, -1.900356650352478, -0.9737789630889893, 0.28676408529281616, -1.3673704862594604, 0.7979887127876282, 0.017050299793481827, 1.9067476987838745, -0.12925614416599274, 0.2224041372537613, -0.18961672484874725, 0.6231638193130493, -0.1481509655714035, -0.8160609006881714, 0.5648369789123535, 0.665774405002594, -0.34213438630104065, -0.9384901523590088, 0.062050510197877884, -0.7411841154098511, 1.7622956037521362, 0.6538503766059875, -0.41505470871925354, -0.19480901956558228, 1.5559250116348267, -0.5778461694717407, -1.030665636062622, -0.13341982662677765, -0.04253266379237175, 0.23381240665912628, -0.1460372507572174, 0.29968205094337463, -0.2219923734664917, 0.24852241575717926, -0.6450459957122803, -0.858522355556488, 0.6400237083435059, 0.6735996007919312, 0.41723111271858215, 0.04545825719833374, 1.3556326627731323, 1.2058002948760986, 0.10050211101770401, -1.2409183979034424, -0.32327064871788025, 1.4189444780349731, 0.7086303234100342, 0.7499717473983765, -0.0656350627541542, -0.08471623063087463, 1.081496000289917, 1.511804461479187, 0.15044623613357544, 0.4805101454257965, 0.30422893166542053, -1.234776496887207, -0.6623984575271606, 1.0955802202224731, -2.677659511566162, -1.236901044845581, -0.0020829287823289633, 0.5094487071037292, -0.5644245147705078, -1.9322211742401123, 0.6151701211929321, -0.2783781588077545, -0.12994162738323212, -0.39247605204582214, -1.9673118591308594, 0.041415996849536896, -1.6849573850631714, -0.6017332673072815, -0.3429190218448639, 0.5984528660774231, 0.2710108458995819, 0.4859650433063507, -0.3420027196407318, 2.287536859512329, 1.9880930185317993, -0.8414833545684814, 0.8040483593940735, -1.0295239686965942, 0.5237481594085693, 0.8531403541564941, 0.23674435913562775, -0.051944222301244736, -1.9508610963821411, -1.8163468837738037, 0.7374419569969177, -0.03710551559925079, -0.7303326725959778, -0.9087796211242676, 0.3951209783554077, 1.369397521018982, 1.196344256401062, -1.0078403949737549, 2.367457866668701, -0.14663846790790558, -1.0431714057922363, -1.7087740898132324, -0.9107440710067749, 0.02843674272298813, -1.4501619338989258, 0.04351579770445824, -1.9645169973373413, 1.3864766359329224, -0.3919193148612976, -0.6903185248374939, 1.7164101600646973, -1.2364890575408936, 1.3473267555236816, 0.22530263662338257, 0.4208335876464844, 0.1412329226732254, -1.1504559516906738, -0.465671569108963, 0.912075400352478, -1.4927095174789429, 1.595948338508606, 0.3571317493915558, 0.14004482328891754, 0.5758375525474548, 0.8741783499717712, 1.033638596534729, -1.5668346881866455, 0.021374497562646866, -0.2677111029624939, -0.024708259850740433, 0.8423511981964111, 1.4001706838607788, 0.14741109311580658, -3.3052403926849365, 1.3520463705062866, -1.0450702905654907, 0.8381090760231018, -0.85423344373703, -1.34226655960083, 1.382991909980774, -0.24640552699565887, 0.6228676438331604, -0.9252332448959351, 1.2077142000198364, -1.0819844007492065, 0.7535541653633118, -0.805126428604126, -0.6494408249855042, -0.5875375866889954, 0.3897396922111511, -0.6671133637428284, -1.6012916564941406, -0.11222238838672638, -2.2881124019622803, 1.0188957452774048, 0.6515628099441528, 2.158416748046875, -0.6050816774368286, 0.1766791194677353, -0.7119110822677612, -0.8981806039810181, 0.7454328536987305, 0.3792259395122528, 1.4719326496124268, -0.15204957127571106, -0.3321937322616577, 0.6189138293266296, 0.9760106205940247, -0.8878551721572876, -0.032154381275177, 0.03660730645060539, -0.20842453837394714, -0.3422076106071472, -0.34215158224105835, 0.0989149734377861, 1.4126049280166626, -0.13411109149456024, -1.0068867206573486, 1.4354007244110107, 0.7162933349609375, -1.2212637662887573, 0.07523026317358017, -0.16602051258087158, -1.2692075967788696, 1.765709400177002, 0.05191279202699661, -1.445950984954834, -0.10022015124559402, 0.078135184943676, 0.4978009760379791, -1.992382287979126, -0.45432156324386597, -0.023407578468322754, -1.2430540323257446, -0.36977943778038025, 0.6910380125045776, -0.142291858792305, -0.3434607684612274, 1.1765793561935425, -0.08929222077131271, 1.3264449834823608, 0.13628220558166504, 0.1567891389131546, -1.9687896966934204, -1.2001928091049194, -1.1160640716552734, -0.320894718170166, 0.8729949593544006, 0.8928861618041992, -0.28830695152282715, 0.6824336051940918, 1.74135422706604, 0.6507585048675537, -1.1612322330474854, 1.1477527618408203, -0.7575135231018066, -0.5134047269821167, 0.6056158542633057, 0.4701566994190216, -0.08635804802179337, -0.461484432220459, 0.9263534545898438, 0.23047325015068054, -0.5536185503005981, 0.6401907801628113, -1.6555131673812866, 0.5130508542060852, -0.04437718912959099, 0.7786433696746826, -0.5589844584465027, 0.6598265171051025, -0.5164164900779724, -0.4811413288116455, -0.9986828565597534, 0.19046472012996674, 0.22451087832450867, -0.3271653354167938, -0.004391566384583712, -0.26778894662857056, 1.5218768119812012, 1.3529196977615356, 0.8260413408279419, -0.18977823853492737, -0.9117039442062378, 0.8269792795181274, -1.526345133781433, -0.0861775130033493, -1.4804807901382446, 1.5133004188537598, 1.1891975402832031, -0.8903583884239197, -0.8278414011001587, 1.357871413230896, -0.6192368268966675, -0.5800334811210632, -0.3106485605239868, -0.4638075530529022, -0.7772859930992126, 0.6492587327957153, -0.05058411881327629, 1.4250965118408203, -0.6315996050834656, 1.047685980796814, -0.5952150225639343, 0.9246660470962524, 0.9663650989532471, -0.7498531937599182, -0.47933298349380493, 0.3203312158584595, 1.5567306280136108, -0.883529543876648, 0.28651291131973267, 0.4394999146461487, -0.6764732003211975, 0.0959348976612091, 0.0012823083670809865, 0.6041656136512756, -1.4545743465423584, -1.4718270301818848, 0.5835921764373779, -0.7895202040672302, 1.3375169038772583, 1.4407743215560913, 0.19142520427703857, 1.0389546155929565, -0.34890520572662354, 0.7429052591323853, 0.46856793761253357, 0.851439893245697, -1.3261606693267822, 0.6868641376495361, -0.8539116382598877, 0.4107457101345062, -1.2961052656173706, 1.2172892093658447, 0.4079698324203491, 0.42173507809638977, -0.8603434562683105, -0.4463496208190918, -1.2375134229660034, -1.496691107749939, 0.17769335210323334, 0.423581600189209, -0.5352318286895752, 1.831952691078186, -0.18269218504428864, -0.06588497757911682, 1.9210772514343262, 0.8362245559692383, 0.16207516193389893, 0.8835166096687317, -1.7185550928115845, 0.5645273327827454, 0.08484528958797455, -1.1403591632843018, -1.1723299026489258, 0.8466807007789612, 0.021986098960042, 1.7342771291732788, 1.389215111732483, 0.7445240020751953, -0.2695848345756531, -0.2981773316860199, -0.05719226226210594, 0.7461428642272949, -0.2977102994918823, 0.07431650906801224, -0.32177406549453735, -0.340770959854126, 1.1089413166046143, -0.38835594058036804, 0.9833405017852783, -1.7880736589431763, 1.0132906436920166, -0.4393560588359833, 1.267491340637207, 1.206488847732544, -0.7185823917388916, 0.48706865310668945, 0.781074047088623, -0.4200156629085541, -0.3487223982810974], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": 3.853172779083252, "dtype": "float32"}, {"__nd__": 2545, "dtype": "int64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(vector_max, json.loads(_TESTS), 'triton', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")